# Демографический советник · ML-05 V7.1.1 — Advanced Data Reconstruction

**Полный автономный Google Colab:** 52 закреплённых источника Stage I (7.2), семь исследовательских модулей, дополнительные строго наблюдаемые реконструкции, сохранение модели/прогнозов, независимая QA, полный evidence в приватном GitHub и автоскачивание компактной аналитики.

**Принцип:** если целевой показатель не подтверждён исходником, опыт фиксируется `SKIPPED_NOT_OBSERVABLE`. Никакой синтетической мобильности, дорасчёта отсутствующих OD-рёбер или автоматического присвоения кампусам адреса по ближайшему населённому пункту. Все результаты V7 сохраняются, а дополнительные эксперименты маркируются отдельно.

**Доступ:** секрет Colab `TRIAD-RU-Colab`. Для запуска: `Runtime → Run all` в свежем сеансе. Автоматическая замена действует только на `stage2/ml05_education/` после технической проверки; исходные Stage I и другие ML остаются неизменными.


**V7.1.1 fixes:** обязательное раннее исполнение DuckDB SQL (до GitHub), атомарная реконструкция E02, независимые индикаторы прогресса и перенос финальных отметок этапов после всех подмодулей. Остальные модели V7.1 сохранены.


## 1. Единственная конфигурационная ячейка
Первый запуск: `MODE="SMOKE"` и `AUTO_PUBLISH_AFTER_FULL_PASS=False`. Для последующей валидации используйте `PILOT10`, затем `FULL`. Точные имена показателей для прогноза приёма можно утвердить только после первичного анализа словаря. Меняется только эта ячейка.

In [ ]:
# Единственная ячейка, в которой пользователь меняет параметры эксперимента.
from dataclasses import dataclass, asdict
from pathlib import Path

@dataclass(frozen=True)
class ML5Config:
    VERSION: str = 'stage2-ml5-v7.1.1-advanced-data-reconstruction-sql-progress-fixed'
    MODE: str = 'FULL'             # SMOKE | PILOT10 | FULL; FULL по всем строкам без загрузки всего E04/E05 в RAM
    REPO: str = 'AntonDozhdikov/Demograf_sovetnik'
    SOURCE_BRANCH: str = 'main'
    INPUT_INDEX: str = 'handoff/07_v2/dataset_index.json'
    EXPECTED_INDEX_BLOB_SHA: str = 'd6625ccddc24f24a1ee451e33c48b736d3a57043'
    TARGET_BRANCH: str = 'ml05-education-20261102T160726Z'
    OUTPUT_PREFIX: str = 'stage2/ml05_education'
    WORKDIR: str = '/content/ml5_v7_1_1'
    FEATURE_PUBLICATION_LAG_YEARS: int = 1
    LABEL_PUBLICATION_LAG_YEARS: int = 1
    HORIZON: int = 1
    ALLOW_RETROSPECTIVE_ONLY: bool = True  # Все исторические данные доступны в текущем, а не историческом vintage.
    REQUIRE_ORIGINAL_E01_FOR_PREDICTION: bool = True
    APPROVE_UNAMBIGUOUS_INDICATORS: bool = False  # Показатель для прогноза утверждается явно, не эвристикой
    APPROVED_VO_ADMISSION_NAMES: tuple = ()  # Точные названия из аудита E02, проверенные по методике
    APPROVED_SPO_ADMISSION_NAMES: tuple = ()  # Точные названия из аудита E01/СПО
    STRICT_OD_COVERAGE_BREAK: bool = True   # При разрыве допускается только описательная диагностика / разведочный тест
    ALLOW_RETROSPECTIVE_TESTS: bool = True  # Не выдавать за operational historical backtest
    MIN_YEARS_FOR_TEMPORAL: int = 3
    MIN_TRAIN_ROWS: int = 100   # For regional ML-5.3 overridden by REGION_MIN_TRAIN_ROWS
    MIN_DEV_ROWS: int = 20
    MIN_TEST_ROWS: int = 20
    MIN_MUNICIPALITIES_E0E1: int = 35
    MAX_REMOTE_INPUT_GIB: float = 40.0
    MAX_READ_ROWS_SMOKE: int = 120_000
    BATCH_ROWS: int = 20_000
    DUCKDB_MEMORY_LIMIT: str = '512MB'
    DUCKDB_THREADS: int = 1
    REGION_MIN_TRAIN_ROWS: int = 30
    REGION_MIN_TEST_ROWS: int = 25
    CATALOG_FULL_DOWNLOAD: bool = True   # FULL: strict 52/52 sha256 verified, no hidden skip
    CATALOG_SCAN_ALL_YEARS: bool = True  # all recorded year labels, without pandas materialization
    STAGE1_DISK_RESERVE_GIB: float = 5.0
    EXTRA_CATALOG_PROFILE_MAX_GROUPS: int = 1500
    MAX_PANDAS_ROWS: int = 150_000
    PANEL_IN_MEMORY_MAX_ROWS: int = 15_000
    PANEL_PREVIEW_ROWS: int = 2_000
    EXPORT_PARQUET_CHUNK_ROWS: int = 10_000
    EXCEL_PREVIEW_ROWS: int = 3_000
    MIN_FREE_RAM_GIB: float = 2.0
    BOOTSTRAP_REPS: int = 399
    ML54_FULL_NO_STRATA_CAP: bool = True
    ML54_FINAL_SEEDS: int = 50
    MAX_COUNT_PREDICTION_MULTIPLIER: float = 10.0
    REQUIRE_UNIQUE_MODEL_ARTIFACTS: bool = True
    REQUIRE_SPATIAL_BASELINES: bool = True
    RANDOM_SEED: int = 20261104
    DEV_SEEDS: tuple = (11,23,37,41,53,67,79,83,97,101)
    FINAL_SEEDS: tuple = tuple(range(101,151))
    AUTO_PUBLISH_AFTER_FULL_PASS: bool = True  # Публикует только финальный проверенный этап V7.
    PUBLISH_RESULTS_TO_PRIVATE_GITHUB: bool = True   # Только отдельный закрытый Release + свой сегмент ветки.
    DOWNLOAD_COMPACT_REPORTS: bool = True              # Скачать отчёты ДО долгой загрузки в GitHub.
    GH_RELEASE_PART_MIB: int = 160                    # Ограничить размер одного Release asset.
    GH_REQUIRE_ASSET_DIGEST: bool = True
    REQUIRE_COMPLETED_COMPUTATIONAL_CONTRACT: bool = True
    PUBLISH_PARTIAL_RUN_WITH_EXPLICIT_FAIL_STATUS: bool = False
    STORE_DERIVED_RESTRICTED_DATA: bool = True  # Только внутри приватного репозитория, при допустимости лицензии.
    PRINT_MAX_ROWS: int = 12

CFG=ML5Config()
assert CFG.MODE in ('SMOKE','PILOT10','FULL')
assert CFG.HORIZON>=1 and CFG.FEATURE_PUBLICATION_LAG_YEARS>=0
assert CFG.LABEL_PUBLICATION_LAG_YEARS>=0
ROOT=Path(CFG.WORKDIR)
for sub in ('inputs','derived','figures','models','exports','logs'):(ROOT/sub).mkdir(parents=True,exist_ok=True)
print('ML-05',CFG.VERSION,'|',CFG.MODE,'| input: immutable Stage I | публикация:',CFG.AUTO_PUBLISH_AFTER_FULL_PASS)


In [ ]:
# V7.1 · embedded deterministic notebook source (no Colab get_ipynb).
# Exact original notebook code reconstructed at export time, without stale Colab snapshots.
EMBEDDED_SOURCE_B85 = "c-q9hYjYb{mLT|jenndDssyAY2vDSCgNAz`5|Xec2^Ij#_E4O35=f9JfdGsIB(W%V$acMYU0v>~t&Zu~nQ~8e@5XGz#zxSxWK&N^^e>?P!u*mwk9#w3-n<EtD)+9tO9T=(@8jHa&pq#R@E@!TEx*?bE)=cJ|6o}M@IQXyj`xQ?JiV~zjqbDu_xcwc`mNi>hyFs_Yq!03#KS#*?6tkI2R|Jg@}pod8MS=TEqs65I{DMdlam)GuTH*w{qW@I^)FA3U;pOh$?MNfo}C<@JhkA%tCL6Y;PaE0@Zg#CZ{J(1D>L~8>(<p=Ay=^e=fgi)<@Q~#-}2kmGK^-`{nnr#j7O8!xI5_I7E`n?M4v8RJozW+_W8-n*I&XICr2lbp#N7e!Yh~xKfEy*?DTwVY0&ewix)3i3v<@V_t5egO!LXf4=2xG{{h-V+t<IcV9uXm2QYJ*<jIlM7<)UuRkbo#b8}e-mjSDM{X1^@84Ug8<T0)62wr>wz5R}tjJ+W=9>4yA8a;I^YWM@s7aEA>G|eX#4h%B_@SVJ};P<1GZ{XM8!#hVbBDKL8Kg0ELYv}sf>tDk}k7;7BUjGU~4=+EV=|2%Oz;RBFXHR|*{Uda~$2mI|wMA$=!u7yAFJK*CSwAboPd_M-_3w7uKJ1Kj^2nloj>U=z($<ac_~v9AU=nTt0pSSXNDbw5p3%PiK+xxRY144E&j}(3^W$9FR@1mIX~jR_RuX!_ixxb3LQsChUE<VXcwCtTGcA>Wc?{1G6Ca*@2cKd0c{6`Su;P7(X#xabw*W0Zw-A|O#sD9JIp8of|H}HL@j-Qcy|V1C)tYYYTBB0GRlc@T`6OqZ{F$+bdLrD%d4EpW9~}#tD(=uX@H1fo!VQ}+uFyI9?DcP6{{fyI!E|WcFX&%b0et%X>%S#Tt1ZvK%>Do!!Ky6mP1y?C)A+cAW3-L%0AWT5152Vw!wx~;pW!F$<T0WhekJHxh?g+Jcks*8*S{s){DKgcj)pi*I4VAdj^GPy_UkVIDu2W#jK$yMv4FGkInM3%mp}w!82tJxSQ|kZKyh@+$@x3ln;&RL5rdx~tf5a@_(M934+&VjZ3ss`6b$k-bo&$U227NYjh07512BT9iS<ddUM(-r)Hi16J8t}+px3XMz%UR2phfGG`lLVZ?)ldL{PiC#_-uK--Y0mL2>ed~m;go|0T&N{p{u`7*}fw=0Iigv1N22A1<j9M#PvNB)WrHEKteV*yVuJvxPE)m@{p*`enOkdgv=rg{S|>YhNDb4_{ab>7h~sz_CQieM7zgyNWP^Lzq(=xoIZ-+k2L2PR>1Se4ql`jdUZd<HPPwb_X9foaBg0Ir5<rYJHoFc!oeqa;ExG47;_|9T%NS<EMK!4KU=Xfh+2YvWf9jxkUu1x{wkH|!cUd_oQZ@IMDWBz-$Vw8AU;Hl6eO5W6rZVMOQ7x(KL1Q@h$J$K<4uAn`+~>^5H`Sg1Sl=*k-C4d{h#3u3NjbJe*mMt=ABT^j(S1RMVz=tUVn)@gPX7g|IqD!RxV<$kiXiQyXv2Vi+I~Az&!tf0QZtHnvsODp0@$v^DP`MB#(F(yb2ot+4v2v6@eb%4)6#F-$%?NC`S_x=4x%ZQo4Piv0D3}a{Iz3xE-IAH<~qfedF3nwQ<ueUvF0G?)8n86?eVdXq2vdy}(yadI)`lG!f({ILafw((wA`8(P6cfIgDHr---KC+pRfTC<S<#IZtLeqnur1L7$E!pw+7sqn}rZa5zHv6xisrazLj{tt-QyyxHHZx&uA;#@yQx(e51oN_3*#Yp;yXv0f+hlF=G4MR<t7kc>mE13O%BMkZ~yfI^)GZx48EO#oRwFYg8PyT(sH9;cgwgy06;n)78*HaPj>h}EhyYq8b(h+bl84o98Vvn~{8QU~;RA=rGXY`B?>MMf2U_)rP;|!uD?kIu_wmPEYM(3Nzg!x7T@I*M>84dO<TuiIy1%V$}-M!&pG>#rQmKU_Ut+DpX&>Qdey4$k(I(#*e+y5B$lY!TM;rC0;(ZtV24d^!@?+Z(Vey6)rjDEp?ZdK}yYHh7(0rzf|ESvA1nZ4e^%v~bmGoIk|XK<jIQH;}{2|n%33`c{VkskyzNbuWsynj5kVdr>MY)k#`ZHvyc^}rH`3qE11Mfz8;F^`y)j~MS~%<z1J<ebUGw?yr~$2(89N35^p=VvP*xWJCr%d7F3)hp|@aAxKHc+g)S{9?D=y)(EwyX^15n!Ju14DR~le)o>Hszz<2zEp9q)yr#3H^WZ$yl!7>SY2D+Xu8$4<;q84vt6&>9&|dh`K#{Txmkp2;E&yIzwO`8{X7`xT~t0=uPilzccot3Yb&*DZsTS-?5=&~%G^S$)o!=u<~wuqUcvY07YhF6%dPo$wy$2k(!T61T+Pp4*1Bz$>o+RR#1eq#m<f4~nYsMjl|msu*DPGgU!A-1pR_(|8%<o^dcAVJ`jLi08vCNX`C+a8!E&`8zB3D0HTL^scx%DEd(|zt1?`RNm2z{VUSVp#RBi&&x+~=y?#GpKy-~Ef{W0`Yh`+N^zE)W|*Bdu$_3D4Z<6!)?ys}dJ(5+XR^;!crrFyI4*49=&E?V1zK@ZyF(L<yb{BtOzK8%UjPb`uZU&_nkam2pjBjPqcCl>Y#x};t(XZ<p*N0jUC!h`|rIUUD;wowJ3!K!Xl*UBqyC0}r_*Xp=s%hjbOoqT2P+H%*|>or*7##(vxTJ^?8ZKHumh_I|71jUPV&(f;@$t29BTy)BX_8p7P^5*#p9z!eU*O0<KV(~`^oj|XR(_D6M)!g#(YPEp`&4oEvV2<O-u;;@hGg+Ep?2>*-Qx!rl&hZ$@$r0=WU9m7pW*b$e`bo5HM4Ke5$Nb}yl$kZw{~e}Qm_7Mt0QuJ%X#z=S3548bcL|2Bmv11Ms+2#_aFn(S;mS<-_jsG(5`_RK1tK#)Kq4M8Nh(86Y#)N*h<7>Mup?_$3_^GE<8Yl%S&V^prysLsrO|98&)J`um;RD4h%RnKImEBQ&>s=y@p{&7Hy95_-4=Y@_F8wwelXT<sp=Y&1v(tf%IbQpUS5$@;BvBYvtEXO>$MLfLXyuDRId+4*2v%Cfmc^%7IK#@xI{*yZoBRGt?hj{ls6Cy;@u>NSgzcP4KSBCOaq{DYNOSSwdzuJy}VLwRx1r40hL0yvIVVq`6Gb)YOPsu`R2M&y;ih3y@7|V=kvL|seO5w=HSv*9>QZT@8<Jb`)lRq(oJo`$yUo7OCKy>b5~&^^^e_^>T0zaUK9(3xz%g7v3>I<P9=@98=)S;A)nqj0b&hiKn=?)wHqvWEZ06<Td9?olcx=Trd0r^XD!UlF3efMt~a-E#RBB$cDjDsvHAl*RYH^CPIqV?rm<9BbK!)@YfL*4Uaw~%k2Y%iZEN56Mpn<;_Ito&-s_Hcfr$pbvfuUsYtI|QxL&XO3oov-Wg-G}qu?%A8y~>_qPnaSM}-9)6Dl78R9!hqAbZ!VD-{><<3_!<u^#QpLf(Kw>*clOawC4877UmH?chR3oMzl0{ENMm_6}Tlx2ly7<2#vb49r%o4jfIn{<96>j+Sn2tbL#zbJI&pDI;7?Hb$JfUawSKU`XRIozr1@tyXISIWMon9z+nl{O-G2k1GrFjD2f07uz>+Bi5%}?ED5X1dU21Ix-7}ty!vVtTh#W0m#Wpv%0=gt<<Bb7ZQ{~avnf?0Fo|>0V{5~-mG3PFKL9_{9<DrI5wEzwQ{4fQeCS|eFxB|wn}S>=n8SJ^BSc9nkrTt-oAx`Gk4j!eASsRI187ZD_5PX?>g^XcHX_}6!L|v#)YatlU>l?%*gBS_!)S`DJ&GS+I!O!70FxepWvQn-R(1zmQYta>54zeqdX1OPpB}`PKB5YoNn09hKtjyS8tV@c+i_SH?A4@g@1{$0IF&>($_pgMgn!Z@SyJZe2{IdOP0`GKedEpCgfyGn3>|11llM;M*;m-*8yYj1i-=6VHFo8fgNH?hF}9w3r{L}<f;F3@-_1|&rvdz(~ZlGLc6HV<px4&r2=?}bfF2i!nKeZT*+I8|6nR#vvEymXobRtnV=V}`ezdcYiA^+P(d&@4bNu+kOhd+a`gt>NoP1h2)mUEijMecqe&tVpzl&`4XDJ@v@Rri;DYfK);HGN4*@S+AdnzTRh#bhaup5&^tqu)0{Tr-gPAN>0JmTsfNjK{!pCJGQ%TH^p%{n(WJ7sKnpe$~g<unQK@=#CL#gO1!Lc7JtSoV&FC<TDkqPV9bWXlUjh}ut>Qx|liMwUaq<&etexnrXRx-+|$E_TVE%-f$#1A^LGB$BYwqpxr2Ye!xi1{@ki@skfSa4<2qp3ph`=z|GpD9vs?3FrPHziavW$-N_p(|7C0B0Laws|GpK5FA&0d3$S?)q)`1OS+fe0;VyXamauAN>2MyTVVs!A@Xjiy0hb7JA9%_U^R1qs-77f!H6A(#rAg1EcN^?&vD4;Rv=MW0Miuc5rq=ZROYx_&b6azF0IZ*s;C6$=KTl>MYG257@#bs4A;^Uw>8Px2I9(>=|_G=Tdd%FSpz8ymR&4zf9Fh7Jz^I-tzah{WkJ?Z6El#y>1^M*lhtPIriaf-Ld#hD+8=0Tf%lw*EHPk!wy6Me%q?t_kb@2A!(<Jf=l>5F3nPuelQZ;_`I(c8p<NZG8lW(dT#}NZy4+j#(^?OW%XKRc^Rc)p*?$N0glz}3vhhgTUQGOS8uSsQeLXutgQg`aJ{RY!foZ;wf#;wNf-5{u7rfkOU`69O{dVKZUddYGVfTw=ykV&<K70W239!UbmZMD;qSRNEPQW>%FPVFm)m}40Ux&gjA1!BzfV8eUeM}x?QAxi6OHZ3xHI#Pt#JkY?UI~8ZsdFI0GNn-ju*Hf5(I!lrFs1Y-*5N)lKmrqQj?)Z{;=n@{5?2~>D}9x48L5m6P=MGOKG!ZA<1YV!RT-2=%Ll^Sgl-mcuFN3!DADFYvp+7a)3xcG!U&a_IWYg&TeVj-m!XqKSQHu-vU|+Q(`dUt2j6MX*0hihS$)B@u3Tsv#<<TMx((fW3!PHFNeDzM75*o$$1z+5)D$)zUKt|Gdwj%*$$_Wz#4%vG%$#9892;|MdRYU{(k01(C8D8#6-`$rP*`B@!n|A--)0I!-MeJ+O@<{(BV1q$CD9G;!RDSg}HX&YI~t|c9ZADi@;zakNtv`RuN0_`gdgJe#z!%X+tK}RLH`xG3q#O9Q@?gy!*-QC$|>dPYmYVP<Q%>Y<tY&ACnr56^;r=a|RzLdY{-Hw0}OF>gC~(O9pYTzvR`j`8#=qvLwLzzOSU;z%$`0mV2|k>2&7bZFM^NbD5*(U_t*(Yk3TirE7={wTj171TIUsI(&}op~B9sZ-n}I;SGR>!<Ug7+hkjf`H;Xg{s4n9m|n3WAIJMbnDk%LjGhZ-_V-5ps^DKWHzHzHt&{(5aLQpTnsA&_rAIlyk8H2?JDqp8ufF5`B}|OCkOzTd1^e(1a}#I*bKrP8)~*-q(j_gg?C!upwbia;?X(==OosdL(g?^TDs>$TDCQU`BXqY5v+p^`tpX`)`DkP1(eP5o-;(B8pS*q|+pN#i!GC`-==<6WtwFEn6TuAaxl0Htf8@aZ)bS?0vB9J}8TD`i!1D&-P@hiV0y9Y#%a!%Y+Hz%Wsak224(wro^R|ofgJbt6d&7Nrz<(W^6?@o%KfsfH4~XnNd?Y>rzdmW*X>Y?L@xf8(svX<`Ids&AUx9(S(;d(7R|mh+Sp3(qTizHh-Czul!jHh@cXxKjJKKBkH2mn`1vCzXNKlM@i;dm)$M*)KJNI!#@}pzl?*;eojk*9>c>WQ5`H;WB^2IfX?d3<u9{Hb6{9ueDg&$+Xj6ZGT{OFHk|9r3wI|ARu2ZR}Lvw-+s&jTLAC)nYWneYk7H{N*I8;to)<;!6-&|WtPy8WHf=J3uA-I=iD30&;>6)sP8b91nuy&#jti!_d<xlXqa{Or(gW%zYy6Ig42j#~-gD58BuYn^6nJMu*2#G=Gqn#npgFjShe*pZo7K9Q;tYDaE_K*1JcK&Ag|qu1l-AYxhg|GA$Jy8Vm*3$0arwtO@n+DN-ze{MUS-Xz%71;D7a##PA0|7%EGaY54-v&hC*wpl$9j*-HMq<f!{RSRuZe=t+&$mQnA>pvu+1h-?#dhnl!IOSTqe(MgRTraZ;q%g<y7v)#So<TGnc8Bn9w~rzg9{rh_b~nJ^XW*9^v8^-IjqO|%pxVlsVf9~(SE6!1y`uYxSdg^+VA@5L$MOQJKXjyB0sep31|mq*1)ehDp~qhM6-gi*&M`gyG|{g-xo{iMu>?6yPy`z6ek8Zig&>X}U^#xGK6^Up#q*x?CgXvmyZAtBBdA{vVCwhaq=&9I;(JPS6cGmzK?lxbve$2pyncXl^hYVLLG#Hte5(O}dw!LkrM&@ysK6LJHsZxv-@iE+S4i-L%cn8MZ5Xv1fK90#;bOdzOH^3)Q}4ES;J#ZQ0P+m_b)N))0rXhuc_?VD`Qwz{;PM}JTj4#p=X-tE+YSH>W8d|W%Xj!;@ae=Gfq+iW^b1woA|VvL@iqJ&PDE=j?-$vA<;Jzuw6=6<<>=y(&4H$RfIt02)97~M7jk=Wm3Pr%9Ig)^z+U0Az;tmZw7-!)p50lClJue&@EtiqM5ccd10Iqa@k2q(j%Qg@O{#P0q=-Qn=*ae5_*_JuqQS7+>&*b)esuB_9)GE*V9-jsx?UnBoL~Ul@!rs0<R8wXlS2gz_5dtmzYnZXLf-<<OFTqQHbE8<nYof(-1GMaqy1t+Odj+F_46~G+ZP&@ro}DYUU#oMF4+gDuQM2k5AE9*ruAiGSL0pMvXl;x5GoxB%W?K_+H`m}a8(ao01_+*W>tbwFM;fwVfbZ3F&<SS7aaH(r*(<GnFZLEIa~&zD;7Q+;A}~X*Fn4kayK)g8Ljs*z&0s=3k+rsGWjzSmlFS&TW3O(83ef<e?08AGrCC&|MTv8-5%aU^0m9&(RkwZT%LO-n+sbcUw&KY3Wb*|d056gZ>1pn`gc}qXEc}$i&n}o?8A-H*kc)ffyeJVc3>gL&65`-p8n24Z+O1ya>+TF;YO&i!@6;M7H;U-PB6<;va{yqz_DanJF_;=Ft_L3x3f7qOF<@^G6W6d4NJ)v5pCVEe;@Wahb{!eOm<2alM$ZZ5~7XTm3VIMZc2Zs-|>cTm28|#G1_uqhBM(e2Y$@t9UK!HXJ^D<Y5l@|8UxU*m?NEXx7&BO_kq5i9c0>CQ@cCg%2OaId!GbwCf~ewhU<HC7cXAPp8{5BHw~ix{gyu*TbWvelxof`uQy@+TDCa!g1|rt`kes?l|)tSBK-l`DG1yhlxYC5`|S*pgPb5j*d7)IEoNc9m-EhC)=68LJ@WS4a0P0Q;lnd@0t_xH_Wf7~EZFDhH-df!i7#Gf*=!CQhaDf9$Q$v$_42Ci;Cj+obFn+N)PuVIT|77_xb5@?+g=Y5GX7)6=GzDuX8VBPd05muJ3(YPz;QQkUr0dc_JyrO>qhq)x+yzWz0ru8@}|fJw&V5t0E?DCW#aUzVH>vIBU_p7fnweRR@Sc4{<E&OW23)aLOq5HgRY9w%KwctZJ%Meqy-g%$7AAa1n_d?O&Y2+G0e>5UKk6tiuwryyVy&S12G<1XoSq^6k8mgD&38=Py;~XrzrZ=Xrf0zv$|R-p{6LeH|P(>gMPQA5@@uNSKzdb{nl<Dy%6o9je&^ujWt%S0RD74-B#@JZ~~H#@SRqlqc0%}fkhj=0amZAh^R*VRYCfd+KtlY)*?ASHA)AE{K-YHEc)8S_(%6=Wo%S!G&bNVd=gfj^2TztN#850<+WyYiGKXqM!AmP)++EV8YyxOqE`5o+y<g=&04d(Qpzu`R?zrQ&)FZSURkQumz4+M^sZNLY}Cb^)>o>{a7xj*@WuwU?%3pJdC)zyi$v1KlR$uswx_a8dLb_$NCI#E9Y_0wzOCqr;NE~eLamZr%pa1%ri;g2IQ2O1s01a>ron)FK<JBtp_8h^Tz_ycBejRQ$+(rxb%Oy*t(k1nkjWgLvj-Ff|MD_Qc6IG~En!Q6&CPkkp$`=Cz=p1gg5x9{jiO6n@L$Ikz2k#XMtsR`7O&*<TZc&#(z)m#j~MT_u#n%1=v9Vi@s?ba280gK2hq#D-F<z@3W(rW$42*)giR=EUQ&qh0lXJu@NrPoF}s~^&(C1EjXvmZN$8++?5e4AU1*|2N8>)d4M?_aXHosoF;EahDe|2_^<}2#?QOTcVux6U%>2bdes2EaMYs>QZP@MI++;|Gy=}A5u$$ZU@3*@<Ab}=PX|(5($vMzhBVG=2AkXg)eY6b1jkseIuOj&s`YffGk$wAiF8^0>ENo}w@A&skLSTQgw+$-_!V@rT=f(pU9{VGpgMmZhet>2{Uo(u-@$ovoi;k-2=pkG|wLiWxZzn*|7aFh>K-@2>P%;APJ{;e1-vi0wr}!C$vhm~J+$Z>SCcPdU;}%E*`AaYzK%gEpCif?$S#NL;0E6cqSh`KS@1Z6gW``=|FtqlOFebv429GEGJ3u%3_LjD$me(K9)!`yWW(FasGz3NKNv=KF8-nEK1aP6bcl>=R$@hcF$al#%frBF*VX7#>Y_p;Bf^b#HrX!h43|P?G_4m9?yQ9Nbh8AG3FqugosbLL<WC_^j2aKjmi7SWaXmkCo8Z^U!7e}#38q|p>L1MaJSEuXuK>FX*JOo0EcDsY(^klCe7|w6I)b4DywsLL6JR{cySnCbDIs83<J$KQ!&86m<cD87ZOG7V5&rHG_2}F7iz^}ro8|j^N5$n&~3Hj*iUC_g{P6A)FVgg}m#GsUy5O#<Uh#PNZorI7g&oy+(NW<@E)9zHV__*zk+uDT_5de-kL}BTxI~v>zN@xX1sGOpoNe~UcqSFVmzNzlrx~d!oV{xCZg+=@~$7-?j=*URTPYW*TM@G9y$f#8sewtfn#W-8J2%a<pM|8t~9O;#WK0@{{f^pk+CIJAfOD0Kgv~QY;S+}9FI(Y}ZWnh8|tkZ;VfWa6}MXuW$v^KlN?xpCJtsI(X)3g*gCSd@M%S)Oj-<POuuJ7&nQ>AG@uMq16>hhcTbeaZ1E(XqbN&L<2c|&naTX&0yY&gPQX@5h6qAzgT*2;8drgxA3Da85iDb!e_-PzMSN3ky;3lZ>I(0>Mz!RY~FI^>`31U?)~Hnt&J-JPbIU_#<0C{YZaA;e<0wRn1tJNwkZiCF2o`DUY2SLYcU3AXR`%|i=Rkt-8z)7Hj^Jjb++tb0f}oA@C|9*k^3qcv#f9S^%dy6bb4us|2&oESS&21M%)c9yE4?2_MI_a_Ra1Wq(tD&-T}4GaZB-G-n~)&Y@3Jd%gDgDNZa^h+9&4T(u81Tc@J-^+cFX!nZ(K&{>&@H1&Qh&c)#FO5$<0acup16KweA3UuNbU(P~`g_CizL-HVA?>s~@^Ba19Yib4Y;}DEePztvb;@zW`$>5rTO?t|Rs>6&(<QrTk*Zr0Kjmw7avWR&Z4$Zj^?I56%%~6vrO(*HFdhg#v^hm{GNR8pL&tN<$U(sm&yX#{*a_<ZyG)ou+CW55AK|l45M8i9`@_d@`A@ci#*}D6xnQ!F$!5bP5ZPIa`O!$Q$NQbSlE>1sDLGw`@3fu{4d8?c6B4K(m55O!5i@1vAB0yPMKsHhf+JwqKP=bR^ns*qC6dtzFYO{Bri~}WE+YKGV`2-wQ++IX(QLYsG}M)bD@Rz^MGX>>9>7MAc3b+y!qD&Mh5JQib-nqqed<V)eix3v7zUQlULCZ>fRLM8r_U85aY+SXOD>FE?N>G$GK4NRL1f#~!v4hEh33_u!8l7NCZ1?mGH{^6YAWI#X3@<G#&=N|v}ZGXz@=k@ZKBDF3T_jdZ<RX3lvc6<pusL`2%-h3VTRdME-Q_EON2fUkSRKcio}#mO(#-PkxuoqlV^)$a{l_{znuJQ+!qEesJmVN-gNCWNfz4jj8tBI9H;SQ&>^L$@Q89)vFD)6NO$D7OLjtACKQF6{KA&htw;?U>Itt`%8e=>+IVgRTl8MyKiYCwZ5kbN?%ytoiN!UMsUw7%NwE-Ht^}X(r^8e^sonK<`U7;&NLa}B{Jm1BPWTaw!`Pp&3#FKP$Ahug69$-qS;=0ZKd^fSvy?DNv8$S0oZ`FI8cyuuAjrXACUE!=Z46{7g83MU&|hXzDwrvoy#sRyO&c}`w8>8I?RHT`)W6$>eVPgOFe1d<i&z{W%faEHLGfu<)^6F3B0rG*Dk(x_%`RVTXZK+uqd^~I3V?JU+$r8AF5-@Jm(-Hbqw0xx=dDsStwS?6hVLlxJ!`j0Dnh3C#{5$}v6>|V0Kya>FJ*SP-w@GxE<LP;Yn_q=G@f1UIFT<<O`}5TDP9TxZQ;8fi-0=$f4DPFNBBh5Kxuw|k1X_BcS*4fc#qlXW+`35tPES7<vWI*bU4FXZ0M6?CpI*C68;_Mv%_4B$dnz6^e3O4d?T_;ekmQsEt=?45l9*Wfiu)axii&OSRx@(Jh8~hlU&00j(uc6+H$Pj@t*D|U1ir&JTGuS_d|ba{_0pao2x5|X{fqRvD+@?6KIq`_(L!O(UXY+7UtkfXl264NV*;{Fp@(Z@#9jx)@X=`BlyauqUB1XdIRa_##*(>=|QHlFEGlj%9<-K19-3UQ4?Komz76osH--_Om0;h8|4*liCQE0@Uya8URzc}DdAxlsw_gORc;UyA>4rH4R@us^g$_vZLYpDXx+&eTQ-`tb){v6E^t%J(t+yNj4B~?N`rf2bIY;p;jR}TmxW0L(Em2fhXT*wq*m*-wbjZR{nYly9=VoN0-wi-D1?i`k@gh@b$>yrFNITU%x4^rflD!4IJj;GRfnF^Dz+5is6uKN$EJHlenDqM7k+J{LwG(X9`pt~^vw#tk{bprNxAwI!kWARBlzEhEncaV1WIQRqciq<+waFxAphI<GJxda>(Aet-G1NBI^qqt+t!(%#O-&=*l|Hg)4hkR9UjlbwNECgINBNR`k(~lA9X{SyV->12%if@QS62YdzIJ3q0ZiU!vHuiymIvYorw(P`W1zE>IN8u)(*_?2Adj^>7LCBf}PS&5*78n`XywIp=orWwz7*BAt96efLds@&A6&zYg{Vi^B4QG>g$@$TKE{@hyw4o7O6QtV%*mZ7DCxoQnyXK>i}mkc38`mW$jMXhB`~OpvUiP`8z<+8`Jzzw*zYsj!3u~F%++fWqdKCG%=q#qdYVvw8)ZfUU)O105WUQBkm6$!09@W6DZ~iox`8l`Y7p?-Y@_qpnXWTr&rR)-PZ8l2tW~<5V;J9xB!Evzop;sx>n$AJ=ZjgRHRE#^HtJMuS9x+XEODHc);nUA8}U1hh#zf%Ib8-vmk&8?f-1!W;x7_s96|^2Ov1-j9BPydn4SU_uAdNRxsZ0`K8+z?sePa-J-G~bFd+cyFPkg7Yl{q{l#reH#C|V!HqQuitpePuXSgKIPBuv`OdrB?_9op;r;h!q1*Rued~1mF^zsXfBq2{y?%F(4to)E5e_HV`CM*3u-N|{+1D;C%8#a|>%8b<EFq!I`9aGYvUC9FQe2r^DTc1j?F;ySv4wO7{qao4+w1oBi@|;XBy?udb;_87$qBrEFcV-Ns6{wuGrZ<^-^qt7EyAL$V1R+fZ@1d*g?2{+K>^ziy*Ba;#ljqX!9~tUnC0ddV3j)L-=n#`{~z!8`yEt3237>9gM<7}4i3gLjG;)MKqm7)%FGlB`JZGD4}ln@HJQImEe;Rgo8>VQL#v<>H)pZa^Y1VId;;P_XMaXyge(R_ujS8d`{R4Q-(TGEhPd6hWyvdfT2TVLq~C+S`tKizi)wcEO1_w%YabTnlb?tOFllVaNHKNhAPDSq`^9%*ck<R{IWJX|NCf6GxxncVo$C|*Q|||+f8h<n1n>-VQBQzB@4xGJ497$w1Y!ufk^!Z=SLCE)ICsOjS0)lW_;cE!n|tTV<?V&=ES(*Ac%Wm@`<GB?J63WS6|DoqL1Z=;&f#GQ(HKX8>aFyt&~RVJg+k*n5%(;ILWMICfx{nrYmmM%XlBNPVezVQrw$Z~eU=#z^$EJa@Ok5u{ukhDYtS2vif><8xLUZ1<nSQ6u?ZV>!Pz=Uuv`e}oOSY#>V?B;q44tkUooRK`)53))cJbqDtWyvT1n3sYDS_oU%)S-*mH5sX*Xof1Q79z2y+^99rD&AzXN}FGeFluqgBEq?cNlh&$IL5!kR=Q3A@C-neeU6n8dU7ULm(2Tz`^1rM*;w9uul@UFbmL7tVtWVCKLG{S~Q6B@s!NOQ!g#r)dFG_CoP5WNV(dmg%!(pXm#;&1~_D>7oCfwr<dCyWv47Dd1Pgg>l1aF(^}MQbdZ6LYU~3Qe8cwbNII$8N>M&M5!0d?C>RJcZictnNXnb2bjQT*YCmnEQ~h|a;e5)ZE86(J&QP`_elg&7OV<O#i>pj)WKUSumlM|#XlqYLW#OsRDaRL#3bU!;E#t*OeeOJDKjzeFVEuaZ=7LvjNhjmNUeX#kfQTQhQDMti*uLrW&{++=6qv>@@YvGuugqV$grhI#={xVCsqDT6COji4AWMj)$vWh6)gqZFB)cRo(MA4rH;LM5Oeb;uh|ZM&D_55lbgk#tQLRLxP2jexMdwgUEjVyh>Nigv$4nFYVfCG{1sjt{$Lw7QqnnsWX5!Lpl)pI{SsUp2&|CuWX`J1%$^R-XdJ}pMZ+(wj@v55!>D8aS7fTd->1jrOpnr|J%ZuLfsYNglz9&Rbnp7%9YuNedv<ysME_++Idbu~J=-A{SxoC7on+Izy<-XqcKK0#bA{Lb*<uBp)1AMhwl_&69aZsbreJ63U#f@=6vt6j&r`02vkVhzeoaR;)C4C_>!HaOcFDedJJlIW51Cu@_R=HfQrJRy_EOEWikZk5zZ>2}A`_X;4X`haTrGNy|JleJhxfdFiN|?EJ~Si_<}EBvJGr(LdM!2;By4s?+u`_)Q;&BVf9u~L`j~;Rq`_%(3f4{Gzuqdz4ikW;NVg-ZMfO`x|MUYz<$py&{Ii%F*K=aQ4rE`4vK}A2`y1gyA!1NWxM-fXbSg0IrZMiO6M<h0MhQJgYWpqwacJ9-?y8}ej&aw;h}XQ)b+6^M_4}0q6v$~W@OvH8eG#{A<pLkb+G5J7(q^-Yx!OaMS8^_y>=`U2KCoQdzL!i=K)D*ekvUI9ogdBGOZ!am>7pbB)wlReAOyviKPT1du~wl59&A!BXG+7irLo%gcDo`6p$46MUU!_CQv$4}!pFT?Dc76VD&?j~W1!d$oAfCI4CjtNE*aqAv^^hWAogNMmz)w+fC*dL(+1)^jhR@U*0|SE%+9D}C?Z33N@lhnG!tZ)h~{*3h0Xl0fZE^6%6q6M_^og#IEreJC<%{5tw&J^HLOr3DvZKMDv(7%r!RRvq7>i}B`tViQ6wrR@c0Ic&Bg5Lh=MXhBRi&Istn9mHf9f1Qn@;N==4juB;cmM<KrL0HKF;N@~bA7Q<>X2a`uvpajKgsoZN8<k7PKN><oUTK|(LYq;DO(mO5y<+o_=zU|3FtQtTa|>$~61VE<W@6DLx8nj4$10A}=;r>pp*l6@0?&1Q3$2s1lvvWjQvOOX}uVVEUKWcdqSPE&GG#<OS}d|nj96aIAckh_lseo*ANOh1KH+m3)J0oc9+^nPy6UY<x@j)!>LS2^Lxbx}BNx$JSxEkoDEL>C0vxJUGX%{5Lah|Vs;Zw=_&4>Fr}zL2N?;Kb+W#K$d%;*UyU$f3pz+L%m-q5xxk&E@Cj9r$M<d|w0)=&$ej9<qP2mqFlh=5pbyY|_(T1TMHcXvY^2Rg1A>gAcZU4mW&O^@Y;CL{)$}$_A)`OR8&}Y|>Fef39ORs`73!Ie7S1PUc<A^(TAWdufgiL)3&L2wYCPh3F1#=wzsKHJozsGhQIL^_1E|V91$fhc9f#uw*MY9?;0yQYi&Zrgs)0Nvx{~OQydr_BuWdb+c{kM4bc%P#Cwg<0w5Fk6doj2U+%xOn7HZYHrRSVWa(nGp_z1Zb3;Wo~l_5jwQ*UY^D@KiOq@@`FJO<%O5rW=X~qkR$+c_`!7pkuG{ZYE{CU4q1a=kXcSe26$N66UM4G9!*wDOz^jDOYmZz%Kj#AQFolzCH!3TYrKYu1ZZtF9wpDIex61XUo8@}e8jN<lzNnGocH55CtbgoMLn^KVO{;6o%8d#(-NqOlZizo81zsMY-^L4SFmmxU#s-W3<Y>*&CX><}e@Yq70~C$D6_(_4@h2VxQ%rWf-guu*YOHzur{o|ZJfhj$#q=}S$#QMu+Db)2#&7d4T1Az5ZPkiVm%q+hAKt9gD;9;E4_%712FlFg)JwJUN~N(>$(#aA$2NM~+M<xelam(@qFSrsBY8`H{(ulrIG;Y?ERQ0G3m}%AwN<Awx4c|;D}YRmjn&mMzygu7WJO4&?dfTIW>LMJ%*AZ6cnxdzHE~TN?;gih@sG>sqqqirYseXUNBu2G9VhBaL*7_lw=dwM05h7U&eytLU1?V8R)+U3zU5Z6VXbYftYon_kuhDy?D5LpwBz@zyj82~Pv0t8*H>z_dM5rn8ylZ!+w{>x@Pt^WogVwIZSEAmou?n|ScUz>pBpka8P5yV*s`vDY!#e20LL<b3%*FSv6f-T7}gr_my)z3uvaI42^#lfpm!gxQFJ4wj18Dl{zupk{{Ca^NZ*ye0DEZY?^><4f{Ju)DiY-pJAvulV|t~vQRr0`E+S}~rV%qy!c#hQ?EjpJYT-WM5;^pdw5AWdUN8Gs3i`V2A2?-&+BIQZVft0_hAW)X;dxdVb(6Nt)#{o~FYv{^u3&>4tcPZ7;oavJXFRP;$h51gx@Ki&FWxGzRF}Cx=*3xsx==mn&Shsl<y4dUOR_l_BZgY&F_n|ZFdK$x$^$rYggd&zft@6-O}f3dWU!WM>mRcs8M)g4d9N8tUEW@_#=E{HZ@V$704Xqj<aa1Z>&%#}&N<!l0{`J2M8%d8rOBjpn*3mDI!$9oLemv>q%QL8y@iVAII9?TU~4<dQC!gz`%-LSmw;zTsxiVrv{BAjHv{Q5Y(i01_@SBY4tgNuV`^zw_Lx|B%BuuIe~P1`O&gnu)Yn<7S+fpe@I16K*K76Fa?=tan~sGA#ZY5et*u%AsnJ}{+Gfp16BFlFJ}LuhfV|UYS3C>X3B=Ue_B(@-Z(;ohkj_zMY0c&47FMzDu7x*OE?r!LwNv5gmFkU~s9l6tiX;()h{vIems9*~Ie<vf*TUC~@Fx;71MF+wnNL+b(zbQF_W_W8Qp18j2A}raa5^s4rZQ<+lzv<go6gb~OIK-~wI_Rf`*DTMrlzv*w#Fdplxz^Y3$}w-c?qQ7Iqg0#*(d)a3c^x&<Oq6SP{5VYj@lXnP_oGe@*KobI3Pg(oj5heCz{+)DqPjsc6G9I(PS@AovgycM;m_H{J0_t-XO`Ds=p=X{O+_lUykNHpPci*as_XYE}oO$<L9B9=u25$=0^eRoPxS)(1RfsCd463tr04H6f|{|JQUN-Q>Jrj5#6FHmQ=$;@hdzzrklog3hKc#jYD@)JOMQt<7t*HqYsb#Da`1cc0<Nis-Mp68KSMz*Zi5N-9t)XDj<$lhR33y4wo=II(cEk?S9Jo!xhQ71+Zb5`6szdgNW*E;_aAN+g3K$?qtkKMajr&7%=-*X%0P0Y?2XeY0EY>oswSb)N8Kg0YalK#qxFl*gNe06Zm@`G;al1AA*=i`!Cs_goOym+e;C&A+%9s8IpV^R*i^VBQ`Q+XV^HP=B1?Q6)_kZCuk$La!yj5$T*(B7-ED21H!=UehdWR{$<g%u6DA-&zwCfzm-c!f%xK^V2UcU&XAVwd>C^^Y>5~RsdAm4!Ys%fHwH>8;BmUfII)4SBWHVy6xktFH?e9;wpw+$R06ZIT#DPagvUpCY!ucG^W$R(NI-Plu>{Q_qXgkOIdWsKI`_xM<~OPrcG<f+f5m&JaISh`mvb^rp(rX#nr1em#Zxyskx#k^sTQdVpJEi&uaKbQ#nrf3o|#*?5`l@T**+ux5v~EOB>jo|7(mw|8GgfW0c&o3(yUj@%QN+j8G8K5A}P1IwsKhfmOM;C^W-COwH#SoN(Y_siQ(lO&Bt7Y>@!h~_{&)JF`9s^-%K@ipL3YK=v*+wFpJ-ich50LrYohKG1TQH7q|Of51;>$Ymdz@<h?7ccjo>Q+k<|_Wn6l~QY-8)T&^!Pv&Q1<xR{>|12oqpwo|=Sc7TtWYy&a81LRW|#8XoCVI=;}V6f98!yRCa2*aNQ{s=*le1+AbrLYASb;#IpB(qP+*@5g8JUgR&dh%x(GR~Mni1qq&f#&g%$$~}2v8mB-qXJCIDa6DGac_$wE5pYn$mZ-Ug9Rpdn)3v~0y7n7sTL^wNKa%2+_SFlpF|cY4&{`-15e^P--$@5C%b4(2fC2UV>2@9ZJV_ZDr==U_C!BlD>o`7dv`n@2F2M~%m>OXu~^Y8-m}51eF>8aVO@2F_M?xLtiF_MaE8fPDsv3~#*9S%D3IYF2X=Wf-W`m@m5VtC;L`PhBfP}3bnJ2qaR`3#hC`XveD-d?EoOBIU1LvAKK0QIpT3#$usd^0)CDb~)pl;CP?(u}$4<|b10di6TR<*$2W@9E>Nyu3u@Vj$0ciW)o=C2TJB_k<=5pS-lFw(Iix=<Q^F}*~WSHpTY;YFGxOs1kx8XQSt~sAAo~Pspx_5II#T0tg;gk)Jn9xY#R9>$NC&VA<zQB`mAbPq~IGmk7y@&vzVu^NJpo0oH2{!RpLXQ80pc8*1$fd4wl;d1Utl>rH^!d4WorQcJ{x=8zyA1!EpB7g_n-J?|G0Y>lNFLTf&-aI!xr-MC9I}h(jMeQ={PWBz4`M0IW`6o?L}sa!`J}(cWM&<U8`Hs)Da^&T-fTA4$-5K!JQT7Dy3idtF*+F%BZ|9Yl|qU))}pmhUr{)VDaCA%OpbDcJO6IW*3!18toaq8Go>v=21S-$#pu7!7GoC;#oqD@kJw4`r7ZJDe3C3yYfp7TwnnK@Y}!5*X&I@lL!w&e4{5nTl4rkEx-2}`)+)^pYxNJ1P5GJ}8^4Jv9SRu0SIMC$)d_!yXoM7DBkF)sYg(Lz@#JWPE<Uwz=OV}>K8EQE;z<tVBJk@09^ylLDh0=TB1XZ1v?S8pmyEO?TPB$+xyuv~l1a%+YQW+6)bK)eDNo-v5c*c4j=-=KPTsyz0lJOX^Cf#0Za=t;JG1yIUdW_%7kzMgS9{h@%cZ?(vqPG_wP^>tp6a(0E1XHG12M*@aZ?8Mz_kn_UiDVFS#fVvn>RP~8Ze5c8P{f-zBwnV_9C;HATHhoM&ZmQ!9T25VPc^SS~T+1h(M1ig()+cK<vJx`d&ge5(#xLbo6#dIKiM~o@{umK_Ep2@!-(7(v0Xuyfq1Sxq0ld*WT;)4bS&H5THH$DIF&IJ&**8&P2z+am*R=0U%49NFm_-x=|%-6-=Q_d7AKIQjkdDDU_YHu33p~$y6qEXNkO$$%5j)k1OFsA@o0;e9ae{462|sRB2pHs&w~%P*Lp9LUr@aN_m-6H2g{yEn;CB<PnS}M~(cWDX_txRp^NNIZL(G)oQa8kp(=1;<sz{^4ii(<(+1^4u=Z8>zCbfGumJ6b<r&G`r8(&8A!yZ8c4s?mZ+^Ou~caKlJ1Y+a&8=h!yJ=1uk8Z8a<_YfZ7z2Lho=-<u!A~zSIXn&sQENfD6)M-RFH-UX`nOoM&4dfLcik<X2$Fovb_%ZaG_Gc&c0Q$v960P?#>VezSaVF0BJ2*D-$FZ&8IcqvzVljkcr<)T&z5|ePzDwQ$3C0X87KgDIuz<Frv6a<;j(2_Wgpjibb=Of{8Cv;=)DI^hJv<8Qe7J<y(qo;8=!Z@*S5&7bQ}Vyo#+2apCxQ%0)(F!1fKrb!1i&Q{Mzk8B0l9#j{OpR!X=VIlI!%VRt6(C3QFOh4c*(cWXxc?hGkZIfm98rYstmuH01O^FPR`l@%M<Q{ay|pF1%hA$e4_PjZQ<TU3`xuW{KyHpNkazEb><BGsV5-RjzM<s<EMCM#@<0vVO1W@VXQaj&h^uDO^`iZY3X1Bo4et2E`6qKQomVeyZl-t1U8hqP2-2|E%&RUtttT4ZE-LdqYhMwMv~L^+tWI3JFC`%F*EMKdl@TrO##GYVYeu%VWWw@8)|XQlQ8FL&i<ku#OB`f6^@o(_jUQ~5fa>kEP6$5OK#1A_#DI(yX2Hyb4yd(%db_i<ovEn4*4z4ozNT`nDX;up7z0h}J|B~X~<w+~~hL<^4cHY_20(#Yy(Q&BvSsuxpEJfTeq2UP7J@o~)&PC_lowTv+1Z@1G$*A)aY&LsA`O&-2)m9EU2RHX5_{_inZ62u=9c6^6-IqU5-u0v+0QEyDRS>cL^fi!vfua?)U*D)PwN#z`sJc?q&SS9W5dHrq&B}02l=2a}7jope76byDFw&<4vkB%r9c$B%5ovQ57C%t)ROFa;AiV~|C5}tc<s8lgUJzz+FfEvVLq2;g^_8*tkN12ghD#b>%W~d@hSQCe{t;21+h=0yz2s10&LRTy_YmGwvrh4bk!1{K^g2&GrZk&<d^3h+XlwTCdcTgo?${Q|HwQSsI?AZ6VrzaAi(s8{(>kerWa}=^ArUi1jEFCvJ?Fzkr#zY0wHoGWSbcyzsA5QN8(|YJhHx*#KKUX|G2S6foy+}wQNlp}3jc8x@dp=yGGjt#^T{}<Nkvn?2Dc+<?l$$p_N6;y=jZIC937<OC^t!Hu=co*$j3X|l^TJv!2ErVU6jiH09gAdijZPBSxpY7<u)$^vHlco~YCYRQ)apL^<JK-foS@|rv@(*u8cD&VB)CrTPB6=N-z*b6`EE8<EyS$rBvfM`nC%k-*p~4OQziRQD!D0CTD1<mo6l#HW#jR1l6kX~LQhCa!7EIb2~3vXB&!IXd9^=N8mjG|m!t-3n!;7wve2BTvW+g5SYA$2;K`_GXAL|!rwxC~?j@Lo?J;L>4^763Svr0qO@vb42$;{Nc`jS$0eS*zX9awkk4cEtae%C|Hp?2t&$lLnk!1D1G-_*>!rQ$<yYVq`sShIoT%5^dTye_Hn3PU2-tbhi@XsPR?noyJQ+MnPb(i=6i-Vg;wT4cPT9L{a10~B&#_LUP7U%L?N;wrnc|P`%C8yfJo=p~qOE;PQqbBc)vXp2+1Ah7T1vxoR$CQ*Np1uA%tVJn&KBgPf#F?Wl!ct}!+?!1)!{<^JmBA!%M+0`E%Y=%N*}dL^d-tkaa4}+5f0~D<sd<`)qwZkDyi<IH>@@IcM1y-#mT;>1EGn(b?*gmtZA#vK>y-DAPa6a|x=8qS#(h+d1Q;&CDo;^N$bl(Fg`}*)jceiAnVS7aB>yRjdL8HnIu{!*E#XZHW}ycVtgF)<?fGpfvRN&!wTt9p68P<662t7X@hICs0`8cPK9!%uvK=U;YZZ-4F`y^A4mtGA*s0u7Ut2U10eKM5#&~1WozqIKSnrVzf|B|aTr63H28&Al7FsE;U8F)Rr^$L#YSioH=~*KxGcg<)^$Yaidpi1$tb-}DqjL6(QZ+{jcYY9_zMM7Un04S>2n6-?FG-E8Mmfv5NDD5^;}I#Lk7Msg-X&BfCAwXbi9K4ljqnHk!1Yv=l)|I6$?DRrQppA}&o*i!(eoYED)E#PN1DT!@R5pZge~EYW>JeH;dF+of)^GVL2~C|smZp;N}_3}D20C}tyf&0rS&5Y8?}{$+??ToG(s8dA)&!dfKl)TCQwdWnz)5odGaRfGx5<hv*=Wth#Kjj<45TB1v*2FzoDLlf)b?PqrfTaKw0}?!sED`D6sz4DdcDxLDBdy;cgk0$-yfc%k({;qM|t~#XBn}@lTr0=v8v#I3;zGL&LgKb9qYcZJqo#0Pi2BZ?Px|G1<#x;+#(>tM$y~b&-9b8#EYWz2ewAq}5J#6JGAgZB%M|Q87I#o>y6|H7hQ|6G))WaQ;_W0i5$=Jxk=%5p0yaCxtE-UFyjL@C{he!Kk}~VyYVq2kyPW=+5?Fa3_<j|GudxLKWSue}6pkT-ixrJDXbsDxr5A4SK!}H>$nY-4V%L_q_3_dp~@|z26J&n+UDn&)g?}cCm=0!<d1mwQ&_;*M7c`qwdogOvwloO|;<tW`0Y#3{Im9NcN+x5jJBsdVb3?45cMvUt^1EPCSa6HHw^<@Rc~BZ1|0CM&TZUwB@ld>HeOplD-tt6cSsWqTBKpfJ;Doa_7bgp=V_v1pdxmm@XRkQ@H+dn{J!GqF+GwzOlr(7;rP8Nv9(vx>u`2BURz2csq!XDLfwKaK%~0I*215{+;%ExD-Pl;cwVc_+?se3<+&Litt7$-~v==il*7@R=SrAmos#srwpmO9m+6oNOy+kg<8;=(HOh~&P#8srCX3?<U*%+C#JkW3LSNRm_lOc*UkKtm3VDYYZz;>Q%n<M&b?iKByfsKRT^=~5MV;LiK@>}+#^mbV6JpPg$8N3Ed0iU9nhascm*OfahYA$UiE^KCzU3&xX?-=Y%!QVNy}J8B*c(yDtVWr2Z4zjpOVqZj8rq?OJG*_h*t!TG?s2wR?CDW20`AOkxu5D6Fp^(X*vTSrDP!pYu?(oFW8q_QN$K}buf_pf{lr1FW5vMTCBY@WzZwOE9&;5M%FPv3>YyHR(~KH9vU+W=<(}lvqsD)LNAv}wn#W&U!sS}!-h!1O*^)=QU$)93o_9USv~PI2)Lzq+~t|7i;8hVQo<JHaV8{h_X5{zVMS4CV+k3J_43+sxsg_^m(DGwKBJ`MN*G(1)+tORgP_aUmvE^zWo~eHCf=xRbFdp>@D_-ocs7tmWq|}taax8k&oRYFQmvO)-5VR_`f?hcD*l{!^B%qa1EML87cbYl7&UzYG8L0IiqFHl7|SX<9Q^)+xJ<u?Tl6Us88xfJEK7A1%p+W|jA~mynDoXa6~JMbpoE+`G%$tS8C7k1;#{QFJhE1*_;h(lx^~jrW>W?(<`maX47;ba3_$7W?^%iY-)lhq>bx@D8R+zz97MxD(z^bFojl80i~m3GB_%~Jh=d%0mK)(9Vrhyo{>BM;x1!3&ISMC5KZM_6$<&!=E2<0}+8rFqiWrQ2<b}8xyom^6F-yU$7rhwc&f<2w2qz1!MMZ{ntf;Ddcx$Cy>nQ1DH;W7TJV>k>XRFe{Pd&l+*l~c0+}`1LCq)1cAr;UipX5vR!8I2WsS`l<7S-d+m|Ow=y<!`<NTsm|N{@y&sX|lgdE2b=vz3b8Ldk`Ny{l_CDvf4k**-nLeOT%nj?5o*Kc^>Blsbr}eJH9SG41|Juj@++!AJDkZ$)9GV@s4NB**8IV_U5!u5_$`l?zOV-1QK^bP!YG!7Kpvyxw33*@Dmq?Dhq~G(6gi)<N7C<j@k%NZCZlFb$(bPTP*9IF2~Nr9D{0ut*?qd%!W+PHZ-t#ljpMz(rK~lX5OFhYTpW_r>DgIK24ELgAgO@8rGng%_XvJq8#HRqqi8*}mlLGh8^9JR2p!l#(_^M=#9HF3g$M1Xx3K_B;0OrYoNCh-BzjGBr7q;$uNzkA;rvC@ick^M-2qGKCD~#4KK%;avFIVI(kX-5iT-XcW*v-qIq~%Ihx_y6#wED$f^Im`CwpsE`k<T*k(?D6zuhDCOxZ7>InyqjF!7!i_>+NF(trJGRoIjxi;|=VaAG?fI{fA6m!k)&cCvKTy66)HnPZ-#rq#4B8D@DM?o2CdK0GUkR_-5Ee!2;>BdHh|rKLRo=A6l&zKHt=ZjHRO0?#q<2>8hjNk9?|Hi-;{b!<1+FnFFM9DJEaiWbkfIdYa2`kPz&iP7K)bIkQ8oNCN`wA+=#d|Wd{ZG(i<NSeQuQIHzNU6EC$}gnMWz&)j2&mIDE1$dO$C3YedQO~qWb*x??c_rj~xZx%I8~cFJJh}GSJFnhbL)aj!qv2-L}Sv=$FJnxg@Nu$p4^%G;pxs`5iR%1ue9uI>edm;l&sl9@*?SC4p6}1bQ+<_#a^#|Ka5SjIaum&x`3gIJ@WwI;db}Pwq2uRlHcZxo%a~nzj1JrOmBH@$BPDxsFvenhoV)6NKtY;%Tjp2^`BS_$)OCowzt$u6vqv54c~7@mq_)VoBA5<hGDyO4<^kUCGWDY^hMSS2xzGOV#x<R)$}#-Kdw>Z+>j2y*Foiud-BItF3<Qu9UCAh2vJ1H@GU2+bn;S+S6s*EbQtk<0wz#pe?n=n4gm}okX)~tk*)ox;3~@D=RAv;xo$2%hhJ}R>eNkaPhpIG8L3IP<dsAHoO5tik-g+V(Dqasa@17H>wRx0bE<A&o!8LDy-(C;cr%N+^ncDzO`OUeQzOp5B7C=1H;zDCe=2Yz!p>*sqbDf?Q~-u28F-ZmRD<TvtC|<rPQ0YQ5R@$tbMRn`*1Bi&!S)|G1H72v^-$4ouNY-D-AWUx`Ws#^ks74y8*Sao`GOjv6CXc)%Wj@GnrPTQ_4`bB^w#0ctTcqv@`PB6Xgk+Fr+3b$U3BwG#U|LC<!efnf6+Z3bwl+R6b7VfKYxYkq!)AO4ov&MIuef<kd=zjip3WF$uP#CGkGs2_D8%dCSb5l(;sLh_;{3zn_wGH_EtMfO!Z(I@Bbgqg%R2J5BZ^j3Q;XUBo{3QFJ7E5qar8d?PgCwc3a%EW7PHecWvV6Liy9DzB{pZ;o`^Hm3N<AoSbYx!Po0>#R~~h%i3c5x-Sy;iD{P0y&om#VmYz1E`8BND+QvF5Y_e+6EB<B0jguRXE@)mGcNyG2mUYE&CGk6&e42Hs?~VJ=ax*A(-39w}N4}*R!}92Eh&?YT+ri52tJzK$YnwfY#kui{@)064-6mA(_^voY$B%gC!oPpwkGi+m2s}iGxZi$J%n`t&&YUN}5zJl5r2<S}Yc+uEOPFael5df3<zN)tM_Sy!-CEbMQ-E{`Sth3;BgMJihE->CAg={_W7PuR6yN?XZHp-L|SOiC^N`xXL6U_M7(^wkrT6ZtV*G!7e5l3P}P(T?!drZ0S}#O~GOwbGW-7U_A(`?cfgX*hS6=Vs{1u^t`7&l*b}kC3<4+mQEwx4}C<xyg1(YzzvTBBELMYSoW&AQfsE10SP^m=Ev><kJ)o~FzC8yQ!jL`2wx6U!sU|Ac+)smtby63V(B2OZdnJg`-ep`17AuQaizd%J6%VW_NQt6L`vYe{u8azA}T1Lv9<?rbx?6EIt9dXw->EGOn77kz^eDhy?raV(*<;G>j_$LH}$$JBJr@nAVxMXe#vCzT8wmhrRio3eHxQKzajcglypg6)%4C=rD#N9{o+%~x^>zPO8$O8%~9j}Rw<Ns4wRRjgvCp$1>I=nQ|*D9Tgger`ruN#(vbI-tTq)j$fhkp>MCH%dm!5?OAwU6PWUZ|s99?~n6!3_KoT)I>e%uud9o2M)_8X`nCt-Fu*J(E@_BC;jqjx4L{d6rmBE=3kDA#Pnu9=zGuWoY(}J1fVAEV7D_9JHtEX73$a#k*@s>a{z=rqpsQ}j$jzV2?%)XRxmgB*5<nI`2l=QvWA36IYLETCwK3U`<Ck&8xo1`*yw}v7Lhuvm2QLRnC0Mmds?y`Y{-XAf(P4+UQP3D-k=yD$sTZEp-d?r#rAiYgkvlQu~zGa=2&_+oGh~j!CJQ3iLATRpl5>hrG#27D9n3NtTiuYS7dEXg3{QvM`wvjlwNy|o7@iU2<kg-F%!})LG-li={4yRq@5#4ek^9EijEZf-RsR`^dtw3{(OLm^T>nYMCan&=GE~$en*M8_xNUH~M$GxtT!)m+$P9G4!#1VwY2j<irg*DRlN94?<2nbsO@xvn8$x$h?Eqs5ba9Rh`IE+N+p_>{=pWSHw0wH1#vl@RA>N`NVuL=PG<cL+^fkG|Bp&_9uZ&Kp)cYiXE(JwnK%#GM1`n)L0-3<ps0d0Ax4!Vc#pyM7WOi4-tB8BfRseE?^U)2v`JZ@SBFNatX|2b<65SKK`4n1+73+4Um-$Y(Qvk6X1<|CE)?G@#r9ZMQghtoMjS;ITsmL<<9Na!$p(L(-dUse{_h1<;oh_`wkU_LC{>+O&6CFsK8>kz{)1{1h7XGmR(wMXOf51e^tXEdZsRW-dPK$qKrYk9c6pGnXS9Z*T-WJLTFQsdZtSD2&dpusHAv?_3$PPjOT?)JVb$TkVs&@vaJQ8~O(3`M1EXzh=NARNjBh~<^0Td!b+1Xva+xHKgmMXHgujY8F6o|HnH)P3O-I*NAf!!<q`NkIV5V-zMIPC6t_V#0?;18!&1>jC^Et?wg`4~N8drf*Q1WjE0=(U#T#rU(3*x!33(1Yw}p8{Bg-!Pveo4<x!C1N-ft(zk*aIA?(oTd_k>6GY2utk*2*E{q;ZgULAPwtcE>vM?7ip^Qg4tM0c3BRC1I7G}uY?)JK4I4wTrs2Tgc{oJ(V`amd4DQ?OMQU>kPg>gmyeqf^zVi%!3i6h}Ph!3<oP+nZ<Ptfb_V<2X0f*G3ufg)#pc+o5$tG)|B(t$>Wa<CL)vA`-IOQgkdS1Zk%wdERcbsy6)__)$^tIN=)dc6u9+@Yf6FejZgAAJJ^Y$Iq@jgzKd3e`smWsy3o1O}q&Uhxw*Az(PWskkld;)3Y~WGmWE{I;M-2PyWp7i|X*^>|ZbpSDQ2WEB?DkL0;c5D$sfBqe&S(I5ycGH+QK0FpK7cj0HXFdRs^JUPShWun08)`^72%<`%+UeO}Fu|>YH?QOqP@GqasB0rBAu{rOIupS??@tRU0avGT@k%?2wi-buNm1wM!mx@{bPvjk6$rm(pacFF%d-xe;UHT1We0j!WQ$!VA?kGY$<Ajwl7kedO;-i(uM>Ji1UVw}<s<j)|`Day#JK{VoYzS7Tz;@d2MKW+so+lzT!Wx`xyw{A+T$=bZ4O|?wCX236d2C}97K3>hf${3(n}iXZemfQwPM=T}pTFf9#gf6D9RJAH<+*p~uC(5{GXIxYmt#z)&DrkVu>62MC0|{mvRVe>xYR%f@b9Uv!!g&+PT=XuQ^dR3N`7wk<d10gpJnrZoZEwEPdGyi+4n{5^Y4X4|C#E1Au_=$z6n|u9A+#PbW?l*=;g<pw~1mOk42Re!n3d{>+cm`3FWecg_L4J0<q8yiq?qRQ9C^(eOWBYP_Sw-RR9}OZ=9z;wP2>~RC^fjzq+<u63*ksAQQ9>bC5}@JvN(e;6S@y&)eH>dzM$s6T=Aih%2hB0Eri}6`<L+KO!?ZbBqOTnyckTvr-o}L?v!TEa6N&C#;6T_<JgRxd1fZ#|Tv9wB$2O1%!$Qb5pt%u{m~rzF35oyis#_TT$jMEYhfUb_?b?S16pZLu&07+yZtBUuFsxrtsu+psW{6c7FMsonMYR7p7X@&^KDTPr=}@^92M2j1a>z3fW`u%tr$yh5!z1W%<m5z(1sk4QpcV^9T!;QY2FyV|8O~O6So@A&9TSF1ETuz>GG~PMGs>H)bIZP4#D)@0DUPGVP<Sbma8|Dw9AR#}7041DvtfWdt?+<MR2)=ZnQ)2rwr#QU^HWev1@eJHA+gr93v5$rBs|@>R2Bt|{cD7ao)8SLCzyLb&V^5q+rKkd)XOuH#rOB2ISwjjWw5OBcmUf}mFs{h^%N<Wa{?TrZH2KY}*jaeXo^A81rwOej+_p2IqNr{xWeVNYaC*i@Vtgic?2CrtBAPMSgOcp)`A<})Fpfz)gC&+H);Efp`Pcpt@6B8R73(;bmt#jrDi8A1;0;lD!^z`4v39Mrj5`=El%#J2)9jw2UP+;&k&hMeKl2u@Lm{w3y+(d)CTYy6!ds6X)bhHxE3@kLP(G0-3kn{jDchqe`0wb4^ss6q_RNF;0MQCZ0VEK~Ftg;+^L%mH<D=vc%Ns(v8=%zoTFBifpcGzYxYL&ZTvwh!Gz`dpFOkfY$d@ilD~T{T}i_)RK0{rkJ#BpAaL39S?d0*3DSuw2<_GLLY%v0_T(JBo{Kiud4HC6Wc6oIIV5^}^4hZK9SSKVsFiQj!9}Q)g4B3yvMHod^%GXkzSeW0!Ct8@d$igQ?q)F>pmjsfd*%^>*wD;No6l$cAyqh-Ev|Vxk9uPbIi)j#-bo;lmSCQ6wiq%$#{L@XH&|BmrYoWiF=z`m$-DbHNo4J*mLPSQe>oZDwpN#fkhT$YMU$4;2WMbz(o=9S+=ZOSuuuDLc&pDfT|{w~hd&)-|f4B+nbRL9m{^cbr}1*B4HYpqbq&hWaVF^l2)m=+k<w;g*+It5nAqxF4(zG(`2^$j<zAH=FtE;(vZ{`}T+Uft}em;S&AB*2#+pY5~Frs+9jA^z+9IHOI*Zk6!=wfnL542e^Iv9**_kuVS^5*D5zonIR4SmGn)2AVOr0c${Z^)4%=#v;RJbui@k+40+KK)mM=((pzwjdb%XPq2S>!9*8tHFJ<~{1jUOm<EmH}iP{g6sSIGeOBTg@{POj~2Xx^Q%kcWM*qU)y=#(UvOB@~LZaaDQK)H_vSidJXe2m)oEk8=Yl1lKII%Y%vuQ@v90hbbZ{nZ0eQ<A3tfWT46=N}N!e@XQ_g!YD33(UeVF-V<sMBEVvf1LdAK-4)!70zc5MCv*69>C8J2n{~FeLF{0?M}Wypguo&nN8)F|23IC0$HtC0w)np%2@(p(G()@y`mBcbz;)WZ_)@l7CjNMBT<1FHK<(HfD^UMSdTSBtn{msrxDYwvD2*{+??Icexw`VqfFI)$r_89&2|Q(eIsklC4OyUo`BGIgT6=a3ylEs5cD-!QH&F6#^1z2l{e|DeAAq*5+<khA*ZYK6Fy4t@-aG;W3wNZY2%z`$lAv!!IU<H;ql*1QLv<(2>WxF%4^Hj<#JPJhA6+2Jdo!cmB$bs>(d7be9^kryjiJhKO_7x2@;MqK{Y?6et<J9h8u}$&=2?zhmjzCP>}UOTnj6Y-EUuiZab4~GWE;fr>KmNICAK@2rWA1auZ<`8~MU~AI7W;iRdvW>0m~Y@^QRE4fQ`*RLd_q#z(TG!4X%s!~_l6IYnBoi5GY+0^}+yz^<dQ>3Xe>0)@MDv$FJo&5`;&|1Aky+BN}FAy)isqg+Q8cde2PBS^kW+raM^#n>BbVF2XDS~Vq*;J}u;U*SDN_Vef=dqYr<DQp4RxPX`UT4TG&RSa#KiFh<|Y<b4yBlv!3W}*cu#-n$TdlEib9b^;*2UXG7rbtZ|@ktpGyK*DO1qnt+<9fIUJQ)d2CRs(#b;xat=0mx!(OzZDp$XDcl8la6tDtd}Sts$FxY+Qy#@(Be%|<Yc91j?H(*MZOhb^YfM)B((ZQ95=pYHk2PCdjD%YJoM89g0{MNT%L$7dZygGMn0NH7{fv6u$jvyGM%B$6G<U$R1+3#Wu%uv%j*#yQ+7S6ClQN+7#DfC-^Qv+G9tojn_SU0Zgg@LW2GMe9c%cW7n6FdnicyTA%`0aENl!Xg7YD1}*Dcl^FT!dzf-8)H5pcH0LTzt8)d(p76R8euBwy94e3HS^<1dq3Wh2s(*u%rHwgXzjwqg8}5@0l(xD5sJSW>C4?P6>8`sfW@`EK8_!yP)oE@dqbN9Iw_!1hLFh6DWjatIoruCLGGhrdR!820J}?me|khe>SgNHb-J$Wm7i@?>y-wUO2Zn~6pZKIs?@93s~oi!?;88+258%Jy}Mqwhv?;c!Jxx6O0T$Lj9b+;Oeo;i)>b}FfgRxH2+&!2$f3`|Qf=*ewZ5t)8jz?oL_e5fSnzc`Xzlg~z_ZPYjBLT|-T=S^Xa?IyqN>`-aMayHDKqq!0C)&5Yb_<->?x`5qF$A2Cv@;@+^kfZ@f#=UAT!k%8Y_9hVc?I5J&75~=}b-4s-D>eXK1)*?sqVM2)d~BIQ5m|_n|Xj1OTPHJ@{lwphI`-uJ4nU`hF)zpNaklsa5Xp;XtHeM8B9Gj1;|9qK_hpJ<AAv%7ZkwaHjbX-;U9zCEI#n3H>SZ>36f6MUe8?W%@2xuf3OpGvA-=4fjC=z<3L9=x1gM*)tJznW9jL26vft=Aooqff>Is`-r(i4TF^X7GVI_ErCL~$cm6!`r}X$u+W87$wrzr#la{Pdoyw+!crYR>Q@ny&grUjN{>XswEU!LF?6ddGYh#`8WKsE1YxOHZdEHEy6Y9Vr`B$q5<3?<K4CG@gv$wO_2pTT>VK@g%DMg*)#9B;#!$P{1`p%#Bv5Fi&<T%y>uj^@q7S!G2uqRVU%RX|jG204ll={IXo$oG(#1!$-B|zjf5)9*UVrs(Sk*M?!}CPRi|{ORuR+<yVej2n1~;46zax7KDa$@*>z(d9gE<sp9?5v96#QZaNXM~6LjD1VpRCR>=b`U@a<hvfYqcmDaLnE}s_{rEm!xFVh`afaafvfo#N4P}e+iTLuv}kLX_TQb-f8m|SZj8wicAkEM}E1OkSbw=!^=0^oiKB?DO>50SvOTLe@Jo;@d^~~nh>x&B@<DqYfB<UfWD9BSn8o>D_lfkrw6$W4U{iAZ(}G)JV03lZa|;Ou(h-C`XNC`rmCWD4Uu+^DcO}y%+Exd-+0|9gN{rtqp8$N?qifMD@Lu7vpA!O0i;XM9D*8Dg(0H!e<H=bS`<2pBS`U2BwvEa+J{EFMH%rFiwKK~iJ*#H8!(|BWgBzCweoS~q>$6=J_ARHG6-$IjCw03A{Hm;&RjYmWknHfoaOcPdhJ$a*}YYZG>7h5d9~7TLS;p~C1u5kH$6<&GYLxL$q;r{4oZYqNc_ObBv28B0+H1;A&BNMfqL?(h+pzQmFrEY61H;f(O?MdR5s@Ygo(gO>0@tuZ5ozO&WxN&g3E<=NuP<1eZYfbNL)-)ho#pgwLKPf93{0>lI@~~{84Oci($4^7%?^duJox3CvHT2Ia8N+ue_2->K7_wMy;eKrtjDxW&u@JY=1?{o}i=MsHLkf68bzTicUBU1H8hiI!JvMVq7+<;_5piXM%0t06miP!v73r)c9=0T(=mF+^~2`0QiJTHo|aG5R<u7hlCEYKA7V^BI%W3NB&S8EMPl{-8zF)d_&CGYJt4aZR6~OcYFdOa6TSb?64`gNfHTU`vf<baE5<CF(Bwkjbxhr;Kj=Z;(Thxx#$?Zv5$MZGv|5l`tO{}$9(~P+(|Y6oU5RynZzQ$^B?F)5*J2BteB}XvTf1{C!PKevYv{X1WfUO|7THah26{IUhZgprHQ=6q5gRCwIfoLIqdEJt#FkJli|r!*YH1J>>Nubw8Kf#bYH;qUJ5(o4}xL;JgSFGInSQ34K&Jlmcvnhpb-V>G7Qd6aP)JPD}#pxT$I#lk2rZ_SW5ZiC0jK(1P@tfW^Wa@>R05yowODH$bRed`Ii4~YrgZN`>hYXzTcbbw7y<mtE`ZcCSG4Fj&4OfrRI%4Id&}eVRxX>{A?w^;7r+T5FFw%-2cUPc&aq7vlqwH&jdD!Ci#Rs-Z>JUP3p%~J6N0t#q}L;I4KCD%$9yuaxt|qkd%^=L>@u18D?~1Rj^Rm;_fFNGDSZ<HIAwej`wHyCE?b?+EbO*i<$GE+{njQWTJN9ybLoh#vhtT7A=#%*k=|Z64O?6*Fp=-@GT}YW*sJ0xT(#0tJEKi_Gq4&lop0Q4C^s<WNK!bkv{FMQs8wsokAwPQEE>iU|>!Y^kdGX{ch+2tuL61xP)Ady{N%NF+D(wbmN6w?sU+_dz1DHwm+nHAoS<4cVntJ5%ivX^G+WwVxe84{p!oKxyr8PV!NMB>qS{cdRNK)H}r=uNMi-`G|X{9fXrs*Z$vlhydT6LDkEfbz}fUbY9Nj)QtGy;R4F<ZWX~bEn}C5#Y5L0K@LcK6IR4f-Ca@{9^SMj4WsK&mt~Kjrk%<r_b=ZFovz_DfN?ksWmv&D>`&@6F$%Am!-_K+t;V9BRac<pTYrfD=66!6y@Gz|k++*Ldob-yWiEqNBz#{u9=kfnsazSUECh64WJma8?92;bw5s->F(xiy$CR8+uTPM^8OB9t<b2PZecIj^R@YIc=RA(s22n-<cVN;?>Z02I#TYv+UU`QApOw`7|kHSfJ?Eb<Z4fr4%`UAYnXMbw`tv?v!V9BoKYW?<(FV*oOISVEr#{h?8($t4-6g?h0VZ(OE+_1HlcLA0j4JPWqw*d5Lw)Dm-IeJ?p_mvd|iMjm-)v=U=CzaiPA<bgPB5}<N<iCFd68kr(_mmn<j)0Z2J0}nhV?ZB8x==!o81JH3&SMzF%S<`Do2ERKr*j7S-9x81IRQ$)5INY^mI32(3dJmG=YNYTeP+@r-5XDLno!I0ua}MUA9<(FU!iMq{?kVjq%k?(q|f&P^Ma<~VpR1k!(qbJ$zEp98)8NjXH#yLBUkN26WYp(3@Ei_NmxOYFS|F;GtRMV%SD+RpE6|=E+5MPQ_?*$wfs^_cWTc2CuX%g$x8uHjxfK7agSrytiuX6QI$34?d)VUbro7$Gf74h^DNj|hXU~Up|RCu&(4xDRTfOk7}YAUVd$(o2$+v?cq!91qq2OX(kRI-*nCrLb`p1u!~|_9pwK`*1@*%R8b_e$CokC#>`UAxIilK7u?1n@obr{b676S;s2SYJi9xd>=jjgKJKS}cLng6s^$wHyLolZ`1qTv<gM__ipY1iIlZ+-~NbVbyqMtYSw-$%69_10KiT7s;4o*K=GM*<BSsz5ECS@0}E0M(pN6R{P-@oVH4MP8*=(iahJIfV<gK-!EvQ|Egxw3nMV4Um|dzqm@-=t3>(86QOJl~(b@qB;owOZb&J+0eW`Sn436iVQvJf~5Cx9)rWQ-|fltsRhJr#JA%8Ad|X+=|nnVPUDXpFCTJTs>ip=U{9~Ef}G!!rg^fBadX!7oo4x7i_Kaq0I_vUqS2$X-p-r-gvAU<w2^|u|P_4&<Ly4LFG|!+nrL34AL+uS%2i?tW>0Ir4du=MOl#6k@CxFHy)>*3n!a>zfd$EYWPvmNT5vj#=9=LPyw}68sy@~d^Y{3T(H~ij5CGol);7rOo53Z$y7I78!9oC+z#}YTgX}utm*AgCdys8YAWg%y`~Lr6iudQiw@;1C8^M@N*%rCuGK5$4}g`ZR~ogITPbyS6z3L^@r9ImB1$A)A4R4#sV&ct@#+UrsWeQmQKYz@*I$WTm(<&LvBZ#M5|#B$(}GcsWF8M;hvkfJq{mEILg*qROO^<=bxO7x;kUp}8?5IKOMkGcl2UPeD*~im#1vmK$>YC6udGXbND6yS>`}?j6=k^8H|%yH-2TI;@`uSt8Zn$RMcpo%cH6XwZ((rZUP8e{T7OYYPg1k`HqMcX)zb91?2f9hm7v!%+&r~1bGf+%i^GnT82W6H(M3W-8otSjjd{zfYfDX(n7E*x69ePR&>MveD&Mid|AI^#&w24;S2)@Nk&LUOzy6g>#`0Ly3yLBw7|s;8|A>%**x{&f3kFS)GW!LWh*IEl=2-Dm-9F(!v~JUuJ{4(TqNIgSHJ!PWTFA$|3Aj*B#XH@r^X?!PJ{{Er2q)ZalNHnH_WKU3u<tPA=?w0S_XhA3IMOj0Se&6pYDRcLe4_X8W;um3Fa?E&RRL7z9E_z%26daF-;1!YyMr7$x@A)0=CcXTJdA$dwBcrU>~KNI*W+?Y*H2tN0K(paK{i{eeww~39m4nNN@#45xO#c&C`pBeg49;XF~Og1Tk30m3g7f0<!>g#wGIQEQKYKARkIgq5wTo6D74_yc&>eX((?9(lOWy#W262>#B)9)a>V7LzT(o>qF@vdeKzokU@Lfx#O+bcFh$4t$)ctBs_|jLrZF`QpN?Zz&b;JGQ=xe@7oW7)W|Dy9)N|q9OZ6+lH4X(0YIuK-_qhWA90+a6SIpv?0=XT35Yt&x`Pq!F2FH#OA3>a9R<XKv!>xVLT&+>AK=!xC(sJe1^$ny!vLK6j%6lFjF^UpsQ_7XCQX!W&mV_J*;P|U{IN_oZ+2Z1+F&RZ6PY)g?@ly+ihCe+I1L1olJd3OVcxowIKpP1veA)&Ev$YF?T&yl4&E=E>#*K3LIlCxkFP@!tz-0flq0t!^Z^QO4$`-Lj8rrJA8wSuUL*cp-XD%E%*+&fdpc}xxu(iZsTj2l=(=cJ6y<V%XHQf)Z&6~B2rW=(5B4o2u=V8F85SgQS39{q>vN`!&-Y}=+(UiXsj%>-Lse2VVciHGC-`_E<m3$9zpS)i87rp{p{2}o^I^4@3UivI+_l)e@w}VT7pq7o-qJN*SSZb398&2~;6jdqNW>0LEh`E-Qc`7kS9wG7MhLM|v>ZBNRzz8{A!C45$L5vq?CN9Pak{5!`lM++R&Mv<s0{RU3Z{aoa1+m4?m9&k-@ju5&6_ruRxm%2i1=mFB&CoxDSY37iP_04<pApT^@zaSnLU*UWPq@~@<PSh2*@bGfpJ{g-K3t}<QtF{G6opErPeSPq#adbKsol{M%XCW6<n$|+1C&Xhj4y<<P%j5UAuKWC0f!YkxdPEx!wSM@tOqs_C1x&0OO+~<Mla|USqt&v7Bf-{c{J|9J+#voK9(g0?<LzM2T-_$sz6)NMu~|yFd;Q&{SAz<pbRb?<!{1aOrR2aXlN^Sl(2i6q8Zi4nWQ5pDHyqi2KKSZf8H-_v6U@ra_Yo)3?PDE_PpQ@I!vxkdgE>tJIHlLgFTnRYyo1M@G4NBF5J@@Lo?wXNR~S>PS#=HSomy$Q0lgb$%t(!<frTl#=xyqmzumclz;XOHYgm>v_XdB0L<xWXY=mBLH061oKJm`n~`&<tugQSl@vD6w{v$O0;+9?-%3PFW2p~sTb%3ZVMKHBl*nNnKRYeyb6B|8Oje)9#pHwghoM_|jGo7HqI{Jg#-YE~G{HAk)YukziUY3NQeBP*4M|A55pyJ16l3kY7!}oc7AJ9ohwNWuF=5_tnv5uU#fT$JGAm9x@1f%595VL_H!LxkqCTsl_<I;f)45W}*%><${?buzF%Gd9`zpd4^c_G^c$_q>geeB#{LCRz&*}6W-z#zCzv-}&j63L2h@!U7=Bk|7SmX)Bu8a5&5YpaOiHr?8DG(EvhlpD11F2lLJVm<6brhX=(l*0>!hM<v2XKJcp_3(hTNB$g6@?TWo__65Q(%XsJ2^@z;XKbH6ay!ut6V=J>xd&q>+0h39p%>U^N$JZIu|{`wn>RTe8mo<yInpMzDSC&lS>>Dboj8tsA#v*+*tlt3W-^1O-bTQz=lG(`p8x&7M$YTu57zdbbFIFrag;;@yt;>N*+C#B$@m)xV(2*H2LHORb-+Z<)i{OhYqUsrUf?|FWXIeX(dia<%2P(@rzIqu4-f{snbIm<-1s>X5?4d15Jy|ancj(=x3X9QO1*Fa>$V>fx4bahLnhuN9K`Cl=>}ATPPkyH74>7pm<8Dut^PMky%l-s1_~ROfcClC4vt&GiebTrIHBW`KjSJo8D4mG)g7>_uQ{y&VW*CJ{&!hY7|;Qyd|tJ7mpk<h8gALi8WMa=AtpCz!w8emQgk|xH-}*yEy4b&Dd}l&3Z=ulr<9k=+*NuL>p^iM3@1S8>cc!QQQ>ezl<i4J*1OpMndVQ9(P<nd4a#jSsvy6$3icdg1(ka@0&d6$I~jDCgA+j6ga1~Pt_GjFMAFg4AbQaI}>R)qj-@g!stZFjZ^Zu(TbyitxGB_o*Fa`{8(reuw12*Co6W~g=#}Y0<hBdRhS!JC^bw4888XWY?vW?Hat+O7ic6IMEUatHT*8JA%->(Bk}Ft^<8l}XFRH+$1+&9Xe{r{xhBMpY1VV*`3#wI{h%e&(2t8)(fPuqai}w*?+*M6f3lb8TQXeIMJX{#(F+}B#@VP+3D4MwRf=j>ERVEqrH&TeW>wd$%t1KTVb*j)d)>Y6xRhc)nUZx_OMI{L-%-$DDY+pc8?A185rW3)f&-^A0;(kONXBkN4`dP~=UmxYg~N$DmUfpD;m#(V9n=*kQV=S?nd}FuVJT{v>$bDg`Z>kmq;y1D@o8OY(GKxh#DO!lpQtTWwZjDhxTHS9Edyq`jrGb>wY<{&7z-?5vY?ut)Q?y|h121%=t#%-lx?uy-kv|Brjw*x3>;(j4avhtasGykPk$?-w<81UW0tnbUOS`4W91S{72~=JXApyxNf3LRNFHYl=jtuwoWKgymqe{!3Fh=$@&!23lVpSroKHo<1%Bz|+njasbxNl4*uvw-Bxev5IW-;?fF<rkKUgOI<Zf*3n@S&BX3e@xlc{LglfB;Y{bA{@P;gogth*%3*!+=HZksmWdB}!ixPbr$ec{3mj!k8}^Y)hU7XLY&=lIT%_6x^0tG>idfd^f_pDiZUcJjASU#F|<;JmwQz%WrxOxM1o0PT#*x)^0mEWx}2_1eZzUezZ!CEQSi>ydBIw@Q3x{6Mj2rrR?xE-_yE^@%KPqeQEo9L;K_Z@!w99x=I?7G~;I5=#kgG);;>aCV3Oh}28TRC%$><@5^!rF(f*==Vd&(w{5UxInJhYV|`G`!U6~rQpNAqlwEfo71eb)2}0~Gj#*?WhuW+B{T5>dV|58$xvDqHw`GFg7Osorztal!In|tK^3?9bC0@Zw0}r-r2YT0_x9aw9A}>B|MydLoHK4fYJwmoN)G7G9Y};EuK5;%l<m;mUJz)Ih(G{AgA{3*bNrIYWH&SMICt*Jo@|nto!x);99xlM*_N$yCSQR13ip%T=cTH;s=68gB_}(xXEJ9jqPx0YpL*)4=jHd13KWyd<hjagLl>PnjEDO%+erf0P%qRcAbmRhLg+y66d@1$SsXQxi#z*A{i8&*ydyinef~~3=VrBHC#cU|d{({8wO9GP@wRT>nZ^oYOp(>m5Ibu55XgDo9Y@9%-SWh9-7h@z7Xl9Qyo3Jz(C@}=;8|0xnY9fqvCtfk;(_DqaEOG0L$XY`vAMoc+Z-zmO3qCec9+!jSb!!N0b$=z+_}7wiUjRHp|8~WWceb=JF7bYKO>~do2em!8tLH(S>d;a0iC5YENqy--G?wAuz$yJ)6i)af}j7Ql%qi)gn*g3)FLmvJUOY+r=0Qd_u?2%AtnBXFsnZ#KVD8?=#iB90bH)V^Z!OV6U?%=FgNd=|K|J?c0PC}M1jxQjn^R`2+{hh=bu1x0!!d?;YZAoo;uh&LLFn8C!~qno7GhvaWu?3I5Qt@RJZOB6iZr$Qbe!?);_JZtY*exoPI)~5O#uV>tv?Vl#XZv{$sc@iYzHM$BFhDNgQsjuW!xzt#}9$P0L^8r=`Pttqvx(!HaQ1$8bRCL<n^5aP33{M__6peLU-9S2$TZJ;qx3uZz~}o>{FG%=fCUM6Eb!!nupgswBKSt&JRM>JnlcLS;f=m&8b>*PTS9>=0|}(s=lnl~M)UEu64c<pi%c>Dm@A`$fB#$*_s3`7|e0|1wp9{ZtEGbbJaM@YjTu_=xXnB+L?fK!go;k4^y(S$Cho(K!ERk=X<8Dg@zRD6yaO?e)J%g`BScR}RC<S6X;W+Iu~7rNu~E^1ptDF@vJVfBcn5NTHdQ|M)A~?~kb#hvSl#QS}kst3Q)InH&-D2$|PIQ~)aM6XY~$s!RM?vR8ukYMA;eqbcbo2-5{hrCY6g7{qtGwpql#?b9-_uDyYk(adp{_V@7m#`xfYXX1hDzeNEP;@eT*&GqfIEj?eB>L#WpW?97X*=lWRxw=+RqqQ_JSj*D-_KlTVkp_Mkw>k}s6BHQFjDDZj8|CjU21~gw^bWQN2MylH9DSW8Zx%<Dx2h}k8h%9g?gY;LR)=n9DH64x*<Yi^aD+g!qj(s${lWu%FQ37P)|SRB^m5(91k7v5ge{mm7c`GzQ;q$75VsIAx;UjfzkZlnFoJ1-iwZ4K-u#YS66a6G`{w6_9z^aub&>AJ|IRD*+Lnir_()VB|HHo`V=E#bw<$3H6^9x@M2+q+s(yKw*&}5WFdJ|Rob62Xz}T&M%AWrZkhh+BZ{DeG*6@c$2wDQTIY;idVu_B_Mz>oiI$)5a_+-F39UP9)f7MgD7bkbc7p-BU%OXyVqrtsKOU&R4wfF)=j|4xyV<@$#eqAJ_jEsC&bt4}h{o8A3&|8XXZ{1wkUdjn3vmBt*mU)o&<jhdEinEH1=kHU_qYqP?x={aQd|N4>5G!~Y()KAGPEW@4DGd<I>|O&KT|sO<|1e|pgX8gcyrB{C*H$MJG|68}nBNo_RGw+b9)#@vg{RVOF^<WU=QHK_=yzMRErk@f-|g=eNdCH{1g&C)ij|If-A*q9)$VXHvK(4tA@I_DoG3G3EVQCKQRY8uMnpcXkP(^(Oi?ZN=d@tFj5a(6(gd42$z}Nb56bNzD-iJUL76Fi|K|F}+qjQdNH}wDG8&1VZE`M{53U4PgN5K)aD4(m`_?*CC*?jd8^trPaBF>YwYuePR5yRHUE2y^)T<ktHL`<yKdf&p6@6DOPOd;MkPPy03*Pw<#uq;=#+(}Bz+~H>f93WK-+59vYw)px(E*CdNjT?U=(F*s)zo!~K{QUHQG7Uv;A%roN~aO($U^^W`9cFiy2>dl46eL_*&;Taa?T=LlmIx%K}W~caqtGhU&!V7sgz}(wqwtm@IiEX=AhvS#z2_af)&lS)5|7)l5_7HCuOz$tj>b7=JKuSZ#Y*La1ak$5GTZVaccbbG2IqZt|z7@kb`vDUc^%4=4<Vh){jQ(?N!=DlNNvS@M(CJ^!D_Xp337JFMA_Hp6aok#(gjQ`26YQc~XSb3DVp!NK<j|=rA5OaA%~q<#GI_qtTj9W#*+TWR{w?Rdf+rS}EyNa>z|fX7eEGjHABER56h{I+8p(eEZL@=7otQJzp^uN5Q<k7CEr-OA2;WNwk%Q?Mv1>uvGE;^h&?z&0SOKJqGX8!zam%`lW^gJHBT`DSI$5PSi4N^!DSz+_hq5vRXRYQ}p2jg>-Zhq2b<KuddYUH){oAN#@{@Io+*p-n>)YECzlQp)edpXkq&c7$49S_vy@e@@9K-0iq&$h-3qm^87SpHj$Y2Br)x|h)H-P@z*ae$FB)H2>Rw}*bkiZT^G%t<-34B)!gRzhs}*J^uXmaH;sB$<#}|UU@`kh?bs^40pD2%bOh356)*d{j&-@X#Psdlu`@NLa;vx^UUG!QKTu;n0|MV#pT^|FT-7<((~7Kl{cki!M~!awnk#R)_3hPyt<x&HBmFOGy&_lu_n>yIt1zbiEawbitXW|SWd~g-$1K?YUG!F#SC_ZE`Et1|zzlMGolR3OukJk2-wBm6BZoE_6%HHFkl{d(_(%3#M~@AUL*P{?rhrA18$Fe{BsR(4unPy$k!buZa9E1qJ@@j=MJ~~xZD&()Peb-ol82&O%PR<&Mk@BKRl(m4oz>;kcCm^ylE~J#RYhO%A5>G8D{(^eT9D~qU`nRW{*tD7LGw3$=}hpVd=4TEEc7ds+)PV*Rjt=^wIU)mg+KJ^py&@dJKoav%{P{Aut~NK=j;+`eyTUOqU!D2n>C^`7k{5duurL~5}ai{f>Y8Apr~uc=rbkNEZU)Rf9vt6CkeZ)xQj&UQjV0GUG!72KH6`VU{MH6xM+E5r`A=+HX%#L&j`Q*_thUn2!)A^@eM)of5SjRXhHdmOTdSm2@1|#c4PdK(BHr0>{ke3B*i$+?T$8OEuKYOp`K=TH`<VOM2rcJRl;zofv;=Uk@TnVUJ$<|{sU0jaXd`WUxu+BbpI+V(Q1%M&8qwH$_yR0r|%diEr}v&rC>F@C$w9Q?%(_!Vc%tEt%Bxc{eSj!u<!T;Ln@nVPsUkkbB(M4tC{eK)Hj)FX?J+YIUKFWxyT(~z$xu`l2h98SZ^5f^9f$kW=+zvcAEOcNF=~${l<R)K_gnJ-L0X6sM(56b8d(kgH7dSDt}_@o<CF<Nh4lNziNsc=7Q-yak_gj5E&`(Wk{aw9h!PETXA|8I@qV)^3p7uab*;VE~t=C=@%+hhzfXNe^e{ApeJ2~HRtNDX)4Tw6f%oN7Gc+lFgfZe2Q&)wR0~uXHty#jQDDCG#(fSzM`;zWcn`*hs5*8v)8JVNnmgK~^mvW?MQ7NGMS&+Fwl3n;_}fXq7_zlpH?jo+m+r^IPI7>dFP)^>?I&^Jq{OJ=3ca3SKZ^krp~OWw?~4J9vK!9DGp_|h{L2>Rd;K2!RGzS_cG~(_FGBg5%ZX6FV;{o#G1v-g>uW+cn#R0M<5qYgdMgLLA0rI=6kR_t0Z7IzEJ0sf4B|upZJg*h8;Jwaxfd3E3%9wK>PHaXIYXZ!TzL$5Gl8Ncc%+5`tV4iylzL$}SA13d(p!QTyNJ4xSR>i*_ggRycCF-E(Y2DsGnK<5%r(|gQc^Xey=GDy=y~6WD$Xi7k#D&SPT0Agf5IVsTDCz#EJIHp)au$jq#;rtC2>;wf;fXrGLKNf?#qZ#P7vO26Pp^1Fz;zL6P`JVvaO>eZi$$CIjyNnsiPM7f5a>);dd&Df#o}UL^%W8+Z8k4Id&>mnHN!!u1K`Hy|T5u0avv&F!gSAWxH0NX0M8^hBLi0Ervj#wnt_oIFhXFW>F+$J|4bR=YWpl2_r1H`3H|43Bq_QL4IMwMOOf<=;>aBZ>|HXfrp*KG^oBgU>a~yaT^gvwx+97-;6cDW@SMKO38%P5z{)|^86WKCDQrSmx~jis)#Z~v(Z*(s_Kjd&D3Wu_jLU;Dim-I$J5qh$a23s|4hWa8Og;|{xcRh*`S32SXM0s0w7`6K2fTxTqp$LagdLbE5LYIfk4tM%8Lr-_>aF*p@o{vm-U|j()OM4fId<<x@f>LTHs{vB04l%HgkCioyB%0%4sha2Ai9>fcla_2X$UsfK^o2mVa19#hJfo*gK9ddUXrUThh){Bh49AknL>#I5U$aAa8KYda$l~sLxDv7qJ|3M=lhM7AuvDY_rpGtd!oluE<mG#9zXWBb*Z;f3dMD9;!RoxA=X+A^ueNQC5aULQYN<b<aP-EQB6y%mq(m9l@O^pq?>O14hk3uf`Qo25UYsu>|rGmWy163N=vAGg-%6LEf@gsNbm?)=}&%7b{|fEc_|l2w;>B7zvdCok3Ozt`?;@t%tCLOWTMjLY_oXeS3Aay7{&hldz~t-CC}#Eaj9kD~(`)D5NZNz<pR=6i*k7r`AU|*H`M{&IApVP(AGfMaDy?R!QusR5_KSIQChB1d44Pp$Bc?4`L9#;1SSVw|`2(N(}-1lQg=G;c3){vo~rS@8`Th&tQ$6;&`VSl#Im7D#$41T?Q4v72cT+OO<mmmKPnfH8K-B!4*3)22Y%5zoO%$e=akJJGP7Ed+0Ra7@<AqkH_oG(KsE&<*T6{&q4$UY`?fDUN2-{6a7!;<m|S=j++xJxXILG$%iRF`$o8KGH04>B*@w7ra>70Z<?3{K|U%ZVT^=n#2)ab4eW>mS3E&a;L44OC-CwD3rrfH6jvb2U8sqiEt1ogG9+{Hi!@|!&zu(c^&yrNUqJ6%%{cA6>{vQ^{=ToX?6gkX50wdtEyi-Q<r|4z=LM?N+!R}~2}8iXU1+l{{JA!pd1C%~H(O?xjdwu8zpcokF)h7d2++TsEqnjZxPLry7qgZ|iwK%NcIX9n=fcF@xsba%4H7TeG$3}TRD`*|5^1=~(+Lo5O@c7D=%-{TdEj6umEi311IJLv0{@5*K|T_W`%f+7{rKJPW>P!8ipz>b2pM9zjymm*E3xRfiZsUFNcwHOoXwZ#ug=WP&s?!XirlmNGM%w1x=3fzo1xqYsVm7bXI#XHyk_DD?y--%|K&80ZXv^K#XX!CsGmaal4e7xj!XpA%<g%RCWWm?0aH&D8%U)#yVb~j#l`TM@Jn)Q@q97`(6?}@)@QWh*aCrn7>^G6t<))oJca5vt7{RnN_6jO+{b$M<2m#ronN@7_gg&KJqsoelxBLGp^Adv!eAG^t-*>esp0MxCy@%=^#4J)r=u&+V>A#OAypz*jE&y#SOH*B9>e&4Ak4VvB=d-jw*-5|#+T2vEMlDU=`%72e`L8gBf07+PUH#(>XI$I7UnFk1@Y~UZ;2d+djXfoWE)98&N2zuC0m`t80PwLP|!Js^LueV594+`MEq;n1^jL?(?qx5Z0O1@2M8EvMEVn5_8qr8>SI$mREVGMvGSfEHJpn<xY=?&bnvGB(Wu#nNsa_5`2oI~M7iiYhUFIBK4EO4Mz`42+Uh!D#mmE&6l@BnGd(U*o1IG2nxh5AI?9}Bi}>dTrT4Nn$_JEO?@<cStK|?wy&RJ4I9EHJq#VB2Q*VvDoLeuoTNeL)gZwMlAk%FOEYr@A3LOoBRO?-sR@#$^rEOA6`(??cy<_VD^#o5jC5qoVkuWK6bLPso-AmtAYo9K5=oT$5AQqZ5(4OAdCm7Rb5~h;;9<;bVJNOzNNk%F1gnlk?j-+>l76^zD*c#(QhNdyTl9@pvI_jaT0pnTHD6%k?=zf?MQ)asgaeMh|M-<WKPd@|}P2h<^qZcC*Mtc+w)BCrdH(|eF2RSl8K2s5Vq*i%E=+C+t^W%z*Wv%mfeDJ&TU!VW;`EMvKsXO&3g%pZ7`^5}gm5yiyjekvWTAI~`Vi6ybBKlLqoA~gkr!#@o?X_S>*&q~>FO=#uI|GnVN$D=s`Gfx8sM`=`%EQk7F!MKN*u_n>*BOou5=zP5z(4d20d>n3^_>fkMGn11E-}s}6boC9pxT~(iGEhF?Q}o$FSGL@Ha;m=9Sy-@hgAvNmbc!HQ1ucmmGz6JUcc@DC|4*uD_N&M_&#FyE~+X;H5^xoov;aZTp_LK`*THSuB!1NZr9D^$7$Z9G;Q4*wYME%5E@hkOplhSjBG1p?^GM;kH6wwQt=+(zP36oQfPP@NW~Y64%iB39~3kN>VA67PPapI=Jng&K?g%J2!~hSE8ZT(nkAVKjQ+ha?bwdCLZQrkWr5@fI!Imf#xEawFZS>4<~j4>gli&%>T=Wqp2#%n2k<l)*UHp~i=JE*j!x3cc%1tr>+<n21z&thtNS<1^@MYX#v^51gm7B^TBfG{l#<`2sCHxq|B|4k;rB=MU-CRYe_Ya&v-jFsuGWiXWibe4qQ{4v%}n0J-JC7%*w55m{LhY^b`&29rn&1Vp8-Fe<bLXEbSnN4G!o$kp}D2n&a5&k&N0k$2O+r&Mt4$|RFiI_+R}E2e<(=QUYl__$MnYusM+tu4);>o#*SvLRta(_4IVIVS*Vm9hRm9&Z-G@_C9jjnRDS`fwBL<~eNScuD!KZQ){ov`)Bp>}b=XL-gmmQzc&=5a=x5NvFoV!v?v1Wp^_|{gI;)EG(e_612~3r&@LOO5njejx1uV@NVd)xEXVS6CsR22s`x_*@RlM~jKR`A7`ce^Hj7I1{!M`K4CSVl$-Sx`|rJd%k!uK-aQC2bMr{0rPk#Mmr5-v6R0`YU<E(DKL(J{~s0(D2#YWlgTob6&BAN`-Y-+(aT0Xut2$V3El^mimzKTE-q9*ckpkvK)mT1R>fbfbBk=AeH>ax0#$lzS8!$F_IjKWEsB2mcpwg1FnINTgRPAX_ws_gX;C+{;%wEqvIz+Kykl-e`)4GAs~YJvie>eDi?^J7RF(^%l<mzT{O~#|^mfw>%^SZxfGMSfnFz2J!yo^IRW57I8D^P#JG-MpAf^Id#sTq!VrC!v31?DbIgF>dMD5(nR?lp(|DmWs<KsdIl2}sI&)PV$dHMT`&+(Bz(zaeKH-)F~vr{BP1tKfmh#%cz<=G;S?~K?(+}4C3f=C^A;%AEJ{rHDeA6NJfk=}zqT@be<5AvpV7Q?HJ|ACc%_L1g6EQ;1GWq^gMmvnyzL!zyE@@~Z?4rYzgGTUKH)GLPlA-Qn?p(-$eM5~y?ebhSDHhj8AntVj|?5~l0A!)z=O*p7hhi@H_&7N+z#YA#Q#YQ?>CD+fw{-WoYHUxjZv+Vx7LRnNIM>a*yttAgMPo8f=>p(gAy(tNNEV}EU7lclbS88nazz*O(RiuXx-B_&ldovp<-<g1{WT?l}MBSOp4i5?xD&(@YLKEDy>_-C9%4MfYCMJk~eGhy2#F_`{iPR)cc>D*z?@)%%W)qtFa&4uB}JF_9vE+nc2q{gwy0kb)~v?b8@+j>hk8B%k|pWaxzXOqWoNISaA;M8wRm8TEhsB=R(F0S%g9qk}STy%rHJCk%0gF6GYE^Y$h;voAa#<1Uvunop<iP^G-6e>&`=Pfm{W}al__Yc#=DEm}HJm1KFtQng(OM0S3sLqYYrIV%!plU8wf+$M&Y%-!G7PEf9GE`L+=B$cPt~gW(b84a64k_6mR5-F9t@^zsaQ8raa$5bn?1j8GWDPi{|LrbWXJqoQ=pB0eI_lH9*yK-1D3MMJux<9`F66uzt;si6M^cfGh%xl%6gp5+K8bh)4NX8DgR@{en9o_D)5<j$d(Dyi8#;AxlG7}`Bm?CcVSCJtHY1Pm9C)ws1En@o2LuG+{D8$E=migB)xS4s(e26N_i=<ZFvr2z$OCa1m8K|Ja-p&annSYF%VA+mXpMTeaet(gedx8Dk-+~{^{hYSCHrxz(LSt~{aUZGG@NwB2Dh;rLk$p`I=ooLmm>oQ4uOvy1MEJr2&rTeb@JDb3iZ!zAho>@eWgo7e?FEohU#>7U2^|xyhVUkl!^{(|L!@P<lez`3>l0Fis3m4bI3ta=cth(q{hY=2-w+X^d=P<6}x)gla^GMkhPt~n90FojSJZ%|28-|e%tN8tJ-iETrmm)$`c3)oAwQlAos&!O{P+PH}7ZLQSHxpW&a%!&Z15@@b)k|r;QxwT9Tly<cp9d}>Jga1-6l_>_$30US;6tI+qcmG?c8gC|V~_zECgkuCJ|b=v7VJH*qHxJODe6yA{^-zmIf+Pj+sQR>Nk8>WvqoJUpVml=?UeK8LW`5{c6{LEOa)?LV7FV;G4AUG7A&T?q#O0xCgv!W{e%ng?HV)TdhI5LD@AMTThSUtE>%}JAy=+v7L!FJPamGHXzx@kuk64iTWQV|IwoX7(|%yp5lmn!4iX+TIzuz4O+oyjf-8uCRUQz|-@71Cr3<iAnVH)SwBK`;-J(0CIDv*yHO22Rf*96ZygXMy)ljE*<nSDgPNq9o;vb9J0yWCZJC0K_6LV`7sepXc=?|UYM$zf?ec@rgbZ44JL)7THyR+9v;6fRy9i5`OQZ+bHA!%$ET76wcggIccsiX@P$nNNG#6aDVLR^m;6%#81TAgUsISZUpj`NfqiN`qTVs2oP(5Lp>5ig_m&8%ovF`yH{ta^h|uhE;LO1T)BN>9^DPp7GLwOYe)gQSy;$PtBy=~oE+q5z*Z_7ZNOB-)(eUo@y9EO?`*c?WY>rf3)VKhKNRqE@*uWzkt72ka~@-pTS4W^WH9{(Dhl6wR0CuNagQqW{g6yzje>&SAuC9Bq56tH7ugHV*RedbGZ_@^<v*^46X8?X8IJ?`yZCAJ#V4qvh3&?JbT)wfB#GkIpby+z|Ic%)+q`8bbnjGgZmyzM1IfD6e%sNN7==@#9dZ3_nd@Pefu|scqH#@yDybwr-wePK-C#_d<Ow0^#tkqDzU$H@rr3*iYb}Zr5uZH#%J$x<{<t)rh~=_kffS;r<0ws#H(jA*?)Uo8SP!k7xSr8KtB0Xk9ZtXH!p8{f*^~4H!C1!`AX@Evj#CAW5*kTd-vqYGVX*0CxaybiXe~Fz%{3U-9CTgT_&Uj@Dg-(nbfEQL0*-fw_pgFjW|Sj+b4pf#bs~1l%k8!~I6D^DdnOmsK3{WxA)5VcxiJ);qm@m{+I;u8F7KUhGLY4!Ffa<M$8Z9&`s~`OZjYiiedu?DYCDM{ofpY1zXJl%!8&dAk>jr8_|Uf9vWJm*>%w$4WCb>&tg(gnt@s?!dQIb&U=O7ptq&6ZhnJLba1|(Z8$WeQ7a3K=%mJJmKl9-og}{^&hODhj6<~qDykrY{qdbaZL+h0PBH^P-}4tJt*f~PeX?PR<nSR>4W`oo_{P74UcibG=nkom7aO<wS0r>x$kcpvb`KKUS`M=P5ZQ9OhuqCM$siM(w&2aIJ*UR!jla8D+SUR?d*<EXp*a;&5jX^Vy0;kvJe3wxZBGH?-*n#m3sR2nSv}Z)Mnny6ZDv4Y|^=-5E{#9lc9;l9BgIpfn@Hk2`QfCQ;&A8I6rlf53#*aDTwpFEt%j8u6adnb)kyzgDJ^i$CvC<Kpf!&NbL{}f%LR^)c|Ik6o$SlIn#S&l=w>8#?Wx}o+|f^p<ljmw<L!L*{=^=aBSa178FD?fj)gsPI;=cFVu|`{RhQIR9ug3ADbjRQEc!%m9OTb=kGuN;5{306-zuL`yUm3h<3#H#1PPH5uQGxk{`b(jDtTVhbSn7OfQW^d;ZZ2jSa~p>Lk|0;#0?(KR*BJJ^qJxCo<a?Xfufq(Z!&d9&pfZE=fCR(|gsMT->4=_~x&2Kgz7|xE9PQn|h)_Sd3s_a+r_+6~ox0a7wPgIor|oyShPksio-dy7u8(brrbK)N8<|2I28cgd8PD0}gi`*QxB{jhm}ZkE5uBo}3IFc*`#rr}>IkJpE?s4SO`|A2zU){dT*9p4j?)lj;ydsOl^m(%`-CgdT#V4SQq)NxkCw2QVcbuNujZC6^BkNzR%Wi!CZS4)N0P89KM*b$Z-INR#g8(~s*Y-p^a&Jyf^`ghB8aL3(6b*AI58pFu(nc=$mojdRR1?g5g$1218LTSGxJBzCqZX{sFivg6-Wax8&YfFz*Lrze-jYYbzL!zlOS7Tbw%a7d9Tr>P0ey^L(IX28R!NiUV218)zSB<Z%1UJbgkW%5~<8Cyt3Y=}F7HWxIFSf>}`H4uvQ;`_o!!4`{UwfUO}m0;#16<&C&j@d|KxQ?xNu<@TXtC$9ltE@AnQ6Cd46bx;)Q!L_pw*~oSOjBjLB#v$=-6^TN(sIo$T)DQIlgNq>^A4;l>|2ov_r<W9^>O7jlz>Q>FL;yw9L4U`E<SsS<4lReUz!yg+vsl|=AJ3KqYR!ex~qP{ADZC@e(2jBIE^$#zM2-uF;>2sKLsU;u0YPRakoo6eom_tL5P@rNpA@@`lKb&FdJ8~F>N1qgwIoUDCsK(nVqT@)LW~kjT3(Atn{$of<f|ewwrGjGu9&OC2TE-+YR(y91omMBeG%>l+C}p{+@q1jrYh~{oKlAFm;$6cHSor^vKY!uWxVOtVJu;8?}{GWvMY2yd9I#Y5mr%<(tc}U2d-5K`mBZGummR3j(U2FE41NZ6(bPLmf0DWklor4|J{8#|BI?+Fo1!!FDZ@Zmdr8nZp;$lsr;V!&ocLrUqSea)_Om0O52*yBd-Tj%mEKk}4ym$5nM@<!zXg>gtW<+uQ5g^=PZQdAqjd2j#4{uC3#fU@fUWkZ3PpO&?3^fjrr>5)92esudlfCTo4i2dPq)$rug*RAxZC?P34W6Pn^#WMcv;daM%ldId(%MS1#?8VNc^H~alz3tcPX<PwM><Vp&2IO?*NSRrc+LC#_6rRTAvhCVm#Zeu^|uG5P_zy-k$PHvQg;2?tEBM`Bab0ut&6IA3vgALSnvlwki#_S;}eIm6+vf1HyLhfN|kY1@5clHnV`oqNQ9O5{=8&L7qFpgbYY7iz+Cf=Wn-N{}j65y`MfpIXO_}_jmY^7*r2TS5D1-7g1*rvb62wAN>wl$_aA>q15B>vUTSooWpFYU#n`!UE<SIfa`3-ScPqCMmF-1maH?~Ok}pc7(B?!~8~o1{FFb!roqR{_pq-skx40s~C@0tA45kFV+#Y0t-uX@A&h8M@~BMs*ALfAL$9AX{0f%=eW_mawc?1gFpo1IbE%k%tgPHE45gRG>W%xTFyZk}02gJGRx7U1V#r-sRP!z@RX0b9B_wL8d(UEJ#cBd;8g9^5NLdYWyxDHGt9G%-fu*ra{ee4dq5{>&;qi&AVDA46N4{a`C<lh%D!LWnFXMqr$LZfu1yNI+F7kdAv1t1Ggsa-`1GzOWt1kvgcHQaggrsGX$7ZgT6D2aAMgH?Y*;#P!vR7Vu#nc>Uh<h9i6Y}zH2B`yR??{HHAXCJ$j(pfHYF-x(6IgIjm{x6`-S|5Wig<hwgTXKDunj$sH!%&tZwWOBmNye0_eLL(mA$*ias}=qxd8pif#NKi{Eaj>wn~O(tnPIx|^DK%^danxTKtAjn$`7lYHi%{im5x-tkxtxhruqRvT39Vx-49G)7zz-iq{PvRgwV;vp17tR;QOe_C8)sm~V{SmCJowJvek9gA_!Tk&FUb^>TUb@z(2NFq~j8rc=xV&WX>zwRBGv(dlZqBEzxls6u;B;}}OV_SkpDUHiuf2u>NNe6MF{hWv%bp4QG$hV59I6v{Tw$>Z3j8I1D#n$5k3I^n0)lVkW%rE^KmrW+`u%P}Ym1EWFafWJ^QE%*F+T$jz8{uL3+3FrnsURYPFSg=Si#|{pANZ^3fND_fIRvN5|Sj$ik!|MeIeYR>6L!2kEvAYDx);=n^|T>&FM5U21OWf`3ON$hW$<}x_1b*wB`atTkrQ;QBHyRu^p6b9>pW}z8d)zdj-+?iSG~H(@4`7%aXY4<FZ4f;5!&x>sR3VlM7%6WfXh^mLx&YC^@DbzBG+aqcq*)=K5Qhd-QG2@oT9UD6~9US-;6XQ3Q8_gAq<e+iRP(`ufV<+LF^<${FjF(rvzPtN75xXTIw#_tK@%;SM;f3!dx=u{8{-NKoqQ^b%z5+68tPduhCHtK{O@M}tu7iRphoL3NO;f3oI<4?n0mk9wSlk@=O_i@O^d*cNw|>s#xasA@<TG{0&Ue`dF(|8{nJ_u|q0&STzG^<dCB(7n8M8|g{3E=><L>8cb-JEA~$a!E^Tma{Rn+aB&<`0(cXQq2!sn^fC)n>ARb`qt()d6lzEn%%GysFA3fIc?w}H=5P-BecZ=@Adh6hk++pA6$CSGm#wm=x_lJ=&Ja3+($XHWkQHxtP_QQicr(IWok{>a;2$E6l8$ZnUGk3y)zv7INg2Wjrf!zQ-Ztb$;^LloeuafzQ`gH21ONBUU=H2?rnJuJtv_6x(tev40u&Xr5l4_>bI_0H5VQO{L{g&dQ_YMHbAbQa8zO_9ZT|LEa^3f&I}Ie3dWm&J&@;A#6h?0jO(p0g%6z31Hsv1Y7$JXjiJM3h#&{(160w$=xOOYu~yh1TPT|ES)?HYHHP;XAyoCg_@Y7ojfL-BYhS;%ccGyFYbDHX@;<qqNUTtiTj&WxTyYZJ=YJ>sS}-T(6V9dZ#7o203E1HM=O1cO{J-RAd#fsR(?Yz2UvPFw7tEHzI(vfNe}oqBjVg$+S^^M_g6IicS>d9laq)<k`a~qC{90rwqA^lV7EffTBrq==kt0vaGm)|71HumYTB3%1NPcD<g)Jb~xNjd5D9&SkOwg2K!XAFhLHco{)m$il@3of+(l?>k_To0?&1uJQehy8{HG7ZUs#UkPH_?sSKmUJh<RF_)%taNRQ0PX4Q#KxG%_1A$Oe6ij@>>}Xs=$<E^`v4!g#LtKjO;EwQt|hX1H^bBw^fP#K?$-x5^juNJF;6FiF?qb1cK*Am}W4t`~&pBS1a_=_${pWt_T_9vx?MJh%7CRN-Bm*{medkwH`U+Y_kvZ`{t;??qFVZb}!RT?mF^6Wn{8qb`f^;8Y9$Lfe0yI>Nh%fIQ!U8ej~Kr1apEOjgqt-Tc^a+VCELJ`>A3_MX{_0YeBV?hZvVl8mZ>Ro&)R>8%tHKJ6ioB`n0$!QcmVx;WZcoGbN>uFhFGSU8<4UHIjn-G>ItHrcwj=s}~-DEtqHO&JdP@4b#_e*IW%}ceE>(nLQN*irbsjjXQ{meEw@<SwCSgkTin``Z~W)_@2ar9+As4BCVZ&_WZvPe?!jp&|;szB~%)botj}Q(*x^M&R6gi9<&HUf-d_4DV$MBA9IFkH(PxKC+~li(3mE!-=};yen4FFYd!fi!{XozQMZIN(LJq%Hi815b9YU;h22|+X50~R9jR+vh;}Buu^s(KN+Xtkll=U}cUQWri9pjT)rQLEaP?Eex!5{b>ol7s7bsGvI;AafHxp>EkPkSZNcH)QA!<@8`@#vE#2M#gR`D;@Df|R6ZoZWA8MUOq9}%LWgW3<gl3)bZ{YLkmaT3tfuB@-Wv8^$K{P|9i7XIoYi(>N##k}B@d<)Ta7MGUmTLdPP7v(+9hCXcE*MVWTfR5_u8xtrNgI4y`5I#b5I8+hi)e3cdfj=5HdT34`k)SzNMTSvu<FKYZGanm0^I09X*6-Q+0O<RUL&T7WD`r&J#_;W(tE%lB>A}%~QU6cTyj%uS5a41c1Y61`pRQ8TvFK2PSxnaQ{c`bD=L<=~mX)DJ1J&>%b5<mnFke%`Fl4$F792B?LL&s~<stA9xOtEgGv^~laIjHaLv5??>{udXb|4N>RR5%6%i0bdQKr&L?zGyv;nBC<TbCR@U6;qMx7}~W##$nPzK`+q`P7t?Kwf;D?cj*28{4J4I(Mc*rg2S*i^iPv2<wnD9r6#S!{%hT9A_2`L+Jabta^bm!;&Hc1*K0pC@7<bN%0p!Jk*sPAPX*{{P+JF$_x~|oXQMKl$<etI<%QJ`Umx_jl_#jDieqJ&#taJwFO_(n2j7b70<s+tQ+L9QpDsE1^%1W%{4dPs-P=#?fpP5<Tzy`h|Ryo*`(gmyotPC0;B)q6!VyL7vpki3n}=SR65DvF@aO5MZU6>|BM!zgk?_fLuw@q64M$@f&Uc<sU|+Nc5x>c{B<|XLx0V|Lj5QG%Amw}$HLK2Kchq~pQ`R!xs4HB+Rk0hH<>iE5#l2<lxV7dGcP*)(2gKqDlee_?fPb@r+X+UvtsB&3Eu*)Z=hR%`ZAmRz|6Kw1_1b=QQC)^eZj6_l6##tY<xf1*(KAt9dt3xcR3CoS$R<=o;4ii8jdUYg%Zl*|B@ef!_+fQODr6i`5*~7An?pV>c1%fX_8JrcRou$C72i8*$V!&-cD1qinN9Srwn`ihc4H$@n5jt8AVDCc&ovozyhTe`5SmXqpPQZogBle{Y0im>_EPu_)&P{ipm8Jp}r+>ecXRfV5|MypU`QQ#j(1w0;}yVd~Bo`1;R+PJmzP3Ja%PTY2+tZ5#NkWIB{&Jn@A1iaJ;cz;d5j(AJ8m)D3urAr2N9x(?7Mi(lbsMg`D;o=|EU1@eAu_i-7rb0zZbTHRSt8obR0tE2K?+-vDvd^u<`<Zyo0N<M4RrxTNnfBRI<r*wu#fp2-^DlY&UIk;E*-Yk|ChQj<({JUcE{%xkXHARJrYOeGIY&R8+0Ya!5HyuP}8Q$|(Xu5Q(qa#F9SDJu*@yX@!444QtS*&hxO%|ro)C_qfk8lsEG*rjN*`lhDHH|NKplg7C;3yp_@(~#StiT+`y8KqoA_Xm)jxkz3JI&Qy_>Nl4}@;IZ(S#;)eP#Ja@xSB-I=RfyCZ>=irXKYe?LUoB~cxqeVZ*zts(Q3(>WD;O<6)A!|<qX(L?l_choCF>_O#pF8JL<k&B|!|8^KIyYH$u88Wp786U7-Vq`Owcj1#@mt*^b$HFsx&CQlbhmq3KubP#oJ)R~jd~?7!z<jx8_?y+9@zrU4NzmBvpLF`L<$&L49&ZaPK(z$E>pK&0oZ5m|?y=rvI`z^{2<d`YI}N5HcmrUqh*UH8y*+USP%8ae`3teLU%%ydT`j&nMNIg`2@QN5|u&MwVviBx_d!5$Wb#w07crgD!@&aV!pZ1h6^QM!K+4}o{?=(X(1K+L&f3E0$JG3bw=SjJ&@7~^(|TQDy4s50v9;dk!iN9=9+b@5v|MBom(yp|f6a&XmB4fpwfHFFdpNR832Gl|y!h^SDKU<SQNKc{7~B=|Q2Uz6V@byTU~EOkV|Zh?Uw(z%sJxoZKJE_}qSl|l}REf4b!%L6e~-Oo-OX%yACytcG_v%0muIgKeIcF_aH29Ko&I7o4jh5=1%6Vr~yEAFzQ(aDDD1QMC$c|_Sbl+kMX<;r#wuXk~Y&S3}9j`{;_hg)nR$6{5&C<ET?NuiN6VXfdal4@C6tSdzq3b2^y9i7W!$N`x(peHT|b?w}ZcP`A>0>{CTxd!a!YiSzYh~=GAA=yw*rj~8~kv8J*$>4-CHn}Wu-i=RZHCxl?3UQ1?=BH?Q68WxlZ4(+;J;LHaxd`Qyr^1a0{fSB@IDP7!OR$F#VXhY>eqbPUFtBmEtt1dJ(Qx?>gac00ZS03=Re^JT*m)O@d@`}vF_vAj)d+{2Xalar%k177tyDR>xQ;}qRf?{RSQX2g+6%Mh1FHDW`sVTv*VnKL(GBx+@i0U5DmezzeVQ2{7w5Pis&8`kdzRXy>ZC6=Iiefl4*xA>qstVxTbyva>}Ns*vVH37jK^5g%R5eIU!5n(?*4<!@G*uJJ=Fd4GW;nsw4+hjcg`ccQ4#oJ#PBE=3qG5-N>w7X+gc@BwKE5fFFHMxNrVgY(Wu{U8(m22b9}4qCigcp?e65yqIvP5^?YW|F@I3ZHoc_Vv?7rY?4uNG4*NS3X+hia4jYfdR*{Wj$xKS1_m7clnd!PZIsd<@m#_xee$MA~R4xSHldvYd{Mg;j5QjU{+?Xp*ww?JY8bWe<(WZ8=Ic<#;C<#7i+ENDH{GBY>w}PJ4cZb#6%g!Almcm(qwxKWMX}|~;&O^qv_xNmpx?IdaNCC&49(q1QArKd9Z|(R^5VkR(3PbN5pNaC({l-v$^7)&!t<ClN2IA7+tuel_7VdF(d2OqTz{eQeg8{|M7_sB(5b$xKmh`vRs`dKv?X^rvA<q?XcHDFSI?Y@Ts{{Vu?G{{@7xP5;1Wp8yC{QMGx=%kNHO;4dW<1mbfk?{mi_XM6&@q>sw7wLfL&r+!mRRy#Qorq7ENRkBtAvb^vF<|CSy(d-ni?ZqPB{ZkDP6uafpgR+Mi(;r3{@=KvwYP?+9JY<^oN9xv$5`H6d=DLOTh1CLc;SWvnlqbRKB4_>=OZls_vh^7e0NK<Gjc*PM*_0LvsQCB<Z$frwt_!JV^|?cgZB8*v)es5ycMSUXY)RLln|xlUzkyWR_+w&C9<_?ke!<v;%S4@AKSXw6f!eq?hF3QDKX+ryC9I@WYT7CofyVSOLO5emlLxkY?yxsZB8&wJU@t3G|wC$7Tc?n|N}LSnUv=vP~FSUXxBHa?y5$enL7G6f1chktL>dVsUwu?M`F#%>0j&^-jY;FFJPMO#h8@TWZ!WG7jVfhTy$h##>0pW*rV%cX43kXR<Ej1^xV%H8P^+_7R@a-XX{`u(e9gWlB{GuCrMf4(NSp&^hjp1R}shb;xcYH;k<pnQFyY#S><9C}Vj5{*Uu@-J?+8d!*kjiH_ii@5y^}VN(BH3L&hs)ftME2viGGkx4#&9#BF%j3>CXwfzUGfiqif7=Hnpej{AE^x&S=5^9>F2c0vY#@Gq&nXnL)3h3-C$AX@>zd0**&XBSfmy7M>i{3p_$wG+sQkIcWYzV}$2Lhyxg3^UkE^dV?K}oAk+FJehppQZ62)j$!OHg@Bs=2`Dp^*y5DnzyitrBGe6?wHedA6C5&K!r8GR4$vsHM;zv$9hRNa36!RKc`uWK|KW9qZLpZKWztjK6C_Axx;En{;LMQ>#6H2TT!p<d+r9M*KfOPW;XJ-+RL5$}sK1H;{rM&Bzbw#)YrZQG&A<f5V|4X|M)=jS}Y1Fl{X>ScQ%Eu(LmuXNbsfqVe7J?X6oY>u;t)*JSbkBJ$-Fe#)u=Bh_UDjf}=+f>Vz#CVpl{Zp-zH>bbrvDXq)co|4~u#2Jlc!!Ejl!fhlRSs73iHyBJepZQcL>KS`~l!Ke<!Z>!k7nO8&GDDeAf7X}47@&Qk885Btq5dlWjySeHH<7p)__s$YDGL%bWAGup2h%Ylgac*bnBcS_u~0Jx4mgQ<B7%K9EM&RVWG1G`C2kz%kmMA=k`mW;H!v1fy~y{DI?NM%4z(raKXS_%i!1HB`B?LG<I8c@gI=jc5c-XDQ4ZbT5Xh^sbTU3!Jq!+Ak+x3ZT7Bc1nBjhiHsku%=G!}Jeu7D>GiXqvHKGA7B)7OyQoBbs_Wr614=yn;oso1#S;!mi^JUW6e?um@kh<8lFNgc)o>U7mwM~@NJ3k?R(7%<}EOJn1yQfGy&;AQ!-?W^vpMSI{WA#jDu_sXc=j>PW6^9YPr$5on<I<l-i;I6kN$($@Kh^!zH!(P16*IBlT3)G5;vJ$my8=A$Q-E-Btr4aLkj7ZVP#;+II2eTD&5kTm;uQ@MDBCZ-s-7B6jMvjct%E^E#+llSYB1QbN2DVvLe=J5SL-`aN>3M^xvu}#<>RE>h(DKzAQ^WDT+fFZwXm1qGx|ykXPA{H@`|VX7UH#@9r%S-5(7cm2l4kzL7K0udi7Aa)ILkSopar=Ca{qVrP%op#<*n-2IKv*O6Yl2C`xveI=+n6-`z7Etuh`lq89dy2}HxY8L!*!Q44Ns2<w6SV=ySr;J^KQ#Vm)Gr7Ppzr>?pqf#psmKPm@?wiX*K8#=R+xTS8|S+6N%MFy*o_YmS-q~tc)*ZJc`5f872=m=L1x992a*~9xQatjn8=gyHRhbw|ZX&fHLv*eCFi>990W*2><5Rw~1AmPpdCsM=dY{d*gs6OxE!66Aq_u!l+A1&<M|IY3z%ve7)w&D|ZWrx51z?=7xazZzlvU6uk1b}C-@i;_`+Q3X-Kf}o<{hIQYfmgb3+XuX^4B&(3#AB-X7i7<RO6Li#HrXy%H~2Z@qX0*EYG9<`fA)t1eYfqr&`zE*deyx7!QIX_Kco^E-u!g_;=1r@CPNrSt`iu%)5U!_Y8Lqs+epddJlf}!S-zb-04<-()s?NcQ-?dUb#AP0Y_G7Dmh%fN+Q%xi;gU53d82~4-N8Jr)^`&E<bHYnkLSOenM3UW?I;=j^@yn959z`3Sm_!BZ%2ChDFrlsrIExhR4iTA-~5S3numv?>POE%5$-WxILYh$$&Bza{{(dx6(Nj#L*PBal}1L0P=ZRSxz;H1?#8;}dP}wX&CTTvoa0F{%JHC|3hcJv@os3m&QXYrCwR+LYBexchIn_p2q8xBu1kE+n~is0Zo39%Xr1%c@5ikp0bSVU`1SOG_{>kQNCNvnKcc5(2^v!nr?1Wy$h|t?VD?C%!P?wI1-rYdM~hCU2ZNQ+rk$q9M^x~&;Da$NX3R>!T@th8BtTuQ4rc0&6fj{HViQ&xtHDHKs;-Ud)*b2dSINVZMDmqvgOhm;I4M|8uP&HjB#+86oz-5tf{vj)dRGVz#qj_bx61*x4nc#sNjCZxpK)yN6KR%u=*7pZ>73c%0Kr9X1MVY@#N>WYkR9p~W`iMo7~gki3CRauYa{ZSaBIbbIUhq!cKrNF1(Q~jySmgUBSc}qjFt4o;;pIl8i!&y7!r2eEp~-bs{9HAd8%dtA6nMDR1jc05M_+U0;#MFvYX!2B?Lx3q(x7LMWqh!2{)kG_Oeo)w5EELnVdgRHd{AZqXC-NXpiEd6EjaobyUmt!nlreSK5u&8uM2#hU<9#_Xxy97xAw+a9-Eq3;$;U(DY2YLK_y_G(Ge0>7M@V{FyUI+ci_?|BFg}=#g_PN&64!YAH635jvkig0e%%6)Gjx#3A7P1xf}i;6qmtiSK7pik|=ih*oC>)`wBDm3-vF$BaOkZw|@59)EKF59hx*{~heIr!apMRRl$z8M2SDd*SEr&VPOW&*G~*009POP<QQgH4r?2R?VxGug=!W3)FRfB6|Ob`u%eus{xJtS}d*@h+1pK3CZg$&D(7RK}lgBCmQ+ZBYm35ci9sxvBWM*ACf2YrzW@}EjYCMzKr9a6p)Cop8vI2SaJ}s(uIQ%_@A^GNf0#tlUPN*E^@4xq7?$sFz$g&ZjL()G)wpeG@I1p&;Jw<pI2XN>|JkOe;J6+?|OJXF3fuj0}pI2Nh*jAPMK1@OCl5fCH5OkyvB&2!&P2Tyz--X(v*R(Mp~&_qj})<+aA=9Jp*Ai@eVL)XBQ*oPpLIT)%6AkrwP?jzf<)tBm19`M}^loYGH`&9{eulu~~slN<d#CC#GLqv`@#Io9p$uhzlk!O11Tf&I7q20<?<4gADlS$J*A*Kl4PqE}SWwa1q;z@J>`Gq_6V4x?90dluoKFoXAHA(M*@Nn|EqA-@qrgs;kQ@Z%5EG=olAaIA$7*k&fiX(&~CtTiV77-+J4+a4ec{^?dJnHxv3oG-ubgRKA+`ihXO}j#l9!dUJUrTY&22UXXA`c`R*F6U4P8nw@IBUfVJUYR1t-$&b4-?<Izw3ujhN(TTVN$Hj4ZI`gWPdw{A3G2mW-0+IsCP!hn-g2OvL41{<V&MlP7Am$&2{`~D58DrCTJ%NisUwelQ!vDsYd@q7)`NsBE6fpq!LB9)oZc!v-J4#~tC8c>Aki=-|KL#e42mZIXh+gEI>+4&yzSzS4tdALR7v@pG$D}>}Y(b9VvR@LY6%J?g5*t%Whxb|?1aTM={uV(3#3vmzIoxv@cMuPRMw`YwtD*XVv1kMPzA`!-ke4)qkVMH*yWKe{P#4iNB<<=Vlq((ex}Dy=f=mJA?u<e+Y3-FHo(XcKJw!V2E6hwP{uqc89nqrgN+tUMexA8A$r{PS@XOHw^1mdUgE3@Y4p9FK*Gag0Vwz)i`-DXd+qgsgsp^M|Q^fv6(8JKbY)tpkjc99iqd2Leo9i2Idj<b8e0mvXW^3KE>VerWUiQ7ht@X{->Xs+iUEm>hJP5N0iSGTdzO__z^%-YBv^p;6+?l0o1=9gQ4{$>x%NP5uSY&Jw?28jx1W7~^AZEH<9JIs1k4r+}xYg<MWAh^?2bY@fZ1L26KEPoG)W3E~9>}HM(IKS_%CWdrOJ405*@C>H%p!*r>EkNIC}7LYkj{e8b7qy)GRtO7b+7UYnAE!0_lQ<Jd8hZ1qr;?V9L)<eYfHCQX0Ovf9;|@Zpwm+tF=B7jI6>?BVY0~X+_S2Y|L;HllhS#3j97@Lyzu22>HLWu<r=1=oC&cM?<0a*=$GgHq6HvH8Ozz{%lHNsA6No@QmJ%7G8p>BV9w^bjZVwNkAbb$W#rbzP8LRKcapr&_Tl{Yb{=%jX6NR1LCAsz;O5&2=H_xm0rrt@g*{A?R%nv9LX&+WP02H>;-a}|vj1B*DFzMz&C_AXzH{U{C;sB{Dd?u9E06oG&X{R3Sxpt04|Pq`6nzs~Z5R|{TH&9k!yJ^VB<{wn-3omIbh7x2zr98}Y|NPut<>(;R(#|;S^_{{DbeG;$i71r!2y369$m(C+<I}decD(NVHSk%yn;#joS;lpqpo(;t1Jf7AnO#NAJ9`AX=qF+S*9(xFk>Zcs3kgC!?u-*@|{;6oDR>-!KT_tXA-Li>Fl53XG%q>@+bweYFJCVZo_FgQ@6y?s#I<)-(FtZGU^w)(wvV_$FjVpjSKr{HcseXy-4P#$>4Kkj}aw*$fS?ypqA_2+WMBa1`G#vT#6vh>{=B7)?6?jT**LS@#-plC@+&G2DwK_d(|8u>^gf2?V~gY;Z2IA0UkS{LzER_tlxqwi8xe?rFRjbqcC4oLhWmYOq*>-yU?U|Va0GO4NR_vJpfTg-pUYZi5^J@Dt7<Xz={;%vHMQqS4bO}1K=c#P;!m^%AiDma;Y(zR;<RC34sYkh{bE`5yQlB?(mEm_T|B;V0v_`xM*BVSahqpyt2Ki17yQ|0bg?<Pca}Bt`a=msmxs|PvPjOTcm;Z8XZLVXmKdI%#KY|u;}5%fUe%0tQurSxYz?NFdUH6*Xhr+^aOA)olmE=;>J8%IoMLe1k%5L60?M3<s8z+3gV_FQRE18H%eX=CU0)etCi<x;hOJJYe|I#DR1x88yxKsDge?Ir(Hvok`tI24UmPA5o1)Mbb132(L{ozm{40{oODXn{etq=)r-qpS1Xko%4u0?E0wKsU@8gQa%a~8+mM5Ii8ba(49r&3!%sm$(o_NqMZfXeHFfxDNgOT2$V}#A3zu@@k(S#2fU;lU2(r4OokQbJp3J%PvgZIA=<?5jg$d_?rO9$K1T)hZ(KF4NgDp(&!<}Iu`ibG^p9ve)di}*2{{Oz=>X|7u<`y~53>VYRJ*i+sGjkU^t4zQ|Ctl(Ok0_vFV1qrF6paVcih4=2(aYiB<QdWJMhOQ`!Mvir*y7;UicQ7EZ<Z*H8`MpSF%yZFCHXG#?Jk^@2Mv&87rj=evES<h>%>E(+ZZIIb?O7#hTHdHr?*ckMU7Wuf<@{@66SE|D?V)8F1mPWrp!n)U}|0GnKN~*tlJlEKwH(Vx?z7z&@>5`WK=q@0$Z$Y>a&8ZIep{0PYS;eZkU~GRi$>CintweIdWMPf6B(tTTy*k+%2W<kG#J*<gCd1#&;y(rt>bHPM7-bXec`1YTO{^yqHEhK9Zw7I-8D$X@G&=#~j!G9{F0r$8roMpov1UxOg1G)X<4Z)x+woRdWIeP20c}OozSJsa_?DVUoo-0qSDc!%8oIld5lcY5dA@KuI-4Gy!Mt?S<hnNfFR<_Hu@widM!{K0+7|%63`t33+2_X_@dj-dx_gBZw|%$J0X7B1XrF?47ya&ABR0>-)aMf?FGQqcV<4!&L>vHs;v^!dkf#!4rXPy^pM6h>}hN#^)7Siw|q<QP2(XC}0GnX`7K8tN4QQ+rrfHoAC|YA9lYr`HeO~xZ?}IE!)Ft_wvq<Y4iM;^-MfJbkH2NP9vNo9>kBO#Aj_j3RH&jr62V>y@HPloqMRx@zc;{>VlogR#%NYz)H@7jOZq&C{GcA|68psEmzkHO1>o0M}N!K5iE>`G+y*<7cW_8x{3$!5HoKhiFW&^AmBTymhU1_c&dsUA~ZE=bQ{Cds0|bx9lGnk59M4{j+J7SRLZkFwP4x`dTx0%@YT$Ti=z@ul{`9Mo*<AKR?|O;eCBg(5r5JP#80tnfTu~4W$Ls<&q(KZc?h1o+_-OH7x&xn>3n(q>df5y%oP;X(z|lM9o_E_??v}|{rkYu$&l)j{!$JqXBf!gu+u$FC%82!hco_<e5KeXvXfiYZa8>B>H4B;r)KtBl7%UpVZk>lrXE=eg>OL?Hs#;uLEOA2lmNP41);Szat_0|d^L*83lXYt<QljnU}ILNl7DL@?@sEGmnL`yfh8WIBV$GAjefA=A*o}AYwzlnxqIZlksxc{u+7!PwByr&8)vlw7ThWLOJxipDPOBpmQc-r4hFEp_u^r>T&aju+L3BI$5qz)U%@9|gU#A)Q8oUQ|7g|gnY_vD0aJdOhC^SZE%bi4Kx6FlsJFFio3TPkBpBgQaOq|A*(?SyzeUH{KH*%qzBqu3OfTFa3!rzxhqq|De{XcyH-X2|gIOPyc$OwyPmM^nt9QoZQp+kHvHCIayYK}F3|K?A-`kge!y%X)B`Q>0YGODO<3XdhAM1~CPpZe6X7}J^Z{Z2aB_a@JBN<rBBFagQh=Qgtnl2Y=8}2<lL0==_vH^Py;6YYI_mKF6Be>u`NjHfu9MR&m6Y5!^u@(C67CS^agN%Uy!jSe;MR(JozBs^Zy6O|PSt^F>Pwt0FT**57<doLMcizh&$;GFa{YW}E2lVB1qPp6nMEa9=$V+X0niovQV7TwfucAyk9hi&aqNi%gyf-_qb9JXtuT`QQXWI1eNs#J2TcydGPjgLHm)8p7Q{h8knAZ4Npcm8U$t<kD$!lh>l;IHhmAG`TF00wZN_**oQpAK=Wmd3#U>KgwH7=eNy_S_9b#1M->HUZG<+X9u7?gmu?AhC6eJyvXLoraD`Rebjs?d5phAKk!ZZ8ArtCurt&DA_-g?pEy)rI}5T<%pU=)d!d!(B@ArOMWdatXl_r618IV^1MuyhkAa4L#!i74OPiaP{hfw^W|<vg&-UYbSfsB@wV6yeWjPBK~eB9nbU$K~OoB8C7maUaG00d$f{U%jm+|Th*Ie(fS)(t3tyfGH|bMR5t~<Dqe1GR#!K+>(*V+bY0~Z@*;+0yCe=rX$IebKSN|1A%{3X)y72+O>+&oSl(kA&YRVhH?lW_iI&`}E=#U{E$H+x7H(v)#~EcY)Iw%xNwgdeNyff0#pIfvy3?x7Zk4>6rVH!z;3S739yKZEX9>Nulwt@y$n~-O@mDyQKmN*ehYFA%h-E#FJq~N@#fg5GfGUN8V4Cw^?}IwiW24tDtvWVB;&a-X%XdtljXF8hbQlT0;axg79bl%D1b!)Cj!OYqIU6t?N25Ob+BX}AgQG;q4e?MLDXbPe=!Jek1g!joK)OCK5xL&O=oxtYvz_zD@aFfs?{WG95!&;I_b{&F`Cm(nj`OGQalDCuQ2c-+Y%rHxZWv4HQZ{N<_on)BMtI2+T`pj;)rr`$-Xf{wSZ|4nUc%UGeM)J*+@EIh6=oETAcZvUM;R{`l#QlHVn}{Yhn-ZlXjW2vYx+tWCpnc^O2bL@n5s^_rg6GLz1*1OhZ7d97<i_5C3Z*AWOpbCG4biHUHFtJF2~MpdUpj^gM|t5j;4SAj?lkXhK)`Vdv|f>YQtfFSn$>DC{l@qDDt6xv${rSrY*a)Q_H<eO`0+z)R*qH4*OB6c~{)kPlFT2=}W)s-9YIFu9R@Ggqfs7LCH~f<n?>q(~`H0sY><_W||GUmy?jsfeAlY!I%X8O6aYC0nY%Tvr;~Rcf({<`d2L?Qp_}&=Um?CDix^HDV?b}-QM9GQJB72Lw(cv?5y4<Uv;_?;(=A%Det=B!l-YrQ=%2v=NP2ARIQWuT8^F39a@7&_n7j|pde#1{PE8DdD^jyP7l8|nalUXGBmHxn>&4|BPx9IAjSU3&1X4TPkdb$1iAi=&D!1N+MB$u$C+1C=o?z+Ga=?^s@7C$B8@7_t}siR)m0D{2ELJx5;o%u?pLo?o~_MgWWU@wP#_{&_OD%`3ewot(zZzGU}A_k#r#yxv^bNFOP1TAKb`QNT`<`RF*dPq^*5F`&|wtDC<S_#h(4u+zW@@xspm(MqXp62Tu_Ke7%pg>;67$TUrcnra|A-{4DM%az{7his&_CJly4CMI=~3!Bb*S!{-H2}1F5C%h^Ug(W%IPhsSd@w7Q8$$2xtMieR=G!Bwz&$e@%+oU~ONr$nP`WK)&$PTy%p9Shf>ixCV&Yj9V?d<%2NY#3+gKjZMfoIj!VQLQ&JM+m|bC7K`IC2+Y`61Ci%{$^;RFc=ah`FMUX{@*h%4v@}_~3V0R(e~%eeD$OgO0s=hdEE3Ns-1;fwyU|_bix&ks_}={W#$Mc<zaYrLCnb#MH2{X>g+^d^q)75K+=cnsh4~3E2ADdIFzLUbi13GLG`ea=hKNgaF)<N{vGsE!izp*6KUEV;*`gT|kKsG878Q{Ur|Arb$5TwQmV=bBwiq!n4ZeRyb#Mst*XQ4S%R%VnxyIGzwd>z|xgd04&E*B}?p5z!J~v$K(DE@O0^mI8NLTp4savvFpv5v{+qPhn#pr<p`@WIDHbE0qf+1=cyUwh_1AZHK+u|NBFX6=2y74|H${C`wJgFZfd`BrCc*P1Wzk9Xf5s2F78iWBsY2e-3K--X1VkV>gLfc4Ri%jjD_n?lDA>ML+`1#$dk?FljvFGX2+UDl^rWsI;M!u*MM3&zXhhArWh0tj*GDGVkGYC0q6c<@<(uHLzSOJ&Opjb+pc&mvTa6!6H-nof&=tZN>VJu?R3^Sx*?lVxMgfBvxoj{rGqTD#ONRWuDrM#TR8t}L25QNCH{lQU}wM;<@?<Dcgr|S~hfYe?5omU>vREX#&sVAY3H+mrg#PBLxsfgE>L>3*}HHuM-O2y@#>lCkyYPo=2tLbGGot}BAD`dXAl`EKs3qOdC&or$Gk|ErBImt!lllt~5stRg&%UJX9)7Q)17XCfw)mGqn*{iKBY1I+j;&D8Eec{;p-i?R->FXke18N|~^$=Y+P~(ACbsl`CX4DzcGzq0p=kVw-GO05AP4zA9o5smB-&F73PQN+o-H$u_2k6xX#L+{CqY1uq^{u6)+FgVgOlxU%k^#tI6whu~Fgb9-)cF7hIn#X4<}z+)kqiIe_^ND{*Ug`8%iGU`s8cD=x6Tx}z+&{`&<7FX7evzl7(`Pe7)**yhO=`A25|<5;jWiv`3j_TLpYrR{pV{E#Gb$r9Z?7~uznOkncWxDgQXC4Lb$Wj;qVs=A?yeGGYqxT#i;@kr_lLCt-YN4JG>dI_nBL`a&47WKYG#0svo=Z7ldDMR`a3|44KL#3`6u|Hwr`aQZNkE+%UAdFH~PukR5K;3dqeg=3NZR#bB&nOsH!j0m_gnx|~WbyrV=z)Pa46)NqmQfLRn8)oLOZgte+||E3TOIjZgfRdVKiCxky6JMX-rRr1a&yJxQ9M>;%2MEDS6u^*x=_yYw5WkfEQ-(rL@k-`-lel>Q~k6o4Yn#u`Rt@Keb&wWsvrb;@Gfsb+lXC;N|@CRZ;@^LDkierA9cWV;uoeJ!GQTZHJo*Xn%*C-qniZiU_uivSnOhn5PrT&vb6}gRu;=8rY>TQ@PjKIoi{17f6-6-yM_B(qWvY}V}&DsyPmw}w4t?K4&SQeR(s2;&RS;9+8UY|w-6BkDDa++YT$mx$dJ&!g?f#vL6LWq%wh5jfqTt#6NqeiC-Ge^6K)JXHM0crnhr92AtxCqT<$SXfs<OQg$y1Ww6E?(Or3yX^`d_&0=fn;!$#4R&=Dm%CC0JN(Zf3`4h$QIHESJs^8aBEZ}D!^d32=l!XWMq)jCnm44iau9x2zm_==)3G0aNUJ@JlIJyW>=p{w?b)Xs}xoy<`|}Ty@!x+9vsTCyq`*y8-0NqLiAx2^dKQ^D6CIvOYNqC=68!ZTFW~7Le6|2{T<?7(m~xNp4*LK^8jsAC5E0Ocuh3JvzTeeO_RrRv)kw#3XnQtJo4t>AL<u=oGy&TOR^Vdx;3*nWM7nFn}x=9Tg1aw>@utlCXYREGXm_WKj<_u5lMLtb=OJ(*aF#y9sg-9l~TJr55;N{K`6Bx2(+)X@Z7BYCgTKKzJiv7jdgE*De%l}G710?s_Dfe3^<V0&Cgz$z2>UzDn^#B!>sl?=%v!z4?G39%r!}z=T4k(ZS%8~N+4U;0(#%bxEl_QXz<}c6#<)X_77z(3$7ZjK@HNOoxNoOhFs&Wq~OY4qls#5arVej;*42nAVR*gbRkF$4{GH(BTAuGo;QA@3VMFdER$y{A}GA|&D%y{dU$KOwz6bAUfo_Zes677*I+?6xAdp>FeGU^j}AN9LY*S;S3oq3F@gfz896`EEm1fYaU38i%ZBaWL#|FYDnDs<5HG(C2Zk(x3<{R*?V7ymL;?xursT;k+st8udc^uW3YZ-n$$&+%X0C3b%Qt;0qJe~kr{FK~=`q_tLpOS+3ltj+<ud~+aVK|rK#AyJg-q#Qd^$_Gn2Wx5fjDneN)DjA#IZ(wk5dFu1Y(jL9Wr>f1U(W%l0ZSOp`oD@=d;y6!aU?8Fk^OfTE&nejrnAVFqKVSv@u=(us~{o>bKB)W7s(%BQL6v6K4nH(RR@AE9&rOBC!a}FP@D*W+k6T;;>FfMaj7VF2ZO8x8+&sQ4iG!yOHC-NBIJnO$=*J=lTqt>&lRnIgNdFZchW{d?8Ky!j*+9uU(suUry70_j+lrG>3ZjjD6!tC0{*lVw96PhpanKGI@Aqj;Bv$&T$jQmqCWnE$xY+v&W>ooSlR+vOGEe91mnm_bt*fAg~rRB{Bp_JI7P*p#offD)K_GW}Wo!PfcibGFyGfS*!x@`QPB*1cO7+!9Np8s{Zx@YX9pMg-gw&lxXlDakd(sH0LA}NW4(_=UAV{==`rsLac5tEpHhzN_2c(rs@bE4E23c8IraeS(f7>ab{Mbpn=9zBJc*5*Y1K`xW4(eRRzmW4>az;!p-?_T*%jKn|nl9&^o~N#}%BY_vyJn>L$A*V|}CQiw$f~co_vSwD5}vjK)!9I{ycRi6-;p&jZG={`&dHH1(7_3Z8xT{1@mzDIlaXFyFjIDkgY#3i_R0M|wfyVSMh#{EaiY5X)q2{R>DgqYoZ4XgPLy3<S1rCZwKgHbt~7J)}fPPedvtqA;WBtF$=eWcoycAj99kN5JIY3DqLyO5xe29({y>cZ8e%sg<*Z^C?j`AD@3=HHGrs`7dE+grd^56rKPXhq|B9=KS#a&ns-JwkAD4yU=4s<`$6eScBCO6cY&(-v#>Yz+3+~kmhe~2|2lv1eK@v??3-g6Y<Zcze6~UZQ7>hQxtCa56+5<Q71GutNzle7jI-?G>~C|#x1%i?u$;Xja3hK40RW(Hv77br@DPJf`$@~jhd{c*V?cWf0%qv39uRI-`0i5#7X_5K}I2x;$8^%?uuJ2nZ09xcr(9nO;pH#Lwzbgr+jS*K%Ev$xA94);&#7}J{;73stjR)w!?F-7wO;gi;nIgOb$<}q#acW5`*Ilslbsadz9~~qAkbdPsmV`$ad0YFLe^CuHfpap-U1(r9CTVRZF6f`Pj8YLJ-A^C*Cx)Kx+~xE9#(9(y2xo+iRSRw6<AWa@DR>2ULZCHd(4c`PJpM+ETQ=wza&%2}<Ddw&l2{nwKC9{q6{ilt>pgBznN|*4u=bvJM1AN>EaVZm%tGvF|52wply&Y>ePwDx1ugZn6Bj)PAKMmk}7O0<&t#lg!5|tL&agofTYvvbQR`bjjL)-m4kD;o4u`tBwm>hT&wll%sn0o+#qig^Y@gjUAhKLpziX<S1C8kOF7Q$0(2dTo;v10YeDM94PWLIGn#yX^bf4MQR%)r=>4uO<{^2K~s;P|7^w<Wqrp*3MF1Gs|H^rqz9GmSVeA6K_=Ckd2T;TbI;-lnh~fLEis>DL~)dJI$4>H<WrockEkWgF9&tvY5vqm-DEnhp;Y%#n()J&a7oYFV+j8*oVh11Ux{C9U%PT4r?%@Q&eMy@RAfpjDboB@2qjq=S<L5WC<gH$18qLfpJ?7~|J{};K@O81Wn{%qMhL@z>nQ|S6uBOwy#5eoQF$$WMWVCtX=0g<q{F{f0@)Y5epuvD;B9tV`%vr6#^^@BpJ0xxjnx(F*7gW{!^wd}q92v7hkEgi6nRbh!gM|#z$Nb!&K1P+uXblBd0Vymme@Whqkl%k@+ph`xP?Arr#(^W$KSFi-S?W!=HAtE`(^I;_WhJknx$HO`8JwC&i~0t0FQaXgm=Ja@@B(ZlD$C1^@x^dR<4feM#?^X|3aRQm?rpZ%9bRq0PmC1fYXb9O49L9=+5^gr(FybKNB}~(YRP@<qG#mTm?ifB79_cmTFh^+FKj7&E-`z0&T2R*UW3@F*~C86+Ci4wnZa(Kz2udg!2~o2mN8^U0LeR`sVTvp_E>YIOv-F!$UafDuU<VTwh&<b52>&5Xmmu>-P6hki@?`y>_3ZJyX>4g`Z*6ZR~TKA}?69v3+A@SvaY$R5`C_-T0JzuF?-zsyATWCx52gha)n5fS9)9Yuno=1#cHEp5*w0aK^31;Q-Mzh9GF;d!z_N{^~YRit3TH-)xNbh(H3ri;hG4dzgN4FgWZ+op$PagqUdi=$ec>BEkHH@%|nN-tEo_yw&`L;3KQ7IdN$kN4NkC`^SV7BR%d}xqQ?kc%&$X?j{l8vPI~^CEn|`)yi`b2&bqHZlc2hi4LjZ!tsU3#Y+_BFV*f+<P`Zfj1tE4pjgo3MyE?|_IszyYdDS4qKEOJ!~1yb)6mfZB^_Mh7paqnoutnYJBf@hH7~A79Ji9F6(2JNp!-$5R$HnE;!PW4w**-&uT@vXTfK@ATy_x19b}n)x6?$kI|he0(Zk~Ku;25NqXFu@7pWVkJ?J$Ct_q_#CJ&_9^<8`97#0wbmErQURC{a3ZzBF5&g;lpOAQ%0?FP*!%7or!4*}o19)Tq#@KYkEc>Td>1}@Q;J(5J5u}2~6f%DFd93i?>t=|b@2lRU+{U0=vgM!61ih9m@kl){^y<HD?WJpQivtFL_`-=R$ftggCR#-ZyN`|67bYtL%n*fYUR85=H=ar;P5tx@lgn%54`t7#Ln`%yWo^yu=vklh^%b8XS1PXj{*$w2)2D5^?ZL(4xkxC`a$^IMG<vfxHwUqQy)$Ef+j}$0RRaS12;B4L5r&72j>4a;f)A^zImP|R_A2ym@JyRG4QbcRptFUe$v}(@0^lLu#?d7F9gxd*u3}Dh#5m8BxoH=I2M_Ot7V=h5YMJ`Fa2ArHm_YQOFR0&w4<9?KMC_;uTIFaF8Oiz?&aWu53J=;kJN3bRajt5L2%q2vc3UdJJq54BnI(tO28inq2dC9co$%fbN0^$7Fw;s46CE<>ilteC@N;N{Cl|<}4+o37-4A-~Q+${<OQn*+3P75}SE3gzt12FtC4al0uR;(&CzOOia-I^sJf#E1Yf5<{XPCMrPiL6=YXT{<a1FhiPN!3@=0kE!<y+l{XZz}A~K|&%*hrgxGq?P6b*qrv0qk?`a1V`4!W^>Albd?kx^ggHO_TAZ%umb@uN*>Lepa~Z#jK>|Q4{XvoVa%x=EN~eM>^cKF)jHA8PhcKHF|Q{yrvf^q-R(C<d7onK{3-sYggEK24t^A^F4ybJYquj<G981(Ir5Xd5y$OzOc5u*apJ;ha5}JudK&fyP=8}MY@9-Y7B&=ej~55VyMgi2tOfYlJuKWwh|@z;#aH#819vVT{7g`s7|8>FwOaElJ;Yb<1^(u0UHkQBbwm3Fl2O_6J2(vn;pq<6x*H7mU*~*@R%j#zs1HR^{I{`}6i#ObMNGcKO{*fwkB4bdDs<`6e6g4Z<R|*7K9@B!O7pCIB7Rd0hxP3HWj4QS!$tiF>aI?U#j`O-rWSanY>=9A$CEj7{{RP{a08HLP<i|PiVdKwfceJO=iuo`YQ$L(4rfuuD_NNi<3o_@2;~Z-UF1Gu$6=s0eHoH}9{q&R=~poeguwidJkcb9uxDUxbz|%8=+<U+)ea$P^VTgTm~V>pXaz$ya;K{&JoYf50CCC&cT!B_cG@Maf)Xcm2Uf|!)ft>40b))y7GIw?JHEVD-@bKg`Q|dpgTnEYE>SkfEeg(9MM(AR%*^dB(lci8p?W`~-6U<l`LMlHF#7(w`mE?uMW%(-#ezQzwWf=1UP#N>-Z7dB({8#yO34AIoUX~sYta^Ioe`4&=TeoCDUf(usK>4<u`foAa5XY3gc6SW;Hi5tf$_ecc0{-~Qt{X2hmPY}U-`C=DaY&E8|XS!;q4PX5Otpg)a3`N)-&Hm)WAC+X(s+4^^YGt|Ici+C9Nx=>cA0dDz+YzD)Nb`=6H;3;7hJeXi%Pi#KDDW%q}|?khxQ3x3*w6QX&C1pzobi-aDtbcSb{F^W^LqtRS0OPNoCU+bMoOoOeuh>o_!u(T0J4<$h@$=Wi5p-p0fW;b^#1FgyHu>$v!;_RQWGX%jj7f+B7DXbJ5!y4ZkOe+WIx|4>W-bq)reQXq<O_(s(XZzZwgJ$($I2Try6Y!R3Q)t;Gn=w|C^(B<S#g&pGxs<YEwDKfI;SKnAcBNRz~;usQc7hke<TpH|wRB0%FcEizu@+WURPDYNlbC>%2Itz5R$}7XV8B#eU2MDT)q4Lz4(9Jj-#SzOJjicRC&i&lok@gV<@{9Ut6S@Zrq<~BiB{5=zWE_KsFFFHQACJEqH}1jU(a+G)G)B@r;Q8_^qal6hSKvPZDU9G5{Fnb0-u&uM@l;G@Mf>3r#4wlJs1NriR-#25hg4hgU884b$6L#9)s`reTkWp%@=IxD`UIgzv7b^4g(|!=8=Vm5#quLi;31r&Ln(d+tx`Q6c0gL$!R(Z^Eg<e<se`#qoPd7m$Y3^i(x1D65Cx9BIACas#iOz|_;lb{5ZSAvDVk)hIy|6Wesx&EoG`{b;n7FD_`-G9&^twqk$B6gJvMLNS-xwfV6*b(n#YM-T*jb!(vQxhs$p;KhIe~qmBaWCI|+3YH!@z}J-kmJQog&j8}nKKDm(h?xa5dylH7xhFzl5uDYVx?nRw*gfWK>8OMe)6n=zSJ`oq=ENvD^7-|RLLSRWB$e4~R-_>Jx@?Cg?iwP1c7n3dZC^*gWN|6#uijqc!}!9y<w9#CeZ+j%!ecm5hjRwiVO_qzT484Ob**3gQfQ$toJ3rFY~p}%Fg(~@hJDHxl=>6oivK44&md$E@A7vnd1pq|jmCnfI|s=6D{h5N8NhZvKAQ3HVt`5<f}TZ%|J`#m1O$~+HibSg8Sx3G#f#swf|gAG827x8#sAYUTp*a6*7(Gl4ad$tZ@ngKZ*8r~~zsk!nWe|6Y6De!SX#)V?VYj!(>0jjhqbVPtt0ZEAzPoqp=cGB0TnQ$JF9+OU&;!H?KjZlT;%RLugd(FFl0PjSS#bMlxJIB}@x>uE4WMI!++lwN+t4C3RQ7ni=!nsm8kP{h}OACS2*heC(b0;2#bLCRmm8S?RDser05vIS3i}YMp%5TIk@y^i(7t6jVW(U&ljCc%h1*b(PggS1`W@m6(>JMPA0OiZgN%8OSd+bLd#6*4rOVM%oj5KuR$y-8q0*`1Q*RB>9-KrO@L<zR`L1PdXX6D?oVWZT8oe_p}l}6amN(Z^+sRSG--w9cdATueazhs%Y-S1z!>Hr=@_F=>$@TT)~;O3xiQ!T_ED%eut^!2ikBV;#lKDymzx1XT03Vsc7SHV{S@Uw0>7tg#F4DqjWURo@7hlS}bNm-6yGY0DshQMOZTH+Qiur?+d^$|~@&<NlMY>RR*=qDhf(VtMC__G+yyjBWcE9KTmid}U6Xx=OAzyshriPZ1}ja|fi#~nDEZlr8Y-^KViTnIR~m|fqCyPbo6zg2-LzC0{mf}?db=pH47c+{AIB86eGICFYQ{}})mLurj1#2RM8b+wy;GscLnSC}gWrs@ovDHaz`$~Y>tfhL8Nj_^5<9KOQVu!2&#JZD!?EV_c%X+IzJ3VYDmyfu%*#$V$58PUQpCn_DtDhZVekLQLzX|!6fiwosa5aZ0m#o{cD_A+q1OFFm0@~!@ey0bG^*$w0$IZf_|FcDjSDH<bAQNSjk0kO`PsG+>ivFgjA(F56H4*jzL#B+-SE(ZJAl5|)l;Z9*@e!djI{}*>J!;fpF;2QjSNn1Q8>jADAN={jT)d@}z3v_hQZ-xHK%#B;O>oYg};75%n16zpJ_@AIpp+*>nh`=F~2JXs28E!2j6lP(I&%uv<`Q!RoK4XEJ;H03)I*oxSUW2cpbK^OM=@iGjV*dyZb`pouy#xn!i&}sk0>Z?8EQ3t3l|$$O2&pdmrKM~3<8~Bc$rC=`;Dm}~MXiWtDSg2?+wpE84m~-LoZ4C3N7TzO-BPg0fXEj00_biLW;R_;rRXv-io_|gC`N(6M5EzRkFMoWSmx)1x7oO_hsTA?CUZ4n_+9qg78QuI+qG_!&eWGbB`0zEl+mYrehiz3y*i%JH<;1?WlAS54v!)yTkKJKOa_Q#?j?gplR}Y|S1(X3#`AfXH1YJ3j_95zt?Mm8h2mB$P%O|>8Kaq{Q-)^Q2Pu7v=>e}mZ9Hb<Uh|Y(Q|K0kDW-pL)6o(V4U(WdPb^9<UdoXJ9T>hSS;sB4ZZPG+%oSW8b>2`*V|J%S*QmmzMZ@PHAmD*k5Y2oi#Dge*E6PgR!-m)jLU47pG)sdI1Rdm6D)$!*=J3ec<J1|YCoSq0BZnGt$kT*Jofyt^QX76K1UDfO-oj1*F>D|nq{cjtdg9_JsnjJ+2_}HG=GiR=odGc3UQAp_@&(!f0TBu5%L8;?3FqbnU;#RVg_No;(4;z!Yjd~;kl{Bp+z4Fu1egwlpHr6L;;Z;`Cv{^n{BC@w<}3^Iw(Go?kTPwhR(+#-8)N^@e`)w8r%ozgVIUMDMSf{SLXzk5PestsLjgo7Jf)@WjvlfvwDO{a7CZ&b`NUqaw?@M-HJtP8r-t*zw??u48kZ$~n=v)Bb(|_G@eCGA%wY$oPCNOSJA}j6fV{9_Y-qM5ryyzxnHDRX96C@UsxTB@hmKW&DtZZOU*jQ4Oc)Q6_`!xC&;mR`o4p^@I~diwd<ZNLhDaIq;{DUmXY^^mSiC@1?lc1&NrzDR2=m?zTXp<7EEJ2oMVe+;W?^&+L8k(}<5K-tKu1dG{(3LI(;wA(=uSK=96;qjs=}c$^x;0&A7Vm0T11xsxc8$_p8;HcKe$P!+*aCbLGMBU=hZNYBgD^24*K0zSX!83QVP;ljU0tpgAgKnPx@VupF+Qz{CMc&ggCp25>)|6H~=Z;bP(bb$xeWBT>=2QX98)0e0s0bYxx!CI=<vceg*0Xm<q^ECg3Tp;D0UsFC{{g1g5#O{yUZZIjQV={-h}4s`28i#X?6#dav$WM4oVA>B>TYr!d}D!mF7109tJI4=L0u4hy737r0}1CKX>7z8mU6F;G6Xq2D}()3^C=fo7Gf^`DYr|BoQDYf})}%IzDgN?~8<MX^-WC*k~j87Cp?#*O1R3FnyJTJd0X5H18{L_n8$oV&18UYNYe;T9Wp8;5(XMhJHkG#ob%I^9;3G;lSj1>l~0aG4pMhHw|^MLz-&C7ioBW&ax>W0Va2O0KAzijSm57?4g9m6HzjCj}2Fb)pLFn@)}R?@az*hbRHJ7TAKlbT102qLgy^`EvQ+0AX`<%Zre<OJSx2>#0@ilH~V<c@hn$4rRN(JGiYeo8-G_S*;jWK4D{6^cu(gPAkdU?xekLd2^-CrZF@YIu#sM4K$8WnG%9^SA88c&@(J`w_rPP^Js{E>B59rdT}z|8AGIhYKAZc3&`%Qaw)fVwVx(gsijC2=!V2;dhD<R%FIOs7mJy3mX>c)INIv=%GUD6%5rV<+uOM8;lI21QX6>eNd44>0D&gS`J}qCai<#1MH{uX>dNvD(H-T+yg$C0_ymYz(C>->3r<fXW+z&$Ro9{_{JUP?iss61ML2=$H|n*`yHGj1The?^XhhTS?jHgDa7<{zj>$Nih~EG*#wZ>nJAycNGg`0W1sjp7H0e#9+mJ*1RvgDYRZO9S;X(D`gQjQ^NCtiLdAA8~ZSl+qn_$tXTrvX#&K#N^BXGby9o#@D+7WR9&;^VvK&=bEjI%G@%t=PrriUJ(J+m$5TG=zf66&k#Z`6Dq&-}IW*cHo{QoHjg(6%w&2c=LnM@4N|BfbWagZ<c=r7VhSJ|%DpxiuE5K8EAwAWjfp5#A3vaHul4?eF<QKX`xGL6s#zge3g+BsPIkOC)o3gskxT)1UZ(;OhzEnYS_&oWw587e>B46#l?M&8O=Q{maw?TO;R=@W{2ONZ0m3ksS!g70fA03?}M}sXv!}*aZ~ps(HJ3)E(cY2~6b(@>M(?C>MeXk3}RSz_FCwzG3i2u1=ozc^E@?TT1UNi?h;sl5Y7eTOBGt-{Ixn)gI8A?`8*@mLTfcoxrS!odn0oP|`0Y4egd-K@Jm`p|^--Xbj`n&Ik~upDRk;O-Vf`>?)mgoOFAkh=&aFrW!rtNYS4zKhGKteBxX;ltvkyWVJA}$y#NsAFqa5XnJKYY&eb!EXmnh#z(HZ<Yo%P&USKss$P<%9*O!sR1y~q+cwr)3<{0Q{MYQfHYTf^yaJ8Or7H`U3XK`@X``6SS8i16wUy<ynv=B<;)`50U@O?>M7r3^ql>*<15oPw7EwTt3?L$R*?`$8Y-KtdX66Fv{|d*-`pOE%&225O)-r2?p2r3tQR<H?YSsd<D;!D0P*G`Ew&|(6RJ(;ZO#bdAWQZJyTGSRcBkjmoSG{%_{%ThJ#@hOuYZs_E^^zBLs|Q9B_c1l-VZEXb?UDYa8_B{jhKUOF=#O!;B4g7law%nyx;#56RK1eQ0d@^}G3=rrIj_g{$?cIZD|EjdSD)KBXwK90wOtSf!5ODt@nPj|Q_x6#Yk7^ay@Xi|b^|aI#;O)fq~If1nX||hQ3NWyU0qxLAy?%fF*h#26no7!p-#OoWKsb=<~h&$sKXLe@}pnR^1^uw(*?fUVeTpvzL*o`t)jun-5<_z6-X|&HsNaY_I-UO@_gg-<sQVe*N<XnV7&@6{XNBreU8{+5>JOx$P*qYW2OGJK&(u$o+$qNDd9SNB?C$q8CR4O&_9&v2PIfk`sqW=RBLHp_PhPPM%N=Sjji>q>PnX5+i;ND6%zwg6MqqNdR#Ci69G-ace8C=%~3?bd(?}K>gHCxSSc@P$)Bu*)o>MwqgFd=9vt=VCE{$u8m&gs`lQ+wea@(0aWNh8<xqwOi~V3bS9O_qoYK73=f9XMhhf=4g8NPMVL}U5p}1(|0&V;HQJpk!gW6MZ=0-jfzS<Fp@*-<j!8qP+PY*OxSOQ_z^<zpwfKS{IXOqM-{w?EeNLZ1lvJ9Z2D~DdVCxaO=r4=*3RaD^kZYKj~4;R5gaxnvP;>neodE&HLDyF)tY<y)-hHI=;T~fjexZ>b{^c6|}d;x=nE-Z%nh#U+OlW&AsQBF!){BNMeOeq;7Kbf{B2a3>ET6^`_kf|gk0Q05vl&mAt>QF>Pbh5KMrs1(tk3*T1fG9Q!5o@U1*()(1W#LCjzgKF(F_08!b^?lrM<gb}*?I}dE=jY~5xNi%7tw|TeJ>rvC#}wYj04%J%w2O>h6tRu?LXkDRtc}(d1dCER|1SLDQeG3^elphXVHUUTuP4i3d46^+4+yvnIAT0-Yw63FWS8<%LKeyl%Z&Mn(}fj;fpKt_yx@z{EHKY|KR+bm44LUgZc3t<enl13&^tBEG)3pK)?%JN{P3+Qmdo-gV1Lgg0pZDf8(X?4T4jv2>@5%mdm%QH@DEa7@Km?IuC=EHdRbu`vJyeplUC?;`j^|8n8pf*Z8a8IBuw)gV54{@V8m*$AnGU&V+Of2LbVxp$O)<uEIL58{S;eW^HA?y5t8q2bX;evumP8rPn-lS&X#B*ugJ$Lv={z72d4fT;E)Bq;$Uk>(Ds_?Xey0$*%kMLLAb^u7Pyv0y~!I%!uhK!JTnyGq5p@DNy$k5cB*>M!lq7H{l9Uv6W4EElN*0BW@JkA%mRwPbLxthnG`ruE{|5i$(stVwf0k+L}{4!GyY1!Ju5qNb*7`B9~W9$tfe98NUHFBfrr-Gex#-0u>|gOa+OHfDEiUQ5gAOB8Dl!!hVrOGO{J`tRh4n^r@gkn~F$F@j@FFBOZk&77!g~d<n|jfoSk%1*^5K&E=aH*f1vbyS9(EBOEcux*`EjKmG8fn=!gSBTml?T!TID(j}M?d+Fu9h9RjCA?IsL`-R9E1^V@&nHAx4g0y`?c)=ff46pMjMUgEoGXCpB0mH*tLlA_H0h9%ZETP7JSsI(BRe*_zgZHfBiQ_KtBxEts-iOrqHy(-NV)2TV;LHGL4U9V)jYfWvvQ+uiH!7vX7vrrq(pVnqO$Vd?sL^HAAC+>%WrLaE9Im4!8&I?_jFwz}?aK9qg@wHfp(Vdo^3H!L@R7gKbA~)2u))V7o5N4ZyV^h~Ou?C+cm)6Zxp%uWx^uKQgLx_t-<WX;8O1go_}m;Oz<J2YWeCecOM!zIZ;j1(j70sEoT0y=Tp9vN?E`@qBZ0Y}>iKXUTbXz8-Yqy8Lq<LS$i#p~kllxLOaDL;oSiI0qZ-P!@&7aT?#pdt=e_8wbmEE|keU!c-DW^TDTst59#aI(%XlUXstW{~AS@8T0H7p}Mya{jv2&cn+m2J&wa<=yaxS~-{Bf$}nUSNJ(P*kpUO;+<JjwYk>)xvyAf?HUt2`pQSFh{0*7~l$?<<0Gl#q<yz%>3>?7TcKjhVEv*9*M?59C{caPzMS4JTCt4ArPJ<4Abwr&tV&lP2MlSHFD27|RE5U+Y{yyjppa7|R5<kC;U}c+ohJUe2t-9{!Zp9%t$+PM<_q4a4XuAfKP}@#W%>!<qNL3EVFOH2pIVX-fh_?aXt5gY6RH3$e=B^5-Ir&A*!afJ=N$oqEP7owTkp_mZh4(hK<<>rA14!#B^ZlutX2jaxizR<<HkGn`;#J52G(U{)Z$t1}vQN%QEk>$(zs*D>-0eHU^i=v;w`OORclG!(i%i6c^9-v}T3qZynp1aE<>&Ma?6j!HOj99aE;wC2ITfAm`;fI}$bV1~ayYH<Lr5&4M_^Iz~mK=b_xL98QsIwsrt)P;r;(5jp{2lrwoEEEDQBkyS&S@s`UwuzA7xE>{x9^HY8<qQ9G0ewNJLX3o$MsjiaLg932xm+n@K(pdVcw!&Vtr2y&B-~BfL%3cpUo}vy_z_gS94(iZuUQU8G3BwW=`cHZ(4X+rf}D_j8^kCz<VX*v)r!o*E#p<H)qOe2R*i?bBA*NXEcIOInZ@&1==0O{8lp@sB)N2bb-lj9P-@N1jrYuNhzui<->~LE|KnPX?wcDjL=9cMRo{f$yz%Zj9eb2dmOr{tDVMXp+-%0T_6X2i0Gq?7w7bN|8Ah5P)Wqv-TqVk*k=3jK_IU^Pc>N}3jN}<G;e;~BO5Rq4roI!Cg)gq}#7#bPROhL>eF2fw$^su##SGA$*1W#c0wU@(=5+bd9(0~EJ{na)gJ(kDvQw_enh?{eWzL9m&{~~BaHa+4gzA0W8L^Jl|L&yu`%>~j!oI<YA-+cBJw(nku2(z|fwQ0si+MRJ>g4^fa)M;?OO!)Qm^@y<-Ih9988XlzjTHtvw3lxsK8-1!E#iM-*oS^St_0iX$|Dy6(ORSJ^|jj#p_+*sKiaErgpsM;QH?t4Bi#p4zYFGpf0>IDcMjsy2>+YiZ=Dao>^iHVEwFmj=4T9OwZ40^i7t^jW1_CO1LB^!qq(m2TA7=VMV98?E~e|cwGJn6gBnzh9dZ&SP6}Z)oA0qhlAw(#ONKaQ@ZanyGxI5=C(QzSu=?ECsPF8quWmMWVMaH1*YTL%Y}~HzuD{cuIgWSgx7=Z1e~d9?T)x&+(-7RL<`IdhwnjWWP|OC8MxZFJ;2Yi)q%&}_6)8F1`(%tU>WJi5#cKS%itcThE@cQuLuuRIm&Rku#Vj6=a0%eQ*-JoyD3PyOW#KZ+Cyvq4S`>^!(Js8}o4bwMjcrT-M%lWQ{?uF&?B5xegoflU36D@un7b;c^eTH2M9{(;nFgT4n{b`7-}Ku0&K4(L@lU|;;LH;+^lks)LG}b#y_1eqF=Ey+pJE>`K06-IgJmZyGW}Z)ZdsSvzi?_uagp`A|MpLuqLY(a{y`{-eo2wiGSTg;kMojfJx@vPC>%s)>RirZgaU$2p8G8&^Oc$c^CElyqz(M^497hrB@vvv+jTlwOj0^%cVo_3GC9qYC&G8VG6eAHSE3NbDb54>DcsJL@bQ9{$n&<ae9ji$TcCyiMqCId`Tb+~*pfjy0|e7BW)@RE43MNyr7vc&V9jY1YF<8zg6ns0uHW8k?(M{!rEvSbSTEL^VAH8>GJ@l6#Xk+-GhtJ&)k3Qsw214v_ARQYwu0|DigT%D{g$<?8EOL(khLn~Ij8K^JK}kUc7k;YzAdHm5+=t7Nw=GHW11n{kW(jFxYiwDws7Xfj1gm!!U@G>6(;*qk%05Ussxqu*r+Cz9gkCD$*zz|c`9;^o2^roLBUP8C-pl<-V~vLSwL}CbXQNG-Xu&#wisyh7$y@a*(Z6oGw&j=6~0D|GxpSfQ)nKS*+r9nBAd{eB<HD<4XxX`mC4=8weq#M-%8G;On$qB$2MwS{x(qAn^~PBGLrp)RA!XLEMSMcfB)zo=#T40hP{8bmQa24jBO7xW23EO^Wlg%%C;w*fgSsi%AWJ;XCiGNZL3cRm0T#4P<^e^3n;`9%&hi_P-gxS{{FR4WucCeToTm`P}1h-BJ~~#UN6Mvx38x!J;bUSr0>zE%*xUzguEA|FMEPg8yRdYq3`9BH>@bW-tAoPE?>R=I*MY;^C~<@;K&ep)?#VDwKHnkdgYzQJB>|rHQ3u|?1<!j()J{?B|UTU->pcy(l2NV4Sn#(;$T3l!Y{KE$$xJSac89TgS7(E`e(*-A;XwuPjD^(&B}bLGLd3N!siz4Yk}NDN~BMPhMv&COooLg4A{&1?ad~NLa1L}S#56KTHn4~$E@`D`_6k?&D}eVod&%X-|Xz&+^z5Y&{jx05fE@a-b<A0+k+!yI62c?Eljz9QCJhP2)++oqiw3=+4ft*N`46EH$r=cRd%UM^E2U4rQ8dvWmHh&i62&>FOM@_{bmnF@q_j$TAyQ(-DeoaIwlran6b$lb{e<k9|th5@w9gN?YCJhK+S8VQofS*N8NoeT7s&Bale<0hDRezh$mYh?YOdMd?k;euj!~H-%`2rg{4<z7k!$-jb}3!(ve~Cm4~BoNV|y=YTA!k4zg&Z`Vp@FZmOS5Pu_!ceFv{oAA7IG1XU+I+1kYw@$|rWdf<1Kklz=7@RrPtAwPykI<*fh@~j<O34S{L7z91*H%52s2V>#xtxF6}FpD8fA-kif#*J48)~f^8Ywo@o^~SVk<vltv-xGmmXFlrI?k}CJ(8vTdTbHpFi^Y}6u!l8q7-I?cKpQx|7@o2(pR=eyu+}jlZ4$9IK;VAx`ImLtHG<eqCIzUy)HxdUJBgZ^$et$Qm&fO{FX;Ior?P{{n%KFOJ)o(iiQMnB_aC+{bPo1UTbx@bBHPLb{c)ik9TZzRvn+`10jEa7%<nUKl<5>p_mc@Sa^2`bg19x)+Jj<7ZXW!;_=8Hh?4?cm^M4X)!c<wrhaGi;^p5*OgiKt%fPYYKDg>yUOny)+7cW)>vMl_@ACd9?hUa~;n2{hWdp^{J1IG>gHi$-0S5zu5`!P85yk&g&_I3PF&!zwb_t*7$Gwf!mnkuza|9&Db)Jgk6+RvPW*i{}y;kuH}SOp*3`Kp}jU8ck{k?NT9(7=6UBmY*o<Ve0**bnTSmWJUS8E8k``<2H}M=Kp7jd>xsbjmap1%%vUqK3)<XxCZIn;IJUO4B=uS$^yt^K|UN;O2+_-1L)s5){Y>38s>KkM!ML%;qu~q8*)*aRr1Xm}6wLDU|<k0s%;7&9+>+dCSnaz(&*9ldH-b5=Bl=Ix&@7gO!^qvjPt+r8GH`am`O|LKGLdX)XqsEgDZv!ZJa3^ov1zx->e>z(-^W9uu=r&u!CFWE$im%K+u($sBfT-&q#uR%&9Ly;{ZgFJcEs`!4m*>F-kb2i3)T&Rxx|Z)<&{xmziRDsdD0)}m><T=s*jVUjm8YcOP2VOXQWHb58AQhjr+d6z6PsSVmcZck3_C}+xwHSLTL(csZz{-v<#=vbSVC#dSf)xrE^Ohf@O&|gX|fStA1<Bejwlu;V%C8Y%TSfwQNWgwF*l8Udv1<UZipG{7y*_zom8>D0YZ|sfM!9A4@*2%6ZBUHy~D2nF?>tMTHBk6e}bckWmh8kzzogr4h$W10lbDuiha-5biHx=97qH%Gbl6Yp|;3p3|$#@nt=pVwAn8LH183f`^$PXbDzSS2)37T;+#CQx6qmqZkQXHc~K8`UT>Gozu;tM^Jj5f)ZgW!9Qj;nnl;(m0D)Rz!C6>RPT5_9o!=I6Avkp<LkMfZ}^8pHkt$JGL7&oC#o6}^S--P|<%EEY49#2Kdo-nIU6-wRaEuPOUJd`tSEPe_x)`yB*hmX#^P?`MJS9>Jc4M#@6+H50cIS*1P`iMhG>vr^Xf#uF3dVxT{`CJOtC-dhoa(8YN9J@hdS8JmkqVz+U3tGSJi0g^7NoQzSJzf7-Pp<tv_?F&{erPPWmjv7nq#8f`>SHFyuHwh}Q*g5wJTh!Pi#!Gp)-Q$B()(67To|Yu%9g~YNbV3`*Iz!=mp3jzXxMhO*CYCCGkikU|nKCbFQMY@bWgQHIoSWGqdwL@Gnb|jPkmCHTMvIHJcx!KCBUQ<*_09T*sAiuq74B|ay%HObTWS9Air^2uwczHwTIQLj)rXThz_wL(Uf}zu_oi8$hAm;ttWmRnZjt(?DgA|{ygq?A(amF@anwXqDQ2FM^C?mXKv3*?@Q53Ge2Jt8YM=-tD!QPbX`(IgJG5Y`zW&+OTCJ>q)@u60tV1)O!x-1j66@>_i&-Q-Hp;${b8cQ*KB-a}7vIpOy1uis*T@SR0w(rk8dim06#7h~Re268qSVcKw=-PdsQBe1E%0K$T6t?emlH}&I`nCMNJ$>~a6$}u5(QUa-h;#T!Gzr-gXzfu$S1RtSrR@j=JJ>B5kTjQ!E{Q$7%8WCB_JjC%ws$G^5S{@QQgPXP`&}6G7)sf`RJydK57UK`<;M1*dumiFV5$jYc+9JGUQigKqygK6=-tSANtq|>l%}6piEdpA4lzn{b{XST<j{*xpMJa5V^AAbI%xhr&@@U;WRlupd_B2zDG-ytOH~5pt`fb6ma~3gbz}Yc)tFc`B5#nguHzs0{50G4&0CFddvR7d9_@)kQtBYokm}vrX{BlarK8b{dkb_*wp!gb!y~>`x!H<tU{9}?vGO1H7Ul0sk9b3v6POEAF7BGq2fK57EYJ!p;`lK`K9@s?<w9@6pM0e_U6H`0DpWSAZ;G0g>bXdJK(2joEty%rlsFHb1P;9sA1Y#h@Y1=JK#Iga7OQxdX)E;i_>FoGW!cpgJ<cq6dHZ(=bt!0bsnQlzm{!ZMUsgO@`-xwsCSUPJIU<&_S-p_p!oP^v$?yoyItRkw;NkKTC0$)kIGmV==+<o4Q51&yG8@Uc-k(Ox|7kEAY4^y`nmNSr0O~F)#FHDoukn-89EvbXP=&X2lH-e5Av(eKVVFchVS@JGLlDot_}V%G~ot(wjeT0#z5fxsmSkL-)Y_o(>sY<-)zR_2;s-wt@`%*PIEIS-FL{fz#NKeR4z`aw0bblVO7q*P)bF{clesuUcU4+!{nbXyX01U;X<dM%8vbjk#|KB@^hGJr13>YvSa-3FVHzFqL60#O>U9_^rH4ib?Ptw6n_8ZpSG%{%e}{u&{v3Xazdf4JrcvQO6DsJgoDyP_pVhs8FxukBw?XOUaO6@n0aU)U(L%cijjQpTUlZ9DWDe=N5sEiHzf9kgGmtHWpZj>eax;O8BT1{yVS=(JmXh9uOCZyQ;$F0N)TrAOi1EOP7t{p;3=4zXUyZ6@v=}ixyXI`>gO_all?y7KcAca-G-ywi_o|8h8h+lQrouX*~fAZNXg($yC-D;k67MG(vld3Ly<w=pTmA3?iFoaM$8!|tw-~?an?kc2ZpCMx~U~snLa}BwXDrK7BT{ht*%rTmcG$1aJvypO=rBZes|rsXO!SA8EGO&cIJD5a1wrXsPtk5LJc^`lGK<hXxaUUjvK=>?}_l@d`iLVPk1ZZDq{Lbx@t)jji>(pDrtT+=q8h(J?R`F2w)&iPD&g*7!5HMw3*j3CCkgirAQ_I$P4D1t8!C6)9Lm^a0I;gYuK-ia)qtpw0UX!rA`#yunN?GRN`>TJYQj7*Uu^bn$QyAJo%LU9G_`>I<^tpaG515MiaL{lf#C#T}8qoyHW}X8T*n9-V5wY)deL<nuwtAcBgR{!vj|LLozY=_oT=*6QeioR8=f^4XC=?Z}&#9uj1q(aBy9_@$I^ww-~WAQG5U-e@|pMP!GF@rSWKnj6c*$C8P`?kkrUaDJ8MG{w8V`p(}JL&v!eaP*_=9ik!WqP)@}Qa)6OjG?FoF&T~(G^>oavvaT;W3yjO2p53f-NmG?Nnf5lf8#~PnjB$IfQQroh4Nj{1?M8fayHWq4Urk}0QvV|MgaOF(BxtTJp+)J{M`(;%E-zn(`>!N+;uiu7gALAl2WosLWwbIPP1s}SIbFEV<Rd->RxB9)oahR6Q`O$f-<;*3bI!VHa*PgEo#`dHSiw<yay%OLKj2Kg?apL0o#Nn^|NWzXz&WrEJd)F-Cy8Da>AGYda4#}r1#r6l$j6mH|4?x`@ET62B{;i&jfSpQABDx7Gj+MjuKHh?egwaH`MXQ7yuW|>I~l-Y7}X>PLl#G3c(5CP)W9KdQ$3}bmTu&qQs2L(RijVL0PSa@9?7JMVSM`P?}?>T4(jm8bIfkvCen2`0$N1`Ru*X&P*lRQye;X`!Kp$9S2=!Fk)lMB^aBQCS*j!xj*!BSK=4+34TBVScI(@_sDjs<h(!%zFA}^;Q0t9eG-0*;_&jzH|0%c}@-)mp0y8X(&bXuuA!~1?;(%Yvu#uiUX}eGp2jUek9-639<_UbLEfeK(>v8^^Dg;UY>qBH52YJ)(1SDjgzUbcKK)GGGz@yDFZ!FF*cq&JpVA5;LlL=;$$iN66R4BW1Y&R+Hg2~^*DW_;-UoImC<C)vQGoMBPh91<*|1*u~fX+?d(!qs45CO@8Si<A;3lT)cSQ5XaklGh4WPYK*ucdsLp_@kRFL|APfli;+NjS%r7Ph;T)g31O-~P#5hkyGgl!1Z={n=4)0u+1D@12ZJrUcqyc?o({dj1*uy&L+2S0)dhQOZ7tJ7M|KJ6p|5J6lcRHKC#G<lMXv4zm&+4FBfHPzyus8N-Kr;yiUcbPntNB4^&jfw-im4xK8|7>Fiz;J(ahloN;q9NbGz{r8+ANwqtj8PzeUX5}$&p?G?7K!$uL7}cq=DD1-Ioa{S9c?%@d+WtiE2OX5^RG0R4L~ie42|k?6ra0_^(7=QYqx29fk!|=crMz8vE}hRpO^DMPnt1s;lgAJtZbBPG4mfCL4x=C}{bbZ177qXNPajP*{L<svBV*bg|K*=%&Ria=bGhm`L*+10%=0`UMcqdfcKJ)<Qv<uW8SP|*zx{!*zCPzI^#{RdN_pLOw3M93-tMi9=8pyUQT1QrmqC~@%PlI=nBOcS)_BTm*~as7Apyt3_Fk4baJ=Vn3X%@bv?nZ{;|iD|r_#QdxuT)~FvgjOkH{?%Ogl#~Yv_`|*0CrsjBl{XPP(V+4WdG~CwO|We}68keWhy4HKe`}{%b@d&ynN=^uw#4;`JqUrF;&HKsuPJpTWO>G(+jodric78keDzlhX9IwcWhEUB4T{zlcNX9vS0t%_-sK@Wx|wP{*{C!#3ljjmL_+-uL0KsneV|HAIhcG$|?*EpX>2$DyaeNiracM+E0df|2P1OcP8fZL7`AwROymQs0Q1n{nga`s!}H4$oH6DnaNi3f4sqP@q+Z>6W+w#rL#ywkQ~;G431X2p-r2xnviK*ofvC4%7$yWw(7ah>@B`LtxH$BJJicTv&E2UZY_L%5s*$Gf#Zfo(h*usZZ$j3O+slnBe(ivU9YLr!`oYank8yrcqJ~A!p5Za7Y1bT%6_r$~tLfB3@L_7QAz39g2G=Lwr=sy0%5vVrxRe$oe_&eooicMRbGy!3kG39?z@6zW%O+Da*=QV`p`HeT#sd-l=b_gE%9sBoc*GodO<(g@iKxh+S7H7Ks9A4IqOQ*UNCV&$u|{jadnrYyb97VvNU03ReL2ov%I#Bep_}$Hn4#^_N$|vxPmUBn)UgvUTDDm-bh#*RQRT&~Ck{(^W_fFbC2H*qRMTO(Tv<V4Brgq1Ldu>DTO@ZT&OtW@I+Axe>g|z7-mM4QluDH>jmqst81&uSCrK=R%J873r=Ont)U;nWi(OvX2Cf{fdx>6xJRE%jpm@AQQTx%$t|`z<Qpvs!Dul&vB;7D{G$rqbM*%fLR?fsq*(Sl1BJCDdF_FFibq*pYqcFLgF$IPsqEO_oabZ{xpzoQNVi==jowi)~xv?<FvLhB{bsHy!@>lEyOJO*J3AqB)#X2W|bP3f-Bhw`Z;QSMI?d6ius}trDcW$Mvh$iRh|%yEI;yA>c$Dk3=swQlL_AWGPp8=je0*B5F<<XT^<{DRm4qxEm*Kd`O2k6`Ko5s|G+w93&cw(8w3IYjW1X`>D*hpDKVC{we6WYih%m=_mc-SQxZ7%%YZ^QaMi!G`LdK3drArfQe2CnzkGgBT;q*0v+tuBz1_ofC%3nods`q9-Mw4ieoq9<&Xd=!p7X%64z1q{6da@_F|F0S!v~GyWMli55JNr^F|5yY#xuVZpywV2HyE8BqZ=98i&5!*WvQ~fbh)w;Y&UihQIR}Qb_L(ZfU^E!3~QeGEoM#JA}7$*Gv0yjITR=5D`C~stH>^NRTa8w7RspKnKS6Do;6c9{qkQVw?UHb3mr|3T*psIr-KL1$I?b4f^t?EE}OJzf!I+m_#DgC3O!!Tp)GZo;Vi%WFG?^(gd&(NaU=4kdv0QsJi(JpL{d5Eu`+60Bna>ySUx5}?!6tCE$B^Z>YemrL$CH^R@)C*T!)LXi;91ghI~>MRMDI4_a-H~IE(Ntw`>f|G6mJZ2U;Q!9~lC{=-?+wXBN}J2|s%nw~Wt|?n#FW@~c+Fz0Q>rhOzLOVn1(`5<n)nkM3}@Q)J-#AE0=^S(9TWCCH&*d1o$Unr@%4AgWOtVA>wwfD54jsSpoh%u*8{lR;%}WyYo$CleGLprRTDtKtvw-l_|ea7_`#8^x&fqREBW4`IqCsHRC`%F_XYSC_*8NT~Eg1#4+U&PNQ_UT4xDv*3&5k5LO#wCl)r<_Re1IjE}BGTi872BKCnC3ixC3E%N0!LISE{oE>ADGE>1og+QAioV7U_YYlahZ_zZK!fiGhl^|kQo1rtGKf#@@GRTG-394>FS0Et<GmnKy)#rzZ+v5Da(=+nJAXsu5^-u0&q2!>^sJ{-Q8voiR%>SyxzESJNZy6~uP>2qyXr?UdqP;QHZQ|&M&Gic_jBQz`AjH-1-z?<gM7Y%o=!MjqoFtbk(9f8lk-bGb9fGqAumlxi?ay7Bc|Yqy!)T9i!0}{&`uiPuxrETFh(&=q~3gHaR&>CEjTvCSISylte0%~x%A>a%as(aRRiH%_JqV<$+$8lkZka`Q9w%9PuSgoG6AUsWU%Hdq98)gNYkX{>>>by$$-G=hnO;6sFsDqVF*xY&g5Jnh5)hI*!^*H`-cdRC;p5ZYquK$P4Dhq1mIlfpA0s>yJLkhQ#KAb3nMtY$z4LYkEM=uH&b*``%|R!jw!ve<1H<Eitf@c|A7pLkMKJ;656X;_)<>ovwVl=UTyfWwAR0knYjy|%+h2=g3*=wvt%+m84r?a!PJcUT|mZNaU)F}#V?yv+^09n8QufEQo9Q*e78qYD_ENunNy7TVLJS$C~&w0ELy4;9v9Y!CC<?uj}Fm~6+Rq|r-h!(kfYWDcl96Wz(djOkJ4R|wHcpls|vvl<XK#v2WqoF4gEf~J)=)J37Fy2AH6t+du)PBNq@A5_X<5Z1tI?%rhxal{Rz8B_b_ry|1PAEcut|!<KrpTkAI*g{<Jh0o!oEt(f5&`U@(fV#BF_{7#+tIMzl&H=2p|NOOh_dm7KuM-6yVNkxryj6XNVnkNV@<qapAu3JpUw<)C6<f7qss#R<+pmUNtF<SDX`W<kp9(yGA;Q6RWEj){-xB<YpJ%6c}RYy{{4H9KC9#{RH_W|SVRVjt7m<J=Umcxv#NX}mY$qVQ*K*>pRwai%6b+`d2@+?U_9cvx{1M_ukTHo}9WHV|@_kDykR1++Iks$oNy@IUOzihDu-x>06xyV;TK+OPvhM}2V$6sAWPrbp64R(h(z6IJD~J@i+CmJ*OXMMX<k6BHkedieR7YZPaR=23L2NurcUWJ+RQx=NE5!s1R5^{0pVTjinyXAZs7&*cjH-8A9t0m8DxckZkke2$o<NWT}ub;^jR{<`J46ISUj3MQkD3)5Z{s+~fp^Lp#)qDvJ&*A84+ANVPkz$vZ*I)-l?dM-x#A|?vNGO4rlv1iM{u9W1<r8L!6EzTil^+fWYYC8&zt0}Dd!X85rMriD#$Enyj#h>IfYMg+~y7FP=_l0Nr5(U-iLKB)jUGtsQbugGnGpF6PF+$J$O>sLlqyLUSH;5}9k8W$p40wDcj}DXBgVE%kkXtRMB(ZwA&PaJx^2B)SWQsU<&F0eC^&3l{Te~=U<fT)*QCqI2T&xcI0|b5Y;k=5w2;opg!g*EIes!tRic(K2)t2uzdKjJ7)YaE+Me6SZp9Vtq$tLUga56fk-9x`-BiuP5PJjH9cBj*x2-)o6_fLzL=#LLi>F|Yr&4YI#STRZQqEV_&?*nQapPd#LtL-)S!@+s>fvrQ3M5QLzAu08IL4u+^lWWbrb~bDNzt|FAN{8&t0YDNBg)g!F6$My&`mLx9xat*00L0qAX3z!pF=J<WWww&~1UbmL0`|4gi=oQy1)m9J#V@3GV}2k4y`Ufja=pqEwy#$MSLhXPcpMu*8u@PpV*MvH!e2_K+(!9Qqp~9Hm8jbP8>NByQYB&FG#r-el+11Wob3T3W=#1R=ycEQS?~=-mHhhUb3^V%X(f3$tKlgIr{vzg{G+6gLA4DimMZNn542bi^^`{UX2&D#mwkgI3WPGndRN!CfY$J^j2q=f#h++N)#OOyzDhB8Vkf@U+*a|4`E@GS!ub9uo`S?YHEK-(O+BoM@AmgY8kavg=W)L`5wb07ILD*$$w0XAgy|;3R;vX3AntXyr=8K@<aju(JreCzt^A5kLsglH%xbZU>a+W?@x+tL{GSa&)~jC;?W>R&BRl=4!W=9klRptkp3j6csR>I=`W^C~MsL_byB9G|p99HvT+T>_`fz?0RGD}!l>=40aK;IJkP9g1JAZ!DIhI1BuiS~nkuZHLU-dmaOPyz0XqfBcatTD^UQ&R`a3(l4G;iNhjkJ7mmT+(6SKf+%jqc+WR11&chJ<@N9t~l$9E~2-!a;JF5^0aiPM%rhz&iag)adMk6)dV#-Kv8I5gm3L@4C+3`X<&jCgr3ea-|TMQy3kW=c(_%7f8Q1Bj`XnchT1Pf_#&n@cGV(m|sYR^Rs~U&40AQ)9{p*sc=-rH46O~329h2`$R_de)H<54FC8UG*UVnc=d5P12)<VIqcv&ufRguX+<Rvco{K~buKy&1xfi!P=?2VCaS8`RHYa9y$7iuP>>9_qhTb!M$UP^5H`x8dQ>h(@Svg}REn((6(@s+Z3*z)0#wvL9-qw6+Pu?8^yPK_i3(64$KGj8Tc;M>+J$fmJB4bT!Tkjc^bY+-iLZdVhA`Sjz_&tsFg|M6mdmALu~p3JF0iDt!fdiHN7ssG6Y1@q1*WL*%xYYPL<pWGx|-taz@tUXq+Jgqxj;s#GsB>%%5*O8^v+iRXR<{T)C^^}=4QOJw*{QTu4YiO59BkTo24PlOaL=E=&Z20!*Ol9+1$Ov-Jgb+4q>?Q)%be+NQ~rh=_jLuLH{6J;eSirljAW@5000CBSVGGUzLA7jt<9}tB9#AYob2<yO3E{q^mRYRr2Tiq6aNM{aB;V3>ieY$90rn4pB}#PFgjzBX~PwOFGxX?+OvdG`8}2uHA}kL)Y`XRA$e09%q8m310s0<$rtmf4uzfFMlVcomtBI;TLwUhYG>}cr-}@bv_2_UW4m+DHYhUNL1c+GHGSTw)kD3zR{T`v`F$zhUu>wJjxIm9vgGAlD0Er2mYiMl=GtSlz*i;6G5j92lX_Nng2Nc&PZiJ916<gS?N=MgBClHWdm<&D;HWia}?ztGjfKz5sqe(G#cnLMNYyglaNky0<)5B?!1>^axaX&)sbX7Xa+n^8zM0K2o*hNI4ib=F1IQ*q~voLgZ~@Tt6B%Eeyg*#aVjj8K3zGZel7h^+20x`<)`HR{oD=JPvIhjUhUl!ZL@57VppRRS7*1cFOhq2PTzk_OvAnCzGoU+n5<Lg2=TTjT0$U^;YfHj`C=^%--leqj?CrB{iue~qkGl+PSB;lum0?6c3QypA#f_OIY-vnV5YpZ69S$)>*Yb^t%JkLa`$XmFJJzbQh*-b%GLRqmDlkpyQF_C65g8b{2adNNr7o3!~7TFOvxdRDzwwcerW<}X;q8{+mTTN7<%FX`+6FPZSjTh$NYpgMV?;{S*yRI(|`fpUFY>@Izk3PN%7!$VkElMaU)U6i$#W&@R9tLz^V|@HR6={h^Cn0E({yENUTedI?#C)Kn<(&zh?!WKJzHObl9vvqoY-Ub3&k^jI#TR-apY9eXV%dZ}hYo3n!gwIH;BXWT$_ELQ8vS!#Nu)qysgYN+nMHR3s<<lE6Pclgj`#|Bh&n;GP_k_9DNCKuUq;q+1bDF8@l@!9g2lPGpTbqjCvw4uNktXhtb{UOs=rsZ6_Xw=W-F>vmr!l_}{|(MgH`v=Of6enP>jgM^&VMu!AgPsf;mjh*s!o;(CrywgWG67=lrBmp}6O-tE%PeDv*Y_2ud%+#lE;^5)HB}qoggG-`wEy>l&_WpsM$xdaY2-P4s81B~}_D=fKBTC>vsunZSL?IGHBH&`U)b0YegEKo7P?#jfOzo(SaJ9a<w!VgO8LRc(`bP6MdD`{{ql5NfDx$BEZB3PPSrE#XlqyAbmh@AJQWQ@{gQS$aUkK}Kch`4zFdRaa4vT)|7J+>*89kWRuvG&whX^h7IKSEfrs%lpgq)gjq6<j%*m&HRj^$7tK2Wisw%4lkOQ4)CERR?mgg;NAF-o7I&v?Bbg7iEYk*%F|bZoaH5f(W?3yWw@ReenU2vXckr~_-^PIGU2wV{_~htkGV)XF=J?e$yhu({Og?PP1?23MD3MW(T)VsuSX9az9k+?^a1VU!2C<b{C9vzpNufp@;$*p2r#>swpf&36#vXGk`4qy3Ui3c?q~OGBm!&XdURnU}A~O{s;q+n0Q^DYo3lv&aO;8YM-eg=(&c1m3y8DK=*=$7Y;nc@RMI+$Q8n2KX~63yEsAxwpBy6J~j$7=%-Gm%`cQaxF}cV+-=?R-?YNjxkq{=O80Iz`Vjsc+tVWHUj>Hc!@_bQea!f)WH=tJ4ynO*r&5#IVzXSf$~kK!^6xmjoDuAtnW}Z>8z6h2p=OpQKzB8KDX+-cZmA8+OwmA_jR$9JcK@`CB|_r=Obqi*TN81YooEc8@M)<XtlnxTd<=Kg8EMIPJMgz4s64~ewA8O`rKYN>vQ{6>hqnwy9GCfp;)cnT)(}(x$7L#>9&#33Kw*FBSYfgREIl|;<ugp{B8qezRiLg93k9nWfah%j6I@SPL5-KkxIRDYrA<D4~Q<R@ZlPo75+T6%YJ;PvE6WHG+5sWfRuwx%<cmlDcD7LYn)Ygsc>FlltuFkw*`vL3~s&`RHEhRGAz4YfOxR(Ne5kKhC|qR$RJEnHRVg8Rb$}HVTrj)k~xWp<izu!3*p80FNVc1+D9$m7-aI&VKVC+jfSuR5LMitPUHt{C9b~<B)hp*-=Uzw3?09<!^|GNZ#3FH^aYsYB;Nx@SEzBjz=oll-X*-{uEdAHGsk>1oR-9LW?(y8)3e&#dXJ`~BvLD*aloPoj&9IT!i(|r{XslH=kWFb#(@)AyqKxrE(W{J!1)M<0iOkhTTMhq54P&tKiX^TMhKC<-JnZ7_=}z0wPIEx7z(`-3SAz*rcf~nz(znTXIvXeEf>6dBOsvSU>QzUK{+@M?=o6Eh_82%*@p!<kl|eBVh+Fpow$OpqUc|R_k#IXhP7cn`|umMxqr<8?$U_JSxlbd9kEowSLw**CrY`i4EZmgUlQOO^rt7BlyHYlmA23t7ytrVk-)HQ7QDC$JAaYMez^E;<eTFX+XCzbq6W~;(+f;|u)ab#ZY!+%r#Czt!41-$&SKIO6N}*Ea0OQgt)b7LLdlfZ52g7MhkOi7K`iC4OeUiNLhWdRJv_Cr1Srk`HQTad3>?a!1GD}yN_NP51O?_j(!!wnvRE=3+|ovxpgl<fcGn-A;)Tf>bf?I`2=q@eE))xazd1P=fDnqwuVk2K8zgnQA%ote-*x#KG`!?7HORD^+mG4#(7w7!CN)cOp_D_1Tg=&~AJoj3GE?*iwGT3)i1H#sJz!BzW1d=CI1`*D2-d1o_#ov46(y6&XadJYXM~}>?l!=_qAMtK8Pmf4+-`Z?URtRTqz{bX)4Ag$UsNGmr-nRe_bSAT!&rMM(dejuI4e{#z7jLkOM8a-zr)mM#A2jpv2@&iXaod&Af)cH<3*|H^Mid~CCU2*Y@=BGKD<?3l#emB*Z0M9V(3K(uwQ3kQaU#2l~3hM5a<u4F;PNnC2gtc3Y@uJ(l1nxij*g+nA^}X+$krM!~|alksqak?FO37F)J7*`q`~({B0DC!@MwBx@N`YY7W4#qKG>eJ2CIVJ;U*@JQ>YK*@llx=0Enytd9y&;f}A@D8hsb3OfM$DC-Za9EhSP-Me6%w=P2&5bAV$agY7|kQFfQndUZ3{oYz*69m<b=8yA*)T>p}nO;$VG2CqsOpn`x<kDe+|Hy;-9)@{M69fmGB`CTE=r#mBfsJo~mIQ+lyNDWxIw5(it?yv^9E#Y?I@=H1$9<5Z$UDSN4oV?jcvMf?!(LLTL|5J2fCo)YM`MesIG<zRMX3WZwr2349L@q2q<0_|)$%g_$`p^!EbTBdz?u2i%Vo{LQin1HkwZ8Yrc>!ICy0`s6sAcJm#Y?*uZG3_{!*o7mi1I+E{)tzhxWZ}m2t-FqXCh6+4?M<d^!8>aO>PRqr4}{VKNa%pc(5uuL=SEfbax;I@nldkDIsRt-YHY>pORlv%7tJyKx(}(|)W)>s-4k;Uo}&Z;Y{z6g1*CJ?~6g;5@@<a<R%Zsn8CvmhHjmRK~_zbM1`|VpEJzVDLw7Pci=nu#(0U+tKu%S`|uwII=W$QT*fe-8;>_-I#)MHkzw-1UO1LuAT`Qp$?+bEfAIBTlKr^8|w|9W>9<X2(T;((8O`;AW(et>eu-56Eavz4`&ICD}ncH95l3}nK)=q&I3QUl3`%=I=)C3q0}19fdK#xy+59eIv^`xM%UvJaEHLlADm2Q!(<9qz|QLGr5~@Z(%C2E2N9;CQhL~Y`!&4uLHJLd+ovi5KPL`L8)P`lKCq)%M5&A<2H#CU{G=oT-DUhcxv2h0%?e(xW`w(Y;@ndK?wN7zP8hiiB+dvU@6CxJUm3tPf+57n;f{OJz2vku?%|!6N?OD;(H_)L3tpkQJ7LEX74nQ7?3Iq%Qy%mE=)@htgECPhwgw%0khJd&PUUr&bTterM(R^%ptM`Z)T#rfay0Cny_O@841~mXR3aizXfg|8M%<297VY?v(IigDQF^t!;sXb$9V94Y*F52xD_nqFa%xBE_bFjSZaQy}>4bvfhMecfLr_NOsM#@ZVk~u9#4=EyK4r+_u&PPtl35kbjrBV>)DL93Ne?+|oO%N;xB8$Qsdr2Z=G$o*HJry}%$$wJvD^wGVixZ(T5V5*c0)Yxu{aH(*$Rg+PTd#AL*FPUX)+ZNSX&{c?6^=cXJW~yqxd>=<``eAArQpg>h2yUW@FkQbz$S~*82APDm=Wq4tE8C2*P!-y|<ORZY}2JTC9T;(MQ7dWYZ+LNNl26{B`0JNf|1y2CmPI49vtudih%q0M0Yv#fi6l1iXGP*#-)Vf(-`Dhy*?{#c+H>E*rwk8JRaJIe29F`RJrU3MI#=%))esk{I~-WF;f>wDjNNKuHfy)01<kLNQR1l^9y4)k_<SSRE!Yg*Z;PG7I;Q=Y&GuT)W#0Q~=>pe+WD%kkm;B-M|q>=iw<w#YDjrEqcu#?%r*Zt1{W-Na8$fAD~|@96o^t<5Lo+V;{XP-S{1GbA{f)u=X}~ExaJ0QS2nD)!%)ONsQuPl|Ku;ZLIBy650l>U0yI^z@_Y@mib{?iI^QzsC8;W$>vxQv%nS;S!=5XlGQPNT}ar)8ZI2K%5mC0WF-g|bY8pF*OL9DgmG67Sf`b|U#JwnkBSVddjiugPt)aP#E$ztibOqsz-5a;{NnBjI)b4WKq05Dj9C<2nKN5ZV%~gloY-hMIAuE5-sbQ`+NCWo^#~|@bnl?uxmSa0iXg3m_DrO3F8OB+J7Oz+a|WC-QVP-%L-Y-7sgnM1j0#r#`gK+-BN8@91;nO<Gj*Rmv9W`8FV1V4OXNgPnzPF>lsMjqLLKFBjalPJ#y;?m&>erUW6>Y3)<KrLke(;+2%JtAt2qB1*zLsc75JB@A*BrWEv};uha!Tb#blj6ie8mIHi3~y8|0Nqf^<C$wzeBfsP;6&E9kaa0lD%%=2I29zbSR+)I--4LP)1gs<VR`tKD1FUo?C5rzA#UZ@J$IXB@AV{h$`JX^raFa;SUWrX2)_F>X`L2_e>gjys^ZUTX#4p3vV8*-f#KI^Y$!W=8Go@!6SqKUMY5$H%1Ep#Sa@_cbl?NOL3ii{H<8fmAio9OT987r(E{$`ReD_|Q<LV;>gI%3Ej*#yJpqOa|A3e+idNXYDb!2=MVFMDH`|r1@Xub+h6iRZeSE2XHrt4$G~Q-AKmw4~gLIq9q`3Ob`fF7V)~3Z4_H`j$M~w$j@^4u=dasBtPfOb(-KKRpUkwpBHAuIDXH%_rj@RI#yK}>C<43pG?tqJqu<_6}nT|r*2lt^c&G!p+3YPzTp!*)s*=o#quq5$S;fs;NAw2cL7=p^m;$F3u=<k7Wr7{p=~1}^Vdp^4lFUNdj(@``3JAo?{4kwWcWZ?GU|0gV?syGlQGgA846Wb7O@qIfOcyH)}m$KIJsL0h~*4*O5<^Uod~T$L7|i44N~b{Ek}gW*tm@|iD&BWb{+mpVE=yb6U72>aOqq~7MPqf!MGH2GK+pMSSNsGWB@p9_Fi&2rB*mZIp`drkuN~8QgI8$0ni!UZ%_INlbziNYF&>BZu-&Vm4o(_0`;dg_$Ix(gm(qZcort1{GY5So_0`ymkQ^$>y1>qIaNw<0v|HL{zCj-t9S;q5D6}TI0uVWpLKw=s`;({2Vf-FPzf1G2?&yq!b&J-gC*pHCdjLoDFq=E@@Z*KBj{wFgRNk<2Ew|e>U|`c)?5O_I_LBt1uIaHumZHuad;7%x)@4qfjQNYPe8kasgB2D>2r_=2zvn<aW0ywyk{QgH)g^d$cCfCF(Sc^QCC^JTwIxqUG#!|-Luvrv!34vVXm=I55_$40)<$CQryL9ApbjHBt*hjU~oIYEVYN7(|J>SHV}mLm%>tgM%WCk5wnpq=5A2~7z3RYV9~kHsH?}Nz?Wp+XZ#pot(p0*in}dhCmZ8gGXGqVG=_W7Bs2LKh%Bt`x8Om4K(>qJtJlKnQG0q6;}e|9$8#nVJ-C5d&ZyteEM@-XZngsdKQ~B~iAA!=X_^NlrutsBP<j@aC}bfq^QFLyJDH6R52Z#4M`u*GKwWYX+xKB#A@g_ss>JkoYci(pBC<Fe^_#GJh-nqf>PDRd#&#BOiSeq|$`#Em8HSi`j9grxG9RLqu|enDHHWn{1P@eeWAM<?#lYXNbtN=TE!=MGZZ~(f8mnY~v2k+dom}I?{jir{y1enxso_|o`NsFc6%Sz0;k<M0TtGnQKG6abDJXY7qAXOPOpiie%JBGppK|tu)%DG_#=H9=+!Sd0h~W?;G$ETC$lH&W$s<(pZ1x~n21E8ej6$Qo;V@>1FLgBMjQ~p{{Fgejg`!0~oh7ayk<zxP>p#C8xWM7gZ^w7*jj(ES=KJV`i70?cnP2OvLdLo!-{;n!RMOv8^tT}p&Wu`*8x`Q9hbL2*yR70%{vPBoICD{q?8Vdc&C7_YRs%k04$=8xj1V8;8A75e@Ey$T!FZ&Em>?h{bUGL^pe=c@5uUz$URB8!)k_9O5MZWGIubX}xPy;Dj!`e5gNR_jaCjV(`g5}GiopEODCr?W@+iOv47UFpkyg?+P$M5_Y?Nxs*}}Jl%#cuWWz!0b+YJoQ*}C&y`UZBgz@`zEXoz0O3^h3tv4UV+(})%5twtaZseHJv{vOwrvyG&z%L3FS71i^%2DYG!O@f(jych*JC-a31#)O(jm#`RDf?hJjSTo5fcaD<Iy=gT#7|o72l7;jV$z;-}C>W85o$SPT)GXnV&<&1&lb@i|@^J!XwQa)C9qequEsXcEV3V;&wilaBX}>13**h*~SbI&r_40RAM8gKvcphADR5lL?+-`E=;rtAg(t@fmxf7{LESBU^>Nl%WtsZ!BQc#`N==J_mrM1F5RBd*;gnZAP=JxtuG&eoSo>WZrllx^QdC@&HHPsot#Qzyya!$SJJTd2}{7`r>#z&(;7g(MG#T8H6gISiMYLWn}iMr-l0i^?!A6Z%`6!5c);fL&5PMZ5-DH)Pg9sW8>9(Mcas8lHKSKn$ym21U6yIQN2J<~X<J(_6uL@}!bxoLNHf*jh_7?$Jq`euBmxv|#V1D?xscXYc}mFBEw*SiKcMQ*K<>zUT}3*l~KdwU&2q1fvq@~ytPF8yhLSQw4wT|j@eXgzZ(Duk<WDefTZ!lE^PKa<c>rMulXHjeNA)tQ<tX5gC2wTb#uVzB66WY);tRxVsMQ{&C&Qu=5a%nGwpIW=BssRZJ(aCQB@Y;>+NJgd3neQHh&BEK4h530F0N+mRLJk_kQZs9c*3z)HlHz%{N*4i3uEG9b98;;6)R#$(J+a|Jlu}n(B|3(IyXmxgqLWa1^Vz}%2$c=1mH`W@f8})62{a9a(@6>nhgpV`L31VV?AQk%Bd;q7Cv=QjHcwXcL&dW72U<GY1NlHm8-iufA3|K{R#qaX0O$l2ZIYq!3QU*Xzf-!pKP0{rL-;kpXITx_0VHzau&QUNOw=rphQ1xbj;9{+{!~Tq7bXiC(7&K>u+C{TTG)t81d<Gr%5Pgh$ZPes)$cvJ@<sX^&tRe!VmI*D3{m<D+`=Jt@;6qp|RizM7R@}7|u&B29(Rg=jqq$wjiy_QftDXL^H$Z%ohZqVBv$TS|8u^5Djz;}XvLZa9l0kO~E+}3sN_8?2i!{wxB`ib**l$KC{lf(1*M4W1%+QR1-tfJH<1o)Sg?ZER7w&YNNe!&$HmJZ!opx3`O^Phok~tn957-T5VLDMOKFcJO(|KBCq5~S@#gHW9R_%CZ5s7@s!my+(v)qzPT{#v!PApe=2mLh~7$zBQpA3=lrymqnmpq)aSnE4Gd)Z<LZeS(I*@w}s`ufJ+c4HCUsgmX78e9y6@S>QDD*Q>edGfGWiYXd8j!^~v{z-pA{t<AkjA234VI*zXS?Zkmhh~N+cO@20#ELI)oVmSKFXGni|FC*yA7t|u7ngHJpHc!fp6)oG7`XCKA6F&#V=|j!SY|<-Tp`O;5Amp2wDz=<z4-8UW3#bc-&p?(p~$thg7tSA_1&Ee*XR^p?A7Hsc?eg@l+3Qy${2IEV8$vB=52fktjORyK)x0Dv<>WhH|a|F#cCMs7oaA+8pYhoEnwyUOE~G9I*(VM2(V<#QT*!fNNMpcrz@7&o=*vi`5TeLnDIe$Xdl6oO0a4htNFZGw%xbi>b~7xe*27Aw!h9qOSK_ozoj@q8sJ}>D6FJ{`IW$c{fq(zUeGYVBF&D7R``S_@pBF{Fu-nq?*}87D77y{)}^NcKu#uv{tbNiJK$`1DY%UI=miBLcr{IO&gUS;y?pkDQAe+Ku65r!Os>Bk>gYRHs(}DJ=mIy~YY(MDO6G_sJ(NHqO#<ff@f=sEEDBm>8gYEMjG_;8IDa6J7te-Dik&0EKR@}1HC@R-@e?UZ6dtnLjGr-vtWdW;$%ZU_=L#_VH-SHFY_H#KZ0=?Om!g*D{*{>2cod5-q3q>dSqsC69AnGw&5~+D-!IdXgXAGF>HT90!(s!pUwKO|r~;<P7XinYyu{bnH@N%@1VE0vBODF)`W}ujydQeHhPW6QIvz&Y8)gw?tv{}BZ)Rc5?=Z9u;%qF(mADcd5eEe9TY#APz{3S0rJs2wah@Y_^5F&yOTq3@e~Lbn2;|ll(D#=PChZ}JLWT+1Ai)?G<o<XR1Ivr(Y63cddQhp@Ypk(Y5e+bfaRwF1onc@t>7C($Jj2GPwL^^~s8H$*WV*)f#&$7{Fc=6lyPp7O6%Swo4J?E}^M$k4=7%X{LF2h-;=Cw^^s#oTN_WwD7daoGRXw9TzVfxEaa8d*wLZ3Aaa%gTDqY@r=eFizBubPkOgs(8d31Muv!E7J8~?~N8{1iH?!ksE76T9@`o|~kXzteEUGVevL;T$QIt{Dw$O5)g6z^<chEd#w^QABcezI80+lBT>5PK1}1)NYIjKKy=rUhW_l1U+#e2Y<NrYHQ6a0LGF>VJhtU!{Rov5h(KkF=jM5FTxG$LLCQHM$mEkKVFy9&y%Y$KzV+94O&^7f(+PLFg}pCCW6_9;nS;ToGTEPKF>V-YXCp4tw$Fo_mcbV3qDcNkXeoM&c1j>>){mareNv7~qM5R{6_6-D&IwBJ}~Ku|QamD{tMs8UE#;LPw0EA(osSA(>2T6?a0@C4m(eQPEI$&4Ef>yBY7^-70!oKp!=X_JVzZnR9Ua!8S=IwL_HcbDNc@k90;}^g=X>nLQUwBYw-6ZZ)gf&_&~nG%V-(#%^OfDDXycc6_ZubSAcQ&9g5hrHUdqYGwLazqwNokCuQCH`nB^V(|x+Wa%y0fwlFW-Stfm)@*SR%v2Z`qv^;wv*1KiWUi1lQ{-M;5m}~UG1UTINU$U6@;=G!{uSR1$#_pG*-dzgAoQTPmQ5HR``{EvhYFfXfFY~z2pvo&RD?r@cnbz_tLB|)PAi^!UV_PBbDFjgr&R<N$5YY`&M?8UR(i!KcK0wjiU0eG>-rpu@MWjmGPWbe&E7}o`a<6F@5U&tgCxKE9{jE25jGm{G&bM~?i~0>FWc^WTMc-yw+RCN_RczdhELo%P&tGd>}LCReN(Ck46rVsa0%sk^F7^>@NuhXGglI5`4;@Ak%x?!o}e4$Y6c$fSq{=g4%5oj%h%3)n99?}#EZ@Vzj}?6B{=s1lT%X)E7`qP?OXRxV1z*m9a)Z_fsdVe^8+8#bZPM8^r5mC7;-A<8H%y=5CQ5KV4<mS{OViPEVX_ehO1yi1y~%Ri-v>I5G~<B<thS}6<31g%h&Je80Y>FB)3wgWe0uvrvds*nGVx^%+>|1^wB{qn2mJP^4z&(;GrTyUzF7XMT#9I!S&KrAdA6(@?8l0QVe7dHYqN3HaS6<G6cdq>7E8UG$Nzxsr_oMDlJ~l!ufIB+-emx0omb*bWm67wp{dLQ^VR5#hnEV`#<}bGV!+Iw4Ww{X*fW0jU;eE-P?^H?X7P&)~aZaVvl@W-}Zz;^v8oFAeAc!vS|k!dEyR>f0EN@PwzZEJLIk@(9DhI&Q29qtk(z5p!SH`c^nygBI(v1d0TuOW&&E@zZz@Aao(>Qq|#+DJWy_O!x=0DPe2r!9W!40=fnpScQ|QxSi@c^m80+guCXqn>C-C2Cj?_hmM{B?6lRE28?HETSaXZ`owY`jAf^y}gnLjTzYoyY5!0<FD2QW<YD>6g2xwD!4t<eS@!@{lY1R0j(jlBq!*(HzW7=Drx8pdp+@IWW_LBkJw*N7`ciXlSanlI6J1vTt4?o0EqW&;~n;Q6mV)|>&5lml(;xZ1Mm>f6jay?+Dih$#ef%6JP>V)9<1TG*v=?{`XS#dCCc*>puP-FzVcs30*yjLkjHu0ecLL@f?qZQptlCc-IlIokE+Fa&JKI0m;+yx<>sXxK`;@wCmSm)<{kHgFiL?U63j$?QSL4%PKN3<%%0B3$^MN=qEsK6Z-i0ixfN8{c4>Moz}dv~+z^F$@6x`c#~*Q*Sn&B_|SP>LdUkDAU@_{06ApdgC5hLke^@%XVy@tRdFEJ-TQz#RK%H8TsZTaD0)!?ZO>5e-X1%ETFNF;>W>4I-C_P%q&DbIDOkie0-xJGQx|(lqslQy@LE_bf4)XEGEES^~r%Jk)0<rpieutBeS2I~H?CMgYA9q}b_|&L3y7GRh3HU?;=PkqslHJoN(df4fAkPk5`bxqyMI*_fjH>dP>E=AP`1M!{k7zyx_Q&Xj3@_xXfj9Yi$upxraOa#(cp5}#)_)OBFfUf;CF30Xxno>om1P21EU#R9MNI>0;(288iECllX@Gd-QmMoS{?TF?#{o`(__${m9e23|nH;be47h}KK27w!g)^0JkWp{}inN}!TTn$IYtz{pYRu?BPI!B0gVFiH>^qHxs?V2{uY9T49OH|CHI9buu2%uXDfz}B1vtMyG3d=KRP%h#)AEY+4CbSb2ZSS=2Wc+$47U0jKzyzCgBU=AE@9JC>F*418j@ZYhjxl&%fa&}cSj4w+AOIC=ko%pI7oxw(yK73M!n_+C{DRH%}Y*&KLTjmM)44-)#SlIA`mnA)o7B==A^?OJBINffJKFK!Z)c%|_H!!bfv5}<nU|AYvBIIIZ(`!mDws;A1@5Jl_+e3g-GXo+P%Hu1Tc;G+#_@9!ymI9}b-l^%92`SviP?nzPNlq2XfcKLeArwxDW7s!KSO9kL!|<X;DWoR)-PEoS2Of`ZE(fdnXkg3ffNK4=DtOw*IqiwR-lQ}@tauiKE*m@(_f6k=*>8Gd^WWi$d}=Y*aMo(JTBejh>jp@l_@|wBBFt_Jb{tYJ!Kw+7P`oh)r261t$W86JuN_KJ8^sTfPHSk18%<~mK?<ZGVy-v>L6SohYH25TehDm6E#>0w=cl<z6JjodV}>ScTzH=&5pbllL_foj*jBbnb8PrDTb}0O1UL^glw!7!7?Wgn+s9+XKZeOh3Ov6DwAZ;ATHwA!&;tBF!@oHtqvUzd>ezB%;Y`x9oG7zm%NIH>Z9$AFVL>#gvq}=naUBhyn{^QnE=Oa!7@LxUyMfN?8?NLG3bE#`8&>T*Cd-9*Zc<xwC?cP&=(S<JvXY_Zc3(#qFH06T(5oaI_Y*wQFuZ+5gu(%5C|G2JKkg3(z}##326u_YX#U$j1$^vt=rc~wf?`(o3erUm=S$I$Q}~QsKxWDzjP)UTMvxPr%&}k&<`OX-#!OM8rnH2&>Ku6~r+qCbrCSw-5V~hpFi|%ib*L>Pc8}A7=kz4ii3}Sk_o9<fDj|`E4wrg~<S<-K-|CX<HbNr=>nlEr5+}13I=uAB25v(jq&6daGK7CzzsruAh!Y_Yc!ynWds1BFCCu+P)#)o#Zs}UB?88sBjxo;q$F_6xjB)++Q)g4IPxyj<q~tdGLbwEf<&VoJFaXQ*Q#g=bf`5L;5%yHUt6vBl^0Zs%xxB2eT<>1Fe5Jj7Mlb7sC<z%n`2YSDQPn?x`86XKD1YZK(UIXJO0E4=C%yhV8287*tNa<iLjPuRg7&&DlN)w6U;$P7v4D+-$3Atz%l{#O5<c@f*pefz>FY~<$wXqvkb7VLV|lq0y!?Mr<_#`fK!^DY7j%}*7a5r~SNsSAVzgAm7@x@EX=Jw{PjK`K{}oNtHxilw)0L|i|C;3({z8xG842s3zhO`D>*Y@Q?St~w*Yp&>id07M7T=N!{WoUjb?U#EqZj15mq}+*ZRB4;zneB3jB@}p4ElRwX_63u!pym<{l{N@8u`|e(j$G$7#uHj@WodM1VIfmzQl7b^Nm2=!0J`*14@fTWFSK^aCE<-d<J62>SLLinKOFEU)9s-9&DBJwSW>leOz+VH{tP?B6NX=ku$%X_sN*wm(O$j;W3i`c3j_X)IqMQ@9ynn!ik7%zqe6R3AHL2*eFYlY^ubrViXZVndK_ZsMPfRLG8QKC#^Q?uoA0{f>%crg>k%DM=NIFQ5D(WX1iD$wd6ZnXVf<pFy-^A6y<Vj)p{i405Yny3(a$zmSN|>4oLl*;|P5QKWd>)JHIEjPq&t%S38@l^y@2awO`f($~&)Dv*nw$hz>ZN)G;=5InMKubMijsseyUJPTCJ@)>nEXOJ$?l{%OA(@ltt{=t@h7N1oMUDNTh^bEwNG`cnGC%<6Pe)F~RzC52^FoS|M!pQcil#e5`e;b8$qRa%T#=oy?Um~u}+IZv*eiqiSPw8ZV>gMRO1bTX9{NN*RTwIyGr9_JOeNY!<6*gXPq7dRJmR5%cz5)m9JoxuqL)3abk+4|(c6`>$RUJgdyZPvP#ybB~7PibXwem_LlL%Ud7sM-%n0g8R`IV+^jR10gLa-+9e761zRjQCPl93OPs=%!b7;CX&BMTDS}<M9;Zuu8MhppRHQQ8J{Q#qDXQ-<LTYGEjS#ue36}Q~X;<{(h56VXhfIaT}Sh?pAxotM1lsGu7SPV6IwPb#EcPbb6EShcaMNS|vm+I0l&u+}IV+(gJnLIf@5BHPQpks)w~U`JQphwmK0>S$0xdjGZ-^r&F|dc(!)lc<=bg7}`-BakyfaO=<3p^0wjyr)$3I-m2WAz&GBP)bh`(XO3!LEM=Y#khtzx-!D#BRxSDz|Jf@p)%Zqz3n=p!kUF%;!2%!)eJnG4CUE|KeonoEha-CRX||YQ126eJi<T7~vtoR%gOTr`%Jec?KChS2nHALP7%+(bTsn5Crfy1qiVS`A!Ktkx#1le0ov(0wtyEceMn}M6Oe5Eg32x%qXx1L&VdrKuH@<nWu_w+o1HugEa50v#s#uH+;4a}d;}(;N#m?rdJ%6~CbbM!hXSc~IEfwyrSVDst<|ip7`TWa7yS6Dc`QTJLxR?~MgcPu3s?90P6}`!Y7K2=|*hb^k{fdi+?iw~)DXo?3lAZ2rYmWMtz&WH=MLYCV!^k&oxo4yBLld9CRmTWh3`V6{Zt-Z!(cG_lQyN?^9@w*>*Kz%*&ZQrItMk_Nw~~V^XVee>rX(_u@Dc0BTg8)kj&3&_8WmMs&sf>{EoKIVWBeOZ);@FKIOJ^scf>O)_JTh8GT8aihUng}U;Y*`cvl=e2L9}GYV}*rRL-}_Ck)`hYIza3N5~w{R7!G%kIVVZU-&bRziDKSx03GL*OJ#q<`CNb>lm|fZwV75(xD9ty25X+qQ)gbvn`M@>|TW|rg#y$8_y29p{YhMoR@m^68fnCYPyn(ouP4^rLyqx3OvSi?|b##hJI>0#vOEzN2~{Qkr+^fgdK_ZS@LiO52&d2B3Bu>Jdg}oVPjT{rT&zYDWv>;j{C2l$<r6F0TugloWf&~<uoh2qmbKLv5FPPI+LRKnQpy?O`Rc?tJ7*;jf-Y2&5<Q>c1#rd7FC|gJ|(1XxGlWXYh-@l3XCeV@q+1FnYh7hT~|jR!>zu%x7}DI%|eVaDOU0(a*Qg$*b|K8NvmOv<h#f)Dwm!U)6r1v<aG92X5^$Z)7C9Wt&3%va_Sni6~E|Egp)oL-iq|0<^S~2$mfH1@K=OB&QnZs6&|ApRo_?441GT411mZ<=MsJy+Xa4^Gyp&CWeWtJV+Q1>C;wT*5A{L|uo)-oq-`3qUS%|vswM@fW4}tb>)KNCEhtLHaC1)^=r~fLmE|g|%o@kR|I%FFOj$QDEgcU`td-flnixRpMW)zAA?enF*3#*1{a>@4#Z0{|g5eWVo6E67eW+VkQz1&&6LiA|P_BEJA_c>i;ty#TVGBMSC^N;U!FMp)gh0Ra@E+d%sc=x<SD+lB3+@rwPdqRwighVsSbgzgzL4xkTM+|37u<@-iYTXeimH5-I2P3nGDB7%<8-5db_Sf7Ag&ENr|gYHnQd9qK&ZM2Miu)Yb7Rhm>yIpOwzjk5(B^3G+9DD<tR98vIvYCUT=0X&7)Ra@YJZMl1dQITo!)cn(-<(s^UbZ?pu>J50L&L_z^H3}GXzO;mfEw<s`*XHY{?6>mN)rg?op4U=<SV6yno)4%#=t@^)A^B9cr7-8!<2wjBzs<<Q)9&iSNdB_&(hitkBYO>%x_b@>lD^@<lex%F`he4MFb6>_pWrsZzRXYmv^7w>m3hwdIaTnRRuID!DFqauG*r!&JrtLrZfZmf5}C?R}$mwOstZe(xLhxpI)}DvjNEqqb}|W^bMw3)JNy<T<_5-#V0AtK}s)&ol~q=2;md^HL}nhF%cX$CJ@vLb|>7KwroxsHw{Q$Gc^yh5sN2KZD7e7P4>9Nyz?KuYO9}sgG^^+YeDk0*`<3p@5<Nnye;&{E(DcKVz*4`l5Zv$`;{m`Z0Ujz4|1KItbU#L{7+hY|0q6bFCM?`Lp+6_B2JWwLh~b>f9OUQ35o^A+pV{#8xJHEaWDjU!fb`_ZpvP$4O`v)%xZd3h&MBc&Bl<zPY=;S_`+fo41(xfdU#}O8}*XNaF{i(LgkrHO)eL3<chHDswm`qsf>LVvYU0&V|K7195t4f1bw{BA2Wj!W@<<3_&oE@BSQr>hM5$F+^&4BKj!vq9E%956Q*>GZxr(G3P5V2bNBCB894(Rz8O6wAN>VKx*f+jZt8*I-AkgbD~ZB1P(KPzqG|Oe+!j{sxu|27A_4>jzu(m8kX0}t|+cuQj(C(CEZ8u))UI<5=jU30_Zo><hYHQ%HiWt2R2@zSn8n&F^A}3qwj6j->I)})G;>3&8f5yNJfqR>zqPT31nq=O&u!5{3^4xik6ZvTcM~T+XzF0E46Y5!7562sMtzrnp9%h#j=lQvBNOV%sY-^>@s?XZfD%P|2ch=s}FZ3@;fBC$w;MK_8}91?T*G;P)X!bf^5%B<~vvlD)4^?dV!HPIwuqK@mLbEHHQ<p*e53)tSSTK4E>5HvjB%-X)k_&4@~(8(_$BjG)f$zUAPwOd`d>?`Awc7M{lYT)Q+IfS58xDHkh>&ozd|zEVkx^7sYnrfY|b)de)VAT&Y3%i~BhwmUY`D(~|*+68m{^rGxeX#&RlhL8z+&CMJ!b)nZh@)TQtzFr_(O8fdAmZHZj!1^6Ces={cekF4VQ@%RK4pX4%?^iFFbtDM5(+$`ROa0;{x^RhET48Tcu2Y*gr%_5cBXB+=1v<Ks(c5S&_Di@2bVt!9chy5AS&VH;Dz0XFNC>-`F!%AUrt4N3AzRIiobX=nbWG;E2x12`uYC079GR}1rUOH6t_lTG5afy@J=cIZT+3wK=lJl!zeq-#PvK_!(80*yNZt`7q4?YI^h|o<B?V+{HdHv>m?lsXg{z4#^Jl>YJ@^FlsurnA<@y4J!_C=3di_T^g=O!)^*?f@Qfm$d5eTzBV!?|B=H&&b5Ym23qrlisiQ>xQTi~XjvLg%J_8@FdixI`t+R*;dZ9tIHE>F(p~eD}HSN$sMM5-;}U$>UJtNx~}?R`IOMOFA8oZ5H*B;VN2<RU&9ph*YaPCp(^LjjX!oyHSb5AI|KogNwG7mBc=;<=hH0GTEQIK7Ts<obk@PH1nIANA=nrwVI#uKED~kjL|UMQ@NN=c$h9Eo$Njy>s<Hp3OpTWCb>8Aq0ejVT>0(k@7!C<&DN%8V&1U_16f~vgGbjKlZZ~enU_;u9e;kYZB84)2m3w<ji^kig51@3cR%ca@Q$(?oOG?|!UdSLT+$Hik)E0R?Fq0YGkr*#`Rikmz@9Za#?k1}{h4Q)PV$-Cqcoo>y5~6k$;ZVrpjWuEyX#BYC}ZAw;U>z<gzl9pd--bp?$%zu$Ebu<QPU#)N7c;Ttv6upYqI<MWtsP$-&FLQihQ#WWX1KgN3czHdhk3l9Ohye6}TG$jvolZi*p41i=lKkqr{cXAMW053O`U55u#wN*<f2KVFjFuv^xlb)@h$iF&>9@_#$`}3oNSTAiLYRyVcw_-D92QvKGq+W#MTtGpnmW+`L1%XJw0y*R5Zr<PUz`Y{qml5l}NSaApEqpMaL-FPm;N-(fa|;CHo(xxSW0hf54u7YL^tN&<EmKqZ6GDdIE)497Plr@pC%G0~{Bq>V5ekU5#)tnD8p`27HAWHOpgmqdhAd%zamHcD<u2e>6{6hUl<+=EFl9!+O(>vT`JPq3#DuA-BiBbgXGuVS3Srz%+eOOXcd1&%99$5T_>_|w45Nh91zU;Y8A`MC`G&h~SRd&u)zA<O9Ch|bC1bqbhs`3c3nTv)!W^ifPce#`h75>`3snKCjn`CI5E1i)#2!jGaeXQXoL&khyWwCP74R9|w$i70MHAP4rF>>9=i@CZ<nH)YZ_>yXdlac8zjS{Rja;=4`I#Kwb#MsTyS`{PDqGvM46m2x?^rp(#AQ7dZ;1Z_P}xNr}%FW2@%gJ>=EZ}`EK<HZqomfX}<#XYTjuE9<!sh`~fw`!`MOPJ8r>Q|3Ha<tf<FI^36+@l$x&y7)OX3cWEPI*|1Of{%4x!;Gu4Pte!n?iE02g|PmgY7agc6*FmF!@XxGcivq227|FG5U#w-RA*esvdO*#VW%xT&T<C<ts~-<)zEWefOs^CgkYPFkBerDhAR|2O`4=Zq$Id*tMJ6xKVpc{Opy?eWWMvhO3|^4BP3w0WANp#6c}8VSOONKW<}C;m*;?;o%_J!T2HPoSlwqsUn3ThZ@1TOdL!+?SGKeO3Tp%IygEeGgRV2)42zDJBkl#!{i}M;&fc1`f%V8MLAAg5-Fz6e>8tU88WWMZRwW20xuHd#ev^nU8ukCn!}o`mP-9;w?9RMe0rH!FI}_&bPTq4xo_*O#GOuU-?vG5|GI$j@6zB<Q`#9`gm3le-lfr@Le75U@vpy!I%O#0vv0kNb>C~%<R2=(!QSQ%H=94+v~wF!v$A|Lzr4Ftqzu@HS?HZo=WY6R^oj1rn8ds`9G5U^f@9)(DA-V8UOcNdgkvlei{3s+k+UF=)Pn(>CBg{?KMW&oujrjmC~GITS7?q=2<v(<0w&NKREgs|mzq9X51)}zh&oxT>F#8?t6((gqkB~`i%12qsN66*jZ>0nrk~;Cj4J9xJ&H1K5^>}@y_jS<%rk4$nb_(ksbQD#H#WtuZ+c&64S3Y8t+r=3aaOhy;>1Uj0{M=Tf~r<|s~mNcF)+bbB04eY#=wKDRK6C$m3uH!el0o9HR1&HVRJGw%Q0DSa<TyQVA7|%2{p>od>0vILa9e-x_1pJKmke{U4v5~tJ0opI=PuuvOSsA9CHQ}3|cde(Q1#^Xpiln_^i{h3?6)7Jw8)FDk|v4wN(lc*@d~Cq8b=B0PG>$tT5sT>cx%H38UsPxigwI2vTBFID(bHj5Wi1!_k9bE$sJ(Aiagf2q&YJ9Yig+5O4Boj1szO>$+SnMbps$2%;7arteR(G?VNmf?sdkzIj)Z+z_AjkI5yXR=Hk|&^KHX9y(e8@!f|tt1Rd9Bg6P72u@KEqzU|BZK-mW8#vR5h;xp%NBD>AeMt{~k+ps<Wxk8pwDV{aFKL(R;OIC5-7P{@*alo1ok4$$EOu0mfT1dS%b8aLEkBH5><uV69%tAGuZrZ#F$Oq}PG<2Vswl@=Xj%zzj1g3xZ|sX<A}j;n@%S}w0oI6}_23qt@gc1LRwwafwy#cxR>qmkyTfQdUBqLeRO?AixMoVOtteKS*3bihZ<-^I$EH2<IJ^8DL*!Lcn@@U^M)^v-y4~E_ft|NXxY-%ISrInTolwxbH-|JxivDI>Bw+|y?f7hrJ+G0m=dm)nhR)8=F>&H1Ofg-@%=L}Q_``f_zBubJ&Qh%6bA*Ei33Ly*=7$B8<XDfD!vxcAJ{=S7(oU~>j{xf7J|2DDIJ`)o0db#>eT9$n$iQAs%PK?3jP5ix*5c;gu1y^=%88jIV%bA_!XU!as0<&DpkmoS91~XWlx$azvr%}&kw#ijp#(L|W|;Xv%-B3@$ecQl6HFtSvbFS%=bBm4>VdR5Lgc8q)7)6Y32)rp+GxIaSAvs}t!2Bh3%_Z5nwArqW6QQeJ0-O|C@bKtiSr8NeQcOlurl<(OlC);u0Uue`g89U+pcmoZC6=q?5u9DZ;@36^VM2}Q{Tw4tVnZ8eK3Hna56#F@uf!jYS2xlok@R;^ss^tb&j59bH~)BQ_iC9C}CV46jn~P*87qK{837AWkBiKwhU=^aMml+jA3jPT(<hH-Nv?=O{^Z@M$2`!DdK9bDZ=T-87&dls%B!PAx7eLD0>cNHbSsBTK9mktkG3X%<dGP3{F>6_*ifN>vNCN{S1TTA?lzonhZF<_RPt`GfZZ-7sqB}_s7lcA3BM6hCa75g?mNS6ctne{3ww_Lbmb<i&3mbZAt-(1?xNcneWU6D!&iB`RD;2x*y<mEvwqEE>&8Q{#mKE{7EuuqOpCeQFI)|%8M<1jR;t~G0Z9OzW2O{YwBec4&?p_G1SBZpIJB_X15N5dX2`}abQmCsCdFeCu13bKE&{IyG0#yK7zwXgq(Xh#-kB1^7m^9x!Rr_w}-&ya04ZKCL;OR>7$Mx=l8`MwaXj`fXvviJ&gFDVn$Z!VanQXIS!GIjh58$NT3%u+fgW7KE*x~x1i`v){QB<9~(J4@Q@CL#sitkgK5TGEM?7wR4%!5!V@CqV_uKwKXN%_e{i9wTBxKyJp@J#!A0re0(B2Z$Ak_#X2(>z*;bpIx7INTwz>VDYGr&s-3UHgpbdF&#dL51G4E||H+GsE?=;qsdAS@nx8vsS9XP7deMx1|wP@TusISF%fX4a)vDpBFeEf&>xqyn2?v>>2_F?&qKt;b*P&WuM{tbfn5TN!m3ZQ!a@&)1yAu8Mp1Oj^b<ke3N)P8!2CBJ?3OG2l{z^SMCm`0!H;URr);?bc^3+=;CyrA~K%|k<%07WJS69<P0ENv*rre}eX$H+%tLw$c#V8PEl{gmNgh{RnAUj8lzxcf_icaAvr2z^A*NUuJ6^=pM!k2pyJdMSAMSwJXCp9I(g8!Y?@qWK9hCWKTX$R8NftG^?#V(6rSxc@6eLubHq(F~3B1wpT?bPG7RZv|jHPW->h5nElx$`z<TwS|NG3a9G}DD{LtX5=STH{}?3LJ@c`XyOEtd=#KXpX~X6_v)t@rH3}lz#ndhMW5kxGwl6WG{<UwmD>OU#|ygr>+b+gs&aTG>2}^ad=v2fyvwSdJw_{gA&)+>#Dz$uH*nD%naaEUc5gUBEMChG<aRPd906@)N4Uv{r))$+KbRFllk0#?eBAFW2`?gyN@5sbc&Xezgi8?ya$uIl?)i}OuUCV4gOwZi8zvcND<E0sA2QF=r^wS>;C$`KBg7D|3(-2}DqFv`j=@(k0*lH>EJr8ucs{U|a*h;!WUC+ot*-(9z7ryQ*&B=wC@f4x@nn`yfnnVTL7|pe+>djG-*}O(@P)v-T9Ktv5K*;h`;c9O<p)j?Uvo7sO6E-ZOT&#AIN&zqh=lgGkw<(1)uNOQMxv7tdf#Rsfy?TSzaxw>tU7D%kIlJ1USRHz=gqy=dCtWsqb5-sDpAVh1MlQ;>*vQRv*&6~6K?>#4C+EYS5zv55~K?o7Rytofq|74Sr9EN%xH_Ey@e)SpdI65Pq@Nydt4Au<PS+`@UX8AK6xRu&M~1L`Y^osFcXjbtyWaIR{XQ8wOTp5$4(5W?zR0U$58I6ivihvb>W-*x&)ww{1gw6ztR<{32v3h<>KJ9kQzi98a`sWl@30BY<P()fiBv$f}#+iYY3HUjbmax+Y9|kLZ5IaKV-#3!M9s<D=Z9=cR3cohU|mPEM2rz%ek}ZM+nhF4wb(;_wzMTL0pj9>mWVCfKc8;21Nu^_XngxSN)r(K*fRwsPNPyVrUTc(;U*7F+YrYoMpI?!r3su*|A4i!0ZG8AaRAUpafP(qbNWD4Wy7=Is;Szzqn`bo2<Pou32jK`&pY*6r$Ez*%8yK)q8K~<d(Iv8k=iYhrm7S1dvV~SXvUrJvDM)+3c`t*`n#fChB!#GtLujjddx<hDR7NTip+>m~hV=Et1+3BW~wb6{XIQ*5mAc!hGdN0gKR=2q<VkF~H{NO8r@KJT2tGw9}pZpCbnN^86TJ?ksx>NGjQ<p?d;>WdR3V@hri-PzAZ3$%eJ1n8`7t?KxGtA;S7xwrpFNtA$thVhBGRCM19I)uD1R6ggk$@W^6MD&^M2P<k5V^j>6KI4mU(`_mZ;O+<rNMW8FvpJ5)a!t76zU>ijK{&CVkB)bC3rdO|vjIo&N;Se);B7As%O6>4i<Dz+OY@ZHT6;D{Sn3H_MHBGLzi4_G?8ITh`CIVYDx*Cf50<5uy{OLx+nH6I#NHWq@`WzYZ$usXjU>~&Xb(2<oH^N0M=y88aUiRUl0o{stTI#a>4#K+Am_ri0XiglrriGm?_)x1Y44Syrg#OgghN@{&9LpK*7>mS?tBJ{W)}XmcIaUzuzjpMvx`fm29fG>f7D$eCdzcqYZr|{)A5pIAb3wARW6Kx$<eX9FuU&O!y!mU_oE3Tg+7-W<Ex2eTI#)-h^Jb&Dw-eWQcABf}^*rAxH%eU&IiF*Aya3A};JwoE>$DGy7<R?L$%Fk&N#wn5!a5j!fir8-v79eF-mGCBW0#~t<^9W{rs_=w&!3k%QwRPTZ;Uw`$m`3OwsU80RfcfK2zH*~d87;ShA*sMjm{3EH|_|_!%STeb7?u0-mZ2qe(E}~g7Nc*lCACFaQTezd>;Jb(fNy5GVFyn&LcCmlj10IX3K!HV*~}o+8SJKND8*9WcKr_W37^+oqK7*Q4?OwUi6#>{NxZam;m<?X4~*~<5v~^N}&nlH6!vdRW+7F8kh?>=$LJ7!dp-<D+?w;L!1Hv3>2%1*pF5&G}HC$3>ncp_y_X5FbM@gF^No2p|KFHn67(w<+SLuIc)FirEHZVnGBgBm1kaix->crI)irqnA9-vj(nv1!2iF~ytK2`6krIjWiufUvUZA@pKE<MJvlt=Q?4rAR}+1b!-q-fA@(zQ%7s42gGE~095Duf$P1?1(wLz^n`E3I8IDeRM;gK1Ijgd>fw@OdUP3T`j~CAicAfAl1_?S|wg;RTENd3b{9tOWnJ^~FF{1wlZD1|NZ308U{9xg^y=5-H-ML(Ot8)2F-ZJm7lgt;Ho+nS)u>F+t&ciF?p~f|MRt=hKQ9w=&-;g847ihkI^-JM>@#@#`^^+(&@6%JBl{f3_@b<ON<+rbQuDzzO!}U_7R9S4Aj2R&#KOs7(v}ZCI_cQW1LJtzek09&6Of~i_U@QqjlK43}k%)I^bWFHj4bWSKd}_$!2%R%zORxTxmWzC7elGeC{dr0V(w~XpxIQW`pJlBd`^S7ssnNcm@&6G&&Gt?(ojI<SDwnR7E?**tU5^irn4*k%fsxowJmg;SyO+Oy`TxB9uP^`a0A?Cfw3*I4&s_J5&tClkUjK|-i<)aoI4*L(`I+T(^741&ee@~)_Y?T&(^r50>Tk$73_YOOb0ZL`%FqYt3+WpqHqd8O>EE!w2RcW=#DB{EK+i-$N>ir_$aHh$7sAdVe*>>L?H!HbIW_h)FuYEFX}Kg{k4N2Fb1g=6iS7M;pORe2^6aw-zq=UoHWU}Td^i$WK6S2SVHEcO2T-G4vpyWfEa1V-)+uno6hjnoB92<v86mhsn5w!aTFCfbIz+V2(zC3(ylK8jH%sLWFoyHaZL3&n4^LB;1FvH6I*_OEw!*C@oZNt@RGQqckf>k%TsYPJ+dq*D;8z4v^8|j0zuY->e8>G$;c+8>=XwmZg&AAw+vqZ68BjQJ+$md1u4^xH*R^LnnYW;5l<S->rHqkrXTeWXA7Xay8hkEYB*v5d{qt@F;YFV2$+Ta&^AWnXC-?gZcp;OD3;&D-$R*!X!{c0z>P)n(s64eecns_uLPhFfhaQavX1R%LUUAo33$TWS4AX5)Q_}4v(^!rdgbrvT^;#e*!&~T+2e+I@z=aWWm3~Hg7FEsw`R)=nLpBurH`ir46pi~Tp|}XB1pm#i1Z67If1Q{T@`V?jX0L9Z{E|%%OZ?ELi6S|@TW;*4TXT9cJvo;1nbiwlCm1=Vm{FnXTW*c^2(oJgGD#*-cqY^Q(d{IK_vWp*d2^?+jo#{wwcCvyEt%~$cY(vcwbA@>=vzq3?Hwd@V39ERyBk(T&IF(ZGI8U2GR8LiMYM9GQo2eOHu>NOrEAxV#SD*lJZl_y>v)<Bry!-=C)W;3=5Tk-!ck0Cc67~Ssf@SckpnP>9E$k)N9}0+>Sq>%2S160_<|USC&HN;amX-@cY*N8$a(x3F<VaqcH{P>Re$-UXtPPbNBAcEk}4)QX=b=EtNRlMH;V#waS6CH;(UKjWBb~5kmbC(Uk3mF(QoItPCMcUNgU(}|IaV~F(8bo7w-IU<xflYt%iA?Wg_r+r-4vaP6i2Gel|HxxvO4}lk&0!(FXHc%*f^#OMJGx431D`E{xjZc>^;P8cr?eOD3q&M^&pg*KfnV4u(>ZM(FQOk}7VqYr%oR7iPNfxZj=TH>PUyVX75XR<U_Kb~16P%JjJDSThmEjKqG>470U7&NtrH56aL&(xY7D3pAl;Sf)~;`Y~m?IoVMd$JqNg4r!t&r8+yx`Kmt`?protW`=}`hz-iZ{h?MMh(GZ_`;hj(JTIPNE+ir7QeJNm55sgWGKO&Dz3ERbM!U>(tpo;VGQul_MTDM{p0m`S_J@TTNC8O!jR*Cut?edWT<<iE^JBAqx3N?7H+9sW+T>0NU#L;I-#dwFIK3-My;-4*u5A38h#dXBmoo+hd{Tl=AbrIDl?m#BX`HnO&<c5_VS)@=C0a++z~MDX>Xj*wAKb42*Kr!4W($*mVQj6;{4(`6j9RT&s{^+iKiXU0hHIU$w0G{H+M=<02d+qrnBLo5|IuCp58(Rd`d_eSRz#`Yu`yC5L+U%KsbwRq`C%s+&w>X1gDO}&W|N1`F$-1fH8KrZ%V@}w<_2h2w+VZVLg?O$Z`Ic~_O|CBbq7*4S|!mE`cW*!L``vgF{}pRMIuvpykD(cE4SuW_WntKf=mwliEJ3%;99fVAEJ(EG=UZCw9_e+J+T~B7Cy5$ACn3-8Fl+p;=h3T$yk)dxgJto2{=`Xo>QtQ(ufd86LGZI&xyPFRH#&6eH;h~)jx3T7YCJn4Fyr{@&#4$TqT#6iab=dh`_9GP{rwJP*4dS3(J9uVezCkVx4aoJ$mej*FrU&`Z!B%?5RSSRfZ&;VbuK&+!-bDw3m!T|4`lyMK~z}3-9i&y(h7$1vsRNh?TXZQ%%DH8lz%xVw?NfcQSx`jK11V)J=}1Ia?>P!X98FsY@WF{IcvWZ5Pju7E0A2O(#aiTrpQ{IVZEHo>ljAsi}vHBB{i3*UQ_(pa;iZWkQ0d-8MsMqVU!#UY@N+37RZNLN#Mt8y@qxWu@q24E6^5`KD}S+E`&3-PnHWX*(sw*)Zbh8m!~XRuTN78Q)pIjbX=z>7Qs5{@JRpqGyBKh&bJjNjnfaaO*Kah{reI;~RniH<XlRkZ^Nv7oM9i6pve3f}JK?IgNsT$Tk!Cwnnrs$-kAh7@;0sXm0&V@}RjEMo8XiF)?IKbbuVrt4~(a;*F5k^!J`)H_c>)E20~w90dFo#VMkkXdIy^Cz=N+oWtlzqp9iK)~YMX<-^LA<+slm+x?HFASX=Pw!&b<FomLGcrJ!a!dFj;gM7}i>#JXqamR?j{yn~C=~K$S&oXCATu)lAPTf~ZM*UT)V=|PP(7#!En~s9D<C#~4F^z<A(-c9y<ZitB+dwD?ND+*erEKXB7^ExSk}<^!P{ozDWb&6^!c_kYH1#uKS@;!HD60Al>L3jBC!~yiYG0@_<n?Jlajk6Bc!pE`X3cw@^1+qEgDdUVGw;FqSH5Z(`4EJsn;fD#2Mz_`dLNMadm162j<8G5juOO-0k$6jWZt<B958TMOMDElf9~|G_PSD~s;ME9kOeJfkux3Um4E-}ALKopWr3@&AyCg29H@x0v}0-R^<OX#LAyH<w<hqU;{uN{B0m_uFckVySpqYaBk2|qM$(O@JEdRbcYvpAOui2A%(Z+tJxlZBq6Z=L2CjL&8rUsI->KBEG~*oVST^P>hCbI@@q!&`bpN4V^|!BXcWhAJc@insA8&pB8@T+d<#OwB9+FYz(&bCns>swQI2Y)-0Yl_CG9yNQ8k_(lhqOrvHh9s1KDpa)Q&b4qsFcDS!YPNlXRE%x{p0nWMk@LmNr059HK!zn9l6gOJ#2eMpd-kd5)HOIm5=TT8<em9LS_S_!<gV;Zq!O|Wo4kw{$FmrLWNE>`{SrI+dw1e#z+0$QH)g#+G7^u@rY<o`cuYXGY2})`OVpM%%mYMIrJBqs2By#jrtZ!3^&)wDKxUs$B>WMXdvh2Ays3|Z`L>Jo2!krBJmJbAARZ>1WCW_B4cSn6OJs1o;4f=BpbqM*<1@gNG6E6Fdno!%*;hVjr$>~M$zHeC*_7sAXXx0Vl_ql1LpH!FJgi?<H#<Ji#F{X0mBQeE=^}A-P3^06pPI6Wt`o;U9>K!S-}}n`C#7s{K9GiFFriMn2z>@)1Qz&hGE>8e2~USSB!pI0;o*o)Ldj1g`Ee}vK#`@F-ivQ8$DZM0y%jK9IMCeA)Ycizz)Aw4u1>H=QR{x82(Jp=k4wK)}8NkN_VA~6^Wd|WOQ&coekmgMaU0yN<?y*jm*haRK&z`rb&er944~|qscv}bUYf2dZ#>z0$_yP=hVbG>0(0{dhZP!0y{rk-`XOF*Z6LIcNNuZl%y_Hck<?lTo5_sLO}C|n`&eX3H3-*>+W{uB;&TKce{NNq=eTe%`S2_fngo128aEJNq0%yN=(s9IQ2mR1yh!3)rKer=mDXS3+;qJjdtfKkbd9@9$<RYsff4rHIjJo?!y~$s)vgAJsnjj<t`yaWLHF2!N%xYIe{cAbkH84UbY)!w_?qMNXE!;L5M`q&0#+o0L#EGv=Mv5M=IDt1rbDu3c{RJrCjD;$W1Mm-3wVnt0DOt&G|fH{mh$c(WDmGes@u7Z0>EZHrVpIAO&b&SRm@5bG;wdGzcS^iQzxPWKftlPiZ|R21DX~`Fg{=v%a(2+$M#@M*a318R5PWUDc|=Z)-d)E5Zs<7{U)R0<=Gj+7J8FTBVq-Pee?#YW&9vKGV@Tt(y8fl>r1;!bm9FYf6nZ#;w3%B8LW*slPO>$v-@^Pjw>-=H_Uayv7>r0$;Ch06M}C5eQoJIy&bnU>XkU;yyF^n8^bZaL8xq!R)f0EU4A?L&J3gj?f84$PRmyW+Fztf46-+#vKnFAp%jzoD#{PjchlZ|7L;-r;+akTU&QGE^XfAEaL+-p8;EfMz%KHIS-H8a6|R)V=gl|_mJO#%NdTp(|{bl)cLmJR*XgEab%vbuoT^?DJ?_+2Js{ygP{M&5RO|Csh6hXq=V84oOVg)-qgun9v>ul>Jm__i1P8_?(lhuUS2$%9@3aE{~m^q<S-wD_|8>5&K6%)WPX1Z!`UtfPZAw}jtQ#a{~)Gf=+;t2M{6mfoGUU6(K63gSL%Y`+sEVMffvVN-X`FnT7%P3=>=69NY5a{N9GzViG_J=t*SsTrrS(s4kBKUQHRgM)rhMbh;MpwpqW6KTaR7RC=&TTvGXEi06EJ4pSXALZsWM}MZZcXS(65=7DQ5{WK)69f=NiiHD3lu#SX*U2LziWA`qYfkRmPfEI(o==XxeN@yXniH8+z?=FY#fmMmMbWJ~fN_X|j0VLr*-ud1%9t_DC^nd7Y31kk<fy=y;zo7gFrb=2OW&?_$3%BPm<e9EV~u*=TFFEygoTYPBLjjshVtB=uyaeSWnclrc8b8XesMc^A)MdKSQP6~d-=`nQ|<h3C}>_;;hG$d<iq8pU0kr-e@V`mv*au|?jG~Q6!q8h7ex{w|I;OHPMo_|Fy%Zj+8%dCud5psK&S&uqKxMbFZzKunb%&dthn7lCA)+qy#LA4QTWxZa3f7e%k+KAj=5EdPO7_CAuhsFXSA|nHz-ZV4`?}kCt5UmpDcfYtbUJ5YVw#2Z?(h_@ZObvSq{^w)KsImYrA{_zEfvB;-KrkNRR9UxhR}%AUBaVuI!c3;Muz!DE%nB*n8qk$ZcS$1Z1MsXKWlvezOH>Pa0Q$!<T*yh1qJ+a6DuTpp5VaA!yiH$vv%^f2MLW`EkdxyL-RulcFHz`@iZ_Sn{Z_ox`-79~<)x}aoF-k)zO`w}vtC>s$>R1wXsZP8^f2Re4iSCl%wPL~Sx$dX`s3FYJUx#ru{-iKSoC98S?zfdtgI&tX*|5OnwEUllNOGrV9M^{LbblmB!FBR?;=i2MYS;jS&-`!DrX05T9gm#iISe6@wAM2m)>G%l8Wq+xu-}Wl!J2UiT<Jg9Hog^?HgK&f|)XSpH@%wZyxSvBIoE1puU1KOODQ>JD!}3mv+@Z<S9_8_s2;=#|Jq>%E*pfe6ZE)?za0_)NMr4Gm+{hca(}^%YRmn{RF(WBh+QQA!?1-st6u>1l4>|SsyA}{W8pRmJU`#Vc!|w8M)awX_$;M$hv37h^zW`Xt2zw*n`Qyw@)o5OPp8T6zD`bLiOk(?EKbklwN^^;;;E>wtt%m_p8Sf%zDboP>FVro;#1e!sTOc@fRM9ZErzGOCxVgIX4)L32*<DJK>e8@KZ$X=rs4j5#yZHvibt!Q>E=ent2CUxsKHFj1gs<aP4Tj(;n=nJ~{MP2RrQ@w)abDrQ}KHZ{v%+g>PMDLvD8uRj^bHMyAKUYQ)VPe~`ijI2M#gq$HUZ>fJ+-Wv~H|vd~b`?hes~V+%;iFQIeQcM>XAkgN2Gv^9Hat~3`Awi$XKU?k*Y*Y*)~WWrjfaj5;~9W`g#Xv?9Ebl_97Gu>S!Ck+kP8<ER|a~|>^G%2P;JPS*UA&@aHb*-|r{*EeRL+mh&Zh`ziGi}S1?t~W70#hjDUSHj)r-f{;SC(r_HPqEq$##p@mQP?C9JN{qdq$C5m@C-lATjZW7t3WeJ`(r)DK?ydcOiD%I7>%C(I}Ty|20haT4klW)Zj){>D<^TO%&@`U>ED7SS%JtTS&4?gTb}wy^~(&JPRDGs1M-AN{Uy$Ua8-xdU>Yud{mA7W{9=otw;|I?Jivk{1?6LiwZlQVdB|NXjqQZ?KEoBU!4znhti<w%je+LSK$3#t1UfxMSDE3ztHer(!(H!odBUAFg&uI*uD_*mZq#x`#A9Vb{FUjk2LRK%?1wHPAOPP;0z~l_rVKf!%6uJ%1-@Y(1zKe7Z9!zC{p4IdUQ*6JPD%K#W$*}9()$#LQvQn6vavSyMx1CtR^^?p)sHdQbmG_u@=v~T95UJ_Xyz-%I>4%6`hu1j7D6bfSPaRD>KxZ7yVrSTFRw7ir`3~^py5Er22G5jF%<`Img5a2lF<s0gl|VnlVgD`7pVYnUb`^r;Y|~BwA6}K;?q9xG8w_s3;C3Kpm&>4c!0_Lj-(dVA1GNj6roK&Nl)6zJ+q@fbNXih6lM@P$BT=yfk*39Za;vfRr*6XtaHP+a5&DyC!nJ^M3l1e@^AGbKv%?juu}Q62A<>1Q6RQ=S-e*5Qeqx@XBnMw=4$cq9$d~zlh55O%Szzwz}e{pnwgX-lTMi#PupL?|PNfb0kcAslw)ohKDdn-*UnaHYb^5w|<0cLdvy^8^q@Au|%oa33?NcD#E}|7+;)?+(Lp6pvGyJQLyJTCovyuC*)=3Yqp5Q%Px1x8rR0n!^C6Jjs6V=lzaTVOXIADVaD@v^=uIh+1sWEKFvfxL}r8Gji|vOWKvTfn)+=~W#kFxh7mPWJ7x0VGjj;#A)5LQ>d3A*@S;9maRCoHNX?)lOnS@>{*l*!xqU9Y2rEIZPv!l@C#MK!cTUdgu|$3hswx6GY?W&(i`BO`!xkt22<g|%gaFvrzW|A;^dmdNU#|ayiSBh0vgfX9OqI+Er1&`o#7v5@Vi~IsBt>g;E8n0k4HjQ7&*WLOQOCi@$_Q0yy_9Pm#tm1tUe3Q|(sufK-Ka`MX6!$)DCP$4I&a#$?k8oVi=#?4a8qWFJd1%aF;<_(mrmf3!hF9vr4?nNF}hX)Ls0PRrP17OM9Eice7<tB`SlA<Q~`s{0+wWmCgaRo#ps*?)&;E65WO_Mzcs41!ftzL6;^0-F@^74Tj9Kw6=ud)R)DT@Sq0t#o4g*LstU42ShP7$;R7zDfbD3Qdfu3MZH*!}Xky%jT}ENo=9aQz@m;mY&QyVe(e(|8^u*;1KznCgd>z+GZ+?$*1~5ZbErSyVbT_g#2EYXp!WUj8%K>MensC==5f5Xs>I9xL16Gv+B31O185pyE5u-G}K(PTms~pC~F!SfHEs!py6c^x;yv*tXS%GW}LsIKZn>8)*;RTN7g;k)hu)?d?xioKOqcI)?R^X#APDiSwKsq1!H3fKz+@b=!3CFbO)*SHG6_6V$TU9`AsdE(-7+q0-DW>k`NrCILwFHEc@#Tmr0`jnVlb@sE{1O5}H|K%#w2@m&K$`IXex4TcY6{qM^WyYJoxe=QfbqJ|*!FaV02zRL{f({(fNU_Y9)MX4z~u>C5FiWZKZY+%(v=s3^H*<=YQcl!Z%tbCGO_!8%b&ww$1txKV7~@~^>F|e+7YmR9Pqv&EuRB@Uvp9mY+bR@(am}_OaljkgB~&<`9?=4M#>-KgApWr6Z71tKCe*~!ZB#`6VJ=}JUgRW)nN9<tauUa&EGYw{n<GJHgL;hw@@%yX{^^b7I4aN?-Q<gm>0pVX2P0)5$7Omz$#x9%YuNvlMnJ5`RmX=r(&wft;Zg}0}Spb+r&sdirsZQ%NN3H;RQKuneTxaO&DqugljtPDKPw%m_Meo=%|AnD&A@y4G{aDOKgJ{s)L^d%^sr`dIKjFZh~l=3EhpgmBz;P>%eKNE9)Ftvs%|&Y;2Itd3O6*#x&#U;^^`Q(&?}`X+X}{&Fblb0ZH#2wvi;ryE2WuQCX*CPLLQC%z3^@i^|2+EibiJ-u+D({`2p-V_zWHXZ%6;BvR^~Q^T6(*-;sj=T3~Oq|j-1Zx2+Sj|3d^L{QMdZgacOfYO0ewca<2FE)eT5CzXQrNkD{gLBI%cy`$6LJD>}@F14ZB%}=^j^Div9Y}~tDm75nDXw~#vu%+{84b)c-_8&Y-{(j~Qig$%wUQk)4$NmKAHlRv($j=tGtSLG=i~&}I@3%5?WkZAigUU}3OLAK1Ao{QLg-+a&0gZko_b*no>!aAR<rrq&bcsnrl(4Xr}_ZVZ4i3q`G<%=qTpU05N;b{zdnpGjI7y*tAGm-5C}hE`Xe}w0lM=?^Q}NoO&-9bi0}9jW4@)>Gv5)yj){xISZ^t29HG3G0tM#d1IwY#6Pm4t@mZ|iLJ}X)sGg*_v`++IpRjHao#nSiPi$Bq>gInZK$eFkE*S6u)k?VMpcE0%&)10l|NQTHYMy^a<r=bJJ1-eCXX@40TFLH}*7w28ant$m+#}xT0;V8MxHSaJ>K@I+Bl#{<LE}rRGVzpV?`!j9{3ve=67<o1y(@f$N@N#O<qPi4rDtCtFh1-NtgrCkl5f9WqG}onxXWG8v{-{xfMKDsQmNOfqu_POIJ0|1R)7OZTq*RMccR<L=|JkDQ*t=52TvwAJj;2}C|_k=#3Y9S55fRSct8_0zMs!au?@1IJ6gm-)uFPshJHOP$CN_b&<;Sb==%(!C4~xd@T~+v3Z4rc5vpH?u5IEhT=3<e>4BuRDAGU=Pt20fpM$fP&Ru^*ZLXip%zv*b`6JE6Q-N~x3^fyXk~i?J+%e3*ghk`(E_66$b0aR80uzIi>URBbIwxbN?`&YF+Xp8EW!cXwq<5`91C!!gwEhfv@!&tcY7Yk|MPqBIxS{jBQ!HVaBW$Os;)Us{S6+G58SqTrfM-UJ_(yypWg*~9tW!Z+ckV0rGG$N?aYeqRA~`_o8UwX|WpJZMSP=)W&yNW9jxM>UcqOG3a0vb#dF2D*V+;t7kOYjej$7cb7ACsk`8=F1GI;*sbdLG*b@(O6cIq#7LOx2MPzZ@(qG{>bA7B8VNSPBsP83#wC4Wp_{1BOqF+YmwKV&>RX8Pd;VW62s`8^)X$Ak(`aP@fanAZUz`aLAH#V<8F+uN$B&tFMQ_(V`k28>VtEq;x)ufBkZ{fHp>jM`7Xa;o%jtq=?~C~vk%Xlx;476xG2yCN$*LO#vtkH)@aR)I@Kv<}(p%zw`~USfE(b2?o<oZf_WGu`wDCtI7`v8@sYG6*tw!P7H1H6Rh7f`M_H1!QJEOBFRWQ$C#8Je>I(Hzlq%9vqzVBShZi_KWlGX8U8h{DhR4S$^0`j|CQy$`g=K@%-1Y89zJEb{o&3*D8$$1x17S);ntLquCQr9gm}kxVl4%Wnly8!O59VuuyV`vT=Ha3{@6t(uDPFezv3$oBrd;TIMujPd;|q24Q|FSg*JbQYjh|R9<qim<a?|DU`+e$@(CVM>^;$vhzuGxS}Knd750SuimIvmeG8Teo&XKp`6Tli_U|dAXA;_OJTaustq*n55%^cn`qd&Pto``bC-P9HyJ9&rQuG4+b`G6FlQpGn}A?g_a&=%8#2p3av?Mm$dvC2ss5o4F0pFS=e`#G>LXQQ?wSEpio9CN9nwvCj~}P<BV3{lKYxetsVpQxLmxi>dn@u%=7Z3!C)<nosE#Gy$^?09x7DbFX3Q#@v{sjE3u$3x!!nAx+5(zE$;ndLSghF=P-A-kKoFx`TkmTwI78+2tb%(vB1Y!QcDHZYxvj1RJS&vmMXVd;5lS1+9`!{|$Z~fY7wO9rhDe(ek%Hyg$E1ILDaG^<!uQd9hM618Ug7&vaW&jt=4DSZ$$c6#*S<tC#c7nm=RtI%E%qhvQnD7<0~z#N@j-Ji(53sn9~6hL)RW!d@ZhklX4O*)&6O;4lh{eC8sM1JkmTgBtOhqxe6y!zx0o85^{8Sqb;k9>VO&lbh9}|Am|;dS&$&ag3$1mM-Qk4M&$2nO!J^dgwL^XImy$7Y6f;z^x*w6C0_?ZTGR*{n^1skddPrx`tf!jU-4B&5OYhTfNeM^|A|IGK0#!w@H?P|HaMZ)%#7rLdRi%ETw$cb)LC2mmPVIAb)NG+IjqYzmiNuT-Xp{>enmHA`90nYdp^`immfcwmYT}Ar&psz!^Y?5peDLfW*HQ^HAE9ML!*c_hoSO>Bfc2I9ifHlw7Zu#45;!S<_dh@TAE>VU;n`oF{nN95G+4%3y}Ia>r@Dq^@0S)fmf%KSEN^aI!)Ibl#jf0_Qq)!%F%c}*N{_5n>gx@UAd_eC+x1xd{DsNx5`C)SHXofC=32U@gS#wIsV)@HxOVF!V!QW8LxE)n#{r8wuv4&l&!sRe1Ytyml%$MN=w2gMjiU~5iXtz|2UL~po-#_OOT$LWr*sukZP&%4*6qbNA~|xu5)tI<;1I(|2c=f83*9MAR1ge4_(Hldh-^>}H^U2XtO!rA@be#rKmQ?>k2ypoiO_cW%Lx7r|0!OePo!iXqB3aA_l0oF2RZU?dH3f(ltN>#cq%~M<&K<1M_ruV?NgE+%R^eEANu-zND+dWCClltudZfS7eRx*Ev&A+6F@iKJu3$5t3e1IIy?)5!u8eqa%DY$!M(LnU5^5E%+#xt&>#G)vA$UJ62~Z`ve|+zvZhOSd5U03ZK>>-(3*rr;NYkgiknkgez+27^9N<)bTEsrS3zB%1NLW^hu{ud<JN<_`1%<#mB%dTfP`WkwKJb*9mE5=((I9j6OGmHE$1p=a<-BkDvh?Wv=pq?m6y}_28*V(2Lu$56!bgA%uoDgPK)qxj`=AsurK_c%OsPQS)PBU+7H^;EmbqzHA7fKq2OPWRBD=NSqlDXk!QuQAYijvv5gE%O2jCM7+MuEWC;AO+o&{xuD=Cu6*ygMtpZDYvvRApazpp`_37gGp8IriWcqdV2A&VH%drF_!p{@p(jODYHe@1o?>@{~ptKq98o>?!uSUuA<x2HZeYsJ+^wY{(RXmfQ!|nW!&;ICmBc)^pR#!1DwNkbP>)Q5j1AA$WMKsq|FKABu%&i-R${P(iVHeXgNDB|+%6YHxR(-vIFR!isv|7)+9?cY)0?tYviH+r)rY|9Xs9i7EPZW#M6w$;<&Y_v9I{BizcRHGhE=RM`Ty$mhmc`nE7?euk`Jt*R<=HYMoB1x3SNvUBsx8;ngPEzRsd3FaA`pUfNI1S<+6lMH8CninAOddNf~2`0J?O6lHbSqJfMNHk65^m>DxV(FMOCdTZypel14{qkSX4ah2Wdg2a=BlUsw${a9Y_Qf8|;esEB`$z<dk02ITb=N>8S51v~-I}B|GX{J8>I+G`=vqqJvqvGji(uObqowHFo<F(eAgDX*_gX=4fodIzr(EH6s^7^&#=yk$Przlm1Ccb5hy^)>98uU}e9AoS-@Eb;{Gp#j8<Y{Z(|j507P)%lYwrO*JwJgQY+%O4dS@zutkhf1~`JIkAHFXEE-v{pF!dxU&4;>~#+2ZTr@UD31r8c#%l06TZw>63YW5nxT2ok`$gj3J%S90glDBkDz>6fn)oQZ41M7lV82i=q%%@Cl1B@18<?J=Et<+B566#xXL~Mx)~nE`!GX@$3W%BGxiT^FxSPW8bMbqJsG704~{y+Hq}!a3<m}8ladsiNt8Qp-Iw$q#!N1U)Q95c^j6NWol<u+{5$(eKb<$&*25--;FmgBlr;*M@P3RjFqtZ`A;+&SYK;By@NPM9)cNi1al2AsZ(PG+^{ov|&VXZH%v)^3LM9p2v9g2hsDiOMuTr(6xRV?c#|^7*uPwoygB#d}Qd89iW~OGYgrma}C2|<O6Q_1z9cgu97V!IQ1#x96My|jI^@HH56Mi)Bc-qnSoeAR#lw|pUAGAHczf6mwTP9y*)ol*O>Fy<k>F8DW_%IhO@3JT0Zu`tp)ulYCD(c9OJKI^pW7Hz4EyT}K90a>8x0F>*o)E)aUiovw^vNywj()$)zm?i}q;rJ)eu=AY#M{k5d+>vDxLCbKZi+DH>Ow0@=LnNO%d7<W`Sc@Rxi63z(={alV(9hxD=yAF2LD~;`<Cz8C$xZRL22^lqwj-Wl6q?dtbWZKG?>=Cd%*h&x7&RoD(XW{b}Zk7u+&(N1oN5kl$P)IOZqb$_xaM?-r2RlIuyhU%8vQ;Tv7GLd=^5AbofGQ{E1%;X8_|QZHL(t<pRfv{aRcl<a~QVp)Pp0{)S(sz)lizxP0BSZz#U*KKqZfuLdTEH-zSIua~FBGfOrgKf6D9Gl57h6z^j7nEZl%!#IZf(PJNSw8&RHMSRacNsIKEdl5IxXL<Qz=UH;b_6PAu9?_rEM|dOLHD+)A0{#+uus@9@XCAVt9DrNbq+2?m?;Db4K9K2DakI4TZS$`xgJ}ops&biX^6xqQx%tl8>iW%U!*n9-pC0yx`w6gvaDF$u$*eER4WCd$7?I;3l4jiNGn)Y-^o5o8oz6;V%#gR}ABW(J=ZZ`6h4Tf%gxA%z&|yU7$SC1TDnqpr+T+Ccf#qM0uH+@I@wS$hRLzd~sH^=TAems#p|?b_T%Pm46M9R!agvTx?^c2Zt5{l!tF(uHR|-|WjMX}2Tqx<H7NbpbIsFxQS%GRE?X-sup$4$Db@o?Jm|#H4Moxa6!csmDC3FmeLVoI!vY<96mO}m_ao<>o^|N_C6`#=c;keMbrs`9QXZS&xEn6Q`?DErGRwK#@MccYz)P~$4CW#%)!zjc5QS(zr2d-b-0I>dboy>3`I1MOclq5pS6dzG3L{7w+4CT$+^w#ajdJoi%19(joeItM3dZo4mEg7-_#(6dSt^IiSDBD4JLoynvYW;`~lD@v0qV;ZurxZ@bLGo~r+6F4O**^NYs)$?N+V7$qjIu2HEis3(y8OnH;FFrus2#udT*!IEA%+?HDdWKg2!OU;<?79`c9U9jL9GzR3~atKzY}sBj#-$=Ro|{IY*5DMPiyNpW3n+(1}M|cP(t#Pk7}*91UqH5PL8CSofa)aq=JzhoIuZ*eJ4cHc-Ga_2bY5M6%ltSFDJW3k*A4lg3v;miI^%Ak`|-{UrYzW1Vu?})RqW%EOoHt9q@lsj(1WP=rH36R6-`mLjOR~AKy_tp^i~{K*#n8e~IGDx13(U)?*cr$YsPy3Pbe+rOX#i0=jbfwdpIf$(3^^0X_R4m|;U929nd_5rZFQy}%+Wp!oD_RYsfRu(ZHHbRJ_uhmj=oFC;=Mnfi%{VIXt;io}j@%}U@Zw?MN_l-?iOtn*PuH6uCo1DH<anrUp96OIjkDeI3fx{QoqXyX5Z4R`eT-y)Ic%JlBe&fN5s?^pc#)-0C1ru$YEKIF32jCHIriaV#6;)^+pzy*@+Uhj4|2=ti?kV}oIcWosI-I|>b^rEGu!u_dN_t@_)QSWKg#3vEQx99|`%;H+BpC3JNYnEW?*+c^TtJVOfXr=3@zzEZ9PW*7KIo$WTvD*U%cyaY$?cU$nE-42;N}mUyyv>K?>W_D}eXf3ZL3t0*7r!3xOK}G?!pq_G+~q6FZ=kdMn)zsmPY(Cd!7wOKdtHv4-cq*S(tf5zT(Xsn!%2IVc)xg^7$6E&>xMrCeJK;!y3XVApb@}!-R=#_GLp^zCV#M{8J42mEu6Q!9|*v(!KqY5J`5Z|s`V3!sFzNh!vkW#blFpn*K@Fq?yG{Vg5Py303WHpU1^Hrrr8Q|i?VUvkyBIV93cQ^Ab{Ab{!36<SqymNs1LGz<Rh(9*;*lahidkXwC*0iz&tgnxPeqV_)i(zXp++V?b0jJamne~cJP0!)>dGc8;#n^je>WC6Z98dsoMexbMaXIDE<WfI3v$x&RN7lovo6pw1|HTW8l9>`<>ZV`8~+0r;<)CPjD*a3N#v5$~a2$U=-1&-`pPX<2j&v7x_nZ)+U6}i;vGiTW^60iDPU;I&rFa!!(QVcPB~gI0G)p#@WJH6uEeBy3GBw2&;^_Fpf7?-i+h0n2)q;lbaE8*?zq+%~rcSywi*CG*8j;6$2|DBJ3lUnuy}#2!;UvpNaa(;ecQ|-&kE;$0{FdpmeP@xL7C^+k+##x(rHV#_>IYB~?w9EV;u})!6t)eXgw$!pe?=%5uwp(OrZjSub;_o5V-VpA?elbrVWW4h&v%Er5x=2ooEm&=6d-PERst6^kVaS|2exd{J}n;R4^APP1)C!$w*!zFvNH4C-Or@9yCN?8AxZ9l%c8Z60-o_-$dbWlq;K?P>}~b`8yl37zmkv)^x?7UJWJu+{j~j-f~%vH`v<Lzgy?DoklEpsOW?q>iKPb{+=_$v9##ALRwqGZ11QCPw@)1?eB78u8!7NTM`ZkS<O@{!W6Wk!h6Z(nl$&P@qOPkGE(D$Fy^0=+O>t(=yE;mrSGTXX@t(TEtBvjdHHT>r?YIpUSRGv<tXoXqG@wbX>zobyU4v9M>57LkHH<V=Loc0tR(AWYawQU*~miP$1>56pJ8+wqq<hn*RLm5tiaZ<Vb8bDoxGI89>qaUGepqDL(t`pPxkL#&!^_W<*1;U41DHDG&bq0SM*OrKxD<75ocZWJ6rH*Tt$b+{91Hn15ZoR7e}WFg>ZWKZoY)UYaju5<5pAz4ixjGn}^>RFwX<9nNFS{aHRix>3e$5M(dr29a*23pshe%~jE4zW-3XN_93f#FY3n(4Z;$jevwx0-<_A$%h);SIdc*Oe;w4m`yCOT_ov@YQWG)%vFe+yb81cx`vPWz_EVdl~-UXN424gNs$Q%Gxv-@37J7VUcvzEqo+oK^%(&iHZp$P92$XD3Kycja1)xa{X8)&CEh4@M3Q%XvuS$j<xEMYgXZ9NS^c_cg!cGClRVG4i2KG97|Mf4Qk`C(yyE$>O4JL9RZ<ifA0B^*{Iu*aC67JAODHdLX+-={>U=t$HTyt_6I7DOh=5V8PUd`qhnK7CH&+)^w}5T&`cbg3pR{gEedT`g?opd6%#*!mkU+bz=Q{%NcK|yW<i+8B6Dz|@tma`K{@bFW$%%Nk4N)+LA)z|f2Th2jr@QAg(FpQ-m~^l}TC1B35mFv6?Y->-!-Nwj-9d5y(`dq$#5|+qH80;ib4mr03(2UEtqWnR%;!vfF;eHsci+=h`GEfvNv?U%32c2YQ@HPd^tAV~v7}*rb!iE-elE40MHiH2ebz!)uySUKa}8NP3f9(^moBZmp$l9O_mg&CSJ%2IRTnV^ZZGKvsv__pI7CP}E@aJ6m3yeR`Z9i~u(L`p<+EpKs&v<?%Hp0!F3Tf!zM5eH#CskT@;yY6<0^^Vq*?qot<k`>Qt@*R>r}O-xP-EpOJIcxQ$@_>ER1|NjiU9EiBJ(i|C9IEqK>tFm1AvHi#cVE^%TwKMM$}`qXNX@gRRt)YN1fFPc_VG(I$tt?7O9-L%ajwUOzlMK)i%*vbWvN7FJ?`IX!yVhcXhd?hWL}nPZ|L7;+CwWr?oEbvQ7H^l6uC%^bzd5jK0J+S+=_OzCXWVP4H2rY#FFXfl=*d7&X*r6FPg5UkUYVS~)bu}X-;3uin`F-nmf=#MKYW<jQ;X+JmYKg%6`VT>{Rpv#*lDZpZ_(b%XGk&_%EG^ZyP04s?7IXNN)!E%&{geau4t?wxKj`Vv8Up2qqs8fMpK?+Uj;P*v$DEb;Q*M~Y8Yp2UFbNq$(cTV#!ow$GI%3N5ox4Wk+5~<cMYHCG~h{P|=%bceoyatK5F6f;s9i#<;CF}-s0Cak)B6p6E?KEFpL>3B4BC*AhRCSjvL#R}-1(8RW;PpW9^4Yopf)Fg(&DEtvxQ*iKGRWfZEE94tmYc_x;Q+#Ctot#aTN%i#mJ6$T(3zWyTm9Z(fJzP8H~kSwpoacrHjj(vPJeZ3HXz~`g?h82T|t+qFwLbU<UlvxSiZE-9KL}<5$q3f@+K3^;FYuv@Je<4r>pfhQ*Z<SaIb*Uy_3YMer<M~-fZT)5v~BOpp~E~(;@illqa)+QZ`k#_?-fl6_KEiWsrmP0%YDY8j{&Sqj7IUpA_v;ag}<@|Bcml9@!0Vlpde282hDn5UH}rAx^Sv;Dp&gu{a_I(x?f7xYksVB<5f@Rb&T)Dvu5ddA!+*x-!GyJdLIrCpBy&6U^(<)+eLdp&?mmKi+nf!V*ilqIu1LT;5kiBwpD+asF$g_*Tg2N>a6%wlP-|Vp)@Rwf8-FGS(YCct23x9QM^j$gj9Di<|S)GgDi};xhHk`ODtgT)H^NAWkEpkk7}deO_kXxa|whm43xvB5QPV$UHf|z2UNEjI9bZfx+97&1yY1aiAz>?FK?i<;0*O-yv0ckRnybFg1Hv0nD{rWKJyxrmB2HJ2C{V8yg%bS4okbnV%s4gGWRHsMhPN^@e7Qw3RpW_J*;B-u~E~Xb^gR?K<H(nLWtu;0XQ1$&3WuLZ^&uO&(fVoVW!e>XJ9~sL0ER<0U30ZlS)tK2?0Nz)JrVj|iqTCcVLXb8&TH0|f<`u*J%Hh2twV_54N31YSDBmY`8=R6;B5#^Ib;0%qgIBQLGPWSr7wtCbuMo1_8R(C>E<hlfMtaTvF%b%fT4RLnWy&8uP`jYf4nb18(3lsafNyMvgG34Gbm$?0N9du)aW<22l=ENxU_5}-AV9N<MXPm?W?XWO)T2Z9R1NXB9_#<ubFLi-2JwwXH(55Yk;4=xJqpb#d5Yzq_+FA=h_s*1rAq0Uc~du{0PzYyne`+~8;u|bRF2awN=7ISxK6e2u74`;*;;y*u`Suu+0i45e~k5RaPbO72l$Wi^?F#=&F36_F{QOC_4x`xr<Id7f(a(T*a4Q6F#YUVNwogIAfHe95)f#1N0d!1u_@#)`pc1wpnN}_;9>0#xgS0(7tPMn$F>@CQ93InwXc!&zCA<VKQXh0WKst8DfYBBEg`nOd$2A5*nDtby9%m#8AIn5*uezvFZ{Q&yYlFFK_ktqz81dIM*__Ewy{)NwpXs)yU<06NC1T0=tPteeI03VOgOw0iRr}W=au+(hb#?VnUBT};>P}7AQdb_<d2%1(hTR<74j?EYiFqWa3EfrC`E7eSup+e&WMg?uEundfuVxp+`?BX!URtRKu^w4fz=nz-WRr)lp&?!sar!1WLtNrQ<zgaAWm|rBao73~YdZ!3yQ;tu^6yc?!HRX7B)9FNKScp%HKTf}Os=}ei7aAyT`Dh}HX_h1w&OP<MfW}~U+*5aoua`O8g4FC7YvZohEAa37>Q5Vfd#|dYn?A>y=g)thCmnEe&{JP)sKZatn0hS7dWn0`r{=kZ$xhMj2(tUGRYHiTBG3FGg@s~edRH&ufTFOI>=EzP^-}iQ$e9n1wmXuug$3P}M32*--;nDL#D4OzcAbIwB(e<MmEsY;m1Y||W1#z8uT~nhH<qe#W4Qu%`pp^!0@gsS<E73Q@|6Wvg3k;N<d#p9hyCeTFP0{;d*!v)c6M6lNRz-UiQkoiXa9uQjEpmZnUle-xp_CS%T5@MoB(;s9-;CeoR0ciGu1)iN<3sZ(oAMMr=EPlU;8-wvwtU|2%=7~MBi9R8w^0`uUO(s{S25CN9{jQxz|Q&HKrA-!5fkOPYHq>k+)$4Flm3(6GA}sPw`)v!TnoIX^ZmHG9G8Ko$m3M4>bm+lymt4zmnwt$6OQ!XX{t0Tl}R${q@(Q;aWIeblLCR)u8D>fc^i9r=YgTG83k0|5q5$J!uN(=P^+DwUHk2$TR;hIiKXUWO}EST>hT<B*0Ur=3s+kRtbuj7s(g4qpJ&Z24#dkwuFm{?X;VF-QIvaij;PCEETgr5a_p0CeA7`&J;};qY@jZ&OKrR>#I;3zhzuKPsWlyi}wti&Jh_UbE1^JRYBZM{s>ZkTv=GCuB}&A7OIvIqfKow)(#$&HxD+sittvBacwIbXkn8m{-CT3gYGY`?bTNj8mDrny-T4fDl5n+ofYgikCQ+Hr3Jh#0-_&!V6f9N(J!H2_CBHwBcd(;LRgg2d$TjCYu}nm*Eh9$=u|q7^<p?r_HKhTbeQJ)P*-9xlbRKuK|L{QKJOmdPwEQHu5~%xL_TRFHCUwqPbtKeS>ZZ%ajK$lVFT{lWxth~>#W8a7E2ej*7E=g{(du7spm@E=-u{`+dFFZQGYX0oQy7M^_rgJb3F*7seD@KFPEUDTr&ERXsT3n(8G;+O^tP(4pnk9hh-|-=_`}&QZl3~k63FTV{><?!Vp(iXS;3amvZLB08cwPwo>rY?m0$Hqlc5`pVzT8UJKfA<9iUjl{qoTffzkd<6`dvhyOGw04Tt#CfM#byRH3_@6^3)C-T8Td9$A)&>3Fje%6EFo?Xm$!1Q#3K<Ys5=fvpr$NG*lUSq!#c!by3?`F<;jUBKb7@*_D+E{sWW%Z{kE>odn2pa3r0jaCvV!(4!G<`+~2kv{gy~}q`G=s4oHoWc=2QX=8%YQ=TSLDW}xPv3TZIIfb#b3y?>5Yx?IZelfjWOHLH0D$CU01!Dtc$b|KE)K7^`Ly0R#ma?V%{o=mk^dnXF8hE5fd+>*;na-^cZh=?8|#bL;@<<*K25I!c>b!@gl!c^x_YYuIMH|aB#Gm91yLX#@***5i&K{@#csftRWC3>Qp8J1A8M5AT6T*I6DJ5FCqY{^=81~!6scec0ZZ&MYo_~R(2uts(=_NkJkK~M^4{3gLRK^s;-v{;p!WWYW)@h7*y&vsvO_oh8io~J+otJ9Y&Riwo16vb#3>?x?vH3Nylo1rhlClgAX13#zp4MZ^t$4YE|r<soVlKN*_UCxKWL3ep$<w(ETPJW$2I&MWN-0N5+}!xu^S_l`n1NPeD%OgLFeWj`UDLZm@$>n&NHYZ%(#`e!=6%)(;VMEAJxX!Gq=@I-w(boB18VUECMA^V6t=v_W@Mj(4D=A1Iv1L#0oGh0HW)^)OQ2*v~$i3i4ETFJY#5{oZK+We$HH=lpgoX|50a%pZY07sJJVqgpR}<T!gH<_U?Wm&tO{Y1)3TwkZQP*c7ic?)hMDMuWu$sQjakENFswhY&DuFW?^D3ARDo!v#9(wh?C?gN9{dv3HnZExH(^vkywvz?H-N7M@g4g<bc&>%hw>lvx_0SBjTWS?^J-7opAZCKktf6Z<*d#H`oAy}IW8I)QR~D2I~@O`e;%G&hs03BVBMX4Do_%S21;%)ltAAD*_|JUGlTF${VWm9nQZ6OjxLuhtQEo9^~o)fLkpm^S9^2W#*h`c+yScXpko)1xAhedRnaz0oVvLGSGr8>^WW!yTsPS<C99Lk)c0%DVIB@W5H&^?kBr=WOyQyd*`-z25WyYl`DJwN@-pw?O0_Cprsl*RGMg8(h6~mEIe@x-V=|pS#k!oXlQro(p*B)>SkudJNBGB4;e;5AiE8;t}MKTku&~`Yem!JRip3f&nGNMFYMR5X^_qKjk`K0;^`hFTGrA?9TSp>FFzTbKeu)=W1!XG))ucSoGC|eF}emM>B&H)xbL}s0IJ=>E}FY?1A`-YriR%g1^2SzGGD||6K|jYM_Od+#23A+t0pM1$7iUrwxVokR1wEcQkSktXXD-@6oqLyQz`iF7M`6UmI;?zB->flm4ye>znMptIh_>w<w!l@T=-NprXSAD}M>giQ^L}zEOY+f?Q(tqKF@se6vB+Pb4^CzkOH+LA$nGslOu(CzIerMw}+8w-!-?ngr=^K!S(UV1aOOA}XCE_`#dj6I(@p1U0tAUicA+9SE@!H#bA3h~+w+&cw|KZGzG=##6>kuS3_F&JxfQfPeyKlWuMmQ%!&$0RbGkU*Hc#71*o3mKiQ$$VWQIAQc#&to%w@uIkiuL}eS|-j1qArvj+)E1hbJ<{dctyLjDeKXGIfeCQ@a3Mb*p1<_`nN#cqn7N-4}QT^!e8Hkr5{0`qVyY2V586Axi@3aRb)zDM3_EWrsdnkJuBhbB?@%EvywhC9y4PQSdcyqbZ;F`9B!=BO0e)G;bx^N2!%}nEx90L{N3C)9eCmFQ*?L)RLQ=FVGpej0EpG&n)GFiF@v#Cwg*A>iLkm-O<>>OdbvF^w*N0hh8ea3yt&o|I5&gD0By;faXG@D9?XpVjiDKp5fFi7^$Z&VJa=0Xg$O{Qk!WNMD&I?(YB+92IA;HybBVku56qS{WdY8f?Ed&?UusO^A4(rO=q<hYX@Fg^+HN&Pj#prq%7bb=G3(W~B}tk)|m4aCfk5Eyw+){wXi?u(OC^Nc~SrQ|AEJ!)0XS6QmTj;x?gPthodtA>JMBb<EdW@zemre@-_Mz7K(Jg59BcBN5WtJE>$W&LJ#nX103da#qXJ4@~2A>dd*_TjkeHIv#o&35q=HPD)bIHQxd2`maDE6WB`>mj@~x`X)~gIG0?y)>=A5l%jE7DUA8h5Blvv2e4x@Fp!uWpT0QW_66|))i`TaM+|6!Q=#;mvqkVG&{GkvZeg82>bi6587IC$N2V*#pP9sbkV=%t(rIu)J7uJ1(t8&CfZ>!3O>wH=LOM}^LSrTSzW==AlZ3Eqn92hqKXb8N)*BrZNc~n=uYF}Hl_lv-K`~it`tCQ`EEZoH5fh_(~fm!zuHeQW7_lOCR~K(y^bl0?ge%h{5UY%_fQf3?WHt32?4kVy)Vy9^0H#tUJ7%QPymMJefjE`m*Jc!t4o<O2OVErf1C%L;KclSsnfokH>XjR3yc!iE*t+THnX6Js)~Kxql1tvHJe*bDe=%|ukq~zwgKA>>6|phOtT5EGL@Qv=WMHNUhawM^C>6@HT0Lp4uTqgXF1R8ZWeVvu_kv+xpH3ly1j#Mrw^4(CWPQ!I)Kzb@zTMhISWJMC*ilfQ}hqn<R8mY6$^L+##WFSYCh2R3Dh>JvkVKtzF-PyRK8>M2nTyd3A-r}>q)&$PG!>$cvYODOSP41<wi9d)jM{lqR3Tt_9~`Hxqgp+OZ(Aal4Y_Ze0!i44Cl#&CC4AxHH4yk=75lK8t6(O2R~bQ0SEWh(6@5ziuR`x8ugQ53T5)*Wq1WH2lS4nS(AYZ7Eo)@X?Jg<dQS|R>eXqVe@HcB)nrnfN0C$*_B&FIZMaX6M1lf`n`-GUXI|$uHjm|qw8B;Tt`jnA=8_l{<&eiu#mMlo7*uE^>Ng3<(Z5<zI@xB_(Zk8UrW_iu>_v?1Rk8+qW9Zuq0mbX2zPUk$4&{l#Hb(Nttx>dr+wS$KboF`2!)qhSbNwdqwS<)hL3LIaRU`uzVpv&ZP6e`gy>^R?ii}QF*=V4tkj1Xb`g*<g24$yMyC%Z`Y&U@OtKU9;RkSX(D<k){3E^<As?O#zBeH(5+12Gj8B4^qSR5sTa@M>+nf+{6$oi_wX-}xk1z`xe-{}hhBkz%|`ztCb^N_H<yNAXVrTUjvfj-$(#*M8^7PTtopeP;WQ92Mx${>;@wFAtgHPq}_ab7hk40VusiA#8A3VNgZExtFZ^AKLbIV>rUqjZfJLWWt{>z#5dy^8Y$B_^^zd}?m$4_h9Fq!>$cQ~kY#kt}xxf~~+s>G5atOYgxGPnGJvB5i~a$`tTTB_E79MD|>So?h~7>69n-w6Y%*XATuSp2}Fe`sjwkA9bl2?oj%}Zm%~?`jXHs4;6D8*Trj^-Dc;Ms_&SdI27g-yP({zxOP4D-+RVHWQOF3Ro1zbS1S`;<}F=HLnY}Qf)<A<73SOWo9|+v<%`X1d2Iwt-{;Nld7Xcd7dfv`RzbZY_tnt=mr3I-VO62E`YL*0nDH*$#AFHH2K1(^y;k>E>spJX$O}(ry1lL{Zo`rT$T77Bshb4QEXHqO=r<?}2S}o~bg8nohU%ysB>2{hy?{62%0P(c+5+4g;RMONO02R5;_onKY@%%*HzV&VoN$l7J<HhLOUC{=@0O8P<eG`+X6UA|-;K;Gz*cCy#Z?G_GTQ^!k!=s3+_f3*<vZ7cQ<X!&mesT{gAiBK3UF$YNY|@@(3rQfAGls^WV+XM_JvhpQCunNRb9JC*U|Wpk~qFazl!qt*hSRr#YVp*Obj6fc=7~aTu{G~FR>%%E2qwDy0Z1Bw6r3&o_kmKn|I20Gdo==-Be0UOoyx2;aJt+xHT4TuC7vK-mONwuu-ogObwd98|xd3?+8{+gFVL{z9}OIEdqo$KCp|VMWD0%jd##^QCX?|tdcTft_8J$C)_~V$|HRlNb4Xo?7zysY-}v!AnH}r$niIrr-s>G4w$V0w;`D^hyE-jn+aeKMy$bG>+jGY*16p2#tI|$V3sfpD1ByiF^}jR&UFg<(^#pjHEymtz(2_AmR1*dW?&=;h?7SU?(J~#ot4Tm@?td`h+)JyI&9k0(I;=ztCd_*SIxDmE^aL7oZZUu+C~HBkOXp=S9l1TSg5R37HTkDT+BDumsi6x!kWpXF{oe*a{DW!oZcN)7siHB31UNWT!1n(oC#4@dmzjzlh#SjQzoTPI|+rKaY_K3B%e%`4KPawn*G&t>*$FvBGDAQiV;-iOs7W-$|g;+<_#TjgWNpbkZ`Xwe83Gz^u0lsJC~;?F*I`lI9!S$vpubQAYU*`Kk4S-y?~&FI0_g*XcrfZRZ2mNY9*%eC493jR<GAqYU;EZ262YH#qaCYw>A)fHq?@XJ@q3ch<I??9quO}RK*xnAu0F85tm6ua4>((l^Y!71WJP)u<&5M?+?^dQ9@T$T#CpagKNPMx*Le3B%rKfj)-J|9T6cba^@~Fv~<nHax<()tYnMeV(aHRHpvcNec{lcsjHV;vsYS|&s&r1b56(fga&cg$MfI8$8V$#tz-z2EP`pLRQ2_dkr(?HNw~ddB?e~16;=H<4JrXyCBmwUPdQeF>swI`U`gZrERsdd2ChV0K8y<1+BIi&559Z;G3D0k(pmoikAEmN<Q}T(#LqvwmXi^z>TMYXC^7dJI+hled!S1m>!Mm;dMjSzWM%_)s07!Sl3sZ~O;djUo0pxoI{oU*^loeWrKGLuaH+MVe=$vY4AS^ojc>?02->-u-BX1@r;9X-M{V-BqSExdr&OLQB>q(DFn>W4O2YMsz@oHRpqXm@kdlf&$=NKUQ(7yFbvn~4pq_6~J$a>3WiAsJW>|uDb1CF_Lrx<00OY<$#kSDu;P-wALnJ4d*tWGjx5#Mm1<=(Q`X9JE1CLuzN$imAMYGE8I7N2^xK)}X9$I9Lqq)<M322r6m=1GGf+<Z*+-R;JFp|(1=R3L|E%(-hJ)Q~2b)>noQ*eBt#Sznemra4}S&=WX<$*S!qCi?(^kfp{uXJ;`Njt)@$ZwUkp+#8}r8OT^@&u$)i~bgamMC&Iwe)_+l|-?s3(I}P^|x(fbZ(=QW@uJ|-S$bclkyaW=RF@JCm8Vq{03BT3+qRssRLLx^z3V-p<r;dZKQCUj(iwMa$*ZO*!Ak$s7atAWXd=Abo_4fKTWw#j?3IZ#?$9NU9DPI@1URM+}fC<4%zMIP8n!3QTVS;_3K;dekzf(+muRvX;aDZB<&o}O>7#-00Z+1vqr|XN#-Pn@OVzy5q!!^#<tKkXkfRAMmlIIk;q(JAwVtCrDT0Hv%1pnw-ho~e!AJ{5sZ!YccG7I1Lo<3Aa(n+Jw2WH#B@xjBR0;1xba|ieAzzuiVC$f-ckUT?2N?9DfnPFx#NO>!fq>$Oj61!mDzlyF$quE45<)JzVX_IRBw-+oIf@gELCq*mf$gE@a3qrMmQ?348<z~IooY+_Xz?*!<<=<81owsZnt5<cEk^of8rW%*4EanBPWVV_TWvOIRXkNCrAPnWtx_4c8$xEn>J;`2k|b;0S3j!@-k}jlU)2=EV+9IIf4iEpWaUMF0Q6I8L82od*JWgs4kpiANu=c_$SLQMr7(zv%I8{ILb5>Fa55aXTBZI&$qBL2|iw1fg#3?n-wxFS;W1k4PkJ_Qa%ZZa(P^TBtR$iKw;2gIG@$NZ8xqKJq$;#)B7C%PmcN65b(~^3iEBTlWeRkSNP!NkCa2(d!61MjdqV=&6;f2qffD6-BZR&Kg>w35A0>ko`<-ey5dWpJ9ayf*lUCdY<Z1B$W~-F=VtW%(FL`T<wf%(<$07$5wS{9IXY!iVze+}vwd)IBmrDpDJTilNz++AyBL|@Cp#0dhn*f-()tPJqQUpcP#J$v&%(S;U{p$|Bzdzmp`*(kTk!{FN;ReS+#gGslRCLe3TOpF3r%bDnY2q`vkp$t2%dQ^<yCBv`TB;}*XGf|^2WwenhY%knF%`0*6o2W;)`?<zdEixK8&KO;cz@ZmSF$DChiTJ9TyfgzaEZ<OqxtEl9un)t{~~6ctP>scq-51t)#6XP}J7uxTqj{1D&Q3U<)cmx7iJjy3K>__TEwNXb@1(bEfo0bv0IuFP%{LmlEHnFbg6D!AJ=00FzQh7GvMlb-3$m+R3B+PN9F}<c?Yl=sakOs)cg^Q{FiZkj3;>U2Ki;&}xDVJ@7kPaWstM!44eDb2%Ck8f3*LTpuwOb7&_HHna{A*>7qvXp|NwU=LYawt-yi2V(Y%LU8L}B_(2*wD!An%_&@Y;D!|G3KzULk=B|7)-l?mgIFp9;XkMzm`Zl!zQ*@g_GKKcv+x#jyQ|5i5&4-W-A?;-CqHkrUge3-7b6FjCNqgBYmr}Fq~sG!XailUgYr7)VLN+?IJDGk!gFs_mM|FHDGU+bT2)(yfu-SnOe-UbCaknc&0&u=LA1k3W(UZ}wjF4ISQtbak6PR9rNtv`s%Zk#UeZG=I2U)K8CzbOyGrkJHo5(tEt&O6Ar2)fm^q13{%Va_bS(P*XVv;DV)HFw1i%V^cfIlkg0bS(NgFxK6K8haB{%RKGl=o_w!J-H_{E^tE$ti~91aS{QPM>K>9`5R+N}LWl8afEg4kw?=W$fPgcd8s^w@){d^<W;d*5VRD&G5SYb0cv_QIio-Vr8PW;2OydYBlfI_T(mKS+<ZiKggE+*y+2xCtYETS@=uXyo3FKDi7UkS-o){7-T~y;lRe^6*Xr*~av5ys#Ird8K*fwdvPp&gTW3CLf@cz>l6e<wN<aiZL=AeZPk%?n#>eWAu|Khu;0?za?KH`NF=jqnw_?gCBcPUZ4MloG=f_U#9&ml;|BPH{0`HqQeGVeC!-!Z`lK+iD0ty2F6SUDSoM>lt0WvzC~===O0Rs9|qJX=<yRtFTanCyky_o<(ci-ovE2uzmM<D^^4uAH8v_s=oEYQIj$RY{~kt;(foc&v0*;{=<`nkOra)l23m$kJbQxmOHNqyh&_G&YoPT7u9+#`{3v+#Z}9z>Fqa=7!~?#8uF`LC37;r`@jV5(%$NmV;9A30Ak+dTpTPjW&*5(fAxpm0Pp+k&Rsxc^PwqWld)k1wHLP(F*un?TKQf((onfipT5AGZ9i-(KcYEDoi4$ru@F@93^VU(&Xkvlba2KT4egXpb-k|*++zy55sb~&lj^+vKBK`JGHVDGT6<`7s^ja7I>vV-rOtp|0ZGLnRUZMvtgr&poo*U$YBN!rTU)*V)!Y(Nkui;ln9$qgXwP@$CU7o%&6>V?#PC$<bX>U*t>7|gL<~o}W<?FQ@SP4a0h|wE|)LaWS$a6>iE_K_lTN7&cV>;enBk~7_C4ND$55HsVXJRajMS}a2U8xKS!QOJNjb>mw)AQd7$B@lLPtp+^1O<#p2yAMqt^2|CDqQ_AwxY2WdRz*)Y?fh@?|K6ci*oC)Wqh;-7^9)xD;NS2M6_jx)E_BmY^Tn#?SLtj5T9%teW%iwk|e=+H0B8!)z)`uHzmKim`54^QIi8tQ3dnu)57Ms*0v%}wG#=X$r!9B${gL?+bbAzP%gNg14RaNcZweDLNq>D9PZEL&u1W!C*<&_+u}ZP0EPMv423{Okii9)0<*8h%mS$CCSk^z5&3P?TEi{0*m7NJ%$v~bWjqsc)sQkwQv$r(J<4j5)ti#3$O}~7Jl~db4Rj!BYQmE*VbSb9tB0i{6dYT(3utqYwVU!Z2LCUi1q_zFAUg}jP+M8A)+GoR8iSAZO8QaYRzcWWtit}WS}37^nzv9Pn@WgE0kr7byTAsc=1G~j;?eeDr#BoFgngHpdl#=^XYrT9ZD^nzZsSbQDB&!VM+OKqPdqly<_RL56o%~~=Gnu>nor9MP_z@?wjYoT^N<6iG^!2Z?g-cN2tU6C9!9M0yR=$?2?X>|ZaLzo#V|VQG`EvZIehjfXysSPSrvXeg-?aise0g_-76P<f8b#D!xu1^&_Ci%vWGXqX87!n@bq8R_>ko)Zj9n0iylu%z~W)&KK9RaZL2699RBZgP!`4GeD(Y@+z@av)4ILt?-=Mxx8p;)%?r(wHk`B*gg{_%KC??@bLiVRoSMdzV=ks;^8Kuh$|!f_Zn@TR*(`>{SH!i}@n*<CG7g()9hdO?3~Qh_QfDbC>7IA(C&n-s?AmdLt;o4|+mZvviA-E>F7%(f*71uRyIDAPuw-=V)FfmF6!tFqHsNjut`9hy@B-(j+9jvxrfwu?8)KVj#8(eIPJh7F?O04X7aTn&zSfl8r^S&D()t#KP7ywnKX^)ZB?f~HQL+r5{XxkiM!+MUd=!>R4*zW!75Y3GeP=F;(a9+St&|H>(ez9)!oOQR3`Z`9z1`gb;!gD1qzUggVGCep4=n7sGm~Losz}4rvCNAl^T+u-0Ds1h-;wThU+Qo?M2IfUuKwpZ?!-*sJCfZ$3#^7P)-HEc@d>hnfr@LAJ*Ji#Cd#NGHI_5X&>EyjF<F+NYWh0+pbKZr5*yWX*RVHbs$icCxoZc_lS}}}Ng00LRF9O)`0uS!w}bY;!LTqr7lq_6>?hql%+a<_t@V@l@SdFi>`q5l`0T<pxnDRb37QWI4koBVbW$>!AGi5-dH2y#_!w3jPsk(EF&|iKj`jnzKzy85mx+SAI4ftyruqxecucaMV`;$47B9ARzoB;WmA$R*g=6=_<=BL;psLD=uV{zF+PjndZdrcV+)N=CRJrN+ph?ni(n&ZT4nA&eWt0t~vuVHO-VpDW%%8KD%)6UTGnx0zpT}G;@0Q=y=o;7H!=~MOX>VAVD*o6eiSE0pV(#GjIKR*<`=v`ZVQR}n4u7{q@42yHvLx@msF2tkmZ24pF1pZZrc}Ikr(C)`>)qRMIttC<#XFZ~5WT$Dh4UycMR(depyc0))E@i;)l&LEbmMX`9Zlu%H70riiOMK5#6Aax`Go*&inRC@2AnGs&OHmxF{hp8$v#eRVLAfYz1J^eTmmAh!gt)>bkyt|!aY;EI+g)9hr})PH`LJqtdfN8nLsSV9-;pg^Vx+uxXYC)C~^32vIxoaJ$o2Na+{$b%<=MAY|1X;rL0%qutDLTDzk-VqO7g!8@!Hy-`sot$!LkVc+KV#v$#DGrz+jOTn;B|7{8Hp;buVxqF8<AIS)jvFvYQUVx<kgu#x!k+^gG{n^(_iB%Ub+&;DHk;h&;)&8Z%xt5AV5bfZ1Id9;mI92OF2UCi8!M?v)jI?A#4v~rRu0+Psm+OrgVk4nvb5lEgIrM>LpXD`=8JNH^^=gLm2`F%{Z2gyONe;SjNRtUEf#HB<x1l-v6j<7Ac)wj`q0XG6oPNb*w;##v0;>*xZ5RRN_5PV*S-TftRZR?^s!1jap+J`p61ECI)fD1zLAdtse`%SZ6OZDxA>XKM1c_4%EQn<V{m!?*xIjESKy0Nt0D4lc$Cx%N*Bba;X$d4uHpz+Si>U#B!)zvpAZM)PyJne291ePl+wd>W!dWJy%s0*jI+unuyy@YPHFvqNE?fO7($1j=JK7?N$lEG68%w!rsx5TeC=I$eetG)!B4`a}&WF-}SWySw1$xtGcU}e56nSj}KbW7f>t*kf7cS8w<Xd{X##wm4&F&TRJ8v);v+PF6Qmj2R^<Z|m(+LfP$`0e3Aj~X$sI_Z}bWU<SLo-{~wyn&`ooP0z>z3pFM{F$Pj2EA|xLky;&N;Mc{!R^Ti_!Yyf3Uj0<%B(M{C&to(ErXbmt?B;;tLu2j0Ha4-e~}T7hM=WsNLs-UvJ1zCIvLmf-cMie7<f_j4LbH7EhgbPMGAn_7|@^z+5<Y<sW4qjnmEgISitp#+lfj?A2P=&;9^Du(wa#-D%00qfo`{?4T0!QXfd!-3pH!~lV4`Ti(fxd6m*C53o!_5M)~ab!1=yVRVwe%?QxGa&p@vBZ-G00N{L?K1sqLQJ2GaMdPreNyPL+>TcsVmOE_PHH!Dsron~)pl8WpoJ7mO2q#$=l(}NG&Nxz_8@yHzpcx_odR=Tqfi>lv}bgW|V<G>^iqB4G~?H;<%-fSn&nI#iBPRhI*iS_eV+31@xwS+$MY|i)7@PD1@v!ZV_SsdA>J^5v8VifUOcX(ws^h80}yVVc9P0whKlVccqPp$CoW^1c>O_f%{PiobE{GKdSZmg}m#bLB^3f3z1w>GNlh-^_^u7uHUco#SLS?O-;Y@VyOY7>v7)Ut{A`Dd57ET;_(Vm67rE5A2-mUvm!G?<svJiSgM9KZ311Zbnp!*&Uu#RQvJZUj1ZJaIX*J7{7^Ft??2QvZ`^@gJ1$62&2NB-+|cp|qbgcM3C8Q&YvW+*QECKx8lym;U%k`7Vs?Y(5{b=R8xPr$f;Op&_CwSg2XHaS`wj`sT{)*jCMbH=;4zQ%0E3&<GQX7?d8ka&gp*6(<@$A_aGv2irT%;CTKVaZC|ex^5?@12@(vnBbrM7$|FS6b(hC?BRuDUDsZt;U7=LMAJF#beuWGfoz`wKZ?>t<#_fJDtbn1^}@3ZV(MQqA}9R$6*-WeXtVA;6+x3${QXKfQ6DJPEA^jiVfUNoA3HG~94({Nqi9su43U|vd>O7}V*xz8WW@A*qY6rE5@ahb<+q0wDU%XsfEM~;flQ?DGX&ZD0_vRXyuVzjzK<p6-e)Q4ebmX{KkOY!W@s^tS`pBxjhc1Z-xKg+k2=7bgEbeXi}2g$zYU8olc~V%d+syJ``*g;J~mFuxC0>pBfB|j)xV%27PlRj03m~{o*U9$1F;9Nm4<`9Dwx7v<J)lFEHo{_OQ@?UlMaP6iA(hZ$Npf3nVHGGgVug>(8NI0?WF9Jh(p%36x^yCvgt%@8Uo1t=FHU0oT*gMt1A^ZS;`{}&2NVCa1>^es|<w@o|{6WH~f2s{(TuiC2eapW!sy>Ea^V7m<v+3T=-NbG5e_5HBJ+n-hy|Gfw{VZ_l$%Ymi53;jbWvputLoz;y!UGgb<-h@&WS*Vocseewv9Uw9WmP1Y(WNw@q~wE9rZtc#P}{*d49gBb22)OV4g7rk#M<#e8s=6dtA#of!-`3d96Kwqs=L`REgJBmA2fCXAXu-@(UUQ_Ucx7EDheU;XU)hb5z>Yk@*m5m=JLW@(*r;yD^i;bZU&>#gsw<W?tXcFSXQ%G`)Jjao`iV?-Rk?DCj9eT*I7*^OKKN8Q_l0<{`Z_?km|w|w|+NoM;x0)KiQLQc0^Qg<5O$pq8MS}2juXrS;rhob4K)o^v^X_`P>c3czLScyQsfjU~kiF~f2%dd@XL!@D7rw?KScAiZ`m>@gov_TS?iVoV{!t~Tsl#Z#0=6D4E+-VMSLT;Eaa0YxkeKP_=o+&x%?!b+3C}q3-U3HJvCO(6Z>p0__@F8u48V*{2sprX`*^P6>IAP9=7o#wX4@(<Cm2KD2kB$~Vm{Bwdk?gVEYkMv12O9hf=Zlp@&XJ&vSMmaA7o(hu_O@uSpClj~HV?~m%lb3s4B=8<Gp`{+s^;${uuXbI!nMmN-tG)eeo9>kC@rJe9mZ_J#?o#SmgK8yrhF_I?)@F{;J-JxNKoJng<Ffp3coR9dCa;g15`ei+a10xx*P;%01fEyUmOnRN0AvCLHlmisy~hEt3PdIyy!A&5}V;omtPFZJlUn~Uhg)7wZvHaiKe_j|K6OxJPm64ok3~0pCs=w5cQxi9l23O*h13JkYht&;cmFy>*zH9?Y)@XYvFwOqnX!cU%fIz)v72XLB76D|B&JIWEkJ+HxJdzDC_JYME$XZdYokjRboB}I~)%9RFtv0ooK6D#NEzPKpQW=e|gGd4*5kygc35MG^uK{=2h}Ge=htY`UUZr*~<~eIZRIzr{NwIFU(|OFpxST${;6oS}?+HjF8d@IR&4MXxb;9b5KO4y0vPIvRk0F`CL6e)&KbMK|r$CJQG%d($s?yI^YCqr$g~P$vx+28731G9Faq;!++tIgTqYQqU26Yl?@Wf9`LOUEj+d7q(urb;*Dg8aTRdZLrmwtrHV}4RH}Sq3|K@n=`?uk;jzd35|_$7Zzn+ansBbIlZ=OAD&jY@=e3D$hIR)|2pxG2@!|jIlX-F4e=i|l$$<oB4qJVDy<RDA8ZQ$*qg2;5kW<7%M8&NT0LQKbiYAEXAG1jzz^c8GOOYd$4-lC1=ny?~@c{OHd)Pjv!mdbhbx}L*jrcGht?||Nu!G-_x`7_%uL#Xgr!G;<xQA|<WawaWa?%(PeGJ2V!buNQ<C9(9XvFaK)Ps*rRPx~xy_4tQXK_a?1ZfBE?xC+QIhdPBjSIKpwf?E9`5KzRHoRHZ5o*3+?H2fllO0k1@Aq0<;6}ht89oN+PmdaLKHg*oAcLENt#sII_wTd^9Ms$FqiiJ&!6qVs<GVCuTS5%-wv<WN&Y_LBPnCyFNE`bTb6q$pCy$)5OJPfy`;<z>HD#ktv!aFOjc-gWPRweU$)TMRv(AFi*_4m~S7JUKsmL<-Ih3(9<X!@@imC-@R@WrLZbH^?(IWZhXMdWI;C_2=pKHiV$?jaL7_0LkN718^%s&xWC^2bLi2@6@O2`u4V=fA44$*_o+*Rx|)k%bdqGL&fYIQ28`no)doGy+xl}zihlP0+=d}wa}Mxr$l$xH#=oVCdTT;aLPFqI5*(u79B{f%P)cCG207Dgvia}%R#Q(I(h);Mbyljm#!ebdbb6t-+tRDSlgk54N)x`xKlAQ1`=Ij)41sjqdobq>GR)msK|_)^3)rj*9HJ@lfS`l#tBi<E7s8*SFO69YLSpO&m4kECc9cU9!jV1#C)uT7F)a59XC+yr%TnKs`NACaTuOFqQGv(H`DGm_@G4*S~hcG}20w-F@urg@qI%sK$Wq)C58xj7F#q={jqaW*0_=?(A+>*m3;-#z>Nvp;%APZXpMrzLeU5#cyST2mek*m$6a49wSTpG|(_2G2}eOo5gCFXq(e<)L%B`EO~<6J)Y>AmkBqdcml_9Dp_E-79ey)8<s6X?~wms7JR8I{_4ZT)0ybQ<5DKQrd%FSB-=OkrZv3b19yv@ONL21ZyP?YwVzi(l^H?o6TUW56D6CRSsY1_c|RE)+AnvuhDi#<Z&A9H;QM$tczGKfdxLUkTPfcLYg(fvQxxylEc5r9xg=IOW84_N%v4#3NRG<!@RQgE*a~F<n%|rMTpQ}MBugW$$*Y3a08bZ?1*W7hZ`;-p!8?tAIj^>LM-LBXPjKCiyo@8F?&-^(b+P9KPi4Iz1*mYe<k7I3C_FCqzRirL5TaO;Q;3{W&9>j4KP>YJUWuB428(0P#=0()ZUorViH=z{p29AiTz1)IAWWR#h3j}4h1Z7tfHd^LI6kbCF2ShZ1?S@#@jBD?VW?3S^C<Z9^tTohRCok@61%E`ePSv7?y{)rl3Xj573`qlGvCkJjfG2Ko{7bU`_tj$JTSpD+iczm64uv>Ess~H&*g7b?!sL`S{G@n<GRGe<|#gs>~qgDSU=E_n&>k1S;JwmJvH1#0M~o$mQt7{ux)oDIM^k28GXDML8wGk=DQSHBT@GbZUa>>4|o&uoKvJ$;l867<jb+!QbO;Z!(jI-m{(xPIBsbKNw!k-|7QShg*OU=z)xIOho76-q8?l7-Sj)yk>@N*=C<-7VV*00>RzLUNwC4B}258Q>NE|Y5l+AY=1;>9rx1<m_bE+qrI9HJ@k1JNy5Sean6&yDKH~qS_6?s>71p?iC{MZ5#;@h3~H`FBdoDE`T@t!E1t;-CtvXgSqjhMr(Be=vJg;;>VMNc40GhD*ixG!hjbxS#lzDAWlHNXd7x1i%-1PP(8Z%i665x;Ik-LGeC<C4>g>qn#`y!e>S2Lu$7{?o_9GxIL|L$L7*c8WOOieWh=)e!EkMsi;OmQSNr%k19HfrWsyRY;DLrIoaWZm5ntlu2%U-+ctzo~}W%2_~S=Vt&qFeon>ZKq~(0xf<ly>35Aec0@g82HsnC07WnZJ~KLg0Bk(=1fPY(tWSm~7HjqxOikrn^t=66*zc_TseL1jQpX4^-Jo_0m%l(TQ464}z*hxsOdX>;FXh(5GTD(f9;fZ&Gx8w}B$619_R`=e!%gb2~!#O)8BI*`7;NBHt6fyGT7&b5Wf_C1cJET6hphqQw&{$(b*1gP0L!Ombpd85N%WuZ+TxdlMS;QYP<?+t5BJDkG28W7r4L)Di`T{TNaTmO?$DE%3*Ddo**|%%+2vdj}3{H1urG-RLzh7bG(SB@!a34}k{}A_b;;>a*!fj)z^|9?AWylBso8j@)4=oM;3MzQ^c1$ponzOr5O12=tW-Lv*n@vKLsE7T#t@UAbO~8;Ldefr$L~s1xbs+0@Dup{PBfZO${AcWruf7JXr5loj3~{H$#ad-fN*fYC^1&u#9X9`-;MA0&2zU(&WBD4u6us|J&fp_n6LNWM8hfKyo=w)J3Q)V5jr9>KMdbzPJu;Z9VQCgppd)1~;>tn<Wj%@?G(c$MhgBND|A1PTy0eX}oX3Fn+rMJbi`DPFoCQwNa$f5W5Z@sgnv;RHde$8@?_!c5aE#3``FzA*{A#Tx}-Trzu&!}?&BTKn<4N0^U1Y<Cl02-^7CR6tzpJF*@<Fm}G?Jt7MZ9Co9Hq>&<*x{`Kcg1J`bo6fazc8o-aKzdu=@H)eP6kI3=w*S%h20}!l)aan*o2KErxP|V?^+9_uoLAQyhKGK{OKV}^;1j=BFl1RwIR*-3spKP`G^}~WHLyUXvXnqKJjOxr3D2VOz5SfhoIhlo5KnIntl&w={R>4-!oJHTh7h=uQ2L9&eEgcugOaU%R0V-}VRd=Awmu2zf8yF`BrbkAk~GDFpfcT~f&<~!B;@~Ds)2n#ifbTpw=~D#M45&RwB^a0F!7df1FSAy)F_iE(E4|uaYHJ1qTpeAYz#p4KR*53!m<#t@R|h6{}w@53@p6#qGit?$NN7Zwfo*ATatwQuh}*&;b2d4w<mGD90w!Wk{o5>abSZ&DN3+mC)$+hwU({FmpWMtOR9Inu2<Hpn?_xj$@bNs42M#RsU_jcau=o9!h+0@Gi!hF2CF|%FuGDhbdaVE(_Kg?Dp<RaE|ZVg?uc`2;HBP6Me|g(nq+u?Rtc7LU_KqO<o{}9hhQ}WF&Spj8z0%FB}uQM4NEwo_Z12QBZJX~O)EiC@bVLae=vKWSMGsTAr(N*Q_CbdYEiDi2S${m?MqYX=U5&Xkt72J)h0JLVO^qx@;lzdU(ysmb}$i2cIanX2Psxhq<O%FoPbZvD8j%aj#*`_{6ne=R4BrKVni@`eR^t&Lfj51Rh912NCny!iz9+SL0R>L!%;a?0SqYptbIsv)0!S4;J=p>MAzEgD^oOEr`OyW6yBqFWRjOCwE7YWfP0*FpBE@b(b47iC`y{aLj2Fks0z>~gM`V=7#7QA<D#S)OM7rUmvxhKe{k-o?a_%pM9vWOE~UDG@vRPAKL=Gz$UC&EJTaQcrLdT;F3~E!2r&$3^%8RtI8Sh12&sc~)tv2PB+t~V8;xqrf#BsayWGl2vEH08cY+YbbgIt-MijZEVMwGjDPEt%tI(eQSH6JeU4}rgp&yd~{f!u_?ipEU4+vmGuF|nt7oQVbW{cDlK|^~;H}?Z(WvM+#<9Yxi`GWZNuT7Mb`+;!Td&HQY{{!1vfmssgeS{p!iJ5+}6N~%cAi)Gg+zjT4n;)4A%kPwu?v8A&G)+U*s*y*G{g-aQ$T~j22j<x$f0X28@Fu6_wI+~YqR94=1OKCk{NP%>x)?XsSE*e5_1aREqDYZWIrJV^F2R#|je|UyB*F&Mmy8_@#3DCm1->&=R*J=uvuJSj5jlE?<uR>XJ1j98Na^7AP8+ir@B`<}MpVu(?%f`jLCgE6)H?*pS7sSEjY=`#AHqA^A$`pl7W0GsqhY&4;08F1-u5q4P69s=&Au`{H8cClD>FGQdy;c<p3G#8z$BzeBfjJvCw;`5f>RuUfpXqt5<8Q`O(V%q1atVtQ-z{va|}ukb0?vArZ|aFl-e>apv0NSwlavwFUE}j!rB$H&c{mTCBvg6WGYlX<kO?MDD6Vf@aa*`Oc7xw>F%|=NjXGhA<n!A<-7Ry9oTFGMC3l?qdb2%bkbQ=Mg`}q4&q^t(lw4cO?a9^?9Jbf`W^NC*%@tkZqjo~J=r|UR1vXUt!<TI4)AE)j)<8Ms#(!ag)j{z{+x%=n~Tx5?#Au2OT2>&dPU4Sgoe+;2)dB@HC-r%P|XQ)&yL@~R|o1v8P{Y}XBu~?AMsn2qV_tl%j;VXxVgvb$@5Rc7oc5hG&XRIUNz32f<2Dp(&`PQ_kWY)o{V7rh>VB+Vo}T0^_#1Ut3c(Qau|kop>?NDhbU4k1W`<mk0LV`707KmdKtKtl$bBXFZPJ7!e}rxeaw=BFHMmqFV^jEK^XZ>2C<5~d4kZii}&ksyLU8*_Xzv0le#Q||L=bp&$W*Fgl07u9_^gsQ2#M&$RH=b80ke)dTe|DADimsQwu5ptt@^8nH>OD@cdKuZn5k}DaSSdf|MMP7#u{u{tW@n+Z8OHe|*g-DZ<V8%aPd8<05{h3TAeaVKk0$8`QXFyL)M|dh619wXv=&FkDJQY%iA8AYEP+6(%ky6?rd)ntMj!6{_L#EmeQ{7&hcTlG^+*fJSALz+S>lZ|3gfJqDyPyIZi4j2oAF`32d29%@W<il#N8ET6Dd<ue>FSyN#`t_4PPRG|F8X|V}qQj)kzu_rvDC*_WnFASi}=f5%M*k_U<W6J&1MC*w<3udNft{Cucv}2+3_Zi^LQ;~)Ck8gF*-(o|ylWG7BO#jCR{}V9<%>J@V>meD`s9T@qY~Lqd_+_IWjtju7rxPCh3VJ)nK*@cu@O#aE`n(X;so6{Dto@G<e((5v*_s0CWbc&fIB7^ToQBzp(=!(@Bc`xQCVk3$?h86;pQV*VOe~2A?FMM+11?0TLjoF&gO77>_W94+>hpkhft*_2Yq<q!o;|q45yQkamCr`1C*c&{6R#`}-s%go3Botlo@(rqKmo8iNXJnYTh(29%!f&RbZhJwNxJNB*5)-jDHg~4T-`C=NJcWHRi`dt)Q-c|QqXhD1gx!kk1`v_aGbD_5G{u=#dvm87#I~4`_$7So*?Hfb1aq%+cL1Gb)yWYvRWwzoyIUxrfk_cTw?ij?kbuss{0$TT@+=42WAidB-rbpnzNxinVCEg4%Pld`mOKbNL_%;_vQV6c>YU71*Mq6&xFZaG45*G?^DSy_vU{n*87hLT$HQ(d_Xssdo6*~@;6m3G%D)PYHI=4?XrbJmvz977xicYts~EN5`}j{!fV0~Vvz~`U~Mg4tX^Nj@_+h)4ve5vzQT3#fYg;)ZZOXP^B$*tqRJTRJP`@*E*<QIks%fqN5Aehu|`H_adUvij>5uqkW<G1Z=s{DJ1ox>CvEev)9hwjOe<){YfF`tm)@!__#G!x)7j=%7h_e%*4Om8n|rsJZ9^$3FMn(t?v4hAWRPuF7i51aBf}z5-d0FWdg)`c0$Q@Ia-G?i-z-Iqc-m{ME!Ebgl=#bTQx)>X1cv?Q-+CSMeH+oOxp=9%c%xcJTn_vSAw-dSh%qevodFgSVbtI6-GQW|BO$%3TV0}}-}S}D-*UcHS_bh^$!+*6NwJSq&Yrp-<^x!Pi!*aqBC|S*iNVD7zsS%yl|mtbxuqbyAhk}kX989nCRT^rvSODCcIhoj5Gl&}PZH2QB@TjC5<L4oc`*b<?_&`OpRlB16l!^rN-d5Kz@q9XxeLX~)y0y}@*xQ3F`MrhwMdLSQW|!QWou{GF>BcrU&yef+&!sn%O;w&ZaKeCX5Y#pLhS5j5qEZW1#M@vAA?;GI2DF7VWjEb^*oT-yLfGKyZQeP+xg$df^NZT9F{ErHrZr3IK0d!3Wt$)KwWTLC|;1?tTRZ;+#}}4{5joE*S?3VBgbR)w_U=&)$*CEF|iNbGgChbme9u}9;%Ckc9!e}5(+&Jhpg2(+DSlbNcyM2;HZUI6c{Id8f+i6JHylmOT{3&@I4#L$!M9}Dfw9%dCeDASJta<ud4+bd5(pF(6%@%mdGDBC}aW6|K!>R$z0^bXFf{vkrm)2f34g=;RT-j@kc=>3eJM+ng^>*j~X7qflIBDc})k4&#Ef6niQg_jiN?4fQlrG1MF6n+r%NSVjwQ$kTm50xcUc|u)qUd7#>*<6sbuiEi<0z)|G(c8Uu+3nwR<|x&D5uMvB7QeaI>B_jK7?yOPJVd)I=6UZ=SoV1)oGXu;NS8!#Y^NhnQFQ-tzl#8zO2sae%k((dFH=#{Vv*vkn*p5<%QgKqP%tRCa2X!IxV>?Avq_G89?keGleL4<q^ua@w`AR7ur50!LzmvWivn1-HH$4^SJ7B3hWKjD;wAz-oe+3q$2x|IlV7KIk<ozZhb1B|I&pMI7%qWfe#?A=bf<-JaC8#Y^^$iYhS`s$n26{n(@i5<<a5mFTeCy)$7-xXh<o};h$BNZ;xy&-|b0z&>T^LzV;@%wx2;r`J!T$sbSxLLXQv*yM3rY^n~Z{3}lia<&-C^(baZ?*?XP=`5cA0$<*5LO6fJ2a{b8}-`yJM(&D5C?9;Afy;jKwZ2Ru<#XZcY0WJHMFMZ;4oINmsQ|h39T~m;AnTZeNqTZ@QhS=DSFn3*#%B}WKqj&rPNOjJ588l)tdPrmUh~SyW8oqQVg4V6zd!xqxYfLEzjDkx7jL5#HI$|)6{rhZ~dU`c_`iYTN6*cG2<!jrh*Ao>2(v-9NG<?lO0fxBaS#odsaLP=)aT16Z|MxmijuTBP;dJq#>79h$t&DRKXn>TpGB1pUku$vZscu$q)Hz{KP6YPbwGaan}NIP>6%mO~8jo9K(NV*L^5iUHkyC#J^gWT4pO^mMPyoq<hYif>JSI(~iT8)DGGrNLVcjoSEgmc-qP)nsg|;&5Hi#YcY-ChYKBcDaPC#4O=LJH4hF8mVBqG`oPxj@q6{^+G?b}#&o9k&<WqyzjEX~aF7@d@HUz<Jkj{R*7zYkg4)Vr_3cePqAlG3Pbfa1>>(yG5msV}fugC#I1f{Fnb*>=D`xwIW1swbHb%lbv-^}<8g#lT_CYd$fybF?1k@tVLwmQae$r#L*GWATljzD-nA{i2Ws|n~Hi0ondZ)C5LD|sDcdC_ojKS_sQbU-6)x>DWjn&+s%{IPSY21v$<2i=&fR$pHI2fsgl0+*61kw?r)O%|-mi<(%*H`O}D6{~&ICrYa=l~?5Cp-A+tvTF(FjlJURat^%Xvn)RY8`xcs^R(M-PL;SMh#zURBu&R;>JR)y0TuoURw}4g89QhN__d_ZXOm2|1GtW8GS>D4_JjFmq&-m3ndkgQwOBDPZZjDdh^PRX->5+_)F=`M+gtZGs!Epx>&@DS{U6VpJS@0&foT%=C^eWj5+Gw?)L6<_4iU=-C1DZ3>`zSius(7lDfXSURk2`m04+b^pqe?Lp*nOsmIhvv(LkhD+>$Nwe<?}J?Ax@NzQ92^{~_$5_aE2t4Jw$af%wfTwSl%78*sR=EKaoqTg-JB#t-vk1az_hVRthTPcS)-c)~XF@v}r5s<8_rYr5mo_a>-J7yF?<UuAj8nML6Vjboe2a5c-zMgsAIGzIXL+_-{6=#d_4W+toU04woze7<1kpX($9T+qT4+Z?LbkKt{+v~Plh2lln+4c2!9q32X_QH4Ca8EQP(oP%%;o;GCN1<b?{U5@87q4x+u~ci^tS*KQL&w_&dePZUefsdq0)0<cbgS77wvzy_D=hCEG<OlF8idM#U%rM<Ad`WIl!AflGTcuPusaE8$U7);nq>jcCQ+IGXjfB2^Q;(%eIiTNMN3KGxq_0aOem1Ac(a)Zz)A?1*ftIif-cnV?v=x%;qJv(J;g_tdcY|Iw3Wd%H6KPKnb5Tv@dn^8?5vVY)5(<x$cM{sT+0@y({&q7D(bs@LR!v*UT9fgi9-Gws8&%;+6JDQnr5X`S;!YIl$u<Ab$566%GL7~O1)ePFe6P>t3koi&VWjxt|SBMo&c?805}Xc!~<>K_iwX+a2UN{H7~b5>dcj7>b05H%=f8}>Yli_Dsg3Hk(mS+^u*Km|BRWnXhiMUG<52r#6(k!%sq<x5>na3wxX3g_HPbS)T7|7ik&xh&k9RS^BUPy!iw)jd4q!vRTW|bGvClEmR1*l|1aFEF1*>WEEbqG*4j^0jS=M81W*#)**`6ZnG%GcKO?$zK|kl1%glBZCiUHr8UTR}ThO9Sh_`HrnhL0dlm!u9uhf>P0%sGgYV**Av!XX;4GuS+a$Mgq;yNDlL3jaPh?s~Uow|41cl>$y;o`=^n~QJ6jkl-_!1bltjhk@0)Ir!-phB>41=nlUMo9491;^InyoGY}Vpz<Yzr(dY$TV~@g9@<rP84V?jG(cxyj-cjLw1*5fA1m{t0D0zI5+}L_psTANy1<GEF}Tg|5X{y0kRp)iZe$qgA>7?Mdh{Hij2b2#fWr9=i*E^lm!;nt0)(Oeigs5wE70A3E7rdm=e)cBo4fsMb^V_LAkif8DyPK0i_?_5m@O5Z*-Cj;FT0oU*0P*VtJwHX;ya-PI+Z%^#(U6`b)VyH<Nuuv=$f!OaN%Tl^fMCQtB@B1%IbPMTHTnkse2LGt}j5x43$vyg=h<;#cxcF}E-E)t^q%7kRq20#m)dT7M^}OMTv(yS*OXXef=wz1xzUNBLM8I{7^#-EOglwHV*nU_R>W9yPnkIwHa4sI}z^r;UZm$_gwJJs-hplK>dMS$$`8xAJ<mI5mSu6t#yD(`7sbGc`S(PEfk{%H{AzeRX3EI#*d*QYQuLfW*+t#nlar!dY0|SXmF#j+%t?J9%qub%IXfQ`qtZ9c9L;d*QyB-;?UpEc}0Ng5F%OEZ3Ia!583p=*#SguEcm~D~pS@_1dkf(Jz}uey@}%L`lcKPGO(CCX?dp<(Vs4(y3kGR%Hnk2tvZopL_kZ%nY*(8ne&JQ1mt^nk-yD=)#JBR;{mw5qxi38t~Inb-fA~71uB#wK%P+!!<^r>1OA2&>o;mxEY{+9asi~K&%TMUN6QCY$Ve*YV`~26s+SjfK)m_Sd1s!u>nm2Q6BI$q>^t@&<^(7pcQokjT!Cj&BjU%7EI;`czf+S^r4}z1g-uv7qh~5K%`&k1!(<3Rb~iH2`Z&b8{6r;t>Td&T8$VOD(Lli3(>8|HUwzotxC!)Pd04>+7@l(6FOkV^WLV+J5S6zJUVUd(N|t-|6$`sse9CIzk8Gv@QJ)};4LNlPxPPsQ-a;GpUt`ca8yfkv&h~W72GZ(4v3L)MRpm4mSjjY1a4~@aClb65*&J9WB${^771&z;0RGLDsY180$nnVoLQMQtu^cQjrE(*Uyualtfs%IgP=#UTE9VeS@HELS`Q<HlZS^_SLc+Klo;W+Vu6&xQs^AW;}buorh#|VP=1S7*QzVH*L<@{`!Ca`HdtX2Ec&jzauvT3SJr?@0auG}R7kMGL0cnJ_pmK{qLfUopoAiHOH2l?yIXL3EH2j?X!MU)D$8&X-_JbMSj#?@Ew-fEP+cjdkZ<^tE~Bi<d!pyDr1rmWR$+5Akl(M@SmY2BCNm0cyS!l8EH5~gN}n(c&mkVv(#9grpdwTfU~0}AD|HjyL}@1(3}J=XxJf_5AYhI?)Zsi`-#`z3e7#!1{{p8YL6_HBE>A3lNt#M3<$`QfK@G!0Q~e|-13QiA3qq1P34&6D%S#vMO4FQc?lfo#4t9h1xT~p*pnv(i^u3dWd1P=7mRM{j2AghXxT&t!>dUy7-Hxf>O0E4~yOqS-a9v?#?{>Ga87@|D#g)eDbrvq=$qEXQ#0sE04cxyV(N%ASTe60};RmV}EJEsSF?qKzK^tR&*up`ARsK{Bi*Sp`8o6ho+Gn0QtO5rBkqcSrVr}I{12xb_wSG&fTQ}cXTV1~iU*)h-OfXoeFN9GptOHeLg<oNgq;8o(#h=xHW<GB=f>zWp1sHiGoQwCe4KVu2N#_!R8j>sh<e(F}=n>h$D6wH_wX(<u?tu4aKS7-sxhkVkq%lEC#_(-V8=Pm+V@eeCJ#+wnk{X9pk>!t5U++`;^S_%2?GmA*ASJDgoJSI&qXS{=5P(cS<|o11Id`x@`l1`9(%tqvCf>O;xPq#LlHqxT?O44%I2s(bTbRgO?kA;PxHU-A2!H<b?TKPb;5kg8p)UA3=PW8R9}0~jc9S;F5ij%;$K-Z{a}izLSO<w3wl%u(sG_J`?`*aVqs^_NQ94nDY@kuAl?^<CGdndwyTC!OlVD$|ttfH84$e6!{+~bp`QR0p$-nC#b>IKHzay^zp4y@{aT7FFufDZWqtMeT8^u^7rU83~kR2<iYtu6mH3VFIg;HM{wVzdu<7XTGe=-wKO-<qd$3P4rM@mlS-F85Zsx5*XC8AwKVo7pREX5?T#PKNjFF`!xC^io0aIrWar1%*+&{F3h|I(?V%*I{d%8`-ZD^0kGfZouP$AN$j+NqZ3hwjlqlxPSX<%{&N8g{pq3pNA~S#GhqW-dGqn}9Wh)CH$Jvp0g8+fs*h0(c==e`P}8kWX2A;^!)Z`Vp6Yzps-5!{W94*n2<I71$<3EEJfs;(Q>P!hfG`jQhd%=@7B^9uoq>11>@SrKFKd^rXfIhkf@QC+i0F%5rIBlsKDlDQ5ayG5w6sQ+^NG7t$#3JQvS1r|3)bg^dGS<8HBc+$(z-|Kx|3D>~YvK8_UjNI;_~2m$#AP;sH|=oT=ec2q;kNuLAVk<V?syKPcVbL;9nx~P-pkZkklQ2}Nx#-dAX1(C{U%1i_8AMT$HFm^?vB9onfa76~CtS)_3*$XH?jm-^S<uxOSlhSU#p4!cvA4)pth^KLc<n6wRQ8{e+*I`HL+nJ{+1*qLpkH#z}<qFa8`Vw56m_{)-BbF*LM`*(GK-S9cc0Kb-Nx1|}Gtaq+=m;n&+JLb_dD<tuZ5Y0Q!kbDPfL~bbwTEMi!oCh=-Yj)hc?Dn)g=wWlNkR8aJIvWBL?P=Bt5Ws^X9#LzXg5iA(Cj(vH(LqrTV)$<cR`Q}y`$FAJXgsu;<}!2R?#d3(@H-pgPd>n&F6e(x{2P)hIfrvveiXV9V_f+SY8I%s<E-g25~YP$snvJi+;^rTQmEruxPmTlQLG$hOc#i7*`xs2s+_bl>Jd)bG$4sFzln5Zh6Kyts-^UI}8#eK8>pX_4?{^%#LGp=ekYN=ygPk4O7ZNCC+y$M8x{qdoid$7%^;0%%~0S>DjaO<N$QDzz9h~<&{PXTbMF^IF{(>5LHjmeW+t70pQ;lKO~{+b<qBs1FsPuCakh}3&YzMW9_C@o<U+Ei}f7V>;8f>p?C;)GJh*h+s16<3oDKFy7<6Nb1k)EK1&u|ZRqy@JfYPay5Y1e&V!ZIbP9SFw&8JOWEKqF8qL=52xB>w^L!_;<K&dP15awQb1k6SC$L(=c~}AhrqoaHPKAYU4}sX=cKh%!*~wWz&<WXziC#rzJ#C~`F6@75Rn^8@+VSITwBuV`sKCrLs!n@1s;k-`;xn(g&y+ZBa(Jq;>Cr$1-FmyWTw8x9UaqV!+$0a+>h<fjg&KO9s9ieIWVOv&O5M1Ww<=`^=`4*R8IfV&3+~wsP1TH~=ck-$pqeVqM`sSS2S!9)OW~QHKKo`qxQpVVdZ(0*{pUYKv+d7+sDpL$!|+f|lwPD!IiIBoTuz|hvqzdCnJEl!RY)Bac*9GGO~Qb)tU!K4NKv}lj6z^(+^m$cVgH*UmPPKw?_ywwIm^CBm78}n944*(E{c=zRvNbBnH>I62?Rz^&uIDvC&d=XpS-?aU0%cMd-SU$6Lqj-1rN+Rc5~c-WJIHLjqU8<AS9mz`mfZr^}jdzvSw%oZ&l{QC};Dk_2I1>(Z}Et^K*SZB97M@g!#6QOn{q$Od`xL15!=pC~BC!h!F+;s(}Oo+#fzD_qWUqOlmZv9z5cGZ^zFOWYs;ah-XCKsT5Zy;fLUj0=BwHe(PhaQY@h<y;ZrU?6eQ<ip&DvEOwbJOp(El@D{7q*4E2+jj*>WS?k3C3Y!)#bR%Y1cCm9Ld@p&cnDBlup&em2cuqzXz8~mQ0FR70CD}XmeUV{O@>Z9#dvukopp@x5Pyxc()TKzB9)F~+5rgaD{ub}OnKb*uZPc4n$Tq>qW2mjCx|RYLK~z9c4=-_nIfBIaT2-<6NML90r)Iiq9BoOp->kG;m3*fE#O!JU-f#i7UnqKb_qD0*x#`Q>=ZYRqYqxO>ua)dgGXzINwv2D;F9xr?vR+;I|Es&!-Z-xF{N2A|=%ARTI9y&tO7SuqfD~z&sBR%?B`!e>mfR(|63JcfE+sJxfhNwaKpQ(nFF}%~tv|N_BU_Cu%kp2K{!!oCxz0I5E@c<!hdAQS%$YOi@}BqdywCgIo#h*%XI;5cQ}yl@KG^1h6CS`<%sKhBdnjASR8}Y+pHL{}8Bq(K{g0tL6~;Ng=0?W6BOjaUTBhzbk&%6F5qe53FXAr7_Sj*ITlv@Augsxrw5y4IK|Rm~K;5rMiuEWE42-y5Ax^fmi>f=U7GB{NYE?^%G_;xqRMMo0Oa7PKuX64*10H@_D2akIV{y7vle7P(Ak@zs5on1=aW)$^IHz?K`%DDQRSdt<+<Yd*TiyQL5W!hIrudULAYi)Rn!m9`qwq{vKnvOupD=#a80-WLI*IYGsj3IhY2jGlE`6mB6_94*v>li`=VuwO4fzORWc<Q&!v9ig$}m?COI<?Te<lz9tg0%7(1Gy_#R4q;{+Dw4nN<D=AD`5Z`UrOfG3s(DR44BqZ^Hq4k%*u8E6pM#cBUwjGSPp_D@qB+Uj^7l>7@IJ){8}zTMfwvZ%NyDXq-puHAoMzc)0a&alt{b6I11MA~<?3O>pXLcyrnc*8L^dFMun5FT-gGoo76QT+&${Ftr+A784qt&Dj-ND0SjF?X6Q$vVW=s*>jZ+fYb3p)5FSuzmU+4O;-LH)rZEefgE9_12^I;*_~3avwFGnpfTC{?$lIs$ZV1sSen1%W386R%KutYq9v2zlq*?y$Kk2H>rbg(lrP@pcyYZPd?#m>yySZH{2IN_XwYSC@+nJO>YGh7X<VP2X*8hKH%K#a$+NLyn+BTQTfVo1p3h_=Q6_Y9b9wE(_ZJcoYnV|7SIJ*?V=Y;{`@=g3TCjB&s&X{`eLPq3Y*D^*{E2cJ6LzEY@duE0`13wkL6Xace>Z!L?SUS7_@ObtBAoE}4kr2~!cX9+*o&PO^A9y!2=G*S1zB=<&};2s?l<yfJ!owW`sk+MiqaHi*wIOI{c6H4T|`WgYvoZ(SOd^W*6p>n+fT;(Tb*vJX}!F?PaQ`yDN8#246L@nZbo64z4m67Y{FMuo5@rZGfTKK1XAr<R3E+0eu5YUOg2oUN%eP@;D3MYP;k%#sF1=t)cqJZ^X@q8?ZFLI&cBe`K0NFUT44>`byBG51yf-4{ac6*AU=eeqf<%_P!`>XU+GG#szPcc`N-Z&v_|9Z39yY-lqVUfCmiRgMHwM#DAc|TCOFB#?_yzS5qsbjTwM50<qDQD$=47ezkDCeh@AIIX*a$O*`H61Xg<}fMIwM`fE5D+jZLIq$f&iHFT%b>uN(9Z6MoT`wm82wU)m?lAZ!;Cd*$|3(<~>2#IpF&ALwbL+h&5r^ybo9nk~$)E=d{^m|Bluy{DsPeWka2O=A7IVjxGY%p&)*#?X7e6}%cjf3u*DO?Hg^YXFOkmoG>`QVXUXp4vKQpah`O#aXNPIh7mu7*)E!hZuPFNc}o?I*!hU8!p`KA`L%Voix#0{QXPWGNK2%^v+UW;6*HXGwhgwQ%ef@aLl)#&L!~MFg1G)h#sN7F02>Pf^Pw9eZys3;@V%-T)60*U2qQX_k!^C`ebeR5msheh^35qa_rJ^^p*Mp$`ET_Nik~4iy4hL1>ul$%DT&CK!JOmyt|99tCwEZxT)muN_K*`;!dqXjY8;dWtqPIC>;$<TWqoe<pb@HbZG&y;}e!i4*A<i-3C?!ATy;BmP)OOXgw{kbE|4=ivRR9v<hr>XO$y6Ta7LjYUV_nCg?rA{q_NsWH(v=NWhVFwv)jwGc*z~lZpKr39Q_`mQSl@={6W=dW3u}7bh))p)~`g$Dc<JRS_QKicbrs-!xjKHQcJ`6<pnIJ!!UgS^;d-cQnWow8LJzvDf}F2Q~K_pvJYgx4C#*`=H<6IUq_D&>s<Ul5E1Bc?cTcN=u<bCCp~x_5A^CD-Wf_QOHXGQejxmMH+JBf;JqfpL@R^GkIjFMd>(3erxLMIi!2mHsw;<UlJRyN07(xy<(F6OCK=HADH)hTOp<|`|~#f9Czwj)c69Ad!w;2+!hUTNq%17AY>kWb^uw4?&O2|2VFYZ)6y*dzukte`)J7%tzuX^@9-eUl6>B_saT?VEd5D3=7?kZ#eRq0OztjPuT4*G*t?uetsHEodb+<iIXnJ{jKguvlNwLQq%XFifTj=~8w?yKhspAfsy!?e)~%!nf`tD#Qh^^Dsi))wpPs-MnW9G%rRuwK<Y<n@kiPoq*`Ln-aQ0tkzehglr%B{B));@-$Q3Do6Gb8rz<D7;JBZC??sa3q3`LDBC|<G&B85oGYO0*;$L>Z06MV=v=7;I~I0;hsFbdTR!}8$F5MHr_?8(*>kdlmLN*M1-?V~k~x3n5`9_<ls&2IlrC)vg@VY0Q?Xz#;e)S%+Ms3r>@D`_FMQ=)LSV0!;(PTjH#M|FaWF%gJd1N{Yz6Hf#!XTMLB7G?yh7g1@0PEO?>G`+huD#$9t59V9_M|DsQCequtX7l^kryOL*(xS~WRE2x3<?cWNJVK&Td_CM=ao<Ei(<c&4_0+g=IKKQo!s)Qz=pB&&iT4r%d4_F@DPQC5M8duW7N!!CSUV0}8vdoP=T&y@K)Vn=cc`&*?%mcb>tQx88fKi8@>}Shhib}ar@dYU*Bti7G&xL1;saGSGzIXZ^x=}DLF9I!!SLmhpa=^C;(!dLFS^l7%nM59etdQ?qilZV#(QP6!yq$vZgjK5bV>N$VxyyIK(HTkPPqwuv3Yo?R}#rvXaI!mjo;?TBE<uz-)!0#@@ql#$ChL0#dadP_1V(OFz<#_t@vIO-h~sJ-mcGYc^N-8%!|12OK3{HpeLQ>p^uhInyL$?!C|^#EqfN#b&t9)!htGUTUwKUWGPaVmNJ@p?J5>eLW1`%N!dva2?3^|-bCBx2^-()qLqw`?pJ2NFe26@=rVA($a~?xwb_Q_eIVS8iuIScEQ`|FxtG)%1vK&+<vWxFRr*6Mh#uDju1ZD8mC}Mk;C3XQFA(kr^FvB3CBiM4KDutPfhuy+kR<lBp)7^nf@o6i<@snVuXFG;(rUz?W}+*N<thTs)0(Bt`ppMPGW%u3r$(*z+bU)LETPflb#MVPP<)9p$29SzZKGx8G9^=`wFpySVF{S2-2w%<0UhDI-lyas=aei}5|@j!GBoAyD8o+}+N^@HPUX9E@eqtj(J%sO!W8#oJPUe(t3EQGs5{T&`V7)imT0~FY+TvBfTAn6d?Wg;ZT+lJY?Be#is=m5H<<&i$C!LZ+|Ox}8lAq!SzWEhqay|Mz5_bNiWp4f*ZZ60dv>{lz0ROt?^E(r<+$JxmqHa(#zj1w+M#`tq0?cnvA@}DB;8uqFIB-&+P?VNIb%Mfe`{A?<~pY{OXIZ_Powx1)GkXMvh@>7T~LR)IZ*A{%{<G>H*+%2B-j|E5wn&GMx^r}x4X2&DrSw%!xH-d0sOi4ij1XIxG_DCTtGK@nVN~S7pj@4Cs-yaUQ)5r%_^oc2{EqYb2JKq5|%d5k<$&;D{+f6^1Y0`J=_nJ7AYr#mj+IPrAEq!GdL3PK^{GWE=^P#oTYd+h8U$LtBn?6K?;-P1><y2JZb&?smaMJ(^sxcPu9)}2HEXXpx_r6_DHaS<aT=@b08`dU47;;{*$bKR#wrV7|P#PUHD82m_IG;&2j8IC-@tyQ`;xW*&pHF-^RoC0;fIu1dgds&i-)ryR%Q>f4@Kbq?TsZ2Kh;vrRogyYREn{(A^lHF7=0aim5N#dx!nq2!241iU3;V<XOECF9rF-b~>FMSe9E9K@offsd~69=~w3C57AEOi&b?dAghqIw8Ojr%k&mnMzLNk-#Z%Y;@)fbL4i6_9mAb-t9|e=li!O^%Y}-%)9+B^e7lEdIe8e-EqAz2d5>GWhX<U|3#~q!V<ZsgOYU!i@NH7OTh2g1JMbB3d-7mLK_To7>bP=n&Tb;~;?~mqDkb!<<+qm?MtPqVzy?#-zE}F#_v)aGmF!M@Z$f$16oq^S4OXkja}B%FO`C!*1s%F4$A$8z^9Y!uVsuX5IZSAqA7YE~IAZL{8LBh>YkWRyG91d@{&uv|6o(0}uWy`qLfh*DGK9h4Y^Ef$`@JoHn{~k4tRs4=y47m$MG3R@nh4N%wdpuP$X11#3&q5U%VC=ib^QvIYuMy*du*yUd9`^GNUG`9_4G|(8-bQ7Q&smbCZfjMS7P5Fr<W;!LdsteVsH@QY=BX%_FJuom4j?5l<bf70mDxY_Ph12S6B!;h}DJC2z>(0GFBhrseLR{i}<;c%8j7!HrfZMH6O%pdb|rF$lO{h&vDv3+^*xNQ#cmqV1`9PrW+U?EpI_Y1~eoBVfn6)+!Dorj2+6ou;8LnME%+8Us|ji<!{7M5_JsTIy$GjI!?*$_*-<Qbd9ENkdQ{pi*?zY8-~ytSFYl-mYsbNDftp;`yzTpfTNrc+RQz!ts2~FClklCO-nKn`PiTu4Sq1xM}VNXM&hHu_tyed%`0X*<6Y}KG#kF4+OCBw^}z%Ywg8N<3L8wdZC-$~_(ro$WFfacpnBLP9x8WkVd>fTSaI5sx-F7xrMNh>x&KVOUAP|gD64`_b`bw_$tA*2z8m*;BKFLu9gHuTgLKt5v&~ZxBiovu(rSYjHLl-l?IGzL%?Ya!)nxu4ApuJH>*&2i8=%)Z?1R>E_s*>kaJ%!pX<t{bpg<;`_HYDk?N(i)c{b8>b+offw4(lOHw$&)n6~}6p_WyxVaT)tD9EwxbV6F|PaK<ePbS3sn!y@v?Ds(ffUkw411l<lg>Yh*$a1^P%9Pt>tH)V4+lO`JcU<^)x}$Tj_8iQUdtYM@*ZNWY+<_r4Z_tK>Nsv~A%9j3Zs2Q2Jkw2r=Op7=1KB2o<B-9B0AZ+l1irXSgZJLcwjRuHwYL}68sVJ?u#Z+2zVUr=ZKYx5w!kKfCmW*^dHdG;4ORL|luP&{wRvG<Xp#^2FC!UlE29-%W>nXCmk7g)K5Pw;@I+;y@&<b(fk>j?GuMsD1j-Zk``eAFY({})1)``kDkJP6pXKLpGkX5P<ncW3TT#i{u3K-`-orYZ#vhELid!Xg*4hG$RZDN9d>{pqL4=GM2(CYwuO;vk4Dc^tIF1f%tD3)p>zK;{kwK#qg^Eqyd(s|asz`^A%7x?}`#SE7bF{~s=zq4IfFCbyz_Zu9m8DBf<!gUlMCmCj-)VG~=boJWgFbk<<?bKoIOw~G@5BMPxnlZf=?{e#u+Fu-As6P8uV93JFV(pL32a5SK^4OcnMyeJ9?|@SS{?8GMBeqzyuPcc-;z>j_jbNRccC;`Yxri_<^<bH_GZBdH?i@?!6hGWoie&Vu0`GG+pp%&BX2^f^`ZreQu?zdP%cE>Q2pxl)U<z*S5>RbzegzKOd|_q&&W-nMhc-<SF7Y}F1o~HnpL2CI>&}t7wD#T-Z29>$lP_4nr_RH3VEzlM4%Oe%<+dg7h~T^E-^I$Jn}-7{XNj|qguPzsd0C9|)r-gmr3l&haRQ91ij@^c=8ZXp%6cuZ!Mg+T<iZVg>-$4{TaJ~y3EQDwQ7;u9TM0$O$t(YqS}^uazB>YLcA$(yZ(OHGq+ABKn0;H>DMc7{$zW1dpeY^VlsFNs<ar#=z4p%Tpvnx2k||lBQo*JDE7BaEb4IjkRT6@$_F^XySN2J^PCU;Ub7Cb@)*NQAAwGOCuykO=*SWPBgolr6DpOswAIuA{kb~#D3oE(z7{`hhO@9{w=<UYNL5I-iP72>#0{{027VcgXG1ds}Q>lD#%5hcRYQcGD#eaPbmA%M3EkBr_2<KAk=p`l1vhfK6F23|HRMHJh(Vvvpn^WXpND41s{7=s5WTp-~5%>!)TbhfJn~dYvzo|x?&z3*j+HhE1)U&f}HV&*e>YG~l$U&>=olfXJ^4c}=y$gA^AK0Rz(3VLy3FMeq2l#{1y)RcWIg%>o!sVz;Rp0sG91(!)^t@pfxprv`jfItlU(FxBb5&gewJNO{HPli+Ly9LfWRy?%<0Ku@X-n>oTrr}kI>cm+hg`rHs&6bHhuP_&ff^P<L4h<dlyEU*hKmo#L$8jIsHIXimCrTMW9k2k?Q7wfcRR2^V471&H5T?jNtGV-S}o$nn?un~G=r2j%vr>|o+aTpExv_LO-4#tML@BjC#>*$^9VOV)~F9)>FZ)ILF|JpT_h6UWl0e|u)_~U95R7cF723swsf6t>LP9&^-FHOsUXYX!3$-$1%wVhyB;dDYOxP-Zmx{5aPAO$xmmpu`CeR^e|IfcELk1}7{4k;{}t~@S!Yd+*NysdT)>oX{gdDv#Akf}8wR-;2L&rqUgD2z1HCrF3aGtUBAREoT_x4uRIe>4h}MeqzhTs=A)=E8oh5;Gn6C2cu0h!4Lm67zh<Gz}3jA74^@zO9pze*s2>KRMaXFCNzxX}AVO0<u?sIZO#4Gzl%<$llMxR`sqWFc9auV2ZODncv#C~#1DHMrL;d1}%R?|!Yy~X!MVY<D1bA|p@rKCoQh#HjAP;Dr;PW8!N|A`8&GWaRnPH`pIta*q$6kVq^doscgdk4LB@Q3er&~WsGgOCb8qmf+lgh6%%@3$aU{D6)B3O=DhVSK;Ah?Vt61IL_R88}(+S%Wd-r!34__<jRKlLsthTJQ<RsPWB$R}0?fg4Otb55N|Et~iSYtXuq@{_OaPGC?Io8_j!_hv+u~brYSBB5N7b^^i)9TRV2NHdJ=>bK(X;8;S`sTth3CTS+h!1XO?|G?i&hP4wgnV@<9kTX0NpXej9^cBy_RIcPnGUhgDSOuL^9I?4TO3Hm2FTN=&R`_ok??T_lT?>6i+B=Jr{)+!1nnyq)JT8$+KC>nKa1qBWb{k8Q_U@1kTV?royCekr!XFb)AkpZ?5ERCbTF!of?^walH{1p){pes#oNLk(yd$`IxE-wh2{5es@oodd2kI$akj9%<`VPYmG(n15$KLjJd2bj|wwfmBR!^=2qR-cWZ9OUvL*o}`c7IzXhgVifoIRJC!U<+M)>F7BT5sUDPIJww^^ps0zXT^+I%}4f3>NULz0T8e^L}OV$exk~2n4}87udNmp{=v_XWDQJ2r+6DGSR9d)E7Vq&BC5Y26;49#@qPzmRSKr8J2s9{N=JAEAr9mhb;^O;v3Y!u>|Jb<N(b^=>~>Whm{`4)O;EUcM-mN%|A-O7O3-NXon)1Gs~GLa)}fBNxBEI&g;ye_PK3&ZUo`stnb~vkqD&|Vl2Rz+{>+YZ_~8{(1|>KzNjjzmAOa!5yR0>T@9wI*=<#}=0lcuvB||5zF_l$y^V*E0l}p#F%@${wgOMV}0+_dacX|@7y+U(nkSVl+YI^D@#R8g{nw-2gGXqPD)?STQ9wAnsL-22rDxufdOM%w6EV|ma*CQ14Cy|ZE;`Bm7I&PG(+s+YLh3O-irKgX4Y_yqcrWfnHy_HPP_|KXQLFfxYr%d6^@&)SM;c-s)LCq%s!?r{B%iM9=#IYPSwd7bla3=|h3hhAV-Pdak+DILeMk*G{$KikV_o{=bwr-7ea1~jnpmnfCPCs(v9tRwU+S6IHPZIAqt(kX4i!Y{X6|=mANtM5|6f_u?ZD}na@Iqp^;N6Wl4B*nGeL_QF$A0gf`Ii1>km{M{S|lz?UG<7U>@RF4xo1qEuZF0M^4``YZ=i;Vilh_Bs3vQ>Em9G0zrVW1dLpWc7}e{t?%8CW5&LTieRO&%A8%y)tIKhmPl2zaet@%7d1B6Q`#tud-PoZ*P2__&9;MODtBOfMPG(&U$F<dqBLcvDLu~h(uDGr(S*AR2?f$O45ioXbZ<#==++skMXy|p8(oCi`LRw?ipS)jw2{gLdX&#}363=cjJ3TQwT}^H{;WG3$gK)C6`IBn0s@>kn^@;0=ad*R=#@`J$#+m!K(<KYRNAR>`$K6*=-aRDN%>LmX7%Q8ty$;rA>2N-nkm*rkO}~v!>S`KTL3z8W76ikAJU}G@#BgD)S1f-*3g@E|8WZH77~L3Kx6qVN6z0vnwX-#74aWN*g&X@)(ez0SuZay>86kEvlx^h$IxmQIv$iB*G0sP5_zZJ$OMTrM5ZzJebzl_E`O+x+oV?2W(ng(sJ{LMd8>dkU8)?KrBTk~0php_tFm}Yod<6kshS%}c>tDlNU)n5#FJ6BXLV(4r?ArOk|DgR=0EG-;z9Pxn&xN_0NEH7pJj*6djpv(&7Cg5Pwm^jWD+eZk!xlPcE$NIEZXEp#?6Lj?b{v&VB7&9A%KDA%Rvn3(l-lU)BPlFT%q*dkKVs|#2FtW?9t5z+i%DQW93eF6o$>@0+qvrxV${tIL@^|oCHsM##)x{r<(zCDB|SL*o8!n-g~AnOD`ibFdMU%>{2*avhnzl=&xMYq%|3|oYBBAkU~h&D%^-3USCSXD(2Zeq!VF({#63tnH#q8o$gRDMc5Dn57@Zw~b<wb?QNzds;iTinCPgeR=v}ea*@c`d@a4MsVLU6im?)g9Z^zl<*a|^k{Izt@c^q_==(j6FiuA2@HVHRe2Z?(6e9s6;rN<-nf)AuThxDhyJolWCNXU>WOyroV@@pncGi4xHO;Sv*jEDJ}pEp$no~6YRmt#>54brE=i@KmCI&w&(9T<;n9kxpJh^{8JZGC0!q!{2r1X8Pcj%r<eU-1)lTh2!k&8vIG=+q;&PCYCys3Oo-Z08|C_&E^)|Md080Xf$>*lzds$pUd**Z2!Mw0UE(2OEDAQ#rXmT{Kg;1o94wo%oK0s|Zsu=ZCGx^C40|1mXQ)J#*Vaxw?sD>{i&6KtP50;3r4w70Vt79Np|>ckzVEBFkM9dvrLOOl1(^=r$F}ykq&P;)&=1OB7G8KT+IRA0aKxMDg~MZ%|pcLcU^G%7u&C8Dc{x{KnBLg=NqmnA@dMhO<!ZKv9F(RY@H;;FdYI8l-zTo*aPf`)?vk*Qy;!dZl1nD~wOY5X-1lC~`tAZR^yfNN=mzuU<wcD~Di|gxh0HMSvr3)Q*MeutO$h6~imcA49ijWEiw#{usI)8<vD04WF8?tt>53c`B9WMU+qZ;_}U<)wR_;QgQC`pp&5s$Ax@XDqJoJcE{zoEBFMN)u_iXB0U_1*qVi!Wot%SRClmmhM5l}yAcG(P6@(BOywFA8$RtK*o<!z)TLgVlMizUFLtT&nx7ZMVYCA&K*a4`1{1QD-)MHRdWaW-2bpn>#))!DfT1uE_0^3NMV=Wd02BY7lF$rw>vEBqJQ4`u3AsoVBtIuGlMfZah-@e;u3@jtNMs`~32mrpFRtx34RE;S4DHwkABt!nh<Rn7tI{9Ke_0yX$Rm*rv;SZgq9KcO?e<~qMOc&yK+DY`Z<Q9qit>YvF8R`L&yNuqI9{giykFME;Wab#c=1St=3msf#|_pyk&_;jIk#x@|3i@(@>|#*5dI9bL%g9Kje0g%^p>$sTw?f<#1J>D(OlCp{<6|2q1Wk9B+SZS;n$Z4fv`nDBw|Vx3FeuU;9mZZa>8Jqy+p96Rz^r!OD`LG6LFf|Hcl0cm?qIvww4;rIV`Gt$bzRs)xLVI4-e5KPRl3N`c(J6NYz{OEjibOrvhq=X)u@R>|f75Is4<;KO--e^3VMH*`Ln->Fm?9e}td^<LqDH-*mZlv58j=e`>R58GnN*Ht0o9pMqfEbcuA%c1F8}L@n?URW?2Q3l)h(pFE+DKI7V@3bS+?$%G25&hTh5CVyq={_@ffb4*^#@7`L>SC?)gqdH~^tuC$Hj}Gp?N%<<`yY%=eHAM;4H~b6251;x)C|{ZYu$vF&y663h8Z<6cCz48Ko}Uh;bCa%I390X?WjzjWhomHD@_?~h3yrz|H14<jaIlruR8_j_LFaKr6{f4gC8Uw;a9xy&?bxiuv)XRlz4Pw!%5BDJb0{%l&%AMB3a<sXtYeK%L4Tmvo#d^J@T{4&^~^(Im9XeJG<WATdkS0WS88oqQLnR8XeBYF>`g#f@>iq|Z@vJVWC8cZ{KE3B<+TsuRCsoj%q~-10BHDI_W~Rl!HVmO4u}i;O1(_-UaOMRv|%|!HkW{<@mIm}$>sIb;!uyxMz6jDryMMwint$d&oAFWD(Rch^;M7J5pH)mB;f~(PC%FVd@I44n@(BIw+JV^*VvRlh?x-v3p7Wn$*D_&zTIr~w?M<dg@~m-9V~;%qhnYDv@z&Fvs7kI;S+@6I&;fxW=X5j1GRyx)+VbzxP|KZpx4+MWXa%hXPg*D;SfX>NTxHh+>Qbyu(k;e%-auJJ*@uGZEfi<6Y@ZEi{mFS31M<os+I16`E>|g#Y<hfBp1~E{NvTP^^mA1^7g@YN596s-Q2wo2Xya9Qu*+PCVX;pGoPKBzP&J)Fzx_9snPrJumy?$(h??y|G7VxY@+r6w_r-?9)Blhs+hTJcUy>%$JTKoW~cM{#TEGs2@vE|AQbNi=DNtuR%WE>Fdu9Se3mI<@Z~%2-W3TsxBA!1_wx7e%-@IOWPah+QqC!}t8d$-UPbG~24}xJbN%Y}JMYZS%uF^Ljjic-X15wwr)RIiKi=7%nw;9)ZcNRz8uGm4z~Ym%?yRxJZzx>0E`d$d>Fk3N{sb>iPflMOpPCw<oS94R-@BVn=TrHV)Q)NFz#7BEjs(uG9?>8lB}cGGnZ#rp#N%dT3sVT1xZpay30!3D#@^tFjD{?^dUF9&*%O$>-jV9WY`%He-lHnHylHYOyD~YQb3JF?*!jE5Kk}Z)v9cLHwg7((q?!#()z(D+=wPth!gO||l5I&sJmP$1GwD2kVJ5;_HRDp2iuNgLq(t4orG3dJqhf1SEQn*;Cj}KG4DnMn`lhyxz<a8u&(<W|w`QSx*z1BkQVTP2KugJ3a$T3*9iv@8Xt#RKrhbcN`fXL%3>uoT|IYW@2b~`LI{Vhi{{o-jJ!S"
def notebook_from_embedded_source():
    import base64, zlib, nbformat
    raw=zlib.decompress(base64.b85decode(EMBEDDED_SOURCE_B85.encode('ascii'))).decode('utf-8')
    nb=nbformat.reads(raw,as_version=4)
    needle='EMBEDDED_SOURCE_B85 = "ML5_V711_EMBEDDED_SOURCE_PLACEHOLDER_a7f1"'
    replacement='EMBEDDED_SOURCE_B85 = "'+EMBEDDED_SOURCE_B85+'"'
    cells=[c for c in nb.cells if c.cell_type=='code' and c.source.startswith('# V7.1 · embedded deterministic notebook source')]
    if len(cells)!=1 or needle not in cells[0].source:
        raise RuntimeError('V7.1.1: invalid embedded source')
    cells[0].source=cells[0].source.replace(needle,replacement,1)
    nbformat.validate(nb)
    if not any('stage2-ml5-v7.1.1-advanced-data-reconstruction-sql-progress-fixed' in c.source for c in nb.cells if c.cell_type=='code'):
        raise RuntimeError('V7.1.1: wrong embedded research version')
    return nb


**Совместимость:** некоторые внутренние имена `V4_`/`V5_` сохранены для воспроизводимости исследовательских процедур. Все результаты, манифест и публикация относятся к V7.1.1.

## 2. Среда Colab, зависимости и статус
CPU достаточен для первичного запуска. Библиотеки устанавливаются только при отсутствии. Результаты SMOKE не публикуются.

In [ ]:
import os, sys, time, json, re, hashlib, base64, io, math, gc, copy, traceback, platform, shutil, subprocess, importlib.util
from datetime import datetime, timezone
from collections import Counter, defaultdict
from urllib.parse import quote

DEPENDENCIES={'psutil':'psutil','numpy':'numpy','pandas':'pandas','pyarrow':'pyarrow','duckdb':'duckdb',
 'sklearn':'scikit-learn','scipy':'scipy','catboost':'catboost','lightgbm':'lightgbm','statsmodels':'statsmodels',
 'networkx':'networkx','xlsxwriter':'XlsxWriter','nbformat':'nbformat','requests':'requests',
 'tqdm':'tqdm','joblib':'joblib','python_calamine':'python-calamine','matplotlib':'matplotlib'}
missing=[pkg for module,pkg in DEPENDENCIES.items() if importlib.util.find_spec(module) is None]
print('Среда: Python',platform.python_version(),'· проверка зависимостей завершена;',
      'отсутствуют:', ', '.join(missing) if missing else 'нет',flush=True)
if missing:
    print('Установка зависимостей; дальнейшее выполнение начнётся только после её завершения',flush=True)
    subprocess.check_call([sys.executable,'-m','pip','install','--disable-pip-version-check',*missing])
    print('Зависимости установлены',flush=True)
import numpy as np, pandas as pd, duckdb, pyarrow as pa, pyarrow.parquet as pq
import requests, joblib, networkx as nx, nbformat, matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, PoissonRegressor, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# Ограниченный DuckDB: промежуточные сортировки/агрегации — в локальные spill-файлы.
DB_TMP=ROOT/'duckdb_tmp';DB_TMP.mkdir(parents=True,exist_ok=True)
def duck_connect():
    con=duckdb.connect(database=':memory:')
    con.execute(f"SET memory_limit='{CFG.DUCKDB_MEMORY_LIMIT}'")
    con.execute(f'SET threads={int(CFG.DUCKDB_THREADS)}')
    con.execute("SET temp_directory='"+str(DB_TMP).replace("'","''")+"'")
    return con

def memory_point(stage):
    gc.collect()
    try:
        import psutil
        proc=psutil.Process(os.getpid())
        available=psutil.virtual_memory().available
        # Colab может использовать cgroup: psutil.virtual_memory() иногда отражает память хоста.
        try:
            cg=Path('/sys/fs/cgroup')
            if (cg/'memory.max').exists():
                cap=(cg/'memory.max').read_text().strip()
                current=int((cg/'memory.current').read_text().strip())
                if cap!='max':available=min(available,max(0,int(cap)-current))
            elif (cg/'memory/memory.limit_in_bytes').exists():
                cap=int((cg/'memory/memory.limit_in_bytes').read_text().strip())
                current=int((cg/'memory/memory.usage_in_bytes').read_text().strip())
                if cap<2**60:available=min(available,max(0,cap-current))
        except (OSError,ValueError):pass
        info={'stage':stage,'rss_gib':round(proc.memory_info().rss/2**30,2),
              'ram_available_gib':round(available/2**30,2),
              'disk_free_gib':round(shutil.disk_usage(str(ROOT)).free/2**30,2)}
        print('RAM',info,flush=True)
        if 'progress_event' in globals():progress_event('MEMORY',f'{stage}: доступно {info["ram_available_gib"]} GiB RAM, RSS {info["rss_gib"]} GiB',announce=True)
        if info['ram_available_gib']<CFG.MIN_FREE_RAM_GIB:
            raise MemoryError(f'Недостаточно RAM: свободно {info["ram_available_gib"]} GiB на стадии {stage}. Последняя диагностическая стадия записана в logs.')
    except ImportError:
        pass

START_TIME=time.monotonic()
STATE={'technical':'NOT_RUN','scientific':'NOT_RUN','publication':'NOT_RUN','overall':'ASSEMBLED_NOT_RUN'}
LOG=[];PANELS={};PANEL_FILES={};TABLE_QA=[];ISSUES=[];SOURCE_AUDIT=[];SEMANTICS=[];QUARANTINE=[]
PREDICTIONS=[];PRED_PARTS=[];PRED_TOTAL=0;METRICS=[];MODEL_RECORDS=[];MODEL_FILES={};FIGURES=[];SPLIT_AUDIT=[]
MODULES={f'ML-5.{i}':{'status':'NOT_STARTED','reason':'Ожидается DATA_GATE','n':0} for i in range(1,8)}

def stamp():return datetime.now(timezone.utc).isoformat()
def note(stage,message,status='INFO'):
    LOG.append({'utc':stamp(),'stage':stage,'status':status,'message':str(message)[:600]})
    if 'progress_event' in globals():progress_event(stage,str(message)[:150],announce=(status!='INFO'))
def fail(stage,why):
    note(stage,why,'FAIL')
    raise RuntimeError(f'{stage}: {why}')
def hash_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024**2),b''):h.update(b)
    return h.hexdigest()
def norm_code(s):
    return s.astype('string').str.strip().str.replace(r'\.0$','',regex=True)
def number(s):return pd.to_numeric(s,errors='coerce')
def safe_year(s):return pd.to_numeric(s,errors='coerce').astype('Int64')
def nonmissing_code(s):
    x=norm_code(s)
    return x.notna() & x.ne('') & ~x.str.fullmatch('0+').fillna(False) & ~x.str.lower().isin(['nan','none','неизвестно','unknown'])
def canonical_hash(o):return hashlib.sha256(json.dumps(o,sort_keys=True,ensure_ascii=False,default=str).encode('utf-8')).hexdigest()
def _disk_schema(df):
    # Примитивные наблюдаемые типы сохраняются, смешанные метаданные — строки.
    fields=[]
    for c in df.columns:
        d=df[c].dtype
        if pd.api.types.is_bool_dtype(d): t=pa.bool_()
        elif pd.api.types.is_integer_dtype(d): t=pa.int64()
        elif pd.api.types.is_float_dtype(d): t=pa.float64()
        else: t=pa.string()
        fields.append(pa.field(str(c),t))
    return pa.schema(fields)

def write_df_chunks(df,path,chunk_rows=None):
    chunk_rows=int(chunk_rows or CFG.EXPORT_PARQUET_CHUNK_ROWS)
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    schema=_disk_schema(df)
    writer=pq.ParquetWriter(str(path),schema=schema,compression='zstd',use_dictionary=True)
    try:
        for i in range(0,len(df),chunk_rows):
            part=df.iloc[i:i+chunk_rows].copy()
            for field in schema:
                c=field.name
                if pa.types.is_string(field.type):
                    part[c]=part[c].map(lambda v:None if v is None or v is pd.NA else
                        json.dumps(v,ensure_ascii=False,default=str,sort_keys=True)
                        if isinstance(v,(dict,list,tuple,set)) else str(v))
                elif pa.types.is_integer(field.type):part[c]=pd.to_numeric(part[c],errors='coerce').astype('Int64')
                elif pa.types.is_floating(field.type):part[c]=pd.to_numeric(part[c],errors='coerce').astype(float)
                elif pa.types.is_boolean(field.type):part[c]=part[c].astype('boolean')
            writer.write_table(pa.Table.from_pandas(part,schema=schema,preserve_index=False,safe=True))
            del part
            if i and i%(chunk_rows*10)==0:progress_event('DISK',f'Parquet {path.name}',min(i+chunk_rows,len(df)),len(df),announce=False)
    finally:writer.close()
    return path

def qa_table(name,df,keys=None,allow_empty=False):
    if df is None:df=pd.DataFrame()
    if not isinstance(df,pd.DataFrame):fail('DATA_GATE',name+': ожидается DataFrame')
    if keys and not all(x in df for x in keys):fail('DATA_GATE',f'{name}: отсутствуют колонки ключа {keys}')
    duplicate_count=int(df.duplicated(subset=keys).sum()) if keys else None
    if duplicate_count:fail('DATA_GATE',f'{name}: дубли ключа {keys}: {duplicate_count}')
    if df.empty and not allow_empty:note('DATA_GATE',f'{name}: панель пуста','WARN')
    TABLE_QA.append({'table':name,'rows':int(len(df)),'columns':int(len(df.columns)),
                     'duplicate_keys':duplicate_count,'status':'PASS' if len(df)>0 else 'EMPTY',
                     'unique_keys':','.join(keys or []),
                     'storage':'PARQUET_FULL_PLUS_PREVIEW' if len(df)>CFG.PANEL_IN_MEMORY_MAX_ROWS else 'MEMORY'})
    if len(df)>CFG.PANEL_IN_MEMORY_MAX_ROWS:
        fp=write_df_chunks(df,ROOT/'derived'/'panels_full'/(name+'.parquet'))
        PANEL_FILES[name]=fp
        PANELS[name]=df.head(CFG.PANEL_PREVIEW_ROWS).copy()
        progress_event('DISK',f'{name}: {len(df):,} строк сохранено на диск; в ОЗУ только preview',announce=True)
    else:PANELS[name]=df
    return df

def limitation(module,reason,required=''):
    MODULES[module].update(status='NOT_FEASIBLE_FOR_PREDICTION',reason=reason)
    ISSUES.append({'module':module,'status':'NOT_FEASIBLE_FOR_PREDICTION','reason':reason,'required':required})

def diagnostic():
    mem='unknown'
    try:
        import psutil
        mem=round(psutil.virtual_memory().total/2**30,1)
    except Exception:pass
    info={'python':platform.python_version(),'cpu':os.cpu_count(),'ram_gib':mem,
          'disk_free_gib':round(shutil.disk_usage(str(ROOT)).free/2**30,1),
          'gpu_available': bool(shutil.which('nvidia-smi')),
          'versions':{}}
    note('ENV',json.dumps(info,ensure_ascii=False))
    print('Colab environment:',{k:v for k,v in info.items() if k!='versions'})
    return info
ENV=diagnostic()

# Прогресс V7.1.1: истинное число завершённых этапов и отдельно текущая операция.
# Этапы меняются только в mark_stage(). SQL/INPUT/DOWNLOAD/MEMORY — подоперации,
# поэтому они не сбрасывают таймер этапа и не подменяют его название.
import threading, html
from IPython.display import display, HTML
PIPELINE_STAGES=('INPUT','CATALOG_52','SEMANTICS','PANELS','FEATURES','CROSS_SOURCE','MODEL_DESIGN','UNIT_TESTS',
                 'SEVEN_MODULES','EXTENDED_MODULES','STATISTICS','VISUALS','EXPORT','QA_AND_PUBLICATION','DOWNLOAD_RESULTS')
_PROGRESS_LOCK=threading.RLock()
_PROGRESS_STOP=threading.Event()
_PROGRESS={
 'completed':[], 'phase':'BOOT', 'operation':'ENVIRONMENT', 'detail':'Подготовка окружения',
 'stage_started':time.monotonic(), 'operation_started':time.monotonic(),
 'current':None,'total':None,'last_render':0.,'last_log':0., 'last_log_text':''
}
try:
    _PROGRESS_HANDLE=display(HTML('<b>ML-05 V7.1.1 · старт</b>'),display_id=True)
except Exception:
    _PROGRESS_HANDLE=None

def _progress_html():
    with _PROGRESS_LOCK:
        s={k:list(v) if isinstance(v,list) else v for k,v in _PROGRESS.items()}
    elapsed=int(time.monotonic()-START_TIME)
    phase_seconds=int(time.monotonic()-s['stage_started'])
    operation_seconds=int(time.monotonic()-s['operation_started'])
    n=len(s['completed']); total_stages=len(PIPELINE_STAGES)
    phase_pct=100*n/total_stages
    current=s['current'];total=s['total']
    numeric= isinstance(current,(int,float)) and isinstance(total,(int,float)) and total>0
    unit_pct=min(100,max(0,100*current/total)) if numeric else None
    unit_label=(f'{int(current):,} / {int(total):,} · {unit_pct:.1f}%' if numeric else
                f'{int(current):,} обработано' if isinstance(current,(int,float)) else 'Ожидание завершения операции')
    # Никакой искусственной оценки процента во время fit/prediction/SHA без известного total.
    current_bar=(f'<div style="width:{unit_pct:.2f}%;height:11px;border-radius:8px;background:#0f9b83"></div>'
      if numeric else '<div style="width:30%;height:11px;border-radius:8px;background:#0f9b83;animation:ml05pulse 2.4s linear infinite"></div>')
    detail=html.escape(str(s['detail']))
    return f"""<div style="font-family:system-ui,Arial,sans-serif;max-width:980px;border:1px solid #cdd5df;border-radius:10px;padding:12px;line-height:1.45">
      <style>@keyframes ml05pulse {{0%{{transform:translateX(-110%)}}100%{{transform:translateX(430%)}}}}</style>
      <div style="display:flex;justify-content:space-between;gap:8px"><b>ML-05 V7.1.1 · Advanced Data Reconstruction</b><span>{elapsed//60:02d}:{elapsed%60:02d}</span></div>
      <div style="margin:9px 0 3px"><b>Этапы исследования</b> · {n} / {total_stages} · {phase_pct:.1f}%</div>
      <div style="background:#e4e9ef;border-radius:8px;height:12px;overflow:hidden"><div style="width:{phase_pct:.2f}%;height:12px;background:#2863b5"></div></div>
      <div style="margin:11px 0 3px"><b>Текущий этап: {html.escape(str(s['phase']))}</b> · {phase_seconds} с</div>
      <div><b>Операция {html.escape(str(s['operation']))}</b>: {detail}</div>
      <div style="background:#e4e9ef;border-radius:8px;height:11px;margin-top:7px;overflow:hidden">{current_bar}</div>
      <div style="font-size:12px;margin-top:5px;color:#65717e">{unit_label} · операция {operation_seconds} с. Известный объём — реальный процент; неизвестный — анимация без процента.</div>
    </div>"""

def progress_refresh(force=False):
    now=time.monotonic()
    with _PROGRESS_LOCK:
        if not force and now-_PROGRESS['last_render']<1.5:return
        _PROGRESS['last_render']=now
    if _PROGRESS_HANDLE is not None:
        try:_PROGRESS_HANDLE.update(HTML(_progress_html()))
        except Exception:pass

def progress_event(stage=None,detail=None,current=None,total=None,announce=True):
    now=time.monotonic()
    with _PROGRESS_LOCK:
        old_operation=_PROGRESS['operation']
        if stage in PIPELINE_STAGES:
            # Не возвращаться в предыдущий этап при nested helper calls.
            if PIPELINE_STAGES.index(stage)>=len(_PROGRESS['completed']):
                if stage!=_PROGRESS['phase']:
                    _PROGRESS['phase']=stage; _PROGRESS['stage_started']=now
        if stage and stage!=old_operation:
            _PROGRESS['operation']=stage
            _PROGRESS['operation_started']=now
        if detail is not None:
            _PROGRESS['detail']=str(detail)[:230]
        if current is not None:
            _PROGRESS['current']=int(current)
            _PROGRESS['total']=int(total) if total and total>0 else None
        elif stage!=old_operation or detail is not None:
            _PROGRESS['current']=None;_PROGRESS['total']=None
        line=f'[{datetime.now().strftime("%H:%M:%S")}] {_PROGRESS["phase"]}/{_PROGRESS["operation"]}: {_PROGRESS["detail"]}'
        if announce and (now-_PROGRESS['last_log']>=1.5 or stage in ('QA_AND_PUBLICATION','DOWNLOAD_RESULTS')):
            _PROGRESS['last_log']=now;_PROGRESS['last_log_text']=line
            print(line,flush=True)
            try:
                log_path=ROOT/'logs'/'live_progress.jsonl'
                with log_path.open('a',encoding='utf-8') as f:
                    f.write(json.dumps({'utc':datetime.now(timezone.utc).isoformat(),
                        'phase':_PROGRESS['phase'],'operation':_PROGRESS['operation'],
                        'detail':_PROGRESS['detail'],'completed':len(_PROGRESS['completed']),
                        'current':_PROGRESS['current'],'total':_PROGRESS['total']},ensure_ascii=False)+'\n')
            except (OSError,NameError):pass
    progress_refresh(force=announce)

def mark_stage(stage):
    if stage not in PIPELINE_STAGES:fail('PROGRESS','Неизвестный этап '+stage)
    with _PROGRESS_LOCK:
        if stage in _PROGRESS['completed']:
            print('Этап уже завершён:',stage,flush=True)
            return
        expected=PIPELINE_STAGES[len(_PROGRESS['completed'])]
        if stage!=expected:fail('PROGRESS',f'Нарушен порядок: ожидался {expected}, получено {stage}')
        _PROGRESS['completed'].append(stage)
    progress_event(stage,f'Этап {stage} успешно завершён',announce=True)

class _ProgressBarFacade:
    def close(self):
        _PROGRESS_STOP.set();progress_refresh(force=True)
MAIN_PROGRESS=_ProgressBarFacade()

def _progress_heartbeat():
    # Обновляет только виджет, не засоряет логи строками [Пульс].
    while not _PROGRESS_STOP.wait(2):progress_refresh(force=True)
_HEARTBEAT_THREAD=threading.Thread(target=_progress_heartbeat,daemon=True,name='ml05_progress')
_HEARTBEAT_THREAD.start()
progress_event('ENVIRONMENT','Среда и зависимости подготовлены',announce=True)

# Лёгкая машинная проверка интеграционных инвариантов в самом ноутбуке.
def validate_table_contract(name,frame,mandatory=(),key=()):
    if not isinstance(frame,pd.DataFrame):fail('CONTRACT',f'{name} is not DataFrame')
    lost=set(mandatory)-set(frame.columns)
    if lost:fail('CONTRACT',f'{name}: отсутствуют обязательные поля {sorted(lost)}')
    if len(frame) and key and frame.duplicated(list(key)).any():fail('CONTRACT',f'{name}: дубли натурального ключа')
    return True

# Проверка дискового ParquetWriter до начала тяжёлой загрузки GitHub.
_probe_path=ROOT/'derived'/'_memory_writer_probe.parquet'
_probe_frame=pd.DataFrame({'region_code':pd.Series(['01000000','02000000'],dtype='string'),
    'test_year':pd.Series([2024,2025],dtype='Int64'),
    'measure':pd.Series([1.5,2.5],dtype=float),
    'period':pd.Series(['historical',2025],dtype=object)})
write_df_chunks(_probe_frame,_probe_path,chunk_rows=1)
_probe_pf=pq.ParquetFile(_probe_path)
assert _probe_pf.metadata.num_rows==2 and _probe_pf.schema_arrow.names==list(_probe_frame.columns)
_probe_read=pd.read_parquet(_probe_path)
assert _probe_read['region_code'].tolist()==['01000000','02000000']
assert _probe_read['test_year'].tolist()==[2024,2025]
assert _probe_read['period'].tolist()==['historical','2025']
del _probe_frame,_probe_pf,_probe_read
_probe_path.unlink(missing_ok=True)
progress_event('MEMORY','Тест потоковой записи Parquet и типов: PASS',announce=True)


In [ ]:
# ML-05 V7.1.1 · обязательный ранний тест SQL DuckDB до скачивания Stage I.
_adr_e02_query='''SELECT CAST(id AS VARCHAR) organization_id, TRY_CAST(year AS INTEGER) observation_year,
       CAST(indicator_name AS VARCHAR) indicator_name,CAST(indicator_unit AS VARCHAR) indicator_unit,
       CAST(area AS VARCHAR) education_area,CAST(specialty_code AS VARCHAR) specialty_code,
       TRY_CAST(indicator_value AS DOUBLE) observed_value
    FROM read_parquet(?) WHERE regexp_matches(lower(COALESCE(CAST(indicator_name AS VARCHAR),'')),
       'прием|приём|принят|зачисл|поступив')'''
_ADR_E02_SUMMARY_SQL = '''SELECT indicator_name,indicator_unit,
        COUNT(*)::BIGINT AS raw_rows,COUNT(DISTINCT organization_id)::BIGINT AS organizations,
        COUNT(DISTINCT observation_year)::BIGINT AS "years",
        COUNT(*) FILTER (WHERE observed_value IS NULL) AS invalid_numeric_rows,
        COUNT(*) FILTER (WHERE observed_value < 0 OR observed_value != FLOOR(observed_value)) AS invalid_count_rows,
        COUNT(DISTINCT COALESCE(education_area,'?')) AS areas,
        COUNT(DISTINCT COALESCE(specialty_code,'?')) AS specialty_codes
        FROM read_parquet(?) GROUP BY 1,2 ORDER BY raw_rows DESC'''

_ADR_E02_TOTAL_SQL = '''SELECT organization_id,observation_year,indicator_name,indicator_unit,
        COUNT(*)::BIGINT AS n_reporting_cells,
        COUNT(DISTINCT COALESCE(education_area,'?')) AS n_areas,
        COUNT(DISTINCT COALESCE(specialty_code,'?')) AS n_specialties,
        BOOL_AND(education_area IS NULL OR trim(CAST(education_area AS VARCHAR))='' OR
            regexp_matches(lower(CAST(education_area AS VARCHAR)),'^(всего|итого|total|all)$')) AS area_total,
        BOOL_AND(specialty_code IS NULL OR trim(CAST(specialty_code AS VARCHAR))='' OR
            regexp_matches(lower(CAST(specialty_code AS VARCHAR)),'^(всего|итого|total|all)$')) AS specialty_total,
        MIN(observed_value) AS min_count, MAX(observed_value) AS max_count
        FROM read_parquet(?) WHERE indicator_name IN (/*VALID_NAMES*/)
        GROUP BY 1,2,3,4'''

def _adr_e02_copy_statement(source, destination):
    """Build DuckDB COPY with paths SQL-quoted; the SELECT itself is preflight-tested."""
    quoted_source="'"+str(source).replace("'","''")+"'"
    quoted_target="'"+str(destination).replace("'","''")+"'"
    expected=_adr_e02_query.replace('read_parquet(?)','read_parquet('+quoted_source+')')
    if expected==_adr_e02_query:
        raise RuntimeError('ADR_SQL: source placeholder substitution failed')
    return f'COPY ({expected}) TO {quoted_target} (FORMAT PARQUET, COMPRESSION ZSTD)'

# Test EXACT production queries before downloading 2.25M E02 rows.
progress_event('SQL_PREFLIGHT','E02: SQL preflight на 3 искусственных строках (без Stage I)',0,4,announce=True)
_adr_fixture=ROOT/'derived'/'_adr_sql_preflight_input.parquet'
_adr_fixture_out=ROOT/'derived'/'_adr_sql_preflight_output.parquet'
_adr_dummy=pd.DataFrame([
    dict(id='0001',year=2022,indicator_name='Число принятых студентов',indicator_unit='чел',area='',specialty_code='',indicator_value=17),
    dict(id='0002',year=2023,indicator_name='Число зачисленных',indicator_unit='чел',area='Всего',specialty_code='Всего',indicator_value=19),
    dict(id='0003',year=2024,indicator_name='Укомплектованность',indicator_unit='чел',area='',specialty_code='',indicator_value=12)
])
try:
    _adr_dummy.to_parquet(_adr_fixture,index=False)
    progress_event('SQL_PREFLIGHT','E02 SQL: фильтрация и COPY',1,4,announce=False)
    with duck_connect() as _adr_con:
        _adr_con.execute(_adr_e02_copy_statement(_adr_fixture,_adr_fixture_out))
        progress_event('SQL_PREFLIGHT','E02 SQL: сводка по индикаторам',2,4,announce=False)
        _adr_fixture_summary=_adr_con.execute(_ADR_E02_SUMMARY_SQL,[str(_adr_fixture_out)]).df()
        if len(_adr_fixture_summary)!=2 or 'years' not in _adr_fixture_summary.columns:
            raise RuntimeError('ADR_SQL: candidate aggregation semantics mismatch')
        progress_event('SQL_PREFLIGHT','E02 SQL: проверка уникальных итоговых строк',3,4,announce=False)
        _sql=_ADR_E02_TOTAL_SQL.replace('/*VALID_NAMES*/',"'Число принятых студентов'")
        _adr_fixture_totals=_adr_con.execute(_sql,[str(_adr_fixture_out)]).df()
        if len(_adr_fixture_totals)!=1 or int(_adr_fixture_totals.iloc[0].n_reporting_cells)!=1:
            raise RuntimeError('ADR_SQL: organization-year aggregation semantics mismatch')
    progress_event('SQL_PREFLIGHT','E02 SQL preflight: PASS',4,4,announce=True)
finally:
    _adr_fixture.unlink(missing_ok=True)
    _adr_fixture_out.unlink(missing_ok=True)
    del _adr_dummy


SQL_PREFLIGHT_PASSED=True
print('E02 SQL: production templates COPY, SUMMARY and TOTAL — PASS before downloads')


## 3. Приватный GitHub: доступ, неизменяемый вход и SHA-256
Токен читается только из Colab Secret `TRIAD-RU-Colab`; скрытого ручного ввода нет. Источники считываются из закреплённого Stage I; чужие сегменты не изменяются.

In [ ]:
progress_event('INPUT','Проверка токена и GitHub без изменения репозитория')

def github_token():
    try:
        from google.colab import userdata
        value = userdata.get('TRIAD-RU-Colab')
    except Exception as exc:
        raise RuntimeError('Нужен секрет Colab TRIAD-RU-Colab и разрешение доступу ноутбука (Secrets).') from exc
    if not isinstance(value,str) or not value.strip():
        raise RuntimeError('Пустой или недоступный Colab Secret: TRIAD-RU-Colab')
    return value.strip()

_TOKEN=github_token()
BASE='https://api.github.com/repos/'+CFG.REPO
SESS=requests.Session()
SESS.headers.update({'Authorization':'Bearer '+_TOKEN,'Accept':'application/vnd.github+json',
                     'X-GitHub-Api-Version':'2022-11-28'})

def api_call(method,url,*,headers=None,stream=False,timeout=(30,600),**kwargs):
    last=None
    for attempt in range(4):
        try:
            if attempt:progress_event('INPUT',f'Повтор GitHub API, попытка {attempt+1}/4',announce=True)
            r=SESS.request(method,url,headers=headers,stream=stream,timeout=timeout,**kwargs)
            if r.status_code in (429,500,502,503,504):
                last=r.status_code;r.close();time.sleep(2**attempt);continue
            if r.status_code in (401,403,404):
                status=r.status_code;r.close()
                raise PermissionError(f'GitHub HTTP {status}: права, доступ или объект отсутствуют; URL скрыт')
            if not r.ok:
                status=r.status_code;r.close()
                raise RuntimeError(f'GitHub HTTP {status}: отклонён запрос; секреты и ответ не выводятся')
            return r
        except PermissionError:raise
        except requests.RequestException:
            if attempt==3:fail('NETWORK','Ошибка запроса GitHub после повторных попыток')
            time.sleep(2**attempt)
    fail('NETWORK',f'GitHub ответил временной ошибкой {last}')

def api_json(method,url,**kw):
    r=api_call(method,url,**kw)
    try:return r.json()
    finally:r.close()

def repo_head(branch):
    p=api_json('GET',BASE+'/git/ref/heads/'+quote(branch,safe='/'))
    return p['object']['sha']

if CFG.PUBLISH_RESULTS_TO_PRIVATE_GITHUB:
    _preflight_repo=api_json('GET',BASE)
    if not _preflight_repo.get('private'):
        raise RuntimeError('WRITE_PREFLIGHT: репозиторий должен быть приватным')
    _rights=_preflight_repo.get('permissions') or {}
    if not (_rights.get('push') or _rights.get('admin') or _rights.get('maintain')):
        raise PermissionError('WRITE_PREFLIGHT: TRIAD-RU-Colab должен иметь Contents:write для приватного репозитория')
    progress_event('INPUT','GitHub: приватность и права на публикацию проверены ДО обучения',announce=True)
progress_event('INPUT','Фиксация HEAD исходного и целевого репозиториев')
SOURCE_COMMIT=repo_head(CFG.SOURCE_BRANCH)
TARGET_HEAD_AT_PREFLIGHT=repo_head(CFG.TARGET_BRANCH)
# Точная фиксация одного исходного снапшота Stage I.
def read_git_blob(path,ref=SOURCE_COMMIT,with_metadata=False):
    p=api_json('GET',BASE+'/contents/'+quote(path,safe='/'),params={'ref':ref})
    if p.get('type')!='file':fail('INPUT','Ожидался Git файл '+path)
    if p.get('content'):
        content=base64.b64decode(p['content'])
    else:
        # GitHub Contents для >1 MiB может не возвращать base64; /git/blobs гарантирует исходный blob.
        blob=api_json('GET',BASE+'/git/blobs/'+p['sha'])
        if blob.get('encoding')!='base64':fail('INPUT','Неизвестный тип Git blob')
        content=base64.b64decode(blob['content'])
    return (content,p) if with_metadata else content

progress_event('INPUT','Чтение dataset_index.json и проверка Git blob SHA')
index_bytes,index_info=read_git_blob(CFG.INPUT_INDEX,with_metadata=True)
if CFG.EXPECTED_INDEX_BLOB_SHA and index_info['sha']!=CFG.EXPECTED_INDEX_BLOB_SHA:
    fail('INPUT','Изменился Git blob каталога Stage I: требуется предметная проверка версии')
INDEX=json.loads(index_bytes)
if INDEX.get('repo')!=CFG.REPO or INDEX.get('version')!='7.2':
    fail('INPUT','Несовместимый каталог Stage I')
ASSETS=INDEX['assets']; ASSET_BY_ID={a['asset_id']:a for a in ASSETS}
if len(ASSET_BY_ID)!=len(ASSETS):fail('INPUT','Повторяющиеся asset_id')
for a in ASSETS:
    if not a.get('sha256_verified') or len(a.get('sha256',''))!=64:
        fail('INPUT','Каталог содержит непроверенный исходник '+a['asset_id'])
SOURCE_MANIFESTS={}
for k,v in INDEX['source_manifests'].items():
    progress_event('INPUT','Чтение манифеста '+str(k),announce=True)
    SOURCE_MANIFESTS[k]=json.loads(read_git_blob(v))
EXPECTED_COUNT=len(ASSETS)
INPUT_SHA={}

def download_stream(r,path):
    part=Path(str(path)+'.part');part.unlink(missing_ok=True)
    try:
        expected=int(r.headers.get('content-length') or 0)
        received=0;last_report=0.
        with open(part,'wb') as f:
            for block in r.iter_content(2*1024**2):
                if block:
                    f.write(block);received+=len(block)
                    if time.monotonic()-last_report>2:
                        progress_event('DOWNLOAD','Получение Release-asset',received,expected,announce=False)
                        last_report=time.monotonic()
        progress_event('DOWNLOAD','Release-asset получен',received,expected,announce=True)
        if part.stat().st_size==0:fail('INPUT','Получен пустой файл')
        part.replace(path)
    finally:
        r.close();part.unlink(missing_ok=True)

def fetch_release_asset(asset_id,path):
    u=BASE+'/releases/assets/'+str(asset_id)
    r=api_call('GET',u,headers={'Accept':'application/octet-stream'},stream=True,timeout=(30,900))
    content_type=r.headers.get('content-type','').lower()
    if 'json' in content_type:
        obj=r.json();r.close()
        if not obj.get('browser_download_url'):fail('INPUT','Не найден адрес закрытого Release-asset')
        r=api_call('GET',obj['browser_download_url'],stream=True,timeout=(30,900))
        if 'json' in r.headers.get('content-type','').lower():
            r.close();fail('INPUT','Private release-asset возвращает JSON вместо бинарного файла')
    download_stream(r,path)
    if path.stat().st_size==0:fail('INPUT','Скачан пустой Release-asset')
    return path

def fetch_asset(a):
    name=hashlib.sha256(a['asset_id'].encode()).hexdigest()[:20]+'.parquet'
    path=ROOT/'inputs'/name
    progress_event('INPUT',f'Исходный набор {a["asset_id"]}: проверка кэша и SHA',announce=True)
    if not path.is_file():
        for _previous_root in ('/content/ml5_v7_1_0','/content/ml5_v7_0_0'):
            _prior=Path(_previous_root)/'inputs'/name
            if not _prior.is_file():continue
            if hash_file(_prior)!=a['sha256']:continue
            try:os.link(_prior,path)  # same Colab storage, no duplicate 1GB download
            except OSError:shutil.copy2(_prior,path)
            progress_event('INPUT',f'SHA-confirmed Stage I cache reused: {a["asset_id"]}',announce=False)
            break
    if path.exists() and hash_file(path)==a['sha256']:
        INPUT_SHA[a['asset_id']]=a['sha256'];return path
    path.unlink(missing_ok=True)
    if shutil.disk_usage(str(ROOT)).free < CFG.STAGE1_DISK_RESERVE_GIB*2**30:
        fail('INPUT',f'Недостаточно свободного диска: требуется резерв {CFG.STAGE1_DISK_RESERVE_GIB} GiB; можно подключить дополнительный диск или увеличить доступное место')
    loc=a['locator']
    if loc['type']=='git':
        # Вся загрузка закреплена SOURCE_COMMIT, а не плавающим main.
        progress_event('DOWNLOAD',f'Git blob {a["asset_id"]}',announce=True)
        data=read_git_blob(loc['path'])
        path.write_bytes(data)
    elif loc['type']=='release':
        progress_event('DOWNLOAD',f'Release asset {a["asset_id"]}',announce=True)
        fetch_release_asset(loc['asset_id'],path)
    else:fail('INPUT',f"Неподдерживаемый locator {loc['type']}")
    if hash_file(path)!=a['sha256']:
        path.unlink(missing_ok=True);fail('INPUT','SHA-256 не совпал '+a['asset_id'])
    INPUT_SHA[a['asset_id']]=a['sha256']
    progress_event('INPUT',f'Проверено {a["asset_id"]} · {path.stat().st_size/2**20:.1f} МиБ',announce=True)
    total=sum((ROOT/'inputs'/f).stat().st_size for f in os.listdir(ROOT/'inputs') if (ROOT/'inputs'/f).is_file())
    if total/2**30>CFG.MAX_REMOTE_INPUT_GIB:fail('INPUT','Лимит локального кэша Stage I превышен')
    return path

def original_spo_workbook():
    x=[a for a in INDEX.get('extra_originals',[]) if a.get('role')=='migration_matrix_original_xlsx']
    if len(x)!=1:return None
    path=ROOT/'inputs'/'spo_original_e01.xlsx'
    if not path.exists():fetch_release_asset(x[0]['locator']['asset_id'],path)
    # В Stage I для оригинала нет SHA в dataset_index: это не равно подтверждению контрольной суммы.
    return path

def source_asset(segment=None,role=None,source_id=None,prefix=None):
    return [a for a in ASSETS if (segment is None or a['segment']==segment)
            and (role is None or a.get('role')==role)
            and (source_id is None or a.get('source_id')==source_id)
            and (prefix is None or a['asset_id'].startswith(prefix))]

def required_one(items,label):
    if len(items)!=1:fail('INPUT',f'{label}: ожидается 1 источник, найдено {len(items)}')
    return items[0]

def read_table(a,columns=None,where=None,limit=None):
    progress_event('SQL',f'Чтение {a["asset_id"]}, фильтр={bool(where)}, limit={limit}',announce=True)
    path=fetch_asset(a)
    cols=a.get('columns_manifest') or list(pq.ParquetFile(path).schema_arrow.names)
    if columns:
        missing=set(columns)-set(cols)
        if missing:fail('SCHEMA',f'{a["asset_id"]}: отсутствуют {sorted(missing)}')
        selected=','.join('"'+c.replace('"','""')+'"' for c in columns)
    else:selected='*'
    sql=f"SELECT {selected} FROM read_parquet(?)"
    if where:sql+=' WHERE '+where
    if limit:sql+=f' LIMIT {int(limit)}'
    con=duck_connect()
    try:
        if limit is None and int(a.get('rows_actual') or 0)>CFG.MAX_PANDAS_ROWS:
            rows=int(con.execute('SELECT COUNT(*) FROM ('+sql+') AS _guard',[str(path)]).fetchone()[0])
            if rows>CFG.MAX_PANDAS_ROWS:
                fail('RAM_GUARD',f'{a["asset_id"]}: запрос даёт {rows:,} строк; запрещено .df() в FULL, используйте DuckDB COPY/агрегацию')
        result=con.execute(sql,[str(path)]).df()
        progress_event('SQL',f'{a["asset_id"]}: получено {len(result):,} строк',announce=True)
        return result
    finally:con.close()

def sql_query(a,query):
    progress_event('SQL',f'Агрегация {a["asset_id"]}',announce=True)
    path=fetch_asset(a)
    con=duck_connect()
    try:
        result=con.execute(query,[str(path)]).df()
        progress_event('SQL',f'Агрегация {a["asset_id"]}: {len(result):,} строк',announce=True)
        return result
    finally:con.close()

stage1_sources=pd.DataFrame([{'asset_id':a['asset_id'],'segment':a['segment'],'source_id':a['source_id'],
 'role':a.get('role',''),'registered_rows':a.get('rows_actual'), 'locator':a['locator']['type'],
 'sha256':a['sha256'], 'columns':','.join(a.get('columns_manifest',[]))[:500]} for a in ASSETS])
qa_table('stage1_sources',stage1_sources,['asset_id'])
source_ids=set(a.get('source_id') for a in ASSETS)
for missing_id in ('D03','D06'):
    if missing_id not in source_ids:
        ISSUES.append({'module':'STAGE1','status':'SOURCE_NOT_INGESTED',
                       'reason':f'В каталоге нет {missing_id}; нельзя подменять внешними данными','required':missing_id})
note('INPUT',f'pin {SOURCE_COMMIT}; catalog {index_info["sha"]}; {EXPECTED_COUNT} assets')
print('INPUT preflight:', len(ASSETS),'objects; git commit',SOURCE_COMMIT[:12],'; no writes')

mark_stage('INPUT')


## 3А. Полная инвентаризация Stage I — 52/52

В режиме FULL каждое зарегистрированное Parquet скачивается, проверяется по SHA-256, схеме и числу строк, затем построчно оценивается годовой охват без переноса гигантских таблиц в Pandas. Единица анализа при этом сохраняется: **проверка файла не приписывает его значения образовательной миграции**. Годовые shards СПО рассматриваются как альтернативные представления, а не добавочные наблюдения.

In [ ]:
# Stage I FULL52 — strict SHA+Parquet+content profiles, disk-conscious, resumable download cache.
progress_event('CATALOG_52','Проверка ВСЕХ файлов Stage I, SHA, схем, лет и строк',announce=True)
FULL52_INVENTORY=[];FULL52_YEAR_STATS=[];FULL52_THEME_STATS=[];FULL52_ORIGINALS=[]

def _catalog_family(a):
    x=a['asset_id'];seg=a.get('segment','')
    if seg=='01':return 'MUNICIPAL_DEMOGRAPHY'
    if seg=='02':return 'MUNICIPAL_ECONOMY_LABOUR_EDUCATION_TAX'
    if seg=='03':
        if a.get('role')=='education_od_matrix_raw_cells':return 'SPO_ORIGINAL_OD_CELLS_NOT_ADDITIVE'
        if a.get('role')=='organization_indicators':return 'SPO_ALL_YEARS_OR_YEAR_SHARD_NOT_ADDITIVE'
        return 'SPO_REGISTRY_OD_REGIONAL'
    if seg=='04':return 'HIGHER_EDUCATION_VPO'
    if seg=='05':return 'GRADUATE_LABOUR_OUTCOMES'
    if seg=='06':return 'MUNICIPAL_SPATIAL_BDMO_TRANSPORT'
    return 'UNKNOWN'

def _source_year_scan(a,p,cols):
    key=a.get('time_key')
    if key not in cols:key=next((c for c in ('year','observation_year','graduation_year') if c in cols),None)
    if not key:return [],'NO_TIME_KEY'
    column='"'+key.replace('"','""')+'"'
    try:
        with duck_connect() as con:
            years=con.execute(f'''SELECT CAST({column} AS VARCHAR) observed_year,COUNT(*)::BIGINT n
                FROM read_parquet(?) GROUP BY 1 ORDER BY 1''',[str(p)]).fetchall()
        return [{'asset_id':a['asset_id'],'year_label':str(y),'rows':int(n),'time_column':key} for y,n in years],'SCANNED_FULL'
    except (duckdb.OutOfMemoryException,duckdb.IOException) as ex:
        ISSUES.append({'module':'SOURCE_AUDIT','status':'TIME_DISTRIBUTION_NOT_AVAILABLE',
                       'reason':a['asset_id']+' '+type(ex).__name__,'required':'more spill memory for time scan'})
        return [],'YEAR_SCAN_UNAVAILABLE'

for i,a in enumerate(ASSETS,1):
    aid=a['asset_id']
    if CFG.MODE!='FULL' and aid not in {'01::0','03::42f47d3cf2159992::000','03::89505d5993e6f4ad::000'}:
        FULL52_INVENTORY.append({'asset_id':aid,'segment':a.get('segment'),'source_id':a.get('source_id'),
           'research_family':_catalog_family(a),'observed_rows':None,'expected_rows':a.get('rows_actual'),
           'physical_read_ok':False,'footer_rows_ok':False,'sha256_ok':False,
           'years_scanned':0,'year_scan_status':'NOT_SCANNED_IN_PILOT',
           'source_sha256':a['sha256'],'level_gate':'PILOT_NOT_FULL'})
        continue
    progress_event('CATALOG_52',f'[{i:02d}/52] {aid}: SHA + контроль Parquet',i,len(ASSETS),announce=True)
    source=fetch_asset(a)  # verifies SHA both fresh and cached; never silently skipped
    footer=pq.ParquetFile(source)
    cols=list(footer.schema_arrow.names);registered=int(a.get('rows_actual') or 0)
    found=int(footer.metadata.num_rows)
    if found!=registered:fail('FULL52',f'{aid}: rows {found:,} != Stage I {registered:,}')
    manifest_cols=list(a.get('columns_manifest') or [])
    if manifest_cols and set(cols)!=set(manifest_cols):fail('FULL52',f'{aid}: Parquet fields != Stage I catalog')
    # Content touch: inspect a physical row through a LIMIT, not whole table in pandas.
    with duck_connect() as con:
        touched=int(con.execute('SELECT COUNT(*) FROM (SELECT 1 FROM read_parquet(?) LIMIT 1)',[str(source)]).fetchone()[0])
    if touched!=int(found>0):fail('FULL52',f'{aid}: нельзя прочитать содержимое Parquet')
    yr,yr_status=_source_year_scan(a,source,cols) if CFG.CATALOG_SCAN_ALL_YEARS else ([], 'SCAN_DISABLED')
    if yr and sum(r['rows'] for r in yr)!=found:fail('FULL52',f'{aid}: сумма годового охвата не совпадает с footer')
    FULL52_YEAR_STATS.extend(yr)
    family=_catalog_family(a)
    FULL52_INVENTORY.append({'asset_id':aid,'segment':a.get('segment'),'source_id':a.get('source_id'),
      'data_role':a.get('role') or a.get('data_role'),'research_family':family,'observed_rows':found,
      'expected_rows':registered,'columns':len(cols),'bytes':source.stat().st_size,
      'physical_read_ok':True,'footer_rows_ok':found==registered,'sha256_ok':INPUT_SHA.get(aid)==a['sha256'],
      'years_scanned':len(yr),'year_scan_status':yr_status,'first_year':yr[0]['year_label'] if yr else None,
      'last_year':yr[-1]['year_label'] if yr else None,'source_sha256':a['sha256'],
      'original_grain_status':a.get('grain_status'),'level_gate':'ORIGINAL_GRAIN_NOT_ASSUMED_ADDITIVE'})
    del footer
    if i%4==0:gc.collect();memory_point(f'catalog_{i}_of_{len(ASSETS)}')

if CFG.MODE=='FULL' and (len(FULL52_INVENTORY)!=len(ASSETS) or len(INPUT_SHA)!=len(ASSETS)):
    fail('FULL52','Неполное чтение/контроль входных файлов')
# Role-specific research trail: every source has a declared analytic role, with no double-counting.
for fam,grp in pd.DataFrame(FULL52_INVENTORY).groupby('research_family'):
    FULL52_THEME_STATS.append({'family':fam,'n_assets':len(grp), 'source_rows_scanned_by_footer':int(grp.observed_rows.sum()),
      'interpretation':'ALT_REPRESENTATIONS_NOT_ADDITIVE' if 'SHARD' in fam or 'CELLS' in fam else 'OWN_GRAIN',
      'sha_verified':int(grp.sha256_ok.sum())})
qa_table('full52_source_inventory',pd.DataFrame(FULL52_INVENTORY),['asset_id'])
qa_table('full52_year_coverage',pd.DataFrame(FULL52_YEAR_STATS),allow_empty=True)
qa_table('full52_theme_coverage',pd.DataFrame(FULL52_THEME_STATS),['family'])
# Original SPO workbook is outside the 52 Parquet assets. Record accessibility separately.
try:
    ori=original_spo_workbook()
    if ori:
        FULL52_ORIGINALS=[{'name':ori.name,'bytes':ori.stat().st_size,'sha256_locally_calculated':hash_file(ori),
          'registered_in_52':False,'status':'CONTENT_AVAILABLE_METHODOLOGY_NOT_YET_IDENTIFIED'}]
except Exception as ex:
    FULL52_ORIGINALS=[{'name':'original_spo_workbook','status':'NOT_AVAILABLE','reason':type(ex).__name__}]
qa_table('full52_raw_originals',pd.DataFrame(FULL52_ORIGINALS),allow_empty=True)
print('CATALOG 52:',len(INPUT_SHA),'SHA verified;',sum(t['observed_rows'] or 0 for t in FULL52_INVENTORY),
      'Parquet rows across files (NOT unique observations);',len(FULL52_YEAR_STATS),'asset-year groups')
mark_stage('CATALOG_52')


## 4. Измерение и определения исходных величин
Для E01 сохраняются фактические заголовки оригинального XLSX и исходных raw-таблиц. Нельзя автоматически называть показателем миграции населения или числом зачисленных наблюдение об образовании и текущем обучении.

In [ ]:
memory_point('before_semantics')
progress_event('SEMANTICS','Аудит определений E01/E02/ВПО/СПО')
# Семантика извлекается из значений исходных таблиц, а не из догадки по названию файла.
E01=required_one(source_asset(role='education_od_flows'),'E01 OD')
SPO_ORG=required_one(source_asset(role='education_organizations'),'E01 organizations')
SPO_IND=sorted(source_asset(segment='03',role='organization_indicators'),
               key=lambda a:0 if 'all_years' in a.get('member','') else 1)
SPO_MASTER=next((a for a in SPO_IND if 'all_years' in a.get('member','')),None)
SPO_REG=required_one(source_asset(role='regional_spo'),'SPO regional indicators')
E02=required_one(source_asset(prefix='04::E02'),'E02 universities')
VPO_211=required_one(source_asset(prefix='04::VPO_2_1_1'),'VPO-1 admissions table')
VPO_213=required_one(source_asset(prefix='04::VPO_2_1_3'),'VPO-1 graduates table')
GRAD_ORG=required_one(source_asset(prefix='05::E04::organization_total'),'E04 graduates')
GRAD_FIELD=required_one(source_asset(prefix='05::E05::field_specialty'),'E05 specialties')
MUN=required_one(source_asset(segment='01'),'Municipal demography')
TRANSPORT=required_one(source_asset(prefix='06::TRANSPORT'),'transport graph')
SPATIAL=required_one(source_asset(prefix='06::D04::D04_member00::spatial_reference'),'geography reference')

# Для FULL/PILOT10 производим сканирование всех строк выбранных источников колонночным движком.
def indicator_catalog(a,fields,cap=None):
    cols=[c for c in fields if c in a.get('columns_manifest',[])]
    if 'indicator_name' not in cols:fail('SEMANTICS','Нет колонки indicator_name: '+a['asset_id'])
    progress_event('SEMANTICS',f'Сканирование словаря {a["asset_id"]}',announce=True)
    path=fetch_asset(a)
    with duck_connect() as con:
        # В SMOKE ограничиваем ввод — это не окончательный словарь.
        sample=f' LIMIT {CFG.MAX_READ_ROWS_SMOKE}' if CFG.MODE=='SMOKE' else ''
        group=','.join('"'+c+'"' for c in cols)
        sql=f'SELECT {group}, count(*)::BIGINT as n FROM (SELECT {group} FROM read_parquet(?) {sample}) GROUP BY {group} ORDER BY n DESC'
        d=con.execute(sql,[str(path)]).df()
    d['asset_id']=a['asset_id'];d['catalog_exhaustive']=CFG.MODE!='SMOKE'
    progress_event('SEMANTICS',f'Словарь готов: {a["asset_id"]} · {len(d):,} записей',announce=True)
    return d

CAT_E02=indicator_catalog(E02,['indicator_name','indicator_unit','area'])
CAT_VPO=indicator_catalog(VPO_211,['indicator_name','admission_type','education_level','education_form'])
CAT_SPO=indicator_catalog(SPO_MASTER,['indicator_name','indicator_unit','indicator_section']) if SPO_MASTER else pd.DataFrame()
CAT_SPO_REG=indicator_catalog(SPO_REG,['indicator_name','indicator_unit','indicator_section'])
CAT_VPO_GRAD=indicator_catalog(VPO_213,['indicator_name','education_level','education_form'])
for name,tab in [('dictionary_e02',CAT_E02),('dictionary_vpo_admission',CAT_VPO),
                 ('dictionary_spo',CAT_SPO),('dictionary_spo_regional',CAT_SPO_REG),
                 ('dictionary_vpo_graduation',CAT_VPO_GRAD)]:qa_table(name,tab,allow_empty=True)

POS_ADMISSION=re.compile(r'(?i)(?:^|\W)(?:при[её]м|принят|зачисл|поступил|поступивш|принятых)(?:\w|\W|$)')
NEG_ADMISSION=re.compile(r'(?i)(?:заявлен|подавш|конкурс|при[её]мн\w* комисс|при[её]м на работу|планируем|места приема|контрольн\w+ цифр|ваканс)')
POS_GRAD=re.compile(r'(?i)(выпуск|окончивш|завершивш)')
NEG_TOTAL=re.compile(r'(?i)(процент|доля|удельн|на 100|отношение|коэффициент|темп|индекс|стоимост|тыс\. руб|млн)')

# Утверждение требует положительного семантического определения, единицы и отсутствия разрезов-двойников.
def classify_indicator(name,unit=None,category=None):
    value='' if pd.isna(name) else str(name).strip()
    u='' if pd.isna(unit) else str(unit).strip().lower()
    cat='' if pd.isna(category) else str(category)
    if POS_ADMISSION.search(value) and not NEG_ADMISSION.search(value) and not NEG_TOTAL.search(value):
        role='ADMISSION_CANDIDATE'
    elif POS_GRAD.search(value) and not NEG_TOTAL.search(value):role='GRADUATION_CANDIDATE'
    else:role='OTHER'
    unit_ok=not u or bool(re.search(r'чел|человек|единиц|численност|шт',u))
    # Автоматическая классификация создаёт кандидатов, но НЕ утверждает научную семантику.
    grade='ELIGIBLE_FOR_ROW_CHECK' if role!='OTHER' and unit_ok else 'QUARANTINE'
    if role!='OTHER' and not unit_ok:grade='UNEXPECTED_UNIT'
    return {'semantic_role':role,'semantic_gate':grade,
            'name_exact':value,'unit_exact':u,'category_exact':cat}

CANDIDATES=[]
for source,frame in [('E02',CAT_E02),('VPO_2_1_1',CAT_VPO),('SPO_ORG',CAT_SPO),('SPO_REG',CAT_SPO_REG),('VPO_GRAD',CAT_VPO_GRAD)]:
    for _,r in frame.iterrows():
        d=classify_indicator(r.get('indicator_name'),r.get('indicator_unit'),r.get('admission_type'))
        CANDIDATES.append({'source':source,'indicator_name':r.get('indicator_name'),'rows':int(r['n']),
                           'level':r.get('education_level'), 'form':r.get('education_form'),
                           'admission_type':r.get('admission_type'), 'unit':r.get('indicator_unit'),
                           'indicator_section':r.get('indicator_section'),'area':r.get('area'),**d})
CANDIDATE_CATALOG=qa_table('semantic_indicator_catalog',pd.DataFrame(CANDIDATES),allow_empty=True)

OD_SEMANTIC={
 'source_id':'E01','origin_definition':'region_of_previous_general_education',
 'destination_definition':'region_of_current_vocational_study',
 'label':'observed_E01_schooling_to_study_value',
 'admissions_confirmed':False,'permanent_migration_confirmed':False,
 'yearly_count_unit_confirmed':False,'unit':'reported_indicator_value',
 'methodological_status':'SOURCE_DESCRIPTION_REQUIRES_PRIMARY_CELL_VERIFICATION',
 'statistics_available_asof':'2026_SOURCE_VINTAGE_ONLY',
 'admission_target_gate':'NOT_CONFIRMED',
 'source_page':'https://tochno.st/datasets/monitoring_spo',
 'primary_workbook':'Stage I extra_originals/migration_matrix_original_xlsx'
}
ORIGINAL_SHEETS=[]
try:
    progress_event('SEMANTICS','Чтение первичного XLSX E01',announce=True)
    p=original_spo_workbook()
    if p:
        xf=pd.ExcelFile(p,engine='calamine')
        for sheet in xf.sheet_names:
            raw=pd.read_excel(p,sheet_name=sheet,header=None,nrows=25,engine='calamine')
            cells=' | '.join(str(v)[:100] for v in raw.to_numpy().reshape(-1)
                           if not pd.isna(v) and str(v).strip())[:2500]
            ORIGINAL_SHEETS.append({'sheet':sheet,'first_cells':cells,'file_sha256':hash_file(p)})
        OD_SEMANTIC['original_opened']=True
except Exception as ex:
    OD_SEMANTIC['original_opened']=False
    ISSUES.append({'module':'ML-5.1','status':'PRIMARY_SOURCE_REVIEW_PENDING',
                   'reason':'Оригинальный XLSX E01 не открыт; ошибка '+type(ex).__name__,
                   'required':'original E01 workbook access'})
qa_table('e01_original_headers',pd.DataFrame(ORIGINAL_SHEETS),allow_empty=True)
# Даже наличие слова «миграция» в заголовке не доказывает, что показатель — поток зачисления.
if not OD_SEMANTIC.get('original_opened'):
    note('SEM-E01','Оригинал не открыт: допустим ограниченный анализ наблюдаемой величины','WARN')
else:
    note('SEM-E01','Оригинальные заголовки записаны; семантика admission пока не подтверждена')

RAW_SPO_CELL_AUDIT=[]
for a in source_asset(role='education_od_matrix_raw_cells'):
    try:
        p=fetch_asset(a);file=pq.ParquetFile(p)
        RAW_SPO_CELL_AUDIT.append({'asset_id':a['asset_id'],'rows':file.metadata.num_rows,
                                   'columns':','.join(file.schema_arrow.names),'sha256':a['sha256']})
    except Exception as ex:
        RAW_SPO_CELL_AUDIT.append({'asset_id':a['asset_id'],'error':type(ex).__name__})
qa_table('e01_raw_cell_sources',pd.DataFrame(RAW_SPO_CELL_AUDIT),allow_empty=True)

# Проверка ручного списка после чтения фактических словарей; не считать пустоту списка отсутствием исходных данных.
def approved_names(source):
    manually={'E02':CFG.APPROVED_VO_ADMISSION_NAMES,'SPO_ORG':CFG.APPROVED_SPO_ADMISSION_NAMES}
    names=tuple(manually.get(source,()))
    existing=set(CANDIDATE_CATALOG.loc[CANDIDATE_CATALOG.source.eq(source),'indicator_name'].dropna().astype(str))
    bad=set(names)-existing
    if bad:fail('SEMANTICS',f'{source}: утверждённые имена не найдены в данных: {sorted(bad)}')
    return sorted(set(names))

SEMANTIC_APPROVAL=pd.DataFrame([{'source':src,'indicator_name':n,'approved_for_admission':n in approved_names(src),
    'semantic_role':r,'unit':u} for src in ('E02','SPO_ORG')
    for n,r,u in CANDIDATE_CATALOG.loc[CANDIDATE_CATALOG.source.eq(src),
        ['indicator_name','semantic_role','unit']].drop_duplicates().itertuples(index=False,name=None)])
qa_table('admission_approval_register',SEMANTIC_APPROVAL,allow_empty=True)
print('Semantic candidates:',len(CANDIDATE_CATALOG),'| E01 original sheets:',len(ORIGINAL_SHEETS),
      '| SPO raw-cell files:',len(RAW_SPO_CELL_AUDIT))

mark_stage('SEMANTICS')


## 5. Исследовательские панели и контролируемые сопоставления
Статистические уровни ВО, СПО, организация, филиал, регион и муниципалитет не смешиваются. Спорные организационные ID исключаются в карантин, ноль отличается от отсутствия.

In [ ]:
memory_point('before_panels')
progress_event('PANELS','Исходные панели: E01, организации, ВПО, E04/E05, муниципалитеты')
# Шесть исходных исследовательских панелей и независимые таблицы ошибок сопоставления.
OD_RAW=read_table(E01,['year','origin_region_code','origin_region_name',
                       'destination_region_code','destination_region_name','indicator_value'])
OD_RAW=OD_RAW.rename(columns={'origin_region_code':'origin_region','destination_region_code':'destination_region',
                              'indicator_value':'observed_value','year':'observation_year'})
OD_RAW['origin_region']=norm_code(OD_RAW['origin_region'])
OD_RAW['destination_region']=norm_code(OD_RAW['destination_region'])
OD_RAW['observation_year']=safe_year(OD_RAW['observation_year'])
OD_RAW['observed_value']=number(OD_RAW['observed_value'])
OD_RAW['sector']='SPO';OD_RAW['origin_definition']='previous_general_education_region'
OD_RAW['destination_definition']='current_study_region'
OD_RAW['target_definition']=OD_SEMANTIC['label']
OD_RAW['known_origin']=nonmissing_code(OD_RAW.origin_region)
OD_RAW['known_destination']=nonmissing_code(OD_RAW.destination_region)
OD_RAW['code_status']=np.select([(~OD_RAW.known_origin|~OD_RAW.known_destination).fillna(True).to_numpy(dtype=bool),
                                OD_RAW.origin_region.eq(OD_RAW.destination_region).fillna(False).to_numpy(dtype=bool)],
                                ['UNKNOWN_CODE','INTRA_REGION'],default='INTER_REGION')
if OD_RAW['observed_value'].lt(0).any():fail('SEM-E01','Отрицательные величины E01: нужна особая интерпретация')
if OD_RAW['observed_value'].isna().any():
    QUARANTINE.extend([{'table':'edu_od_year','reason':'missing_value','row':int(i)}
                       for i in OD_RAW.index[OD_RAW.observed_value.isna()].tolist()[:100]])
OD_RAW['explicit_zero']=OD_RAW.observed_value.eq(0)&OD_RAW.observed_value.notna()
OD_RAW['missing_edge']=False;OD_RAW['suppressed']=False
OD_RAW['source_asset_id']=E01['asset_id'];OD_RAW['source_vintage']=INDEX['created_at']
# Год может быть статистическим периодом учёбы; не переименовывать в admission_year.
OD=qa_table('edu_od_year',OD_RAW,
            ['observation_year','origin_region','destination_region'])
OD_INTER=qa_table('edu_od_interregional',OD[OD.code_status=='INTER_REGION'].copy(),
              ['observation_year','origin_region','destination_region'],allow_empty=True)
OD_INTRA=qa_table('edu_od_intraregional',OD[OD.code_status=='INTRA_REGION'].copy(),allow_empty=True)
OD_UNKNOWN=qa_table('edu_od_unknown_code',OD[OD.code_status=='UNKNOWN_CODE'].copy(),allow_empty=True)
if sum(map(len,[OD_INTER,OD_INTRA,OD_UNKNOWN]))!=len(OD):fail('DATA_GATE','OD: категории не исчерпывают данные')

counts=OD.groupby(['observation_year','code_status'],dropna=False).agg(
    rows=('observed_value','size'),total_value=('observed_value','sum')).reset_index()
qa_table('od_composition',counts,allow_empty=True)
EDGES=OD_INTER[['observation_year','origin_region','destination_region','observed_value']].copy()
EDGES['edge_id']=EDGES.origin_region.astype(str)+'|'+EDGES.destination_region.astype(str)
edge_years={int(y):set(g.edge_id) for y,g in EDGES.groupby('observation_year')}
coverage=[];transitions=[]
for y in sorted(edge_years):
    keys=edge_years[y];past=edge_years.get(y-1,set())
    coverage.append({'year':y,'observed_edges':len(keys),'new_vs_previous':len(keys-past) if past else pd.NA,
                     'lost_vs_previous':len(past-keys) if past else pd.NA,
                     'intersection_vs_previous':len(keys&past) if past else pd.NA,
                     'jaccard_vs_previous':len(keys&past)/len(keys|past) if past and (keys|past) else np.nan,
                     'total_observed_value':float(EDGES.loc[EDGES.observation_year==y,'observed_value'].sum())})
    if past:
        transitions.extend([{'year':y,'edge_id':e,'transition':'NEW'} for e in sorted(keys-past)])
        transitions.extend([{'year':y,'edge_id':e,'transition':'NOT_REPORTED'} for e in sorted(past-keys)])
coverage_df=qa_table('od_edge_coverage',pd.DataFrame(coverage),['year'])
qa_table('od_edge_transitions',pd.DataFrame(transitions),allow_empty=True)
if len(coverage_df)>1:
    coverage_df['edge_growth_ratio']=coverage_df.observed_edges/coverage_df.observed_edges.shift(1)
    coverage_df['potential_break']=coverage_df.edge_growth_ratio.gt(1.5) | coverage_df.edge_growth_ratio.lt(0.67)
    if coverage_df.potential_break.any():
        ISSUES.append({'module':'SEM-E01','status':'COVERAGE_BREAK_UNRESOLVED',
         'reason':'Изменение регистрации OD-направлений: прямое межгодовое сравнение требует аудита полноты',
         'required':'Единые определения и покрытие матрицы по годам'})

# Реестр ВО из E02: комбинация исторического года, организационного id и географического положения.
# E02 длинная таблица показателей (2.25 млн строк): реестр читаем DISTINCT в DuckDB,
# не загружая миллионы повторяющихся идентификаторов в оперативную память.
with duck_connect() as con:
    progress_event('PANELS','E02: DISTINCT организаций из полной таблицы, без переноса 2.25 млн строк в pandas',announce=True)
    vo=con.execute('''SELECT DISTINCT id,name,inn,ogrn,region,oktmo,institution,parent_id,parent_name,year
                     FROM read_parquet(?)''',[str(fetch_asset(E02))]).df()
if CFG.MODE=='SMOKE':vo=vo.head(CFG.MAX_READ_ROWS_SMOKE).copy()
for c in ['id','parent_id','oktmo','inn','ogrn']:vo[c]=norm_code(vo[c])
vo['observation_year']=safe_year(vo['year'])
vo=vo.drop(columns=['year'])
vo=vo.loc[vo.id.notna()&vo.observation_year.notna()].drop_duplicates().copy()
vo['sector']='VO';vo['organization_id']=vo['id']
vo['campus_id']=vo['id'] # Только при явном собственном id в исходнике; головной id не переносить на филиал.
vo['parent_group_id']=vo.parent_id.fillna(vo.organization_id)
vo['entity_id']='VO|'+vo.organization_id.astype(str)
vo['campus_geo_status']=np.where(nonmissing_code(vo.oktmo),'REPORTING_OKTMO_NOT_VERIFIED_CAMPUS','UNKNOWN')
vo['entity_match_confidence']=1.0

sporeg=read_table(SPO_ORG,['year','region_name','region_oktmo','id','name','city','lat','lon','type','head','address','branches'])
sporeg['observation_year']=safe_year(sporeg.year);sporeg=sporeg.drop(columns=['year'])
sporeg['id']=norm_code(sporeg.id);sporeg['region_oktmo']=norm_code(sporeg.region_oktmo)
sporeg['sector']='SPO';sporeg['organization_id']=sporeg.id
sporeg['entity_id']='SPO|'+sporeg.id.fillna('UNRESOLVED')
sporeg['campus_geo_status']='POINT_WITHOUT_MUNICIPAL_OKTMO'
sporeg['entity_match_confidence']=np.where(sporeg.id.notna(),1.0,0.0)
sporeg['parent_group_id']=sporeg['head'].fillna(sporeg['id']).astype('string')
sporeg['name_normalized']=sporeg.name.astype('string').str.lower().str.replace(r'\s+',' ',regex=True).str.strip()

# Проверять уникальность натурального ключа после исключения полных дублей.
# Исторические конфликты не удаляются молча: полные дубли снимаются, остальные в карантин.
def quarantine_conflicting_registry(df,sector):
    key=['entity_id','observation_year']
    if df.empty:return df
    df=df.drop_duplicates().copy()
    unknown=df.entity_id.astype(str).str.contains('UNRESOLVED',regex=False) | df.entity_id.isna()
    if unknown.any():
        QUARANTINE.extend(df.loc[unknown].head(500).assign(source=sector+'_MISSING_ID').to_dict('records'))
        df=df.loc[~unknown].copy()
    counts=df.groupby(key,dropna=False).size()
    badkeys=set(counts[counts>1].index)
    if badkeys:
        badmask=pd.MultiIndex.from_frame(df[key]).isin(pd.MultiIndex.from_tuples(list(badkeys),names=key))
        QUARANTINE.extend(df.loc[badmask].head(1000).assign(source=sector+'_CONFLICTING_ID_YEAR').to_dict('records'))
        df=df.loc[~badmask].copy()
    return df
vo=quarantine_conflicting_registry(vo,'VO')
sporeg=quarantine_conflicting_registry(sporeg,'SPO')
for x,tab in [('VO',vo),('SPO',sporeg)]:
    if tab.empty:ISSUES.append({'module':'ML-5.2','status':'NO_REGISTRY',
          'reason':f'Исторический реестр {x} пуст после контроля конфликтных идентификаторов','required':'unique stable IDs'})
registry_columns=['sector','observation_year','entity_id','organization_id','parent_group_id','name','region','oktmo',
                  'city','address','campus_geo_status','entity_match_confidence']
vo2=vo.rename(columns={'region':'region','oktmo':'oktmo'})
spo2=sporeg.rename(columns={'region_name':'region','region_oktmo':'region_oktmo_source'})
reg_parts=[]
for frame in (vo2,spo2):
    part=frame.copy()
    for col in registry_columns:
        if col not in part:part[col]=pd.NA
    reg_parts.append(part[registry_columns])
REGISTRY=qa_table('edu_org_registry',pd.concat(reg_parts,ignore_index=True),
                  ['sector','entity_id','observation_year'],allow_empty=True)
REGISTRY_AUDIT=pd.DataFrame([{'sector':s,'records':len(g),'unique_org_ids':g.entity_id.nunique(),
    'years':g.observation_year.nunique(),'confirmed_campuses':int(g.campus_geo_status.eq('CONFIRMED_STUDY_CAMPUS').sum())}
    for s,g in REGISTRY.groupby('sector')])
qa_table('registry_coverage_audit',REGISTRY_AUDIT,allow_empty=True)

# Словари по учреждениям: ни один показатель не принимается по непустоте конфигурационного кортежа.
def selected_names(source,role='ADMISSION_CANDIDATE'):
    # В словарях сохраняются кандидаты, но в прогноз попадают только утверждённые точные имена.
    sub=CANDIDATE_CATALOG[(CANDIDATE_CATALOG.source==source)&
         (CANDIDATE_CATALOG.semantic_role==role)&
         (CANDIDATE_CATALOG.semantic_gate=='ELIGIBLE_FOR_ROW_CHECK')]
    candidates=set(sub.indicator_name.dropna().astype(str).unique())
    if source in ('E02','SPO_ORG'):
        approved=set(approved_names(source));unapproved=sorted(candidates-approved)
        if unapproved:ISSUES.append({'module':'ML-5.2','status':'UNAPPROVED_ADMISSION_CANDIDATES',
          'reason':f'{source}: {len(unapproved)} неутверждённых исходных имён','required':'утвердить точные имена в конфигурации'})
        return sorted(candidates&approved)
    return sorted(candidates)

# Панель приема ВО на уровне E02: все положительные candidates + точный натуральный ключ + единица.
# При неоднозначности исходных area/specialty/indicator нужна quarantine, а не сложение строк.
def selective_records(a,cols,names):
    if not names:return pd.DataFrame(columns=cols)
    esc=lambda t:"'"+t.replace("'","''")+"'"
    name_list=','.join(esc(t) for t in names)
    return read_table(a,cols,where=f'indicator_name IN ({name_list})',
        limit=CFG.MAX_READ_ROWS_SMOKE if CFG.MODE=='SMOKE' else None)

vo_admission=selective_records(E02,['id','parent_id','year','name','region','oktmo',
 'area','specialty_code','specialty_section','indicator_name','indicator_unit','indicator_value'],
 selected_names('E02'))
if len(vo_admission):
    vo_admission['entity_id']='VO|'+norm_code(vo_admission.id)
    vo_admission['observation_year']=safe_year(vo_admission.year)
    vo_admission['value']=number(vo_admission.indicator_value)
    vo_admission['indicator_grain']='E02_ID_YEAR_AREA_SPECIALTY_INDICATOR'
    # Сохраняем полный programme-разрез до отбора только общих строк для орг-панели.
    vo_program_candidates=vo_admission.copy()
    # area — раздел мониторинга, а не доказанный подитог. Ограничиваем только specialty_code.
    for categorical in ('specialty_code',):
        v=vo_admission[categorical].astype('string').str.strip().str.lower()
        keep=v.isna() | v.eq('') | v.isin(['всего','итого','total','0','00'])
        rejected=vo_admission.loc[~keep].copy()
        if len(rejected):QUARANTINE.extend(rejected.head(100).assign(source='VO_NON_TOTAL_'+categorical).to_dict('records'))
        vo_admission=vo_admission.loc[keep].copy()

# SPO данные организации сопрягаются только по уникальному совпадению региона/названия/года.
spo_admission=selective_records(SPO_MASTER,['year','region_name','name','property','indicator_section',
 'indicator_name','indicator_unit','org_value'],selected_names('SPO_ORG')) if SPO_MASTER else pd.DataFrame()
if len(spo_admission):
    spo_admission['name_normalized']=spo_admission.name.astype('string').str.lower().str.replace(r'\s+',' ',regex=True).str.strip()
    spo_admission['observation_year']=safe_year(spo_admission.year)
    spo_admission['region_name']=spo_admission.region_name.astype('string').str.strip()
    lookup=sporeg[['region_name','name_normalized','observation_year','entity_id']].drop_duplicates()
    count=lookup.groupby(['region_name','name_normalized','observation_year'],dropna=False).size().rename('matches').reset_index()
    lookup=lookup.merge(count,on=['region_name','name_normalized','observation_year'],validate='many_to_one')
    ambiguous=lookup[lookup.matches!=1].copy()
    if len(ambiguous):QUARANTINE.extend(ambiguous.head(500).assign(source='SPO_MATCH').to_dict('records'))
    lookup=lookup[lookup.matches==1].drop(columns=['matches'])
    spo_admission=spo_admission.merge(lookup,on=['region_name','name_normalized','observation_year'],
                                     how='left',validate='many_to_one')
    spo_admission['value']=number(spo_admission.org_value)
    spo_admission['indicator_grain']='SPO_ORG_YEAR_PROPERTY_INDICATOR'
    # property в мониторинге СПО может обозначать форму собственности организации,
    # а не разрез значения. Не удаляем реальные организации по этому полю.

# ВПО-1: полная точная агрегация, частями на диск. Лимит 512 МБ используется только для лёгких запросов.
VPO_FULL_PROGRAM_DESCRIPTION=pd.DataFrame()
VPO_FULL_PROGRAM_PATH=None
_vpo_names=selected_names('VPO_2_1_1')
if _vpo_names and CFG.MODE=='FULL':
    _vpo_path=Path(fetch_asset(VPO_211))
    _base=ROOT/'derived'; _base.mkdir(parents=True,exist_ok=True)
    _shards=_base/'vpo_year_shards'; _shards.mkdir(parents=True,exist_ok=True)
    VPO_FULL_PROGRAM_PATH=_base/'vpo_full_program_description.parquet'
    _esc=lambda value:"'"+str(value).replace("'","''")+"'"
    _where_names='indicator_name IN ('+','.join(_esc(x) for x in _vpo_names)+')'
    _src='read_parquet('+_esc(str(_vpo_path))+')'
    # Каждый год рассчитывается независимо; результаты объединяются без повторной агрегации.
    # Группы год×индикатор×специальность нельзя дробить между выходными файлами.
    _select='''SELECT TRY_CAST(year AS INTEGER) AS observation_year, specialty_code, indicator_name,
        object_level, education_form, education_level, admission_type,
        COUNT(*)::BIGINT AS observed_records,
        MEDIAN(TRY_CAST(indicator_value AS DOUBLE)) AS median_reported_value,
        COUNT(DISTINCT object_name)::BIGINT AS distinct_geographic_names,
        FALSE AS totals_additive_confirmed'''
    _start_filter=f'({_where_names}) AND TRY_CAST(year AS INTEGER) IS NOT NULL'
    memory_point('before_vpo_full_aggregate')
    progress_event('PANELS','ВПО-1: проверка годов и полной численности',announce=True)
    with duck_connect() as _con:
        _con.execute("SET preserve_insertion_order=false")
        _years=[int(r[0]) for r in _con.execute(
            f'SELECT DISTINCT TRY_CAST(year AS INTEGER) AS y FROM {_src} WHERE {_start_filter} ORDER BY 1'
        ).fetchall()]
        _expected=int(_con.execute(f'SELECT COUNT(*) FROM {_src} WHERE {_start_filter}').fetchone()[0])
        _bad_years=int(_con.execute(f'SELECT COUNT(*) FROM {_src} WHERE ({_where_names}) AND TRY_CAST(year AS INTEGER) IS NULL').fetchone()[0])
    if _bad_years:
        ISSUES.append({'module':'ML-5.4','status':'VPO_UNDATED_EXCLUDED',
            'reason':f'ВПО-1: {_bad_years} строк без корректного года исключены из динамики',
            'required':'проверка первичных дат'})
    # Очистка только технических файлов этой конкретной агрегации.
    for _old in _shards.glob('vpo_*.parquet'): _old.unlink()
    _parts=[]

    def _vpo_copy(where,output):
        # Точный MEDIAN и COUNT(DISTINCT) без замены статистик приближёнными.
        _sql=f'''COPY ({_select} FROM {_src} WHERE ({where}) GROUP BY 1,2,3,4,5,6,7)
                    TO {_esc(str(output))} (FORMAT PARQUET, COMPRESSION ZSTD)'''
        output.unlink(missing_ok=True)
        try:
            with duck_connect() as _con:
                _con.execute("SET memory_limit='3GB'")
                _con.execute('SET threads=1')
                _con.execute('SET preserve_insertion_order=false')
                _con.execute("SET max_temp_directory_size='40GB'")
                _con.execute(_sql)
            return output
        except Exception:
            output.unlink(missing_ok=True)
            raise

    for _y in _years:
        _filter=f'{_start_filter} AND TRY_CAST(year AS INTEGER)={_y}'
        _outfile=_shards/f'vpo_year_{_y}.parquet'
        progress_event('PANELS',f'ВПО-1: полный расчёт за {_y} год',announce=True)
        try:
            _parts.append(_vpo_copy(_filter,_outfile))
        except duckdb.OutOfMemoryException:
            progress_event('PANELS',f'ВПО-1 {_y}: дробление по индикаторам и специальностям',announce=True)
            for _i,_indicator in enumerate(_vpo_names):
                _ifilter=f'{_filter} AND indicator_name={_esc(_indicator)}'
                _ipath=_shards/f'vpo_year_{_y}_indicator_{_i}.parquet'
                try:
                    _parts.append(_vpo_copy(_ifilter,_ipath))
                except duckdb.OutOfMemoryException:
                    # hash(specialty_code) фиксирован для каждой группы; 16 частей не пересекаются.
                    for _bucket in range(16):
                        _bfilter=f'''{_ifilter} AND hash(COALESCE(CAST(specialty_code AS VARCHAR),'__NULL__')) % 16={_bucket}'''
                        _bpath=_shards/f'vpo_year_{_y}_indicator_{_i}_part_{_bucket:02}.parquet'
                        _parts.append(_vpo_copy(_bfilter,_bpath))
    if not _parts:
        ISSUES.append({'module':'ML-5.4','status':'VPO_NO_MATCHES','reason':'Нет строк по утверждённым именам индикаторов'})
        VPO_FULL_PROGRAM_PATH=None
    else:
        _filelist='['+','.join(_esc(p) for p in _parts)+']'
        with duck_connect() as _con:
            _con.execute("SET preserve_insertion_order=false")
            _con.execute(f'''COPY (SELECT * FROM read_parquet({_filelist})) TO
                          {_esc(VPO_FULL_PROGRAM_PATH)} (FORMAT PARQUET, COMPRESSION ZSTD)''')
            _actual=int(_con.execute('SELECT SUM(observed_records) FROM read_parquet(?)',
                                     [str(VPO_FULL_PROGRAM_PATH)]).fetchone()[0] or 0)
            VPO_FULL_PROGRAM_DESCRIPTION=_con.execute(
                'SELECT * FROM read_parquet(?) LIMIT 2000',[str(VPO_FULL_PROGRAM_PATH)]).df()
        if _actual!=_expected:
            fail('VPO_DATA_GATE',f'Потеря или дубль строк ВПО-1: источник {_expected}, агрегация {_actual}')
        _pf=pq.ParquetFile(VPO_FULL_PROGRAM_PATH)
        PANEL_FILES['vpo_program_description']=VPO_FULL_PROGRAM_PATH
        PANELS['vpo_program_description']=VPO_FULL_PROGRAM_DESCRIPTION
        TABLE_QA.append({'table':'vpo_program_description','rows':_pf.metadata.num_rows,
            'columns':len(_pf.schema_arrow.names),'duplicate_keys':None,'status':'PASS',
            'unique_keys':'','storage':'DUCKDB_YEAR_SHARDED_EXACT_AGGREGATE'})
        progress_event('PANELS',f'ВПО-1: {_expected:,} исходных строк, {_pf.metadata.num_rows:,} групп; контроль PASS',announce=True)
        del _pf
    _parts.clear()
    gc.collect()
# Полная агрегация выше; ниже только просмотр небольшой выборки, который не является полным источником.
vpo_admission=selective_records(VPO_211,['year','object_level','object_name','object_oktmo','university_type',
 'education_form','education_level','specialty_name','specialty_code','indicator_name','indicator_value','admission_type'],
 _vpo_names) if CFG.MODE!='FULL' else (read_table(VPO_211,['year','object_level','object_name','object_oktmo','university_type',
 'education_form','education_level','specialty_name','specialty_code','indicator_name','indicator_value','admission_type'],
 where='indicator_name IN ('+','.join("'"+x.replace("'","''")+"'" for x in _vpo_names)+')',limit=2000)
 if _vpo_names else pd.DataFrame())
if len(vpo_admission):
    vpo_admission['observation_year']=safe_year(vpo_admission.year)
    vpo_admission['value']=number(vpo_admission.indicator_value)
    vpo_admission['geography_grain']='REPORTED_OBJECT_LEVEL_'+vpo_admission.object_level.astype(str)

# Автоматический агрегат для прогнозирования строится только когда один уникальный показатель
# на ID/year, и источник действительно указывает итог, а не несопоставимые категории.
def unambiguous_org_admission(raw,sector):
    if raw.empty or not {'entity_id','observation_year','indicator_name','value'}.issubset(raw):return pd.DataFrame()
    x=raw.dropna(subset=['entity_id','observation_year','value']).copy()
    integerish=np.isclose(x.value.to_numpy(float),np.rint(x.value.to_numpy(float)),atol=1e-8,rtol=0)
    noncount=x.loc[~integerish].copy()
    if len(noncount):QUARANTINE.extend(noncount.head(200).assign(source=sector+'_NONINTEGER_ADMISSION').to_dict('records'))
    x=x.loc[x.value.ge(0)&integerish].copy()
    if x.empty:return pd.DataFrame()
    keys=['entity_id','observation_year']
    counts=x.groupby(keys,dropna=False).agg(n=('value','size'),n_indicators=('indicator_name','nunique')).reset_index()
    usable=counts[(counts.n==1)&(counts.n_indicators==1)][keys]
    good=x.merge(usable,on=keys,validate='many_to_one')
    if good.duplicated(keys).any():fail('SEMANTICS',f'{sector}: неоднозначный показатель приёма после фильтра')
    good=good.assign(sector=sector,target_definition='source_approved_count_of_admitted')
    bad=x.merge(usable.assign(used=True),on=keys,how='left')
    bad=bad[bad.used.isna()].drop(columns=['used'])
    if len(bad):QUARANTINE.extend(bad.head(100).assign(source=f'{sector}_MULTIPLE_ADMISSION_VALUES').to_dict('records'))
    return good[['sector','entity_id','observation_year','value','indicator_name','target_definition']].rename(columns={'value':'actual'})

org_parts=[unambiguous_org_admission(vo_admission,'VO'),unambiguous_org_admission(spo_admission,'SPO')]
ORG_YEAR=qa_table('edu_org_year',pd.concat(org_parts,ignore_index=True) if any(len(p) for p in org_parts)
                  else pd.DataFrame(columns=['sector','entity_id','observation_year','actual','indicator_name','target_definition']),
                  ['sector','entity_id','observation_year'],allow_empty=True)
qa_table('vo_admission_raw_candidates',vo_admission,allow_empty=True)
qa_table('spo_admission_raw_candidates',spo_admission,allow_empty=True)
qa_table('vpo_admission_audit',vpo_admission,allow_empty=True)

# Образовательные программы: источники E02 и ВПО-1 сохраняются без географической реконструкции OD×programme.
parts=[]
if 'vo_program_candidates' in globals() and len(vo_program_candidates):
    v=vo_program_candidates[['entity_id','observation_year','specialty_code','specialty_section','indicator_name','value']].copy()
    v['sector']='VO';v['geography_grain']='ORGANIZATION';parts.append(v)
if len(vpo_admission):
    v=vpo_admission[['observation_year','specialty_code','indicator_name','value','geography_grain']].copy()
    v['specialty_section']=pd.NA;v['entity_id']=pd.NA;v['sector']='VO_VPO';parts.append(v)
PROGRAM=qa_table('edu_program_year',pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(),allow_empty=True)

# Выпускники: FULL обрабатывает каждую строку E04 и E05 через DuckDB, в RAM — только итоги.
# Оригинальные E04/E05 остаются в Stage I (SHA фиксируются). Отчёт об исходных записях явный.
GRADUATE_GROUP_SUMMARY=pd.DataFrame();GRADUATE_FIELD_SUMMARY=pd.DataFrame()
GRAD_E04_VALID_ROWS=0;GRAD_E05_VALID_ROWS=0
GRAD_COLS=['object_level','object_name','oktmo','gender','education_level','year','university',
           'count_graduate','percent_employed','average_salary_fact_avg','average_salary_fact_med']
E05_COLS=['object_level','object_name','year','education_level','study_area','specialty_section',
          'specialty','specialty_code','count_graduate','percent_employed','average_salary_fact_med']
if CFG.MODE=='FULL':
    progress_event('PANELS','E04/E05: загрузка оригинальных файлов, агрегация на диске',announce=True)
    path_e04=fetch_asset(GRAD_ORG);path_e05=fetch_asset(GRAD_FIELD)
    with duck_connect() as con:
        progress_event('PANELS','E04: полная агрегатная обработка DuckDB',announce=True)
        GRADUATE_GROUP_SUMMARY=con.execute("""SELECT TRY_CAST(year AS INTEGER) AS graduation_year,
                 object_level,gender,education_level,COUNT(*)::BIGINT AS n_source_records,
                 MEDIAN(TRY_CAST(count_graduate AS DOUBLE)) AS median_reported_graduates,
                 MEDIAN(TRY_CAST(average_salary_fact_med AS DOUBLE)) AS median_observed_salary,
                 MEDIAN(TRY_CAST(percent_employed AS DOUBLE)) AS median_employment_percentage
          FROM read_parquet(?) WHERE TRY_CAST(count_graduate AS DOUBLE)>=0
          GROUP BY 1,2,3,4""",[str(path_e04)]).df()
        GRAD_E04_VALID_ROWS=int(GRADUATE_GROUP_SUMMARY.n_source_records.sum())
        progress_event('PANELS','E05: полная агрегатная обработка DuckDB',announce=True)
        GRADUATE_FIELD_SUMMARY=con.execute("""SELECT TRY_CAST(year AS INTEGER) AS graduation_year,
                 object_level,education_level,CAST(specialty_code AS VARCHAR) AS specialty_code,
                 COUNT(*)::BIGINT AS observed_records,
                 MEDIAN(TRY_CAST(count_graduate AS DOUBLE)) AS median_reported_graduates,
                 MEDIAN(TRY_CAST(average_salary_fact_med AS DOUBLE)) AS median_wage
          FROM read_parquet(?) WHERE TRY_CAST(count_graduate AS DOUBLE)>=0
            AND specialty_code IS NOT NULL AND TRY_CAST(year AS INTEGER) IS NOT NULL
          GROUP BY 1,2,3,4""",[str(path_e05)]).df()
        GRAD_E05_VALID_ROWS=int(GRADUATE_FIELD_SUMMARY.observed_records.sum()) if len(GRADUATE_FIELD_SUMMARY) else 0
    progress_event('PANELS',f'E04/E05: обработано {GRAD_E04_VALID_ROWS:,}+{GRAD_E05_VALID_ROWS:,} допустимых строк',announce=True)
    GRADUATE_GROUP_SUMMARY['aggregate_totals_identified']=False
    GRADUATE_GROUP_SUMMARY['employment_asof']='2024-12-31'
    GRADUATE_FIELD_SUMMARY['aggregate_totals_identified']=False
    GRADUATE_FIELD_SUMMARY['employment_asof']='2024-12-31'
    qa_table('graduate_descriptive_statistics',GRADUATE_GROUP_SUMMARY,allow_empty=True)
    qa_table('graduate_specialty_description',GRADUATE_FIELD_SUMMARY,allow_empty=True)
    qa_table('graduate_full_scan_audit',pd.DataFrame([
       {'asset_id':GRAD_ORG['asset_id'],'source_rows':int(GRAD_ORG['rows_actual']),
        'valid_rows_in_statistics':GRAD_E04_VALID_ROWS,'method':'DUCKDB_FULL_SCAN_MEDIAN','raw_stage1_sha256':GRAD_ORG['sha256']},
       {'asset_id':GRAD_FIELD['asset_id'],'source_rows':int(GRAD_FIELD['rows_actual']),
        'valid_rows_in_statistics':GRAD_E05_VALID_ROWS,'method':'DUCKDB_FULL_SCAN_MEDIAN','raw_stage1_sha256':GRAD_FIELD['sha256']}]),allow_empty=True)
    # Доступен только явно обозначенный preview, не выдаваемый за полную базу.
    grad=read_table(GRAD_ORG,GRAD_COLS,limit=1000)
    grad['graduation_year']=safe_year(grad['year'])
    grad['employment_asof']=pd.Timestamp('2024-12-31')
    grad['salary_reference_year']=2024;grad['employment_location']=pd.NA
    for c in ['count_graduate','percent_employed','average_salary_fact_avg','average_salary_fact_med']:
        grad[c]=number(grad[c])
    grad['source_id']=GRAD_ORG['asset_id']
    GRADUATES=qa_table('graduate_outcomes_preview',grad,allow_empty=True)
    del grad;gc.collect()
    note('MEMORY','E04/E05 FULL: все строки агрегированы в DuckDB; в Pandas только сводные таблицы и 1000 исходных строк')
else:
    grad=read_table(GRAD_ORG,GRAD_COLS,limit=CFG.MAX_READ_ROWS_SMOKE)
    grad['graduation_year']=safe_year(grad['year']);grad['employment_asof']=pd.Timestamp('2024-12-31')
    grad['salary_reference_year']=2024;grad['employment_location']=pd.NA
    for c in ['count_graduate','percent_employed','average_salary_fact_avg','average_salary_fact_med']:
        grad[c]=number(grad[c])
    grad['source_id']=GRAD_ORG['asset_id']
    GRADUATES=qa_table('graduate_outcomes',grad,allow_empty=True)
    gradf=read_table(GRAD_FIELD,E05_COLS,limit=CFG.MAX_READ_ROWS_SMOKE)
    gradf['graduation_year']=safe_year(gradf['year']);gradf['employment_asof']=pd.Timestamp('2024-12-31')
    gradf['specialty_code']=norm_code(gradf['specialty_code'])
    for c in ['count_graduate','percent_employed','average_salary_fact_med']:gradf[c]=number(gradf[c])
    gradf['source_asset_id']=GRAD_FIELD['asset_id']
    qa_table('graduate_specialty_outcomes',gradf,allow_empty=True)

# Освободить полный первичный E04/E05 сразу после расчёта сводных таблиц.
for _unused in ('gradf',):
    if _unused in globals():del globals()[_unused]
gc.collect();memory_point('panels_after_graduates')

# Муниципальная демография: точные исходные ОКТМО и исторические года; значения не размножаются по вузам.
mun=read_table(MUN,['oktmo','region','municipality','year','population','migration','migration_rate','births','deaths'])
mun['oktmo']=norm_code(mun.oktmo);mun['observation_year']=safe_year(mun.year)
for c in ['population','migration','migration_rate','births','deaths']:mun[c]=number(mun[c])
mun=mun.drop(columns=['year'])
mun['geography_grain']='MUNICIPALITY_FROM_D01'
mun=mun.dropna(subset=['oktmo','observation_year']).copy()
mun_unique=mun.groupby(['oktmo','observation_year']).size()
if (mun_unique>1).any():
    ISSUES.append({'module':'ML-5.7','status':'CONFLICTING_MUNICIPAL_KEYS',
      'reason':f'D01 дублирует ОКТМО×год: {int((mun_unique>1).sum())} ключей; исключены',
      'required':'reconciliation of historical municipality borders'})
    ok=mun_unique[mun_unique==1].index
    mun=mun.set_index(['oktmo','observation_year']).loc[ok].reset_index()
MUNI=qa_table('municipal_demography',mun,['oktmo','observation_year'],allow_empty=True)

# Только verified campus география; наличие без ОКТМО никогда не трактуется как отсутствие.
conf=vo.loc[vo.campus_geo_status.eq('CONFIRMED_STUDY_CAMPUS'),
  ['entity_id','observation_year','oktmo','parent_group_id']].dropna(subset=['oktmo']).copy()
conf=conf[nonmissing_code(conf.oktmo)]
conf=conf.drop_duplicates(['entity_id','observation_year'])
conf=conf.groupby(['oktmo','observation_year'],as_index=False).agg(
    confirmed_vo_sites=('entity_id','nunique'))
MUNI_EDU=MUNI.merge(conf,on=['oktmo','observation_year'],how='left',validate='one_to_one')
MUNI_EDU['education_coverage_known']=MUNI_EDU.confirmed_vo_sites.notna()
# Missing is unknown, NOT zero campuses.
MUNI_EDU['confirmed_spo_sites']=pd.Series(pd.NA,index=MUNI_EDU.index,dtype='Int64')
qa_table('municipality_education_context',MUNI_EDU,['oktmo','observation_year'],allow_empty=True)

qa_table('entity_match_quarantine',pd.DataFrame(QUARANTINE),allow_empty=True)
print('Panels: OD',len(OD),'inter',len(OD_INTER),'VO+SPO registry',len(REGISTRY),
      'org_year',len(ORG_YEAR),'program',len(PROGRAM),'graduates',len(GRADUATES),
      'municipal',len(MUNI_EDU))

# Финальная отметка этапа перенесена после всех его подмодулей


## 5А. Advanced Data Reconstruction · реконструкция первичных показателей и географии
Только наблюдаемые записи: E01 на общем покрытии, все кандидаты приёма E02, географические соответствия СПО с D04 и раздельные флаги пространственной неопределённости.

In [ ]:
# ML-05 V7.1.1 · Advanced Data Reconstruction: audited targets, balanced OD and spatial links.
# Reconstruction is evidence-gated. No inferred school admission, residence, campus or employment location is recorded as observed.
progress_event('PANELS','V7.1: первичная реконструкция по опубликованным наблюдениям',announce=True)
ADR_PREPROCESS_AUDIT=[]
ADR_ADMISSION_YEAR=pd.DataFrame()
ADR_ADMISSION_CANDIDATES=pd.DataFrame()
ADR_GEO_SPO=pd.DataFrame()
ADR_GEO_VERIFIED=pd.DataFrame()
ADR_OD_BALANCED=pd.DataFrame()
ADR_OD_PAIRWISE=pd.DataFrame()
ADR_SEMANTIC_PROVENANCE=[]

def adr_text(s):
    return (s.astype('string').fillna('').str.lower().str.replace('ё','е',regex=False)
            .str.replace(r'[^\w\s-]',' ',regex=True).str.replace(r'\s+',' ',regex=True).str.strip())

def adr_numbers(s):
    return pd.to_numeric(s.astype('string').str.replace(',','.',regex=False),errors='coerce')

def adr_log(stage,source,status,n_rows=0,rule='',reason=''):
    ADR_PREPROCESS_AUDIT.append({'procedure':stage,'source':source,'status':status,
        'n_observations':int(n_rows),'strict_rule':rule,'limitation':str(reason)[:300]})

# ML-5.1 / ML-5.3 / ML-5.6. All-year balanced panel on *actually observed* interregional E01 edges.
# This is an independent coverage-stability study, not synthetic completion of the OD matrix.
_adr_od=OD_INTER[['observation_year','origin_region','destination_region','observed_value']].dropna().copy()
_adr_od['observation_year']=pd.to_numeric(_adr_od['observation_year'],errors='coerce')
_adr_od=_adr_od.loc[_adr_od.observation_year.notna() & _adr_od.observed_value.ge(0)].copy()
_adr_od['observation_year']=_adr_od.observation_year.astype(int)
_adr_keys=['origin_region','destination_region']
_adr_years=sorted(_adr_od.observation_year.unique())
if _adr_od.duplicated(['observation_year']+_adr_keys).any():
    raise RuntimeError('ADR E01: duplicate observed OD edge-year')
if len(_adr_years)>=2:
    _adr_counts=_adr_od.groupby(_adr_keys).observation_year.nunique()
    _adr_shared=_adr_counts[_adr_counts==len(_adr_years)].index
    ADR_OD_BALANCED=_adr_od.set_index(_adr_keys).loc[_adr_shared].reset_index() if len(_adr_shared) else _adr_od.head(0).copy()
    _adr_bal=ADR_OD_BALANCED.groupby('observation_year',as_index=False).agg(
        n_common_edges=('observed_value','size'),total_on_common_edges=('observed_value','sum'),
        median_on_common_edges=('observed_value','median')) if len(ADR_OD_BALANCED) else pd.DataFrame()
    if len(_adr_bal):
        _adr_bal['all_years_shared']=len(_adr_years)
        _adr_bal['scope']='OBSERVED_BALANCED_EDGE_INTERSECTION_NOT_NATIONAL_TOTAL'
    qa_table('adr_od_balanced_edges_by_year',_adr_bal,allow_empty=True)
    if len(ADR_OD_BALANCED):
        qa_table('adr_od_balanced_edges',ADR_OD_BALANCED,['observation_year']+_adr_keys)
    _adr_pairs=[]
    for _left,_right in zip(_adr_years[:-1],_adr_years[1:]):
        if int(_right)!=int(_left)+1:continue
        _x=_adr_od.loc[_adr_od.observation_year.eq(_left),_adr_keys+['observed_value']].rename(columns={'observed_value':'prior'})
        _y=_adr_od.loc[_adr_od.observation_year.eq(_right),_adr_keys+['observed_value']].rename(columns={'observed_value':'current'})
        _both=_x.merge(_y,on=_adr_keys,validate='one_to_one')
        _adr_pairs.append({'prior_year':int(_left),'current_year':int(_right),
           'prior_observed_edges':len(_x),'current_observed_edges':len(_y),'common_edges':len(_both),
           'coverage_of_prior':len(_both)/len(_x) if len(_x) else np.nan,
           'coverage_of_current':len(_both)/len(_y) if len(_y) else np.nan,
           'MAE_persistence_common':float(np.abs(_both.current-_both.prior).mean()) if len(_both) else np.nan,
           'sum_prior_common':float(_both.prior.sum()),'sum_current_common':float(_both.current.sum()),
           'coverage_break_at_2023':int(_right)==2023,
           'claim_scope':'COMMON_REPORTED_EDGES_ONLY_WITHOUT_MISSING_ZERO_IMPUTATION'})
    ADR_OD_PAIRWISE=qa_table('adr_od_pairwise_comparable',pd.DataFrame(_adr_pairs),allow_empty=True)
    adr_log('E01_STABLE_OD',E01['asset_id'],'COMPLETE' if len(ADR_OD_BALANCED) else 'NO_BALANCED_EDGES',
        len(ADR_OD_BALANCED),'intersection of observed OD keys across all available years',
        'No permanent-residence movement or year-of-admission claim')
else:adr_log('E01_STABLE_OD',E01['asset_id'],'SKIPPED_NO_TIME_SUPPORT',len(_adr_od),'at least two years')

# ML-5.2: exhaustively inspect admission-labeled E02 rows at original organization+indicator+year grain.
# Single row may be admitted only when explicit source terminology, count unit and uniqueness all hold.
_adr_e02_out=ROOT/'derived'/'adr_e02_admission_candidates.parquet'
_adr_e02_tmp=ROOT/'derived'/'adr_e02_admission_candidates.incomplete.parquet'
_adr_e02_tmp.unlink(missing_ok=True)
# Stage I E02 read starts only AFTER SQL preflight succeeds.
_adr_e02_path=fetch_asset(E02)
progress_event('PANELS','E02: запись всех подходящих исходных строк на диск',announce=True)
with duck_connect() as _adr_con:
    _adr_con.execute(_adr_e02_copy_statement(_adr_e02_path,_adr_e02_tmp))
    _adr_total=int(_adr_con.execute('SELECT COUNT(*) FROM read_parquet(?)',[str(_adr_e02_tmp)]).fetchone()[0])
    ADR_ADMISSION_CANDIDATES=_adr_con.execute(_ADR_E02_SUMMARY_SQL,[str(_adr_e02_tmp)]).df()
    _adr_e02_tmp.replace(_adr_e02_out)
progress_event('PANELS',f'E02 проверено {_adr_total:,} кандидатных строк; индикаторов {len(ADR_ADMISSION_CANDIDATES)}',announce=True)
if len(ADR_ADMISSION_CANDIDATES):
    _adr_label=adr_text(ADR_ADMISSION_CANDIDATES.indicator_name)
    _adr_unit=adr_text(ADR_ADMISSION_CANDIDATES.indicator_unit)
    ADR_ADMISSION_CANDIDATES['explicit_count_noun']=_adr_label.str.contains(r'численност|количеств|число',regex=True)
    ADR_ADMISSION_CANDIDATES['explicit_admitted_people']=_adr_label.str.contains(r'принят|зачислен',regex=True)
    ADR_ADMISSION_CANDIDATES['ambiguous_meaning']=_adr_label.str.contains(r'заявлен|конкурс|контрольн|план|квот|приемн|доля|процент|на работу',regex=True)
    ADR_ADMISSION_CANDIDATES['unit_is_people']=_adr_unit.str.contains(r'чел|человек',regex=True)
    ADR_ADMISSION_CANDIDATES['semantic_candidate']=ADR_ADMISSION_CANDIDATES.explicit_count_noun & ADR_ADMISSION_CANDIDATES.explicit_admitted_people & ~ADR_ADMISSION_CANDIDATES.ambiguous_meaning & ADR_ADMISSION_CANDIDATES.unit_is_people
    ADR_ADMISSION_CANDIDATES['user_or_method_approval']=ADR_ADMISSION_CANDIDATES.indicator_name.isin(CFG.APPROVED_VO_ADMISSION_NAMES)
    ADR_ADMISSION_CANDIDATES['strict_source_label_supported']=ADR_ADMISSION_CANDIDATES.semantic_candidate & ADR_ADMISSION_CANDIDATES.invalid_numeric_rows.eq(0) & ADR_ADMISSION_CANDIDATES.invalid_count_rows.eq(0)
    ADR_ADMISSION_CANDIDATES['automatic_official_approval']=False  # Source label != externally verified cohort methodology.
qa_table('adr_organizational_admission_dictionary',ADR_ADMISSION_CANDIDATES,allow_empty=True)
adr_log('E02_ADMISSIONS',E02['asset_id'],'SCANNED_FULL',_adr_total,
    'count-unit, label, original reporting grain, single unambiguous organization-year observation',
    'Candidates without methodology approval are described, not asserted as admissions')

# Persist full candidates, but only an explicitly approved indicator with a unique reporting total can enter a new target.
_adr_approved=ADR_ADMISSION_CANDIDATES.loc[
    (ADR_ADMISSION_CANDIDATES['strict_source_label_supported'] & ADR_ADMISSION_CANDIDATES['user_or_method_approval']),
    'indicator_name'].dropna().astype(str).unique().tolist() if len(ADR_ADMISSION_CANDIDATES) else []
if _adr_approved:
    with duck_connect() as _adr_con:
        _adr_con.execute("SET memory_limit='1536MB'")
        _adr_literal_names=','.join("'"+x.replace("'","''")+"'" for x in _adr_approved)
        _adr_sel=_adr_con.execute(_ADR_E02_TOTAL_SQL.replace('/*VALID_NAMES*/',_adr_literal_names),
            [str(_adr_e02_out)]).df()
    _adr_sel=_adr_sel.loc[_adr_sel.organization_id.notna()&_adr_sel.observation_year.notna()&
        _adr_sel.n_reporting_cells.eq(1)&_adr_sel.area_total.fillna(False)&_adr_sel.specialty_total.fillna(False)&
        _adr_sel.min_count.notna()&_adr_sel.min_count.ge(0)&
        _adr_sel.min_count.eq(_adr_sel.max_count)&_adr_sel.min_count.mod(1).eq(0)].copy()
    _adr_sel=_adr_sel.drop_duplicates(['organization_id','observation_year'],keep=False)
    _adr_sel['entity_id']='VO|'+norm_code(_adr_sel.organization_id)
    _adr_sel['actual']=_adr_sel.min_count
    _adr_sel['target_definition']='SOURCE_LABEL_REPORTED_ADMISSION_COUNT_NOT_OFFICIAL_COHORT'
    _adr_sel['methodology_approval']=_adr_sel.indicator_name.isin(CFG.APPROVED_VO_ADMISSION_NAMES)
    _adr_sel['scientific_scope']='EXACT_E02_SOURCE_LABEL_AND_UNIQUE_ORG_YEAR_TOTAL'
    ADR_ADMISSION_YEAR=qa_table('adr_admission_organization_year',_adr_sel,
          ['entity_id','observation_year'],allow_empty=True)
else:
    adr_log('E02_ORG_ADMISSIONS',E02['asset_id'],'SKIPPED_NO_SEMANTICALLY_UNAMBIGUOUS_TARGET',0,
      'explicit source reported admitted count, unit people, unique organization-year TOTAL row',
      'No organization-year admission target invented from section/area counts')

# ML-5.7: relate *reported SPO coordinates* to D04 settlement identifiers, with an ambiguity flag.
# This never silently converts a spatial proxy into an officially verified campus location.
try:
    from sklearn.neighbors import BallTree
    _adr_sp=sporeg[['entity_id','observation_year','region_name','city','lat','lon']].copy()
    _adr_sp['lat_num']=adr_numbers(_adr_sp['lat']); _adr_sp['lon_num']=adr_numbers(_adr_sp['lon'])
    _adr_sp=_adr_sp.loc[_adr_sp.lat_num.between(40,85)&_adr_sp.lon_num.between(19,190)].copy()
    _adr_sp['region_key']=adr_text(_adr_sp.region_name)
    _adr_sp['city_key']=adr_text(_adr_sp.city)
    _adr_sp=_adr_sp.loc[_adr_sp.region_key.ne('')&_adr_sp.city_key.ne('')].copy()
    _adr_grid=fetch_asset(SPATIAL)
    with duck_connect() as _adr_con:
        _adr_d=_adr_con.execute('''SELECT CAST(source_oktmo AS VARCHAR) source_oktmo,
             CAST(region_name AS VARCHAR) region_name, CAST(municipality_name AS VARCHAR) municipality_name,
             CAST(settlement_name AS VARCHAR) settlement_name,
             TRY_CAST(latitude AS DOUBLE) latitude,TRY_CAST(longitude AS DOUBLE) longitude,
             CAST(coordinates_valid AS VARCHAR) coordinates_valid
             FROM read_parquet(?) WHERE TRY_CAST(latitude AS DOUBLE) BETWEEN 40 AND 85
               AND TRY_CAST(longitude AS DOUBLE) BETWEEN 19 AND 190''',[str(_adr_grid)]).df()
    _adr_d['region_key']=adr_text(_adr_d.region_name)
    _adr_d['city_key']=adr_text(_adr_d.settlement_name)
    _adr_d['mun_key']=adr_text(_adr_d.municipality_name)
    _adr_d['source_oktmo']=norm_code(_adr_d.source_oktmo)
    _adr_d=_adr_d.loc[nonmissing_code(_adr_d.source_oktmo)&_adr_d.region_key.ne('')].reset_index(drop=True)
    _adr_match=[]
    for _reg,_rows in _adr_sp.groupby('region_key',sort=False):
        _places=_adr_d.loc[_adr_d.region_key.eq(_reg)].copy()
        if len(_places)<2:continue
        _tree=BallTree(np.radians(_places[['latitude','longitude']].to_numpy(float)),metric='haversine',leaf_size=40)
        _dist,_ix=_tree.query(np.radians(_rows[['lat_num','lon_num']].to_numpy(float)),k=2)
        _places=_places.reset_index(drop=True)
        for _j,_rr in enumerate(_rows.itertuples(index=False)):
            _nearest=_places.iloc[int(_ix[_j,0])]
            _d1=float(_dist[_j,0]*6371.0088); _d2=float(_dist[_j,1]*6371.0088)
            _city=str(_rr.city_key)
            _city_match=(_city==str(_nearest.city_key) or _city==str(_nearest.mun_key))
            _high=bool(_city_match and _d1<=2.0 and _d2-_d1>=0.50)
            _adr_match.append({'entity_id':_rr.entity_id,'observation_year':_rr.observation_year,
                'reported_city':_rr.city,'reported_region':_rr.region_name,
                'candidate_municipal_oktmo':_nearest.source_oktmo,
                'nearest_centroid_km':_d1,'second_nearest_centroid_km':_d2,
                'city_name_consistent':bool(_city_match),'spatially_unambiguous':bool(_high),
                'source_id_spo':SPO_ORG['asset_id'],'source_id_geography':SPATIAL['asset_id'],
                'identity_status':'PROXIMITY_MATCH_NOT_OFFICIAL_CAMPUS_LOCATION' if _high else 'UNRESOLVED'})
    ADR_GEO_SPO=pd.DataFrame(_adr_match)
    if len(ADR_GEO_SPO):
        # Inconsistent historical assignments for the same institution-year are quarantined.
        _dup=ADR_GEO_SPO.duplicated(['entity_id','observation_year'],keep=False)
        ADR_GEO_SPO.loc[_dup,'spatially_unambiguous']=False
        ADR_GEO_SPO.loc[_dup,'identity_status']='CONFLICTING_HISTORICAL_LOCATION'
        ADR_GEO_VERIFIED=ADR_GEO_SPO.loc[ADR_GEO_SPO.spatially_unambiguous].drop_duplicates(['entity_id','observation_year']).copy()
    qa_table('adr_spo_geolocation_candidates',ADR_GEO_SPO,allow_empty=True)
    qa_table('adr_spo_geolocation_high_confidence',ADR_GEO_VERIFIED,allow_empty=True)
    adr_log('SPO_GEOCODE',SPATIAL['asset_id'],'PROXIMITY_RECONSTRUCTION',len(ADR_GEO_VERIFIED),
        'same region, city/settlement match, <2km, nearest separation >=0.5km',
        'Geocoded municipality is a reconstructed spatial proxy, not officially confirmed campus placement')
except (KeyError,ValueError,TypeError) as _geo_error:
    adr_log('SPO_GEOCODE',SPATIAL['asset_id'],'SKIPPED_INCOMPATIBLE_SPATIAL_FIELDS',0,'valid coordinates/municipal code',str(_geo_error))

qa_table('adr_preprocessing_provenance',pd.DataFrame(ADR_PREPROCESS_AUDIT),allow_empty=True)
progress_event('PANELS',f'V7.1: common OD={len(ADR_OD_BALANCED):,}; approved admissions={len(ADR_ADMISSION_YEAR):,}; SPO geo proxies={len(ADR_GEO_VERIFIED):,}',announce=True)

mark_stage('PANELS')


## 6. Временные признаки, экономика и транспорт
Все признаки имеют исторический период и осторожный лаг доступности. Дистанции между городами остаются прокси, не истинным расстоянием регионов.

In [ ]:
memory_point('before_features')
progress_event('FEATURES','География, экономический контекст и исторические признаки')
# Экономический и пространственный контекст: только фактические данные, с пометкой уровня агрегации.
NAME_MAP=pd.concat([
    OD_INTER[['origin_region','origin_region_name']].rename(columns={'origin_region':'region_code','origin_region_name':'region_name'}),
    OD_INTER[['destination_region','destination_region_name']].rename(columns={'destination_region':'region_code','destination_region_name':'region_name'})
],ignore_index=True).drop_duplicates()
NAME_MAP['region_name_norm']=NAME_MAP.region_name.astype('string').str.lower().str.replace('ё','е').str.strip()
conflicts=NAME_MAP.groupby('region_code',dropna=False).region_name_norm.nunique(dropna=True)
if (conflicts>1).any():
    QUARANTINE.extend(NAME_MAP[NAME_MAP.region_code.isin(conflicts[conflicts>1].index)].head(150)
                      .assign(source='REGION_CODE_NAME_CONFLICT').to_dict('records'))
    NAME_MAP=NAME_MAP[~NAME_MAP.region_code.isin(conflicts[conflicts>1].index)].copy()
NAME_MAP=NAME_MAP.drop_duplicates(subset=['region_code'])
if NAME_MAP.region_code.duplicated().any():fail('GEOGRAPHY','Конфликтная карта кодов субъектов')
# Название региона должно задавать единственный код. Иначе исключаем сомнительные соединения.
_name_counts=NAME_MAP.groupby('region_name_norm',dropna=False).region_code.nunique()
_ambiguous=set(_name_counts[_name_counts>1].index)
if _ambiguous:
    QUARANTINE.extend(NAME_MAP[NAME_MAP.region_name_norm.isin(_ambiguous)].head(100).assign(source='AMBIGUOUS_REGION_NAME').to_dict('records'))
    ISSUES.append({'module':'GEOGRAPHY','status':'AMBIGUOUS_REGION_NAME','reason':f'{len(_ambiguous)} названий соответствуют нескольким кодам','required':'territorial crosswalk'})
NAME_LOOKUP=NAME_MAP[NAME_MAP.region_name_norm.notna() & ~NAME_MAP.region_name_norm.isin(_ambiguous)].copy()
REGION_CONTEXT=[];DISTANCE=pd.DataFrame()

try:
    raw=read_table(TRANSPORT,['region_from','region_to','distance_km','transport_type'],
                   limit=CFG.MAX_READ_ROWS_SMOKE if CFG.MODE=='SMOKE' else None)
    raw['origin_region_name_norm']=raw.region_from.astype('string').str.lower().str.replace('ё','е').str.strip()
    raw['destination_region_name_norm']=raw.region_to.astype('string').str.lower().str.replace('ё','е').str.strip()
    raw['distance_proxy_km']=number(raw.distance_km)
    raw=raw[(raw.distance_proxy_km>0)&raw.distance_proxy_km.notna()]
    d=raw.groupby(['origin_region_name_norm','destination_region_name_norm'],as_index=False).agg(
        distance_proxy_km=('distance_proxy_km','median'),distance_measurements=('distance_proxy_km','size'))
    o=NAME_LOOKUP[['region_code','region_name_norm']].rename(columns={'region_code':'origin_region',
        'region_name_norm':'origin_region_name_norm'})
    t=NAME_LOOKUP[['region_code','region_name_norm']].rename(columns={'region_code':'destination_region',
        'region_name_norm':'destination_region_name_norm'})
    DISTANCE=d.merge(o,on='origin_region_name_norm',validate='many_to_one').merge(
        t,on='destination_region_name_norm',validate='many_to_one')[['origin_region','destination_region',
        'distance_proxy_km','distance_measurements']]
    DISTANCE=DISTANCE.drop_duplicates(['origin_region','destination_region'])
    note('GEOGRAPHY','Расстояние — медиана транспортных пар городов; НЕ истинное расстояние субъект-субъект')
except Exception as ex:
    note('GEOGRAPHY','Транспортный контекст недоступен: '+type(ex).__name__,'WARN')
qa_table('region_distance_proxy',DISTANCE,allow_empty=True)
# Карантин дополняется при географическом сопоставлении, исходная панель обновляется после этапа.
PANELS['entity_match_quarantine']=pd.DataFrame(QUARANTINE)

# Региональный контекст собирается только из источников с явно указанным субъектным grain.
D05_LABOR=source_asset(source_id='D05')
for a in D05_LABOR:
    if 'object_level' not in a.get('columns_manifest',[]) or 'indicator_name' not in a.get('columns_manifest',[]):continue
    try:
        path=fetch_asset(a)
        with duck_connect() as con:
            sql='''SELECT object_name, object_level, year, indicator_name, indicator_unit,
                         TRY_CAST(indicator_value AS DOUBLE) AS indicator_value,
                         COUNT(*) as duplicates
                    FROM read_parquet(?)
                    WHERE regexp_matches(lower(indicator_name), 'безработ|заработ|доход|зарплат|численность рабочей силы')
                    GROUP BY ALL'''
            v=con.execute(sql,[str(path)]).df()
        v['source_asset_id']=a['asset_id']
        v['observation_year']=safe_year(v.year)
        # Нельзя считать объект регионом только по совпадению названия; требуется документированный уровень.
        v=v[v.object_level.astype(str).str.contains(r'регион|субъект|region',case=False,regex=True,na=False)].copy()
        if len(v):REGION_CONTEXT.append(v)
    except Exception as ex:
        ISSUES.append({'module':'ML-5.1','status':'ECONOMIC_SOURCE_GATED',
            'reason':a['asset_id']+': '+type(ex).__name__,'required':'correct unit and regional grain'})
ECON_RAW=pd.concat(REGION_CONTEXT,ignore_index=True) if REGION_CONTEXT else pd.DataFrame()
qa_table('economic_indicator_candidates',ECON_RAW,allow_empty=True)

# Точное сопоставление названий; отсутствие точной пары = NA, не произвольное присвоение.
ECON_PANEL=pd.DataFrame()
if len(ECON_RAW):
    e=ECON_RAW.copy()
    e['region_name_norm']=e.object_name.astype('string').str.lower().str.replace('ё','е').str.strip()
    e=e.merge(NAME_LOOKUP[['region_code','region_name_norm']],on='region_name_norm',how='inner',
              validate='many_to_one')
    e=e.dropna(subset=['indicator_value','observation_year'])
    e['indicator_key']=e.indicator_name.astype(str)+'|'+e.indicator_unit.astype(str)
    # Только единичное значение: похожие категории и единицы не складываются.
    natural=['region_code','observation_year','indicator_key']
    multiplicity=e.groupby(natural,dropna=False)['indicator_value'].transform('size')
    if 'duplicates' in e:
        multiplicity=multiplicity.where(e['duplicates'].fillna(1).le(1),other=2)
    rejected=e.loc[multiplicity.ne(1)].copy()
    if len(rejected):
        ISSUES.append({'module':'ML-5.1','status':'ECONOMIC_GRAIN_CONFLICT',
           'reason':f'Исключено {len(rejected)} неоднозначных строк регион×год×индикатор',
           'required':'verified regional stratum and indicator definition'})
        QUARANTINE.extend(rejected.head(200).assign(source='ECON_CONFLICTING_REGION_YEAR_INDICATOR').to_dict('records'))
    e=e.loc[multiplicity.eq(1)].copy()
    if len(e) and e.duplicated(natural).any():fail('ECONOMY','Конфликт натурального ключа регион×год×индикатор')
    if not e.empty:
        top=e.groupby('indicator_key').region_code.nunique().sort_values(ascending=False).head(3).index
        e=e[e.indicator_key.isin(top)]
        ECON_PANEL=e[['region_code','observation_year','indicator_key','indicator_value','source_asset_id']]
qa_table('regional_economy',ECON_PANEL,allow_empty=True)

# Подготовка лага/скользящих сумм без доступа к целевому году.
def historical_pair_panel(observed,entity_fields,target_col,feature_lag=None,horizon=None):
    fl=CFG.FEATURE_PUBLICATION_LAG_YEARS if feature_lag is None else feature_lag
    h=CFG.HORIZON if horizon is None else horizon
    if observed.empty:return pd.DataFrame()
    o=observed.copy()
    if 'observation_year' not in o:fail('TIME','Нет observation_year')
    o['observation_year']=safe_year(o.observation_year)
    if o.duplicated(entity_fields+['observation_year']).any():fail('TIME','Дубли натурального временного ключа')
    out=o.copy()
    out['target_year']=out.observation_year.astype('Int64')
    out['origin_year']=out.target_year-h
    out['information_cutoff_year']=out.origin_year
    out['historical_year']=out.origin_year-fl
    out['historical_year_2']=out.historical_year-1
    out=out.rename(columns={target_col:'actual'})
    hist=o[entity_fields+['observation_year',target_col]].rename(columns={
        'observation_year':'historical_year',target_col:'prior_value'})
    hist2=o[entity_fields+['observation_year',target_col]].rename(columns={
        'observation_year':'historical_year_2',target_col:'prior2_value'})
    out=out.merge(hist,on=entity_fields+['historical_year'],how='left',validate='many_to_one')
    out=out.merge(hist2,on=entity_fields+['historical_year_2'],how='left',validate='many_to_one')
    out=out[out.prior_value.notna()&out.actual.notna()].copy()
    out['feature_available_year']=out.historical_year+fl
    out['target_available_year']=out.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS
    out['horizon']=h
    out['source_vintage']=INDEX['created_at']
    out['vintage_warning']='RETROSPECTIVE_ONLY_HISTORICAL_VINTAGES_NOT_ARCHIVED'
    out['split_id']='UNASSIGNED'
    if not (out.feature_available_year<=out.information_cutoff_year).all():
        fail('TIME','Будущие признаки попали в historical panel')
    return out

OD_MODEL=historical_pair_panel(
    OD_INTER[['observation_year','origin_region','destination_region','observed_value']].dropna(subset=['observed_value']).copy(),
    ['origin_region','destination_region'],'observed_value')
if not OD_MODEL.empty:
    OD_MODEL['entity_id']=OD_MODEL.origin_region.astype(str)+'|'+OD_MODEL.destination_region.astype(str)
    OD_MODEL['sector']='SPO'
    # Маргинали/число партнёров строго за доступный прошлый год.
    counts=OD_INTER.groupby(['observation_year','origin_region']).agg(
        origin_mass=('observed_value','sum'),origin_destinations=('destination_region','nunique')).reset_index()
    dests=OD_INTER.groupby(['observation_year','destination_region']).agg(
        destination_mass=('observed_value','sum'),destination_origins=('origin_region','nunique')).reset_index()
    outm=counts.rename(columns={'observation_year':'historical_year'})
    inm=dests.rename(columns={'observation_year':'historical_year'})
    OD_MODEL=OD_MODEL.merge(outm,on=['historical_year','origin_region'],validate='many_to_one',how='left')
    OD_MODEL=OD_MODEL.merge(inm,on=['historical_year','destination_region'],validate='many_to_one',how='left')
    if len(DISTANCE):OD_MODEL=OD_MODEL.merge(DISTANCE[['origin_region','destination_region','distance_proxy_km']],
                                          on=['origin_region','destination_region'],validate='many_to_one',how='left')
    # Лагирование экономических факторов проверяется датой доступности. Никаких future margins.
    if len(ECON_PANEL):
        e=ECON_PANEL.pivot(index=['region_code','observation_year'],columns='indicator_key',
                           values='indicator_value').reset_index()
        keys=[k for k in e.columns if k not in ('region_code','observation_year')]
        for direction in ('origin','destination'):
            f=e.rename(columns={'region_code':f'{direction}_region','observation_year':'historical_year',
                                **{k:f'{direction}_econ_{i}' for i,k in enumerate(keys)}})
            OD_MODEL=OD_MODEL.merge(f,on=[f'{direction}_region','historical_year'],
                                    how='left',validate='many_to_one')
    OD_MODEL['target_id']=OD_SEMANTIC['label']
    flagged=coverage_df.loc[coverage_df.potential_break.eq(True),'year'] if 'potential_break' in coverage_df else pd.Series(dtype=int)
    OD_MODEL['coverage_break_target_year']=OD_MODEL.target_year.isin(flagged).astype(bool)
    OD_MODEL['coverage_break_historical_year']=OD_MODEL.historical_year.isin(flagged).astype(bool)
qa_table('forecast_od_candidates',OD_MODEL,allow_empty=True)

# Отток: СУММА известных межрегиональных образовательных маршрутов, не migration_residence.
OUTFLOW=OD_INTER.dropna(subset=['observed_value']).groupby(['observation_year','origin_region'],as_index=False).agg(
    observed_interregional_routes=('observed_value','sum'),observed_destinations=('destination_region','nunique'))
OUTFLOW['sector']='SPO';OUTFLOW['target_id']='E01_observed_interregional_routes_from_school_region'
qa_table('outflow_observed',OUTFLOW,['observation_year','origin_region'],allow_empty=True)
OUTFLOW_MODEL=historical_pair_panel(OUTFLOW[['observation_year','origin_region','observed_interregional_routes']],
                                  ['origin_region'],'observed_interregional_routes')
if len(OUTFLOW_MODEL):
    OUTFLOW_MODEL['entity_id']=OUTFLOW_MODEL.origin_region.astype(str)
    OUTFLOW_MODEL['sector']='SPO';OUTFLOW_MODEL['target_id']='E01_observed_interregional_routes_from_school_region'
qa_table('forecast_outflow_candidates',OUTFLOW_MODEL,allow_empty=True)

ORG_MODEL=pd.DataFrame()
if len(ORG_YEAR):
    for sector,g in ORG_YEAR.groupby('sector'):
        m=historical_pair_panel(g[['entity_id','observation_year','actual']],['entity_id'],'actual')
        if len(m):
            m['sector']=sector
            m['target_id']='verified_organizational_admission_count'
            if ORG_MODEL.empty:ORG_MODEL=m
            else:ORG_MODEL=pd.concat([ORG_MODEL,m],ignore_index=True)
    if len(ORG_MODEL):
        rg=REGISTRY[['sector','entity_id','observation_year','parent_group_id','region']].rename(
            columns={'observation_year':'target_year'})
        ORG_MODEL=ORG_MODEL.merge(rg,on=['sector','entity_id','target_year'],how='left',validate='many_to_one')
qa_table('forecast_org_candidates',ORG_MODEL,allow_empty=True)

# Профиль программ по одному источнику; несопоставимые классификаторы не агрегируем.
PROGRAM_PROFILE=pd.DataFrame()
if len(PROGRAM) and {'specialty_code','value'}.issubset(PROGRAM):
    pp=PROGRAM[(PROGRAM.sector=='VO')&PROGRAM.specialty_code.notna()].copy()
    pp=pp.dropna(subset=['value','observation_year'])
    pp['specialty_code']=norm_code(pp.specialty_code)
    pp=pp[pp.value.ge(0)]
    # Не складывать разные индикаторы в одном направлении/году: оставляем строгую согласованную совокупность.
    natural=['entity_id','observation_year','specialty_code','indicator_name']
    multiplicity=pp.groupby(natural,dropna=False).size().rename('records').reset_index()
    ok=multiplicity.loc[multiplicity.records.eq(1),natural]
    excluded=int(len(pp)-len(ok))
    if excluded:ISSUES.append({'module':'ML-5.4','status':'AMBIGUOUS_PROGRAM_GRAIN',
      'reason':f'{excluded} строк нельзя однозначно агрегировать по программе; карантин без произвольного сложения',
      'required':'programme/level/form/classifier disaggregation'})
    pp=pp.merge(ok,on=natural,how='inner',validate='one_to_one')
    # Суммы на одном и том же идентификаторе индикатора допустимы лишь как суммы строк с непересекающимися организациями.
    grp=pp.groupby(['observation_year','specialty_code','indicator_name'],as_index=False).agg(
        observed_value=('value','sum'),organizations=('entity_id','nunique'))
    if not grp.empty:
        grp['sector']='VO';grp['source_grain_gate']='UNIQUE_ORG_YEAR_SPECIALTY_INDICATOR_NOT_POPULATION_TOTAL';PROGRAM_PROFILE=grp
qa_table('program_profile',PROGRAM_PROFILE,allow_empty=True)
# ВПО-1 может содержать реальную специализацию при отсутствии утверждённой организации-приёма: не отключать описание.
VPO_PROGRAM_DESCRIPTION=pd.DataFrame()
if len(vpo_admission):
    z=vpo_admission.dropna(subset=['observation_year','specialty_code','value']).copy()
    z['specialty_code']=norm_code(z.specialty_code)
    if len(z):
        VPO_PROGRAM_DESCRIPTION=z.groupby(['observation_year','specialty_code','indicator_name','object_level',
          'education_form','education_level','admission_type'],dropna=False,as_index=False).agg(
          observed_records=('value','size'),median_reported_value=('value','median'),
          distinct_geographic_names=('object_name','nunique'))
        VPO_PROGRAM_DESCRIPTION['totals_additive_confirmed']=False
if VPO_FULL_PROGRAM_PATH is None:
    qa_table('vpo_program_description',VPO_PROGRAM_DESCRIPTION,allow_empty=True)
else:
    VPO_PROGRAM_DESCRIPTION=VPO_FULL_PROGRAM_DESCRIPTION
    note('VPO','ВПО-1: полная агрегированная панель находится в evidence-Parquet; Pandas содержит только preview')

progress_event('FEATURES',f'Готовы исторические панели: OD {len(OD_MODEL):,}, отток {len(OUTFLOW_MODEL):,}',announce=True)
print('Feature coverage:',len(OD_MODEL),'OD historical rows,',len(OUTFLOW_MODEL),'outflow,',
      len(ORG_MODEL),'organization; economic indicators',len(ECON_PANEL),
      'distance pairs',len(DISTANCE))

memory_point('after_features')
mark_stage('FEATURES')


## 6А. Исследовательское использование всех групп источников

Этот блок выводит дополнительные контекстные показатели без смешения пространственных уровней: экономические и образовательные индикаторы БДМО, муниципальную историю ОКТМО, географические справочники, все E04/E05 и все разделы ВПО. Крупные группировки записываются непосредственно в Parquet. Математически корректное сопоставление сохраняет исходный `indicator_code` и ограничения на сложение пересекающихся разрезов.

In [ ]:
# FULL52 second pass: thematic *derived* panels at the original statistical grain.
# Each of 52 datasets has already passed a physical SHA + row-coverage audit above.
memory_point('before_cross_source');progress_event('CROSS_SOURCE','БДМО, ВПО, E04/E05, СПО и история ОКТМО',announce=True)
CONTEXT_PARQUET_FILES=[];D02_PROFILE=[];VPO_ALL_PROFILE=[];SPO_YEAR_CHECK=[];FAMILY_E04E05=[];SPATIAL_CROSS=[]
BDMO_EDU_PROXY=pd.DataFrame();BDMO_EDU_PROXY_CANDIDATES=pd.DataFrame()
BDMO_SPATIAL=pd.DataFrame();D04_SPATIAL=pd.DataFrame();HISTORY_MUNICIPAL=pd.DataFrame();BDMO_PROXY_FULL_VALID=False
GRAD_ALL_ASSETS=[]

def _sql_literal(v):return "'"+str(v).replace("'","''")+"'"

def thematic_sql_disk(asset,query,out_stem,preview=1500,mem='2GB'):
    # SELECT query may contain __INPUT__ placeholder; never use SELECT * to pandas.
    p=fetch_asset(asset);out=ROOT/'derived'/'full52_context'/(out_stem+'.parquet')
    out.parent.mkdir(parents=True,exist_ok=True)
    sql=query.replace('__INPUT__',_sql_literal(p))
    tmp=out.with_suffix('.partial.parquet');tmp.unlink(missing_ok=True)
    with duckdb.connect(database=':memory:') as con:
        con.execute(f"SET memory_limit='{mem}'")
        con.execute('SET threads=1');con.execute('SET preserve_insertion_order=false')
        con.execute('SET temp_directory='+_sql_literal(DB_TMP))
        con.execute('COPY ('+sql+') TO '+_sql_literal(tmp)+' (FORMAT PARQUET, COMPRESSION ZSTD)')
        data=con.execute('SELECT * FROM read_parquet(?) LIMIT '+str(int(preview)),[str(tmp)]).df()
    tmp.replace(out);pf=pq.ParquetFile(out)
    CONTEXT_PARQUET_FILES.append(out)
    return out,data,int(pf.metadata.num_rows)

# A. Every economic/BDMO source: content-level year/indicator/region coverage (but no false additive sums).
# 5-NDFL/7-NDFL cut across multiple tax forms; count/coverage ≠ observed number of residents.
for i,a in enumerate(source_asset(segment='02') if CFG.MODE=='FULL' else source_asset(prefix='02::labor'),1):
    aid=a['asset_id'];cols=set(a['columns_manifest']);progress_event('CROSS_SOURCE',f'D02 [{i}/11]: {aid}',i,11,announce=True)
    ind=next((c for c in ('indicator_code','indicator_name') if c in cols),None)
    yr='year' if 'year' in cols else None
    geo=next((c for c in ('region_name','report_region_name','object_name') if c in cols),None)
    if not ind or not yr:continue
    select_geo='CAST('+_sql_literal('UNSPECIFIED_LEVEL')+' AS VARCHAR)' if not geo else 'CAST("'+geo+'" AS VARCHAR)'
    level_where="AND regexp_matches(lower(CAST(object_level AS VARCHAR)),'регион|субъект|region')" if geo=='object_name' and 'object_level' in cols else ''
    query=f'''SELECT CAST("{yr}" AS VARCHAR) observed_year,
      {select_geo} AS reported_geography,CAST("{ind}" AS VARCHAR) indicator_key,
      COUNT(*)::BIGINT AS reported_records,
      COUNT(DISTINCT CAST("{geo}" AS VARCHAR)) AS n_locations
      FROM read_parquet(__INPUT__) WHERE "{ind}" IS NOT NULL {level_where}
      GROUP BY 1,2,3'''
    if geo is None:query=query.replace('COUNT(DISTINCT CAST("None" AS VARCHAR))','COUNT(*)::BIGINT')
    try:
        p,preview,rows=thematic_sql_disk(a,query,'d02_'+str(i).zfill(2),preview=800,mem='2GB')
        D02_PROFILE.append({'asset_id':aid,'rows_in_source':a['rows_actual'],'year_indicator_geography_groups':rows,
            'profile_parquet':p.name,'status':'GROUPED_ALL_SOURCE_ROWS_NO_DOUBLE_COUNT'})
    except Exception as ex:
        ISSUES.append({'module':'D02','status':'SOURCE_PROFILE_FAILURE','reason':aid+' '+type(ex).__name__+': '+str(ex)[:130],
                       'required':'repair dedicated source query; full SHA inventory still valid'})
        D02_PROFILE.append({'asset_id':aid,'rows_in_source':a['rows_actual'],'year_indicator_geography_groups':None,
                            'status':'GROUP_PROFILE_FAILED'})
qa_table('d02_indicator_coverage',pd.DataFrame(D02_PROFILE),['asset_id'])

# B. All 11 E02/VPO tables: count by publication year and exact indicator name, each input processed independently.
for i,a in enumerate(source_asset(segment='04') if CFG.MODE=='FULL' else source_asset(prefix='04::E02'),1):
    cols=set(a.get('columns_manifest') or [])
    if 'indicator_name' not in cols:continue
    query='''SELECT TRY_CAST(year AS INTEGER) observation_year,CAST(indicator_name AS VARCHAR) indicator_name,
      COUNT(*)::BIGINT observed_rows
      FROM read_parquet(__INPUT__) WHERE indicator_name IS NOT NULL
      GROUP BY 1,2'''
    try:
        p,preview,rows=thematic_sql_disk(a,query,'vpo_'+str(i).zfill(2),preview=500,mem='2GB')
        VPO_ALL_PROFILE.append({'asset_id':a['asset_id'],'input_rows':a['rows_actual'],
                                'year_indicator_groups':rows,'parquet':p.name,'status':'PASS'})
    except Exception as ex:
        VPO_ALL_PROFILE.append({'asset_id':a['asset_id'],'input_rows':a['rows_actual'],
                                'status':'PROFILE_FAILED_'+type(ex).__name__})
        ISSUES.append({'module':'VPO-1','status':'SOURCE_PROFILE_FAILURE','reason':a['asset_id']+' '+type(ex).__name__,
                       'required':'retry annual indicator group scan'})
qa_table('vpo_all_sections_audit',pd.DataFrame(VPO_ALL_PROFILE),['asset_id'])

# C. SPO all-years and independent yearly shards; diagnose overlaps without summing them.
spo_assets=source_asset(segment='03',role='organization_indicators')
annual=pd.DataFrame(FULL52_YEAR_STATS)
master_id=SPO_MASTER['asset_id'] if SPO_MASTER else None
if len(annual) and master_id:
    ref=annual.loc[annual.asset_id.eq(master_id)].set_index('year_label')['rows']
    for a in spo_assets:
        cur=annual[annual.asset_id.eq(a['asset_id'])]
        for year,sub in cur.groupby('year_label'):
            n=int(sub['rows'].sum());v=ref.get(year,pd.NA)
            SPO_YEAR_CHECK.append({'asset_id':a['asset_id'],'year':year,'rows_in_representation':n,
               'rows_in_master':v,'same_count':bool(n==v) if pd.notna(v) else None,
               'treatment':'NOT_ADDITIVE_WITH_MASTER'})
qa_table('spo_annual_vs_master',pd.DataFrame(SPO_YEAR_CHECK),allow_empty=True)

# D. All four E04/E05 datasets: by actual year, educational level, gender or area (never sum intersecting cross-sections).
for i,a in enumerate(source_asset(segment='05') if CFG.MODE=='FULL' else source_asset(segment='05')[:1],1):
    cols=set(a.get('columns_manifest') or [])
    group=['year','education_level']+[q for q in ('gender','study_area') if q in cols]
    keys=','.join('"'+k+'"' for k in group)
    q=f'''SELECT {keys},COUNT(*)::BIGINT source_records,
          MEDIAN(TRY_CAST(count_graduate AS DOUBLE)) median_reported_graduates,
          MEDIAN(TRY_CAST(percent_employed AS DOUBLE)) median_employed_percent,
          MEDIAN(TRY_CAST(average_salary_fact_med AS DOUBLE)) median_wage
          FROM read_parquet(__INPUT__) GROUP BY {keys}'''
    try:
        p,d,n=thematic_sql_disk(a,q,'graduates_source_'+str(i),preview=2000,mem='2GB')
        FAMILY_E04E05.append({'asset_id':a['asset_id'],'input_rows':a['rows_actual'],
           'group_rows':n,'group_columns':','.join(group),'status':'PASS_NOT_ADDITIVE',
           'employment_asof':'2024-12-31','geography_of_work_known':False})
        qa_table('graduate_family_'+str(i),d,allow_empty=True)
        PANEL_FILES['graduate_family_'+str(i)]=p
    except Exception as ex:
        FAMILY_E04E05.append({'asset_id':a['asset_id'],'status':'FAILED_'+type(ex).__name__})
        ISSUES.append({'module':'ML-5.5','status':'FAMILY_PROFILE_FAILED','reason':a['asset_id']+' '+type(ex).__name__,
           'required':'recheck exact statistical strata'})
qa_table('e04_e05_full_family_audit',pd.DataFrame(FAMILY_E04E05),['asset_id'])

# E. Historical municipal reference: full SQL join, no 431k+ rows in Pandas.
for key,asset_prefix in [('D04','06::D04::D04_member00::spatial_reference'),
                         ('BDMO_REG','06::BDMO6::BDMO6_member00::municipality_registry_observed'),
                         ('BDMO_HIST','06::BDMO6::BDMO6_member00::municipality_history_observed')]:
    a=required_one(source_asset(prefix=asset_prefix),key)
    cols=a.get('columns_manifest') or []
    chosen=[x for x in ('source_oktmo','region_name','municipality_name','settlement_name','observation_year',
      'coordinates_valid','latitude','longitude','census_year','observed_name_change','observed_type_change',
      'boundary_change_not_verified') if x in cols]
    if chosen:
        frame=read_table(a,chosen,limit=2000)  # explicitly labelled preview; full source profiled above
        if key=='D04':D04_SPATIAL=frame
        elif key=='BDMO_REG':BDMO_SPATIAL=frame
        else:HISTORY_MUNICIPAL=frame
        qa_table('spatial_'+key+'_preview',frame,allow_empty=True)

registry_asset=required_one(source_asset(prefix='06::BDMO6::BDMO6_member00::municipality_registry_observed'),'BDMO REG')
municipal_path=fetch_asset(MUN); registry_path=fetch_asset(registry_asset)
query=f'''SELECT CAST(d.oktmo AS VARCHAR) oktmo, TRY_CAST(d.year AS INTEGER) observation_year,
    MIN(CAST(d.region AS VARCHAR)) region, MIN(CAST(d.municipality AS VARCHAR)) municipality,
    MIN(CAST(b.municipality_name AS VARCHAR)) municipality_name,
    COUNT(DISTINCT CAST(b.municipality_name AS VARCHAR)) n_registry_names
    FROM read_parquet({_sql_literal(municipal_path)}) d
    INNER JOIN read_parquet({_sql_literal(registry_path)}) b
      ON CAST(d.oktmo AS VARCHAR)=CAST(b.source_oktmo AS VARCHAR)
     AND TRY_CAST(d.year AS INTEGER)=TRY_CAST(b.observation_year AS INTEGER)
    GROUP BY 1,2'''
try:
    with duckdb.connect(':memory:') as con:
        con.execute("SET memory_limit='2GB'");con.execute('SET threads=1')
        joined=con.execute(query).df()  # upper bound: 31,445 D01 rows
    joined['municipality_names_equal']=joined['municipality'].astype(str).str.lower().str.replace('ё','е').str.strip().eq(
       joined['municipality_name'].astype(str).str.lower().str.replace('ё','е').str.strip())
    joined['spatial_gate']='EXACT_OKTMO_YEAR_NAME_COMPARE_ONLY_NOT_CAMPUS'
    SPATIAL_CROSS=joined
    qa_table('spatial_exact_crosswalk',joined,['oktmo','observation_year'],allow_empty=True)
except Exception as ex:
    ISSUES.append({'module':'GEOGRAPHY','status':'CROSSWALK_FAILED','reason':type(ex).__name__+': '+str(ex)[:140],
       'required':'inspect municipality-year and historical OKTMO definitions'})
    qa_table('spatial_exact_crosswalk',pd.DataFrame(),allow_empty=True)

# F. Educational infrastructure D02::education at municipality×year×indicator_code, conservative uniqueness.
edu_source=required_one(source_asset(prefix='02::education'),'BDMO educational indicators')
edu_cols=edu_source['columns_manifest']
# Selection is a *hypothesis candidate*, not an automatic campus register.
pattern='(профессиональн|высш[еиг]|колледж|техникум|университет|образовательн.*организац)'
q=f'''SELECT CAST(oktmo AS VARCHAR) oktmo,TRY_CAST(year AS INTEGER) observation_year,
  CAST(indicator_code AS VARCHAR) indicator_code,CAST(indicator_name AS VARCHAR) indicator_name,
  CAST(indicator_unit AS VARCHAR) indicator_unit,
  COUNT(*)::BIGINT n_raw_rows,
  MIN(TRY_CAST(indicator_value AS DOUBLE)) AS min_reported,
  MAX(TRY_CAST(indicator_value AS DOUBLE)) AS max_reported,
  MEDIAN(TRY_CAST(indicator_value AS DOUBLE)) AS median_reported
  FROM read_parquet(__INPUT__)
  WHERE regexp_matches(lower(CAST(indicator_name AS VARCHAR)), {_sql_literal(pattern)})
  AND oktmo IS NOT NULL AND year IS NOT NULL
  GROUP BY 1,2,3,4,5'''
try:
    if CFG.MODE!='FULL':raise ValueError('MUNICIPAL_ECONOMIC_FULL_SCAN_ONLY')
    fp,preview,n=thematic_sql_disk(edu_source,q,'bdmo_education_municipal',preview=1000,mem='2GB')
    # Bounded filtered result only. If high-cardinality, use SQL to filter exact one-row groups first.
    with duckdb.connect(':memory:') as con:
        con.execute("SET memory_limit='2GB'");con.execute('SET threads=1')
        nuniq=int(con.execute('SELECT COUNT(*) FROM read_parquet(?) WHERE n_raw_rows=1 AND min_reported=max_reported',[str(fp)]).fetchone()[0])
        if nuniq<=CFG.MAX_PANDAS_ROWS:
            BDMO_PROXY_FULL_VALID=True
            BDMO_EDU_PROXY_CANDIDATES=con.execute('''SELECT * FROM read_parquet(?)
              WHERE n_raw_rows=1 AND min_reported=max_reported AND median_reported>=0''',[str(fp)]).df()
        else:
            BDMO_EDU_PROXY_CANDIDATES=preview.copy()
            ISSUES.append({'module':'ML-5.7','status':'PROXY_PREVIEW_ONLY',
               'reason':f'{nuniq} unique municipal education rows exceed RAM cap','required':'SQL-only E0/E1'})
    BDMO_EDU_PROXY_CANDIDATES['source_definition']='BDMO_MUNICIPAL_EDUCATION_INDICATOR_NOT_CAMPUS'
    qa_table('bdmo_education_candidates',BDMO_EDU_PROXY_CANDIDATES,allow_empty=True)
except Exception as ex:
    ISSUES.append({'module':'ML-5.7','status':'BDMO_EDUCATION_REVIEW_SKIPPED_OR_FAILED',
       'reason':type(ex).__name__+': '+str(ex)[:150],'required':'review BDMO indicator units/levels'})
    qa_table('bdmo_education_candidates',pd.DataFrame(),allow_empty=True)

# Human-readable quality conditions, all sourced from actually processed files.
qa_table('municipal_infrastructure_candidates',BDMO_EDU_PROXY_CANDIDATES,allow_empty=True)
print('CROSS SOURCE:',len(D02_PROFILE),'D02 members;',len(VPO_ALL_PROFILE),'E02/VPO;',
      len(FAMILY_E04E05),'E04/E05;',len(SPATIAL_CROSS),'exact municipal code-year matches;',
      len(BDMO_EDU_PROXY_CANDIDATES),'municipal infrastructure candidates')
memory_point('after_cross_source')
# Финальная отметка этапа перенесена после всех его подмодулей


## 6Б. Содержательная проверка 52/52 источников
Пофайловое содержание без подмены единиц измерения; годовые СПО и raw-матрицы не суммируются повторно.

In [ ]:
# ML-05 V4 · independent full-content utilization register (52/52).
# These are measurements of the SOURCE itself, not automatically legal predictive features.
progress_event('CROSS_SOURCE','V4: физическое обследование значений всех наборов Stage I',announce=True)
V4_SOURCE_PROFILES=[];V4_SOURCE_ERRORS=[]
for _i,_a in enumerate(ASSETS,1):
    _id=_a['asset_id'];_p=fetch_asset(_a);_cols=list(pq.ParquetFile(_p).schema_arrow.names)
    _time=next((x for x in ('year','observation_year','graduation_year') if x in _cols),None)
    _measure=next((x for x in ('indicator_value','observed_value','count_graduate','population','value','migration') if x in _cols),None)
    _y=f'TRY_CAST("{_time}" AS INTEGER)' if _time else 'NULL::INTEGER'
    _v=f'TRY_CAST("{_measure}" AS DOUBLE)' if _measure else 'NULL::DOUBLE'
    _q=f'''SELECT {_y} year_label,COUNT(*)::BIGINT source_rows,
      COUNT({_v})::BIGINT numeric_values,
      SUM(CASE WHEN {_v}<0 THEN 1 ELSE 0 END)::BIGINT negative_values,
      SUM(CASE WHEN {_v}=0 THEN 1 ELSE 0 END)::BIGINT zero_values,
      SUM(CASE WHEN {_v} IS NULL THEN 1 ELSE 0 END)::BIGINT null_or_non_numeric,
      MIN({_v}) AS minimum_observed_value,
      MAX({_v}) AS maximum_observed_value,
      AVG({_v}) AS unweighted_mean_of_source_records,
      STDDEV_POP({_v}) AS dispersion_of_source_records
      FROM read_parquet({_sql_literal(_p)}) GROUP BY 1'''
    _out=ROOT/'derived'/'v4_source_values'/f'source_{_i:02d}.parquet';_out.parent.mkdir(parents=True,exist_ok=True)
    _tmp=_out.with_suffix('.partial.parquet');_tmp.unlink(missing_ok=True)
    progress_event('CROSS_SOURCE',f'V4 [{_i}/52] число, пропуски, нули: {_id}',_i,len(ASSETS),announce=(_i==1 or _i%5==0 or _i==len(ASSETS)))
    try:
        with duckdb.connect(':memory:') as _db:
            _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
            _db.execute('SET preserve_insertion_order=false')
            _db.execute('SET temp_directory='+_sql_literal(DB_TMP))
            _db.execute('COPY ('+_q+') TO '+_sql_literal(_tmp)+' (FORMAT PARQUET, COMPRESSION ZSTD)')
            _r=_db.execute('SELECT COALESCE(SUM(source_rows),0),COUNT(*) FROM read_parquet(?)',[str(_tmp)]).fetchone()
        if int(_r[0])!=int(_a['rows_actual']):raise AssertionError(f'Content count {_r[0]} != {_a["rows_actual"]}')
        _tmp.replace(_out);CONTEXT_PARQUET_FILES.append(_out)
        V4_SOURCE_PROFILES.append({'asset_id':_id,'segment':_a['segment'],'source_id':_a.get('source_id'),
           'rows_actual':int(_r[0]),'time_field':_time,'numeric_field':_measure,
           'groups_by_year':int(_r[1]),'sha_verified':_id in INPUT_SHA,'content_status':'FULL_VALUE_COVERAGE_PASS',
           'model_eligibility':'REQUIRES_TARGET_SEMANTICS_AND_GRAIN','derived_parquet':_out.name})
    except Exception as _ex:
        _tmp.unlink(missing_ok=True)
        V4_SOURCE_ERRORS.append({'asset_id':_id,'error':type(_ex).__name__,'details':str(_ex)[:280]})
        V4_SOURCE_PROFILES.append({'asset_id':_id,'segment':_a['segment'],'source_id':_a.get('source_id'),
           'rows_actual':None,'time_field':_time,'numeric_field':_measure,'groups_by_year':None,
           'sha_verified':_id in INPUT_SHA,'content_status':'FAIL_CONTENT_SCAN','derived_parquet':''})
_v4_used=pd.DataFrame(V4_SOURCE_PROFILES)
qa_table('v4_all52_source_value_audit',_v4_used,['asset_id'])
qa_table('v4_source_scan_failures',pd.DataFrame(V4_SOURCE_ERRORS),allow_empty=True)
if CFG.MODE=='FULL' and (len(_v4_used)!=52 or len(V4_SOURCE_ERRORS)):
    fail('ALL52_CONTENT','One or more Stage I members failed full numeric content scan: '+str(V4_SOURCE_ERRORS[:2]))

# Source -> defensible research use. Each stage is recorded independently, no claim of 52 model predictors.
_v4_roles=[]
for _a in ASSETS:
    _seg=_a['segment'];_r=_a.get('role','') or ''
    _topics={'01':'Municipal demography and ML-5.7','02':'Economic/education context and ML-5.1/ML-5.7',
      '03':'SPO OD, organizational panels and network ML-5.1/2/3/6',
      '04':'VO monitoring, admissions and specialties ML-5.2/5.4',
      '05':'Graduate employability and outcomes ML-5.5',
      '06':'Historic OKTMO, places and geographic accessibility ML-5.1/5.7'}
    _direct=_a['asset_id'] in {E01['asset_id'],E02['asset_id'],VPO_211['asset_id'],
              GRAD_ORG['asset_id'],GRAD_FIELD['asset_id'],MUN['asset_id'],TRANSPORT['asset_id']}
    _v4_roles.append({'asset_id':_a['asset_id'],'segment':_seg,'role_in_program':_topics.get(_seg),
       'content_scanned':True,'scientific_content_used':'YES_SOURCE_SPECIFIC_PROFILE',
       'main_prediction_input':bool(_direct),
       'not_main_input_reason':('Derived/alternate reporting representation or incompatible key/grain;' 
                                'not pooled into target by assumption' if not _direct else ''),
       'prevent_double_counting':_seg=='03' and (_r=='organization_indicators' or _r=='education_od_matrix_raw_cells'),
       'source_sha':INPUT_SHA.get(_a['asset_id'],'')})
qa_table('v4_all52_research_use',pd.DataFrame(_v4_roles),['asset_id'])
print('V4 / source use:',len(V4_SOURCE_PROFILES),'complete value-audits;',len(V4_SOURCE_ERRORS),'failures')


In [ ]:
# V7.1.1 · источники: обследование содержимого и проверяемые аналитические роли.
# Ни один из 52 наборов не объявляется модельным признаком только потому, что SHA сошлась.
progress_event('CROSS_SOURCE','V7: контроль научного обследования всех 52 источников',announce=True)
V7_SOURCE_AUDIT=[]
_v7_profile={r['asset_id']:r for r in V4_SOURCE_PROFILES}
_v7_inventory={r['asset_id']:r for r in FULL52_INVENTORY}
_v7_roles={
 '01':'Муниципальная демография: временные величины и состав E0/E1',
 '02':'Доходы, занятость, бюджет и образование: раздельные источниковые распределения',
 '03':'СПО, OD, реестры, сопоставление матриц и годовых вариантов',
 '04':'Мониторинг ВО, приём и структура специальностей',
 '05':'Выпуск, занятость и зарплаты: отдельные непересекающиеся страты',
 '06':'История ОКТМО, география, поселения и транспорт'
}
for _a in ASSETS:
    _id=_a['asset_id'];_p=_v7_profile.get(_id,{});_i=_v7_inventory.get(_id,{})
    _full=_p.get('content_status')=='FULL_VALUE_COVERAGE_PASS'
    _asset_name=_p.get('derived_parquet','')
    _stats_path=ROOT/'derived'/'v4_source_values'/_asset_name
    V7_SOURCE_AUDIT.append({
      'asset_id':_id,'source_group':_a['segment'],'rows_in_manifest':int(_a['rows_actual']),
      'sha256':_a['sha256'],'verified_sha256':INPUT_SHA.get(_id)==_a['sha256'],
      'footer_rows_verified':bool(_i.get('footer_rows_ok')),
      'all_values_profiled':bool(_full),
      'numeric_profile_file':_asset_name,
      'numeric_profile_exists':bool(_stats_path.is_file()),
      'time_field':_p.get('time_field'),'numeric_field':_p.get('numeric_field'),
      'source_specific_statistical_use':_v7_roles.get(_a['segment'],'UNREGISTERED'),
      'predictor_role':'NOT_DETERMINED_UNTIL_MODEL_FIT',
      'scientific_constraint':'NO_CROSS_SOURCE_ADDITIVITY_WITHOUT_GRAIN_AND_UNIT_VALIDATION'
    })
V7_SOURCE_AUDIT=pd.DataFrame(V7_SOURCE_AUDIT)
qa_table('v7_all52_scientific_source_audit',V7_SOURCE_AUDIT,['asset_id'])
if CFG.MODE=='FULL' and (len(V7_SOURCE_AUDIT)!=52 or not
        (V7_SOURCE_AUDIT.verified_sha256 & V7_SOURCE_AUDIT.footer_rows_verified &
         V7_SOURCE_AUDIT.all_values_profiled & V7_SOURCE_AUDIT.numeric_profile_exists).all()):
    raise RuntimeError('V7: хотя бы один источник не прошёл полный числовой/временной/хеш-контроль')
print('V7: исходники 52/52 проверены, содержательные профили раздельны; модельная трассировка будет сформирована после обучения')

mark_stage('CROSS_SOURCE')


## 7. Предрегистрация, временные разбиения, модели
Сопоставляются persistence, среднее двух наблюдений, затухающий тренд, Ridge, CatBoost; PPML только для доказанных неотрицательных счётных величин. TEST не выбирает победителя.

In [ ]:
progress_event('MODEL_DESIGN','Разбиения, метрики, базовые методы, Ridge/CatBoost')
# ML-05 · план эксперимента и функции, одинаковые для всех доступных модулей.
EXPERIMENT_PLAN={
 'version':CFG.VERSION,'mode':CFG.MODE,'horizon':CFG.HORIZON,
 'source_commit':SOURCE_COMMIT,'source_index_blob':index_info['sha'],
 'feature_lag':CFG.FEATURE_PUBLICATION_LAG_YEARS,
 'label_lag':CFG.LABEL_PUBLICATION_LAG_YEARS,
 'candidate_models':['persistence','rolling_mean_2','damped_trend','ridge','elasticnet','catboost','lightgbm','ppml_if_observed_integer_counts','negbin_fixed_dispersion_if_integer_counts'],
 'admission_approval_required':True,'unknown_edges_never_zero':True,
 'E01_is_not_certified_admissions':True,
 'validation':'DEV only if its label is available by TEST origin; otherwise fixed preregistered comparison, no selection',
 'seeds_dev':CFG.DEV_SEEDS,'seeds_final':CFG.FINAL_SEEDS,
 'test_keys_policy':'strict common support; no synthetic missing OD edges',
 'source_vintage':INDEX['created_at'],
 'publication_policy':'only FULL + local checks + explicit opt-in + science gate',
}
PLAN_HASH=canonical_hash(EXPERIMENT_PLAN)
PRED_KEYS=['module','target_id','sector','entity_id','origin_year','horizon','target_year',
           'split_id','model','seed','information_cutoff_year','source_vintage',
           'geography_grain','observation_definition','actual','predicted']

# Предварительно заданные признаки; семантика источника не попадает в X как oracle.
CAND_NUM=['prior_value','prior2_value','origin_mass','destination_mass',
          'origin_destinations','destination_origins','distance_proxy_km',
          'confirmed_vo_sites','prev_population','prev_migration_rate',
          'origin_year']
CAND_CAT=['origin_region','destination_region','region','sector','parent_group_id']

def feature_spec(frame):
    nums=[c for c in CAND_NUM if c in frame.columns and number(frame[c]).notna().any()]
    cats=[c for c in CAND_CAT if c in frame.columns and frame[c].notna().any()]
    for c in frame.columns:
        if c.startswith(('origin_econ_','destination_econ_')) and number(frame[c]).notna().any():nums.append(c)
    if 'prior_value' not in nums:fail('MODEL','Нет prior_value: нельзя оценить историческую инерцию')
    return nums,cats

def x_prepare(frame,nums,cats):
    x=frame[nums+cats].copy()
    for c in nums:x[c]=number(x[c]).astype(float)
    for c in cats:x[c]=x[c].fillna('__MISSING__').astype(str)
    return x

def effective_mae(y,yp):
    y=np.asarray(y,dtype=float);p=np.asarray(yp,dtype=float)
    mask=np.isfinite(y)&np.isfinite(p)
    if not mask.any():return {'MAE':np.nan,'RMSE':np.nan,'WAPE':np.nan,'n':0}
    y,p=y[mask],p[mask]
    return {'MAE':float(np.mean(np.abs(y-p))),'RMSE':float(np.sqrt(np.mean((y-p)**2))),
            'WAPE':float(np.abs(y-p).sum()/np.abs(y).sum()) if np.abs(y).sum()>0 else np.nan,'n':int(len(y))}

# Проверка, что один и тот же target не оказался одновременно в train/dev/test.
def temporal_partition(frame,min_train=None,min_test=None):
    if frame.empty:return None,'EMPTY_FRAME'
    f=frame.sort_values('target_year').copy()
    years=sorted(int(x) for x in f.target_year.dropna().unique())
    if len(years)<2:return None,'INSUFFICIENT_TARGET_YEARS'
    test_year=years[-1];test_origin=test_year-CFG.HORIZON
    tr=f[(f.target_year<test_year)&
         (f.target_available_year<=test_origin)].copy()
    test=f[f.target_year==test_year].copy()
    min_train=int(CFG.MIN_TRAIN_ROWS if min_train is None else min_train)
    min_test=int(CFG.MIN_TEST_ROWS if min_test is None else min_test)
    if len(tr)<min_train or len(test)<min_test:
        return None,f'LOW_SUPPORT: train={len(tr)},test={len(test)}'
    # DEV может существовать только если доступен к моменту финального решения.
    available_dev=sorted(int(y) for y in tr.target_year.unique())
    dev=None
    if len(available_dev)>=2:
        dv=available_dev[-1];dev_origin=dv-CFG.HORIZON
        early=tr[(tr.target_year<dv)&(tr.target_available_year<=dev_origin)].copy()
        potential=tr[tr.target_year==dv].copy()
        if len(early)>=min_train and len(potential)>=CFG.MIN_DEV_ROWS:
            dev=potential;tr=early
    if tr.duplicated(['entity_id','target_year']).any():fail('SPLIT','Неверная кратность train')
    if test.duplicated(['entity_id','target_year']).any():fail('SPLIT','Неверная кратность test')
    if (tr.target_available_year>test_origin).any():fail('SPLIT','train label published after test origin')
    if (test.feature_available_year>test.information_cutoff_year).any():fail('SPLIT','future feature in test')
    if dev is not None and (dev.target_available_year>test_origin).any():fail('SPLIT','dev label leaked to test')
    out={'train':tr,'dev':dev,'test':test,'test_year':test_year,'test_origin':test_origin,
         'protocol':'DEV_SELECTED' if dev is not None else 'FIXED_NO_DEV',
         'retrospective_vintage':True}
    SPLIT_AUDIT.append({'module':f.get('module',pd.Series(['UNSET'])).iloc[0],
       'protocol':out['protocol'],'train_rows':len(tr),'dev_rows':0 if dev is None else len(dev),
       'test_rows':len(test),'test_year':test_year,'test_origin':test_origin,
       'labels_known_asof_test':True,'source_vintage_at_test':'NOT_ARCHIVED'})
    return out,None


# V4: fixed-dispersion NB GLM, not mislabeled as an estimated NB2 dispersion model.
from sklearn.base import BaseEstimator, RegressorMixin
class NegativeBinomialFixedDispersion(RegressorMixin, BaseEstimator):
    """NB2 (alpha fixed), penalized MLE with stable log-link and declared prediction-support bound.

    Count targets must be observed nonnegative integers. Fits a low-dimensional, numeric-only
    design with L2 penalty; never densifies the high-cardinality categorical one-hot matrix.
    The predeclared upper bound is 10×max(train count): clipping is reported as diagnostic,
    so predictions cannot silently explode into 1e68 while still receiving PASS.
    """
    def __init__(self, alpha=1.0, penalty=0.5, support_multiplier=10.0):
        self.alpha=float(alpha);self.penalty=float(penalty);self.support_multiplier=float(support_multiplier)
    def fit(self,X,y):
        from scipy.optimize import minimize
        from scipy.special import expit
        a=np.asarray(X,dtype=np.float64); y=np.asarray(y,dtype=np.float64).reshape(-1)
        if a.ndim==1:a=a.reshape(-1,1)
        if a.shape[0]!=len(y) or not len(y) or a.shape[1]>64:raise ValueError('NB2 incompatible design')
        if not np.isfinite(y).all() or (y<0).any() or not np.allclose(y,np.round(y),atol=1e-8,rtol=0):
            raise ValueError('NB2 requires observed integer nonnegative counts')
        a=np.clip(np.nan_to_num(a,nan=0.0,posinf=0.0,neginf=0.0),-8.,8.)
        a=np.column_stack([np.ones(len(y)),a]); r=1.0/self.alpha
        # Stable NB log-likelihood: (y+r)*softplus(eta-log(r))-y*(eta-log(r)), up to constants.
        scale=max(1.,float(np.mean(y+r)));x0=np.zeros(a.shape[1]);x0[0]=np.log(max(.001,float(np.mean(y))))
        def fun(beta):
            e=a@beta-np.log(r)
            loss=np.mean(((y+r)*np.logaddexp(0.,e)-y*e))/scale+0.5*self.penalty*np.dot(beta[1:],beta[1:])
            g=(a.T@((y+r)*expit(e)-y))/(len(y)*scale)
            g[1:]+=self.penalty*beta[1:]
            return float(loss),g
        bounds=[(-22.,22.)]+[(-6.,6.)]*(a.shape[1]-1)
        opt=minimize(fun,x0,method='L-BFGS-B',jac=True,bounds=bounds,options={'maxiter':350,'ftol':1e-10,'gtol':1e-7})
        if not opt.success or not np.isfinite(opt.fun):
            raise RuntimeError('NB2 regularized likelihood did not converge: '+str(opt.message))
        self.coef_=opt.x;self.fit_converged_=True;self.fit_objective_=float(opt.fun)
        self.train_max_=float(np.max(y));self.train_mean_=float(np.mean(y))
        self.n_fit_=len(y);self.count_truncated_=0;self.max_raw_prediction_=np.nan
        return self
    def predict(self,X):
        a=np.asarray(X,dtype=np.float64)
        if a.ndim==1:a=a.reshape(-1,1)
        a=np.clip(np.nan_to_num(a,nan=0.,posinf=0.,neginf=0.),-8.,8.)
        eta=np.column_stack([np.ones(len(a)),a])@self.coef_
        # The model's response space is constrained by a *predeclared* historical support bound.
        # This is not replacement of missing observations. Truncation frequency is an output QC.
        max_mean=max(1.,self.train_max_)*self.support_multiplier
        clipped=np.clip(eta,-30.,np.log(max_mean))
        self.count_truncated_=int(np.count_nonzero(clipped!=eta))
        self.truncated_fraction_=float(self.count_truncated_/max(1,len(a)))
        self.max_raw_logmu_=float(np.max(eta)) if len(a) else np.nan
        pred=np.exp(clipped)
        if not np.isfinite(pred).all():raise FloatingPointError('NB2 unstable response')
        return pred

def pipeline_fit_predict(model,train,test,seed=11,allow_ppml=False,nonnegative=True):
    nums,cats=feature_spec(train)
    if set(nums+cats)&set(['actual','target_year','target_available_year','observed_value']):
        fail('LEAKAGE','В модель попала фактическая целевая переменная или период метки')
    Xtr=x_prepare(train,nums,cats);Xte=x_prepare(test,nums,cats)
    y=number(train.actual).to_numpy(float)
    if not np.isfinite(y).all():fail('MODEL','Невалидные training labels')
    if model=='ridge':
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                                        ('scale',StandardScaler())]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.5)),
                      ('model',Ridge(alpha=30.,solver='lsqr'))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='elasticnet':
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                                        ('scale',StandardScaler())]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.7)),
             ('model',ElasticNet(alpha=1.0,l1_ratio=.35,max_iter=4000,random_state=(None if int(seed)<0 else int(seed)),selection='cyclic'))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='lightgbm':
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median'))]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.6)),
             ('model',LGBMRegressor(n_estimators=220,num_leaves=15,max_depth=5,learning_rate=.05,
                       reg_lambda=8.,min_child_samples=15,verbosity=-1,n_jobs=1,random_state=(None if int(seed)<0 else int(seed))))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='ppml':
        if not allow_ppml or np.any(y<0):raise ValueError('PPML requires verified nonnegative counts')
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                                        ('scale',StandardScaler())]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.5)),
                      ('model',PoissonRegressor(alpha=3.,max_iter=200))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='negbin':
        if not allow_ppml or np.any(y<0) or not np.isfinite(y).all():
            raise ValueError('Negative binomial GLM requires observed nonnegative finite counts')
        # NB2: numeric-only design; avoids high-cardinality one-hot IRLS explosion.
        # All numeric inputs are observed past information, not current target.
        obj=Pipeline([('prep',ColumnTransformer([
                 ('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                          ('scale',StandardScaler())]),nums)],sparse_threshold=0.)),
            ('model',NegativeBinomialFixedDispersion(alpha=1.,penalty=0.5,
                          support_multiplier=CFG.MAX_COUNT_PREDICTION_MULTIPLIER))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        cats=[];artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats,
                  'dispersion':'FIXED_ALPHA_1_PENALIZED_NB2',
                  'extrapolation_rule':'PREDICT_MEAN_AT_MOST_10X_MAX_OBSERVED_TRAIN_COUNT',
                  'prediction_guard_events':int(obj.named_steps['model'].count_truncated_),
                  'prediction_guard_fraction':float(obj.named_steps['model'].truncated_fraction_)}
    elif model=='catboost':
        med={c:float(Xtr[c].median()) if Xtr[c].notna().any() else 0. for c in nums}
        for c in nums:
            Xtr[c]=Xtr[c].fillna(med[c]);Xte[c]=Xte[c].fillna(med[c])
        obj=CatBoostRegressor(iterations=60 if CFG.MODE=='SMOKE' else 260,
                  depth=5,learning_rate=.05,l2_leaf_reg=8.,loss_function='RMSE',
                  random_seed=int(seed),verbose=False,thread_count=1,used_ram_limit='2gb',allow_writing_files=False)
        obj.fit(Xtr,y,cat_features=cats);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats,'medians':med}
    else:fail('MODEL','Неизвестная модель '+model)
    yp=np.asarray(yp,dtype=np.float64)
    if len(yp)!=len(test) or not np.isfinite(yp).all():raise FloatingPointError(model+' invalid prediction array')
    if nonnegative and (yp<0).any():yp=np.maximum(yp,0)
    return yp,artifact

def reload_predict(artifact,test,nonnegative=True):
    X=x_prepare(test,artifact['nums'],artifact['cats'])
    if artifact['kind']=='catboost':
        for c,v in artifact['medians'].items():X[c]=X[c].fillna(v)
    p=artifact['pipeline'].predict(X)
    return np.maximum(p,0) if nonnegative else p

def baseline_prediction(test,kind,nonnegative=True):
    a=number(test.prior_value).to_numpy(float)
    if kind=='persistence':b=a
    elif kind in ('rolling_mean_2','damped_trend'):
        other=number(test.prior2_value).to_numpy(float)
        b=np.where(np.isfinite(other),(a+other)/2 if kind=='rolling_mean_2' else a+.35*(a-other),a)
    else:fail('BASELINE','Неизвестный baseline '+kind)
    return np.maximum(b,0) if nonnegative else b

def predictions_frame(frame,module,model,yp,seed=-1,protocol='ROLLING_TIME'):
    out=pd.DataFrame({'module':module,'target_id':frame.get('target_id',pd.Series(['UNDEFINED']*len(frame))).to_numpy(),
     'sector':frame.get('sector',pd.Series(['UNKNOWN']*len(frame))).to_numpy(),
     'entity_id':frame.entity_id.astype(str).to_numpy(),
     'origin_year':frame.origin_year.to_numpy(),'horizon':CFG.HORIZON,'target_year':frame.target_year.to_numpy(),
     'split_id':protocol,'model':model,'seed':int(seed),
     'information_cutoff_year':frame.information_cutoff_year.to_numpy(),
     'feature_available_year':frame.feature_available_year.to_numpy(),
     'target_available_year':frame.target_available_year.to_numpy(),
     'source_vintage':INDEX['created_at'],
     'geography_grain':'ORIGIN_DESTINATION' if module=='ML-5.1' else ('REGION' if module=='ML-5.3' else 'ORGANIZATION'),
     'observation_definition':frame.get('target_id',pd.Series(['observed_value']*len(frame))).to_numpy(),
     'actual':number(frame.actual).to_numpy(float),'predicted':np.asarray(yp,dtype=float)})
    for c in ('origin_region','destination_region','parent_group_id','region'):
        if c in frame:out[c]=frame[c].astype('string').to_numpy()
    return out

def record_prediction(frame):
    # Прогнозы всех сидов сохраняются немедленно; массивы не остаются в RAM.
    global PRED_TOTAL
    out=ROOT/'derived'/'prediction_parts';out.mkdir(parents=True,exist_ok=True)
    path=out/f'part_{len(PRED_PARTS):05d}.parquet'
    write_df_chunks(frame,path)
    PRED_PARTS.append(str(path));PRED_TOTAL+=int(len(frame))
    del frame
    if len(PRED_PARTS)%10==0:
        gc.collect();memory_point(f'predictions_saved_{len(PRED_PARTS)}')

def fit_and_record(module,model,train,test,seed=-1,nonnegative=True,allow_ppml=False,protocol='ROLLING_TIME'):
    sector=str(test.sector.iloc[0]) if 'sector' in test and test.sector.nunique()==1 else 'MIXED'
    if model in ('persistence','rolling_mean_2','damped_trend'):
        pred=baseline_prediction(test,model,nonnegative);artifact=None
    else:
        pred,artifact=pipeline_fit_predict(model,train,test,seed=seed,allow_ppml=allow_ppml,
                                         nonnegative=nonnegative)
        ident={'module':module,'sector':sector,'protocol':protocol,
               'target_id':sorted(test.target_id.astype(str).unique().tolist()) if 'target_id' in test else [],
               'model':model,'seed':int(seed)}
        short=hashlib.sha256(json.dumps(ident,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:16]
        name=f'{module.replace("-","").replace(".","_")}_{re.sub(r"[^A-Za-z0-9_]+","_",sector)[:28]}_{re.sub(r"[^A-Za-z0-9_]+","_",protocol)[:32]}_{model}_{seed}_{short}.joblib'
        path=ROOT/'models'/name
        if name in MODEL_FILES or path.exists():
            raise RuntimeError('DUPLICATE_MODEL_ARTIFACT: '+name)
        joblib.dump(artifact,path,compress=3)
        loaded=joblib.load(path)
        rp=reload_predict(loaded,test,nonnegative=nonnegative)
        if not np.allclose(rp,pred,rtol=1e-7,atol=1e-7):fail('MODEL_RELOAD',name+' discrepancy')
        MODEL_FILES[name]=str(path)
        MODEL_RECORDS.append({'module':module,'sector':sector,'protocol':protocol,'target_id':ident['target_id'],'model':model,'seed':seed,'path':name,
            'file_sha256':hash_file(path),'feature_num':artifact['nums'],'feature_cat':artifact['cats'],
            'training_rows':len(train),'test_rows':len(test),'reloaded':True,
            'prediction_guard_events':artifact.get('prediction_guard_events',0),
            'prediction_guard_fraction':artifact.get('prediction_guard_fraction',0.)})
    if len(pred)!=len(test):fail('SUPPORT','Модель изменила число тестовых ключей')
    record_prediction(predictions_frame(test,module,model,pred,seed,protocol))
    met=effective_mae(test.actual.to_numpy(float),pred)
    METRICS.append({'module':module,'sector':sector,'model':model,'protocol':protocol,'seed':seed,
       'target_year':int(test.target_year.max()),'source_vintage':INDEX['created_at'],
       **met})
    # Процент неизвестен (ветвление зависит от данных); выводим точный счётчик.
    progress_event(None,f'{module}: {model}, рассчитано {len(METRICS):,} метрик, сохранено {len(MODEL_FILES):,} моделей',
                   current=len(METRICS),total=None,announce=False)
    return pred

mark_stage('MODEL_DESIGN')


## 8. Встроенные интеграционные проверки без GitHub-публикации
11 автоматических тестов на малом искусственном наборе проверяют чистоту времени, восстановление моделей и карантин. Их численные значения не используются как научные результаты.

In [ ]:
progress_event('UNIT_TESTS','Проверка моделей, утечек и повторного чтения')
# Локальные встроенные тесты математической и организационной логики; не являются научными наблюдениями.
SELF_TESTS=[]
def unit_result(name,condition):
    SELF_TESTS.append({'test':name,'result':'PASS' if bool(condition) else 'FAIL'})
    if not bool(condition):fail('UNIT',name)

# Временной split — метки всегда доступны к соответствующему моменту принятия решения.
_fixture=pd.DataFrame([{'entity_id':f'TEST_{e:02d}','observation_year':y,
                        'response':float(e+2*(y-2010)), 'sector':'UNIT'}
                       for e in range(35) for y in range(2012,2026)])
_panel=historical_pair_panel(_fixture[['entity_id','observation_year','response']],['entity_id'],'response')
_panel['sector']='UNIT';_panel['module']='UNIT';_panel['region']='UNIT_REGION'
_panel['target_id']='ARTIFICIAL_TEST_ONLY'
_split,_reason=temporal_partition(_panel)
unit_result('TIME_PANEL_EXISTS',len(_panel)>100)
unit_result('NO_FUTURE_FEATURES',bool((_panel.feature_available_year<=_panel.information_cutoff_year).all()))
unit_result('SPLIT_AVAILABLE',_split is not None)
unit_result('TRAIN_LABELS_KNOWN_AS_OF_TEST',bool((_split['train'].target_available_year<=_split['test_origin']).all()))
unit_result('TEST_KEYS_UNIQUE',not _split['test'].duplicated(['entity_id','target_year']).any())
if _split['dev'] is not None:
    unit_result('DEV_LABELS_KNOWN_AS_OF_TEST',bool((_split['dev'].target_available_year<=_split['test_origin']).all()))

# Модельный прогноз должен повторяться без дообучения после восстановления состояния.
_train=_split['train'];_test=_split['test'];_y_ridge,_ridge_art=pipeline_fit_predict('ridge',_train,_test)
unit_result('RIDGE_RELOAD_EQUAL',np.allclose(_y_ridge,reload_predict(_ridge_art,_test),atol=1e-8,rtol=1e-8))
_cb_y,_cb_art=pipeline_fit_predict('catboost',_train,_test,seed=CFG.DEV_SEEDS[0])
unit_result('CATBOOST_RELOAD_EQUAL',np.allclose(_cb_y,reload_predict(_cb_art,_test),atol=1e-7,rtol=1e-7))
unit_result('MODEL_OUTPUT_FINITE',np.isfinite(_y_ridge).all() and np.isfinite(_cb_y).all())
_en_y,_en_art=pipeline_fit_predict('elasticnet',_train,_test,seed=-1)
unit_result('ELASTICNET_SENTINEL_NEGATIVE_SEED_SAFE',np.isfinite(_en_y).all() and
             np.allclose(_en_y,reload_predict(_en_art,_test)))
_nb_fixture=_train.copy();_nb_test=_test.copy()
for _q in (_nb_fixture,_nb_test):_q['actual']=np.maximum(np.rint(_q.actual),0)
_po_y,_po_art=pipeline_fit_predict('ppml',_nb_fixture,_nb_test,seed=-1,allow_ppml=True)
unit_result('PPML_OBSERVED_INTEGER_FIT_AND_RELOAD',np.isfinite(_po_y).all() and
             np.allclose(_po_y,reload_predict(_po_art,_nb_test)))
_nb_y,_nb_art=pipeline_fit_predict('negbin',_nb_fixture,_nb_test,seed=-1,allow_ppml=True)
unit_result('NEGBIN_FIXED_DISPERSION_RELOAD',np.isfinite(_nb_y).all() and
             np.allclose(_nb_y,reload_predict(_nb_art,_nb_test)))
unit_result('BASELINE_NOT_FUTURE',np.allclose(baseline_prediction(_test,'persistence'),_test.prior_value))

# Спорный id×год отбрасывается с содержательным карантином, не объединяется по имени.
_quarantine_n=len(QUARANTINE)
_bad_registry=pd.DataFrame([{'entity_id':'X','observation_year':2024,'name':'Вуз 1'},
                            {'entity_id':'X','observation_year':2024,'name':'Вуз 2'},
                            {'entity_id':'Y','observation_year':2024,'name':'Колледж'}])
_clean=quarantine_conflicting_registry(_bad_registry,'TEST_ONLY')
unit_result('AMBIGUOUS_REGISTRY_QUARANTINED',len(_clean)==1 and _clean.entity_id.iloc[0]=='Y')
del QUARANTINE[_quarantine_n:] # тестовые записи не являются реальными спорными организациями

qa_table('embedded_unit_tests',pd.DataFrame(SELF_TESTS),['test'])
assert all(r['result']=='PASS' for r in SELF_TESTS)
print('ML-05 embedded tests:',len(SELF_TESTS),'PASS (контрольные данные; не научные результаты)')
del _fixture,_panel,_split,_reason,_train,_test,_ridge_art,_cb_art,_en_art,_en_y,_po_art,_po_y,_nb_art,_nb_y,_nb_fixture,_nb_test,_y_ridge,_cb_y,_quarantine_n,_bad_registry,_clean

mark_stage('UNIT_TESTS')


## 9. Семь предметных блоков ML-5.1–ML-5.7
Научные блоки выполняются на фактически наблюдаемых источниках. При отсутствии target или подтверждённого grain предусмотрен описательный результат или мотивированный отказ от прогноза.

In [ ]:
memory_point('before_model_training')
progress_event('SEVEN_MODULES','Расчёт семи модулей; расширенные статусы',announce=True)
# Семь предметных экспериментов. Наличие модели не является условием выполнения описательной части.
MODULE_DIAGNOSTICS=[];CONFIRMATORY=[];HYPOTHESES=[];MODULE_SUBTASKS=[]

def blocked(module,why,required=''):
    limitation(module,why,required)
    MODULE_DIAGNOSTICS.append({'module':module,'prediction_feasible':False,
        'n_observed':0,'reason':why})

def two_way_mae_bootstrap(base,model,groups_a,groups_b,reps=399,seed=20261104):
    # Two-way pigeonhole bootstrap — строки получают произведение кратностей независимых блоков.
    a=pd.Series(groups_a).fillna('__UNKNOWN__').astype(str).to_numpy()
    b=pd.Series(groups_b).fillna('__UNKNOWN__').astype(str).to_numpy()
    unique_a=np.unique(a);unique_b=np.unique(b)
    if len(unique_a)<8 or len(unique_b)<8:return {'delta_MAE':np.nan,'ci_low':np.nan,'ci_high':np.nan,'p_value':np.nan,'clusters_a':len(unique_a),'clusters_b':len(unique_b)}
    v=np.asarray(model,dtype=float);u=np.asarray(base,dtype=float)
    d=v-u;delta=float(d.mean());rng=np.random.default_rng(seed)
    boot=[]
    for _ in range(reps):
        ca=Counter(rng.choice(unique_a,len(unique_a),replace=True))
        cb=Counter(rng.choice(unique_b,len(unique_b),replace=True))
        w=np.array([ca[x]*cb[y] for x,y in zip(a,b)],dtype=float)
        if w.sum():boot.append(float(np.average(d,weights=w)))
    if len(boot)<100:return {'delta_MAE':delta,'ci_low':np.nan,'ci_high':np.nan,'p_value':np.nan,'clusters_a':len(unique_a),'clusters_b':len(unique_b)}
    arr=np.asarray(boot)
    p=min(1.,2*min((np.sum(arr<=0)+1)/(len(arr)+1),(np.sum(arr>=0)+1)/(len(arr)+1)))
    return {'delta_MAE':delta,'ci_low':float(np.quantile(arr,.025)),
        'ci_high':float(np.quantile(arr,.975)),'p_value':float(p),
        'clusters_a':len(unique_a),'clusters_b':len(unique_b)}

def test_predictive(module,raw,nonnegative=True,count_confirmed=False):
    if raw.empty:return blocked(module,'Нет валидной целевой панели после семантического контроля',
       'entity_id, observed target years, known historical features')
    progress_event('MODELS',f'{module}: подготовка временного теста ({len(raw):,} строк)',announce=True)
    frame=raw.copy();frame['module']=module
    split,reason=temporal_partition(frame,min_train=CFG.REGION_MIN_TRAIN_ROWS if module=='ML-5.3' else None, min_test=CFG.REGION_MIN_TEST_ROWS if module=='ML-5.3' else None)
    if split is None:return blocked(module,reason,'train + independent test + as-of labels')
    train,dev,test=split['train'],split['dev'],split['test']
    cand=['persistence','rolling_mean_2','damped_trend','ridge','elasticnet','catboost','lightgbm']
    if count_confirmed:cand+=['ppml','negbin']
    if CFG.MODE=='SMOKE':cand=['persistence','rolling_mean_2','ridge']
    if CFG.MODE=='PILOT10' and len(train)>10000:
        train=train.sample(n=10000,random_state=CFG.RANDOM_SEED)
    primary='persistence';dev_scores={};dev_selection=False
    if dev is not None:
        # DEV labels действительно доступны к времени TEST-предсказания.
        for model in cand:
            try:
                if model in ('persistence','rolling_mean_2','damped_trend'):
                    p=baseline_prediction(dev,model,nonnegative)
                else:
                    p,_=pipeline_fit_predict(model,train,dev,seed=CFG.DEV_SEEDS[0],
                                             allow_ppml=count_confirmed,nonnegative=nonnegative)
                dev_scores[model]=effective_mae(dev.actual,p)['MAE']
            except Exception as ex:
                note('DEV',module+' '+model+': '+type(ex).__name__,'WARN')
        if dev_scores:
            priority={m:i for i,m in enumerate(cand)}
            primary=min(dev_scores,key=lambda m:(round(dev_scores[m],8),priority[m]))
            dev_selection=True
            # dev labels допустимо добавить в обучение только при as-of совпадении.
            train=pd.concat([train,dev],ignore_index=True)
    else:
        note('TEMPORAL',module+': DEV не опубликован к дате прогноза; модели предзаданы, без выбора по TEST','WARN')
    pred_by_model={}
    for model in cand:
        progress_event('MODELS',f'{module}: расчёт {model}',announce=True)
        seeds=[-1]
        if model=='catboost':
            if CFG.MODE=='SMOKE':seeds=[CFG.DEV_SEEDS[0]]
            elif CFG.MODE=='PILOT10' and dev_selection and primary=='catboost':seeds=list(CFG.DEV_SEEDS)
            elif CFG.MODE=='FULL':seeds=list(CFG.FINAL_SEEDS)[:CFG.ML54_FINAL_SEEDS] if module=='ML-5.4' else list(CFG.FINAL_SEEDS)  # 50 seeds for all FULL temporal tasks
            else:seeds=[CFG.FINAL_SEEDS[0]]
        preds=[]
        for i_seed,seed in enumerate(seeds,1):
            progress_event('MODELS',f'{module}: {model}, сид {i_seed}/{len(seeds)}',i_seed,len(seeds),announce=(i_seed==1 or i_seed%5==0 or i_seed==len(seeds)))
            try:
                p=fit_and_record(module,model,train,test,seed=seed,
                    nonnegative=nonnegative,allow_ppml=count_confirmed)
                preds.append(p)
            except Exception as ex:
                ISSUES.append({'module':module,'status':'MODEL_FAILED',
                     'reason':model+': '+type(ex).__name__+' — '+str(ex)[:180],
                     'required':'Inspect modeling frame; no fabricated substitute'})
                if CFG.MODE=='FULL' and model in ('negbin','ppml','ridge','elasticnet','lightgbm'):
                    note('MODEL_QA',module+' '+model+' failed, recorded as scientific FAIL','FAIL')
                if model==primary:raise
        if not preds:continue
        pred_by_model[model]=np.mean(preds,axis=0)
        if model=='catboost' and len(preds)>1:
            record_prediction(predictions_frame(test,module,'catboost_ensemble',pred_by_model[model],-1))
            METRICS.append({'module':module,'sector':str(test.sector.iloc[0]) if 'sector' in test else 'UNKNOWN','model':'catboost_ensemble','protocol':'ROLLING_TIME',
                 'seed':-1,'target_year':split['test_year'],
                 **effective_mae(test.actual,pred_by_model[model])})
    if 'persistence' not in pred_by_model:fail('MODEL','Нет сопоставимого исторического baseline')
    y=test.actual.to_numpy(float)
    base_err=np.abs(y-pred_by_model['persistence'])
    primary_err=np.abs(y-pred_by_model.get(primary,pred_by_model['persistence']))
    if module=='ML-5.1':
        ga=test.origin_region;gb=test.destination_region
    elif module=='ML-5.3':
        ga=test.origin_region;gb=test.target_year.astype(str)
    else:
        ga=test.parent_group_id.fillna(test.entity_id) if 'parent_group_id' in test else test.entity_id
        gb=test.target_year.astype(str)
    if dev_selection and primary!='persistence':
        ci=two_way_mae_bootstrap(base_err,primary_err,ga,gb,
               reps=99 if CFG.MODE=='SMOKE' else CFG.BOOTSTRAP_REPS,seed=CFG.RANDOM_SEED)
    else:
        ci={'delta_MAE':np.nan,'ci_low':np.nan,'ci_high':np.nan,'p_value':np.nan,
            'clusters_a':len(pd.Series(ga).dropna().unique()),
            'clusters_b':len(pd.Series(gb).dropna().unique())}
    chosen=effective_mae(y,pred_by_model.get(primary,pred_by_model['persistence']))
    original=effective_mae(y,pred_by_model['persistence'])
    CONFIRMATORY.append({'module':module,'sector':str(test.sector.iloc[0]) if 'sector' in test else 'UNKNOWN','test_year':split['test_year'],'test_n':len(test),
        'choice_basis':'DEV_ASOF' if dev_selection else 'NO_CONFIRMATORY_COMPARISON_NO_DEV',
        'selected_model':primary,'baseline_mae':original['MAE'],'selected_mae':chosen['MAE'],
        'delta_mae':chosen['MAE']-original['MAE'],**ci,'source_vintage_warning':True})
    progress_event('MODELS',f'{module}: тест завершён, инерция MAE={original["MAE"]:.3g}, выбранная={chosen["MAE"]:.3g}',announce=True)
    MODULES[module].update(status='TESTED_FULL' if CFG.MODE=='FULL' else 'TESTED_PILOT',
        reason=('Ретроспективный тест с неподтверждёнными историческими версиями источника; '
                'DEV-проверка доступности соблюдена; улучшение не гарантировано'),
        n=len(test),test_year=split['test_year'],selected=primary)
    MODULE_DIAGNOSTICS.append({'module':module,'prediction_feasible':True,
        'n_observed':len(raw),'n_test':len(test),'test_year':split['test_year'],
        'dev_available_asof':dev_selection,'selected_model':primary,
        'mae_baseline':original['MAE'],'mae_selected':chosen['MAE']})
    if not dev_selection:ISSUES.append({'module':module,'status':'SEVERE_TEMPORAL_LIMIT',
       'reason':'Нет DEV-меток, опубликованных до cutoff TEST; сравнения альтернатив только разведочные',
       'required':'older archived vintages or longer observations'})
    return split

# ML-5.1: истинная единица E01 пока не подтверждена как приём; регрессируем именно E01_observed_value.
if len(OD_INTER):
    MODULE_DIAGNOSTICS.append({'module':'ML-5.1','prediction_feasible':None,
        'n_observed':len(OD_INTER),'target_definition':OD_SEMANTIC['label'],
        'unknown_codes':len(OD_UNKNOWN),'diagonal_excluded':len(OD_INTRA),
        'coverage_breaks':int(coverage_df.potential_break.fillna(False).sum())})
    test_predictive('ML-5.1',OD_MODEL,nonnegative=True,count_confirmed=bool(OD_MODEL.actual.ge(0).all() and np.isclose(OD_MODEL.actual,np.rint(OD_MODEL.actual),atol=1e-8,rtol=0).all()))
    if coverage_df.potential_break.fillna(False).any():
        ISSUES.append({'module':'ML-5.1','status':'UNRESOLVED_YEAR_COVERAGE_BREAK',
          'reason':'Смена числа OD-пар в 2023 г. не доказана как изменение реальной мобильности; модельные результаты — условно разведочные',
          'required':'semantics/coverage harmonization across 2022–2023'})
else:blocked('ML-5.1','Нет истинно наблюдаемых межрегиональных E01-рёбер')

# ML-5.2: отдельно ВО/СПО; не моделировать учреждения, если не удалось установить уникальную величину.
if len(ORG_YEAR):
    for sector,group in ORG_MODEL.groupby('sector') if len(ORG_MODEL) else []:
        if len(group):
            result={'status':'NOT_STARTED','reason':'source not evaluated','n':0}
            try:
                test_predictive('ML-5.2',group,nonnegative=True,count_confirmed=True)
                result=MODULES['ML-5.2'].copy()
            except Exception as ex:
                result={'status':'FAILED','reason':f'{sector}: {type(ex).__name__}: {str(ex)[:100]}','n':0}
                raise
            finally:
                MODULE_SUBTASKS.append({'module':'ML-5.2','sector':sector,**result})
            MODULE_DIAGNOSTICS.append({'module':'ML-5.2','sector':sector,'n_organizations':group.entity_id.nunique(),
                    'n_observed':len(group)})
    if MODULES['ML-5.2']['status']=='NOT_STARTED':
        limitation('ML-5.2','Установлены потенциальные индикаторы, но недостаточно исторических пар для модели',
                   'admissions×organization×year with unambiguous units')
else:
    limitation('ML-5.2','Предметный каталог E02/VPO/SPO прочитан, но однозначное зачисление организация×год не подтверждено',
               'source-admission indicator and organization with unique key')

if MODULE_SUBTASKS:
    qa_table('organization_sector_subtasks',pd.DataFrame(MODULE_SUBTASKS),allow_empty=True)
    statuses=[r['status'] for r in MODULE_SUBTASKS]
    if any(x.startswith('TESTED') for x in statuses):
        MODULES['ML-5.2']['status']='TESTED_FULL' if CFG.MODE=='FULL' else 'TESTED_PILOT'
        MODULES['ML-5.2']['reason']='Отдельные сектора: '+', '.join(f"{r['sector']}={r['status']}" for r in MODULE_SUBTASKS)

# ML-5.3: только межрегиональные маршруты из региона получения общего образования.
if len(OUTFLOW_MODEL):test_predictive('ML-5.3',OUTFLOW_MODEL,nonnegative=True,count_confirmed=bool(OUTFLOW_MODEL.actual.ge(0).all() and np.isclose(OUTFLOW_MODEL.actual,np.rint(OUTFLOW_MODEL.actual),atol=1e-8,rtol=0).all()))
else:blocked('ML-5.3','Недостаточно наблюдений агрегированного межрегионального E01',
             'origin schooling region, destination study region, comparable years')

# ML-5.4: реальная программа, не фиктивное происхождение по специальности.
if len(PROGRAM_PROFILE):
    pp=PROGRAM_PROFILE.copy()
    pp['denominator']=pp.groupby(['observation_year','indicator_name']).observed_value.transform('sum')
    pp['relative_share_qualifier']='CONDITIONAL_ON_EXACT_INDICATOR_AND_OBSERVED_PROGRAM_RECORDS'
    pp['share_observed']=np.where(pp.denominator>0,pp.observed_value/pp.denominator,np.nan)
    pp['herfindahl_component']=pp.share_observed**2
    concentration=pp.groupby(['observation_year','indicator_name'],as_index=False).agg(
        reported_programs=('specialty_code','nunique'),reported_value=('observed_value','sum'),
        HHI=('herfindahl_component','sum'))
    qa_table('program_concentration',concentration,allow_empty=True)
    qa_table('program_distribution',pp,allow_empty=True)
    MODULES['ML-5.4'].update(status='DESCRIPTIVE_VALID',reason='Программы — условный срез конкретных индикаторов и кодов; OD×programme отсутствует',n=len(pp))
    MODULE_DIAGNOSTICS.append({'module':'ML-5.4','prediction_feasible':False,
       'n_observed':len(pp),'reason':'Нельзя прогнозировать выбор программы по OD без OD×programme'})
elif len(VPO_PROGRAM_DESCRIPTION):
    MODULES['ML-5.4'].update(status='DESCRIPTIVE_VALID',
        reason='Реальные направления ВПО-1 по категориям и статистическим уровням, без двойного суммирования и ненаблюдаемого OD×programme',
        n=len(VPO_PROGRAM_DESCRIPTION))
    MODULE_DIAGNOSTICS.append({'module':'ML-5.4','prediction_feasible':False,
      'n_observed':len(VPO_PROGRAM_DESCRIPTION),'reason':'Представлены медианы исходных разрезов; точные когорты программы не установлены'})
else:limitation('ML-5.4','Доступные словари программ не дали согласованного временного показателя',
                'stable specialty codes, verified target, level and year')

# ML-5.5: полная E04/E05 обрабатывается потоково в DuckDB; не сравнивать preview со всей базой.
if CFG.MODE=='FULL':
    if GRAD_E04_VALID_ROWS>0 and not GRADUATE_GROUP_SUMMARY.empty:
        MODULES['ML-5.5'].update(status='DESCRIPTIVE_VALID',
            reason='Полные E04/E05: статистические группировки всех наблюдений в DuckDB; employment_asof 2024-12-31; RETENTION_NOT_OBSERVABLE',
            n=GRAD_E04_VALID_ROWS)
        MODULE_DIAGNOSTICS.append({'module':'ML-5.5','prediction_feasible':False,
            'n_observed_e04':GRAD_E04_VALID_ROWS,'n_observed_e05':GRAD_E05_VALID_ROWS,
            'employment_asof':'2024-12-31','reason':'Срез на одной референсной дате; место работы не подтверждено'})
    else:
        limitation('ML-5.5','Агрегат E04 по всем строкам пуст после валидации показателя')
elif len(GRADUATES):
    gr=GRADUATES.dropna(subset=['count_graduate']).copy()
    gr=gr[gr.count_graduate.ge(0)]
    gr_summary=gr.groupby(['graduation_year','object_level','gender','education_level'],dropna=False).agg(
       n_source_records=('university','size'),median_reported_graduates=('count_graduate','median'),
       median_observed_salary=('average_salary_fact_med','median'),
       median_employment_percentage=('percent_employed','median')).reset_index()
    gr_summary['aggregate_totals_identified']=False
    qa_table('graduate_descriptive_statistics',gr_summary,allow_empty=True)
    if 'graduate_specialty_outcomes' in PANELS:
        f=PANELS['graduate_specialty_outcomes'].dropna(subset=['graduation_year','specialty_code'])
        f=f[f.count_graduate.notna() & f.count_graduate.ge(0)]
        source_groups=f.groupby(['graduation_year','object_level','education_level','specialty_code'],dropna=False).agg(
          observed_records=('specialty','size'),median_reported_graduates=('count_graduate','median'),
          median_wage=('average_salary_fact_med','median')).reset_index()
        source_groups['aggregate_totals_identified']=False
        qa_table('graduate_specialty_description',source_groups,allow_empty=True)
    MODULES['ML-5.5'].update(status='DESCRIPTIVE_VALID',
      reason='SMOKE/PILOT: описательная статистика по ограниченным данным; retention неизвестен',n=len(gr))
    MODULE_DIAGNOSTICS.append({'module':'ML-5.5','prediction_feasible':False,'n_observed':len(gr),
      'employment_asof':'2024-12-31','reason':'Одна дата занятости; нет географии работы'})
else:limitation('ML-5.5','E04/E05: нет валидной описательной статистики')

# ML-5.6: динамическая сеть, петли исключены; сетевые изменения с учетом неодинакового покрытия.
NETWORK=[];NETWORK_EDGES=[];COMMUNITIES=[];NET_SETS={}
for year,sub in OD_INTER.dropna(subset=['observed_value']).groupby('observation_year'):
    progress_event('NETWORK',f'Год {year}: расчёт наблюдаемой сети',announce=True)
    g=nx.DiGraph()
    for r in sub.itertuples(index=False):
        if pd.notna(r.observed_value) and r.observed_value>0:
            g.add_edge(str(r.origin_region),str(r.destination_region),weight=float(r.observed_value))
    if nx.number_of_selfloops(g):fail('NETWORK','Обнаружена диагональ в межрегиональном графе')
    if not g.number_of_edges():continue
    pager=nx.pagerank(g,weight='weight')
    undirected=g.to_undirected()
    try:comms=nx.community.louvain_communities(undirected,weight='weight',seed=CFG.RANDOM_SEED)
    except Exception:comms=list(nx.community.greedy_modularity_communities(undirected,weight='weight'))
    membership={node:i for i,community in enumerate(comms) for node in community}
    NET_SETS[int(year)]=set((a,b) for a,b in g.edges())
    for node in g.nodes():
        incoming=g.in_degree(node,weight='weight');outgoing=g.out_degree(node,weight='weight')
        vals=np.asarray([float(d['weight']) for _,_,d in g.out_edges(node,data=True)],float)
        sh=vals/vals.sum() if vals.sum()>0 else np.asarray([],float)
        out_hhi=float((sh*sh).sum()) if len(sh) else np.nan
        out_entropy=float(-(sh*np.log(sh)).sum()) if len(sh) else np.nan
        NETWORK.append({'year':int(year),'region_code':node,'in_strength':incoming,'out_strength':outgoing,
          'outbound_HHI_observed':out_hhi,'outbound_entropy_observed':out_entropy,
          'pagerank':pager.get(node,np.nan),'community_id':membership.get(node,-1),
          'n_in':g.in_degree(node),'n_out':g.out_degree(node),'observed_edges':g.number_of_edges()})
    for a,b,d in g.edges(data=True):NETWORK_EDGES.append({'year':int(year),'origin':a,'destination':b,
        'value':d['weight']})
    COMMUNITIES.append({'year':int(year),'communities':len(comms),
        'n_regions':len(g.nodes()),'n_edges':g.number_of_edges(),
        'coverage_break_warning':bool(coverage_df.loc[coverage_df.year==int(year),'potential_break'].fillna(False).any())})
qa_table('education_network_year',pd.DataFrame(NETWORK),allow_empty=True)
qa_table('education_network_edges',pd.DataFrame(NETWORK_EDGES),allow_empty=True)
qa_table('education_network_communities',pd.DataFrame(COMMUNITIES),allow_empty=True)
if len(NET_SETS)>=2:
    stability=[]
    for a,b in zip(sorted(NET_SETS)[:-1],sorted(NET_SETS)[1:]):
        x,y=NET_SETS[a],NET_SETS[b]
        stability.append({'year_from':a,'year_to':b,'edge_jaccard':len(x&y)/len(x|y) if x|y else np.nan,
                          'n_edges_from':len(x),'n_edges_to':len(y),
                          'coverage_break_warning':bool(coverage_df.loc[coverage_df.year==b,'potential_break'].fillna(False).any())})
    qa_table('network_stability',pd.DataFrame(stability),allow_empty=True)
if len(NETWORK):
    MODULES['ML-5.6'].update(status='DESCRIPTIVE_VALID',reason='Сетевые показатели наблюдаемой E01 без самопетель; покрытие лет различается',n=len(NETWORK))
else:limitation('ML-5.6','Нет валидных взвешенных межрегиональных ребер E01')

# ML-5.7: одинаковые муниципалитеты для E0/E1; пропущенный факт нахождения кампуса нельзя считать нулём.
context=MUNI_EDU[MUNI_EDU.education_coverage_known].copy()
if (context.oktmo.nunique()>=CFG.MIN_MUNICIPALITIES_E0E1 and
        context.observation_year.nunique()>=CFG.MIN_YEARS_FOR_TEMPORAL and
        context.confirmed_vo_sites.nunique()>1):
    x=context[['oktmo','observation_year','migration_rate','population','confirmed_vo_sites']].rename(
      columns={'oktmo':'entity_id','migration_rate':'actual','population':'prev_population'})
    # Не подтягивать число университетов из целевого года: lag, as-of.
    m0=historical_pair_panel(x[['entity_id','observation_year','actual']],['entity_id'],'actual')
    lag=context.rename(columns={'oktmo':'entity_id','observation_year':'historical_year',
                                'confirmed_vo_sites':'confirmed_vo_sites',
                                'population':'prev_population'})
    m0=m0.merge(lag[['entity_id','historical_year','confirmed_vo_sites','prev_population']],
        on=['entity_id','historical_year'],validate='many_to_one',how='left')
    m0['target_id']='municipal_migration_rate';m0['sector']='MUNICIPAL_CONTEXT'
    qa_table('municipal_e0e1_model_frame',m0,allow_empty=True)
    # Прогнозная процедура E0/E1 — одна и та же модель и поддержка; отключаем инфра в E0.
    split,reason=temporal_partition(m0)
    if split:
        tr=split['train'];te=split['test'];features0=['prior_value','prev_population','origin_year']
        features1=features0+['confirmed_vo_sites']
        for version,features in [('E0',features0),('E1',features1)]:
            model=Pipeline([('imputer',SimpleImputer(strategy='median')),
                            ('scaler',StandardScaler()),('reg',Ridge(alpha=20.))])
            model.fit(tr[features],tr.actual)
            predictions=model.predict(te[features])
            record_prediction(predictions_frame(te,'ML-5.7',version,predictions))
            METRICS.append({'module':'ML-5.7','sector':'MUNICIPAL_CONTEXT','model':version,'protocol':'COMMON_SUPPORT_MUNICIPAL',
              'seed':-1,'target_year':split['test_year'],**effective_mae(te.actual,predictions)})
            fp=ROOT/'models'/f'ML_5_7_{version}.joblib';joblib.dump({'model':model,'features':features},fp)
            reload=joblib.load(fp)
            assert np.allclose(reload['model'].predict(te[features]),predictions)
            MODEL_FILES[fp.name]=str(fp)
        MODULES['ML-5.7'].update(status='TESTED_FULL' if CFG.MODE=='FULL' else 'TESTED_PILOT',
            reason='E0/E1 на одинаковом подтверждённом составе ОКТМО',n=len(te))
    else:limitation('ML-5.7',reason,'more confirmed municipality-years')
else:
    limitation('ML-5.7',f'Недостаточно подтвержденного кампусного покрытия: {context.oktmo.nunique()} ОКТМО; неизвестно != 0',
               'verified municipality campus registry with historical validity')

# Все семь модулей получают явный предметный статус; ни один не теряется.
for key,result in MODULES.items():
    if result['status']=='NOT_STARTED':
        limitation(key,'Данные просканированы, но применимый протокол не смог сформировать результат')
qa_table('seven_module_status',pd.DataFrame([{'module':k,**v} for k,v in MODULES.items()]),['module'])
qa_table('module_diagnostics',pd.DataFrame(MODULE_DIAGNOSTICS),allow_empty=True)
print('MODULES:',{k:v['status'] for k,v in MODULES.items()})

memory_point('after_seven_modules')
mark_stage('SEVEN_MODULES')


## 9А. Дополнительные исследования ML-5.2–ML-5.7

Строго отделяются проверенный приём и неутверждённые кандидаты, описательная структура программ и индивидуальный выбор, занятость выпускников и территориальное закрепление. Региональные и муниципальные экспериментальные протоколы запускаются только на проверенной общей области наблюдения. Сетевые показатели проверяются на устойчивость. Отрицательные результаты фиксируются без замены тестовой модели задним числом.

In [ ]:
# ML-05: substantive completion of all seven modules with explicit evidence gates.
progress_event('EXTENDED_MODULES','Дополнительные исследования ML-5.2–ML-5.7 и сетевой стресс-тест',announce=True)
EXT_MODULES=[]

# ML-5.2: distinguish candidates, verified admission, and non-additive section splits.
if 'CANDIDATE_CATALOG' in globals() and len(CANDIDATE_CATALOG):
    cand=CANDIDATE_CATALOG.loc[CANDIDATE_CATALOG.semantic_role.eq('ADMISSION_CANDIDATE')].copy()
    sector_rows=cand.groupby('source',dropna=False).agg(indicator_names=('indicator_name','nunique'),
                         rows_in_source=('rows','sum')).reset_index() if len(cand) else pd.DataFrame()
    if len(sector_rows):
        sector_rows['approved_entity_year_targets']=len(ORG_YEAR)
        sector_rows['interpretation']='SOURCE_CANDIDATES_ONLY_NOT_VERIFIED_ADMISSIONS'
    qa_table('ml52_admission_candidates',sector_rows,allow_empty=True)
    if len(sector_rows) and not len(ORG_YEAR):
        MODULES['ML-5.2'].update(status='DESCRIPTIVE_ONLY_TARGET_UNAPPROVED',n=int(sector_rows.rows_in_source.sum()),
          reason='Семантические кандидаты действительно найдены, но утверждённой численности приёма организация×год нет')
        EXT_MODULES.append({'module':'ML-5.2','evidence':'SEMANTIC_CANDIDATES_NOT_COUNTS',
          'n_candidates':int(sector_rows.indicator_names.sum()),'prediction_status':'NOT_FEASIBLE'})

# ML-5.4: whole-dataset specialty histories, not a preview of the first 2,000 aggregate rows.
ML54_HISTORY=pd.DataFrame()
if VPO_FULL_PROGRAM_PATH and Path(VPO_FULL_PROGRAM_PATH).exists():
    p=Path(VPO_FULL_PROGRAM_PATH)
    q='''SELECT observation_year,CAST(specialty_code AS VARCHAR) specialty_code,
      CAST(indicator_name AS VARCHAR) indicator_name,
      SUM(observed_records)::BIGINT rows_in_source,
      COUNT(*)::BIGINT independently_reported_strata,
      MEDIAN(median_reported_value) median_of_source_stratum_medians
      FROM read_parquet(?)
      WHERE specialty_code IS NOT NULL AND TRY_CAST(observation_year AS INT) IS NOT NULL
      GROUP BY 1,2,3'''
    with duck_connect() as con:
        n=int(con.execute('SELECT COUNT(*) FROM ('+q+')',[str(p)]).fetchone()[0])
        if n<=CFG.MAX_PANDAS_ROWS:
            ML54_HISTORY=con.execute(q,[str(p)]).df()
        else:
            out=ROOT/'derived'/'ml54_full_history.parquet'
            con.execute('COPY ('+q.replace('read_parquet(?)','read_parquet('+_sql_literal(p)+')')+
              ') TO '+_sql_literal(out)+' (FORMAT PARQUET,COMPRESSION ZSTD)')
            PANEL_FILES['ml54_specialty_history']=out
            ML54_HISTORY=con.execute('SELECT * FROM read_parquet(?) LIMIT 2000',[str(out)]).df()
    qa_table('ml54_specialty_history',ML54_HISTORY,allow_empty=True)
    if n:
        MODULES['ML-5.4'].update(status='DESCRIPTIVE_FULL_SOURCE_NOT_OD_PROGRAMME',n=n,
          reason='Всё содержимое VPO-1 сгруппировано по годам/кодам/показателям; составное медианное значение не равно численности зачисленных')
    EXT_MODULES.append({'module':'ML-5.4','full_specialty_year_indicator_groups':n,
          'forecast_valid':'NO_VERIFIED_ADMISSIONS_TARGET','unit':'source_records_and_medians_not_additive'})

# Exploratory specialty continuity forecast for stable time series (not a verified admission forecast).
# Predictions are directly row counts of source records in a published statistical strata grid.
ML54_TEST=pd.DataFrame()
if len(ML54_HISTORY) and 'rows_in_source' in ML54_HISTORY and 'observation_year' in ML54_HISTORY:
    z=ML54_HISTORY.copy();z=z.dropna(subset=['observation_year','specialty_code','indicator_name'])
    z['observation_year']=pd.to_numeric(z.observation_year,errors='coerce')
    z=z.sort_values(['specialty_code','indicator_name','observation_year'])
    g=z.groupby(['specialty_code','indicator_name'],sort=False)
    z['lag1_rows']=g.rows_in_source.shift(1)
    z['lag1_year']=g.observation_year.shift(1)
    last=int(z.observation_year.max()) if len(z) else 0
    ML54_TEST=z[(z.observation_year.eq(last))&(z.lag1_year.eq(last-1))&z.lag1_rows.notna()].copy()
    if len(ML54_TEST):
        ML54_TEST['abs_error_persistence']=(ML54_TEST.rows_in_source-ML54_TEST.lag1_rows).abs()
        ML54_TEST['measure_definition']='VPO_REPORTING_STRATA_COVERAGE_ONLY'
        qa_table('ml54_history_persistence_diagnostic',ML54_TEST,allow_empty=True)
        EXT_MODULES.append({'module':'ML-5.4','coverage_persistence_year':last,
          'test_cells':len(ML54_TEST),'MAE_source_row_coverage':float(ML54_TEST.abs_error_persistence.mean()),
          'scientific_gate':'NOT_ADMISSIONS_OR_STUDENT_FLOW'})

# ML-5.5: all 4 E04/E05 sources, male/female strata kept separate, no artificial employment location.
GRAD_DISPARITIES=pd.DataFrame()
family_frames=[]
for i in range(1,5):
    name='graduate_family_'+str(i)
    path=PANEL_FILES.get(name)
    if path and Path(path).exists():
        with duck_connect() as con:
            f=con.execute('SELECT * FROM read_parquet(?) LIMIT 30000',[str(path)]).df()
        f['asset_id']=source_asset(segment='05')[i-1]['asset_id']
        family_frames.append(f)
if family_frames:
    GRAD_DISPARITIES=pd.concat(family_frames,ignore_index=True,sort=False)
    GRAD_DISPARITIES['employment_reference']='2024-12-31'
    GRAD_DISPARITIES['outcome_grain']='MEDIAN_OF_PUBLISHED_AGGREGATE_RECORDS'
    qa_table('ml55_outcome_disparities',GRAD_DISPARITIES,allow_empty=True)
    EXT_MODULES.append({'module':'ML-5.5','source_families_analysed':len(family_frames),
        'output_strata':len(GRAD_DISPARITIES),'time_reference':'2024-12-31',
        'retention':'NOT_OBSERVABLE_WITHOUT_WORK_LOCATION'})
    MODULES['ML-5.5'].update(status='DESCRIPTIVE_ALL_FOUR_FAMILIES',
       reason='E04/E05 всех четырёх типов изучены по исходным разрезам, без двойного суммирования; закрепление неизвестно',n=len(GRAD_DISPARITIES))

# ML-5.6: reciprocity, removal robustness, SCC/WCC and stable network coverage.
ML56_STRESS=[]
for yr,sub in pd.DataFrame(NETWORK_EDGES).groupby('year') if len(NETWORK_EDGES) else []:
    g=nx.DiGraph()
    for r in sub.itertuples(index=False):
        if r.value>0 and r.origin!=r.destination:g.add_edge(str(r.origin),str(r.destination),weight=float(r.value))
    if not g.number_of_nodes():continue
    pg=nx.pagerank(g,weight='weight');leading=max(pg,key=pg.get)
    reciprocal=sum(1 for u,v in g.edges() if g.has_edge(v,u))
    w0=len(max(nx.weakly_connected_components(g),key=len))
    s0=len(max(nx.strongly_connected_components(g),key=len))
    h=g.copy();h.remove_node(leading)
    w1=len(max(nx.weakly_connected_components(h),key=len)) if h.number_of_nodes() else 0
    edges_before=g.number_of_edges();edges_after=h.number_of_edges()
    ML56_STRESS.append({'year':int(yr),'n_nodes':g.number_of_nodes(),'n_edges':edges_before,
       'reciprocity_directed_edge_share':reciprocal/edges_before if edges_before else np.nan,
       'largest_weak_component':w0,'largest_strong_component':s0,'removed_top_pagerank_region':leading,
       'weak_component_after_removal':w1,'edges_after_removal':edges_after,
       'lost_edges_percent':100*(edges_before-edges_after)/edges_before if edges_before else np.nan,
       'interpretation':'STRUCTURAL_STRESS_NOT_EMPIRICAL_MIGRATION_DISRUPTION'})
qa_table('ml56_network_stress',pd.DataFrame(ML56_STRESS),allow_empty=True)
if len(ML56_STRESS):
    MODULES['ML-5.6'].update(status='DESCRIPTIVE_FULL_NETWORK_STRESS',n=len(NETWORK),
      reason='Динамические сети, PageRank, Louvain, концентрация, reciprocity и устойчивость удаления центрального узла; сопоставимость лет ограничена')
    EXT_MODULES.append({'module':'ML-5.6','n_years':len(ML56_STRESS),'validation':'OBSERVED_EDGES_ONLY'})

# ML-5.7: BDMO education-infrastructure proxy E0/E1, same OKTMO-year rows, no fabricated university locations.
ML57_SUPPORT=pd.DataFrame();ML57_RESULT={}
if BDMO_PROXY_FULL_VALID and len(BDMO_EDU_PROXY_CANDIDATES) and len(MUNI):
    e=BDMO_EDU_PROXY_CANDIDATES.dropna(subset=['oktmo','observation_year','indicator_code','median_reported']).copy()
    e['oktmo']=norm_code(e.oktmo);e['observation_year']=safe_year(e.observation_year)
    e=e[e.n_raw_rows.eq(1)&e.median_reported.ge(0)].copy()
    if len(e):
        mig=MUNI[['oktmo','observation_year','migration_rate','population']].drop_duplicates(['oktmo','observation_year']).copy()
        mig['observation_year']=safe_year(mig.observation_year)
        # only same historically observed municipality, lookback=one year at origin.
        mig=mig.sort_values(['oktmo','observation_year'])
        mig['prev_year']=mig.observation_year-1
        past=mig[['oktmo','observation_year','migration_rate','population']].rename(
             columns={'observation_year':'prev_year','migration_rate':'lag_migration_rate','population':'lag_population'})
        mig=mig.merge(past,on=['oktmo','prev_year'],how='inner',validate='many_to_one')
        for c in ('migration_rate','lag_migration_rate','lag_population'):mig[c]=number(mig[c])
        last=int(mig.observation_year.max()) if len(mig) else None
        # Candidate selection entirely on PRE-TEST historical years; never on test labels.
        prior=e[e.observation_year.lt(last)] if last is not None else e.iloc[:0]
        stats=prior.groupby('indicator_code',dropna=False).agg(n_muni=('oktmo','nunique'),
           n_years=('observation_year','nunique'),n_rows=('median_reported','size'),
           n_distinct=('median_reported','nunique')).reset_index()
        stats=stats[(stats.n_muni>=CFG.MIN_MUNICIPALITIES_E0E1)&
                    (stats.n_years>=CFG.MIN_YEARS_FOR_TEMPORAL)&(stats.n_distinct>1)]
        if len(stats):
            stats=stats.sort_values(['n_muni','n_years','n_rows','indicator_code'],ascending=[False,False,False,True])
            selected=str(stats.iloc[0].indicator_code)
            x=e[e.indicator_code.astype(str).eq(selected)][['oktmo','observation_year','median_reported']].copy()
            x=x.rename(columns={'observation_year':'prev_year','median_reported':'lag_education_proxy'})
            # municipality's published education indicator is used at t-1 only, not t.
            ML57_SUPPORT=mig.merge(x,on=['oktmo','prev_year'],how='inner',validate='many_to_one')
            ML57_SUPPORT=ML57_SUPPORT.dropna(subset=['migration_rate','lag_migration_rate',
                               'lag_population','lag_education_proxy'])
            ML57_SUPPORT['indicator_code']=selected
            ML57_SUPPORT['source_definition']='BDMO_PUBLISHED_EDUCATION_PROXY_NOT_VERIFIED_CAMPUS'
            qa_table('ml57_common_support',ML57_SUPPORT,allow_empty=True)
            tr=ML57_SUPPORT[ML57_SUPPORT.observation_year.lt(last)].copy()
            te=ML57_SUPPORT[ML57_SUPPORT.observation_year.eq(last)].copy()
            if (len(tr)>=max(50,CFG.REGION_MIN_TRAIN_ROWS) and
                len(te)>=CFG.MIN_MUNICIPALITIES_E0E1 and tr.oktmo.nunique()>=CFG.MIN_MUNICIPALITIES_E0E1):
                # Identical E0/E1 keys and model, change ONLY historic education covariate.
                estimates={};baselines=te.lag_migration_rate.to_numpy(float)
                baseline_score=effective_mae(te.migration_rate,baselines)
                for vers,features in [('E0',['lag_migration_rate','lag_population']),
                                      ('E1',['lag_migration_rate','lag_population','lag_education_proxy'])]:
                    model=Pipeline([('imputer',SimpleImputer(strategy='median')),
                                   ('scale',StandardScaler()),('ridge',Ridge(alpha=20.0))])
                    model.fit(tr[features].astype(float),tr.migration_rate.to_numpy(float))
                    pred=model.predict(te[features].astype(float));estimates[vers]=effective_mae(te.migration_rate,pred)
                    fp=ROOT/'models'/('ML_5_7_proxy_'+vers+'.joblib')
                    artifact={'pipeline':model,'features':features,'proxy_indicator_code':selected}
                    joblib.dump(artifact,fp,compress=3);rp=joblib.load(fp)['pipeline'].predict(te[features].astype(float))
                    if not np.allclose(rp,pred):fail('ML-5.7','Model serialization discrepancy')
                    MODEL_FILES[fp.name]=str(fp)
                    MODEL_RECORDS.append({'module':'ML-5.7','sector':'MUNICIPAL_BDMO_PROXY','model':vers,
                      'seed':-1,'path':fp.name,'reloaded':True,'training_rows':len(tr),'test_rows':len(te),
                      'feature_num':features,'feature_cat':[],'file_sha256':hash_file(fp)})
                    out=te[['oktmo','observation_year','migration_rate','prev_year']].copy()
                    out=out.rename(columns={'oktmo':'entity_id','observation_year':'target_year',
                         'migration_rate':'actual','prev_year':'origin_year'})
                    out['information_cutoff_year']=out.origin_year
                    out['feature_available_year']=out.origin_year
                    out['target_available_year']=out.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS
                    out['target_id']='D01_MUNICIPAL_MIGRATION_RATE'
                    out['sector']='MUNICIPAL_BDMO_PROXY'
                    pf=predictions_frame(out,'ML-5.7',vers,pred,seed=-1,protocol='RETROSPECTIVE_COMMON_SUPPORT')
                    pf['geography_grain']='MUNICIPALITY';record_prediction(pf)
                    METRICS.append({'module':'ML-5.7','sector':'MUNICIPAL_BDMO_PROXY','model':vers,
                       'protocol':'RETROSPECTIVE_COMMON_SUPPORT','seed':-1,'target_year':last,
                       'source_vintage':INDEX['created_at'],**estimates[vers]})
                ML57_RESULT={'status':'RETROSPECTIVE_PROXY_E0E1','year':last,'n_train':len(tr),
                  'n_test':len(te),'n_muni_test':te.oktmo.nunique(),'indicator_code':selected,
                  'inertia_MAE':baseline_score['MAE'],'e0_MAE':estimates['E0']['MAE'],
                  'e1_MAE':estimates['E1']['MAE'],'delta_MAE_E1_minus_E0':estimates['E1']['MAE']-estimates['E0']['MAE']}
                MODULES['ML-5.7'].update(status='RETROSPECTIVE_MUNICIPAL_PROXY_TEST',n=len(te),
                    reason='E0/E1 на общих ОКТМО с опубликованным BDMO-показателем: не доказана фактическая образовательная площадка; причинные выводы запрещены')
if not ML57_RESULT:
    ML57_RESULT={'status':'NOT_FEASIBLE_WITH_VERIFIED_GEOGRAPHY',
      'reason':'Недостаточная муниципальная область наблюдения или отсутствующий сопоставимый показатель инфраструктуры'}
EXT_MODULES.append({'module':'ML-5.7',**ML57_RESULT})
# ML-5.1 generalization checks: both time and territory blocked, no test-based hyperparameter selection.
SPATIAL_MODEL_RESULTS=[]
if len(OD_MODEL):
    frame=OD_MODEL.copy()
    frame['module']='ML-5.1'
    years=sorted(int(y) for y in frame.target_year.dropna().unique())
    if len(years)>=2:
        ty=years[-1];origin=ty-CFG.HORIZON
        origins=sorted(frame.origin_region.dropna().astype(str).unique())
        destinations=sorted(frame.destination_region.dropna().astype(str).unique())
        def stable_holdout(code,salt):
            return int(hashlib.sha256((salt+'|'+str(code)).encode()).hexdigest()[:8],16)%5==0
        oo={r for r in origins if stable_holdout(r,'ML5_ORIGIN_HOLDOUT')}
        dd={r for r in destinations if stable_holdout(r,'ML5_DEST_HOLDOUT')}
        protocols=[('TERRITORY_ONLY',frame.origin_region.astype(str).isin(oo),
                    ~frame.origin_region.astype(str).isin(oo)),
                   ('CROSSED',frame.origin_region.astype(str).isin(oo)&frame.destination_region.astype(str).isin(dd),
                    ~frame.origin_region.astype(str).isin(oo)&~frame.destination_region.astype(str).isin(dd))]
        for protocol,is_test,is_train in protocols:
            tr=frame[frame.target_year.lt(ty)&frame.target_available_year.le(origin)&is_train].copy()
            te=frame[frame.target_year.eq(ty)&is_test].copy()
            meta={'module':'ML-5.1','protocol':protocol,'train_rows':len(tr),'test_rows':len(te),
                  'n_origins':te.origin_region.nunique(),'n_destinations':te.destination_region.nunique(),
                  'date_origin':origin,'test_year':ty,'selection_basis':'PREDECLARED_GEOGRAPHIC_HASH'}
            if len(tr)<CFG.MIN_TRAIN_ROWS or len(te)<CFG.MIN_TEST_ROWS:
                meta['status']='NOT_FEASIBLE_SUPPORT';SPATIAL_MODEL_RESULTS.append(meta);continue
            # Persist the baseline on precisely the same test keys as each spatial model.
            pbase=fit_and_record('ML-5.1','persistence',tr,te,seed=-1,
                                 nonnegative=True,protocol=protocol)
            reference=effective_mae(te.actual,pbase)
            meta['baseline_mae']=reference['MAE'];meta['status']='TESTED_RETROSPECTIVE_EXPLORATORY'
            # Single fixed seed; no TEST choice; same held-out keys for all models.
            for model in ('ridge','elasticnet','lightgbm'):
                try:
                    yp=fit_and_record('ML-5.1',model,tr,te,seed=CFG.RANDOM_SEED,
                                     nonnegative=True,allow_ppml=False,protocol=protocol)
                    mt=effective_mae(te.actual,yp)
                    meta[model+'_MAE']=mt['MAE'];meta[model+'_minus_baseline']=mt['MAE']-reference['MAE']
                except (ValueError,RuntimeError) as ex:
                    ISSUES.append({'module':'ML-5.1','status':'SPATIAL_MODEL_FAILURE',
                        'reason':protocol+' '+model+': '+type(ex).__name__,'required':'inspect spatial support'})
            SPATIAL_MODEL_RESULTS.append(meta)
qa_table('ml51_spatial_protocols',pd.DataFrame(SPATIAL_MODEL_RESULTS),allow_empty=True)
if len(SPATIAL_MODEL_RESULTS):
    EXT_MODULES.append({'module':'ML-5.1','protocols':len(SPATIAL_MODEL_RESULTS),
      'tested':sum(x['status'].startswith('TESTED') for x in SPATIAL_MODEL_RESULTS),
      'interpretation':'SPATIAL_GENERALIZATION_RETROSPECTIVE_CAVEATS'})

qa_table('module_52_extension',pd.DataFrame(EXT_MODULES),allow_empty=True)
qa_table('seven_module_status_final',pd.DataFrame([{'module':k,**v} for k,v in MODULES.items()]),['module'])
print('Extended modules:',[(k,v['status']) for k,v in MODULES.items()])
# Финальная отметка этапа перенесена после всех его подмодулей


## 9Б. Региональный приём по специальностям и вузовский контингент
Подтверждённые количества по ВПО-1, отличные от отсутствующего приёма организации.

In [ ]:
# V4: measured organization outcomes and complete regional specialty admissions, with separate scientific targets.
progress_event('EXTENDED_MODULES','V4: реальный приём по специальностям ВПО-1; организация — отдельная величина',announce=True)
V4_SUBEXPERIMENTS=[]

def v4_block(name,reason,required='',rows=0):
    V4_SUBEXPERIMENTS.append({'subexperiment':name,'status':'NOT_FEASIBLE_FOR_TARGET',
       'reason':reason,'required':required,'n':int(rows)})
    note('V4_DATA_GATE',name+': '+reason,'WARN')

# Source VPO_2_1_1 has exact textual 'Принято, всего'. This is NOT a campus/branch observation.
_v4_vpo_p=fetch_asset(VPO_211)
V4_ADMISSION_PATH=ROOT/'derived'/'v4_regional_specialty_admissions.parquet'
_v4_s=f'''SELECT TRY_CAST(year AS INTEGER) observation_year, CAST(object_level AS VARCHAR) object_level,
    CAST(object_name AS VARCHAR) object_name, CAST(specialty_code AS VARCHAR) specialty_code,
    CAST(education_level AS VARCHAR) education_level,CAST(education_form AS VARCHAR) education_form,
    CAST(admission_type AS VARCHAR) admission_type,CAST(university_type AS VARCHAR) university_type,
    COUNT(*)::BIGINT records_in_cell,MIN(TRY_CAST(indicator_value AS DOUBLE)) minimum,
    MAX(TRY_CAST(indicator_value AS DOUBLE)) maximum
  FROM read_parquet({_sql_literal(_v4_vpo_p)})
  WHERE TRIM(CAST(indicator_name AS VARCHAR))='Принято, всего'
    AND regexp_matches(lower(CAST(object_level AS VARCHAR)),'регион|субъект')
    AND specialty_code IS NOT NULL AND year IS NOT NULL
  GROUP BY 1,2,3,4,5,6,7,8'''
_v4_tmp=V4_ADMISSION_PATH.with_suffix('.partial.parquet');_v4_tmp.unlink(missing_ok=True)
with duckdb.connect(':memory:') as _db:
    _db.execute("SET memory_limit='2048MB'");_db.execute('SET threads=1')
    _db.execute('SET temp_directory='+_sql_literal(DB_TMP));_db.execute('SET preserve_insertion_order=false')
    _db.execute('COPY ('+_v4_s+') TO '+_sql_literal(_v4_tmp)+' (FORMAT PARQUET,COMPRESSION ZSTD)')
    _v4_counts=_db.execute('''SELECT COUNT(*),COUNT(*) FILTER (WHERE records_in_cell=1 AND minimum=maximum
      AND minimum>=0 AND ABS(minimum-ROUND(minimum))<1e-8),COUNT(DISTINCT observation_year)
      FROM read_parquet(?)''',[str(_v4_tmp)]).fetchone()
_v4_tmp.replace(V4_ADMISSION_PATH);CONTEXT_PARQUET_FILES.append(V4_ADMISSION_PATH)
qa_table('v4_specialty_admission_gate',pd.DataFrame([{'source_id':VPO_211['asset_id'],
   'full_source_rows':int(VPO_211['rows_actual']),'exact_name':'Принято, всего',
   'regional_specialty_strata':int(_v4_counts[0]),'valid_singleton_nonnegative_integer_strata':int(_v4_counts[1]),
   'years':int(_v4_counts[2]),'source_grain':'REGION_X_SPECIALTY_X_EDUCATION_LEVEL_X_FORM_X_ADMISSION_TYPE_X_UNIVERSITY_TYPE',
   'warning':'NOT_ORGANIZATION_ADMISSIONS_AND_NOT_OD_BY_SPECIALTY'}]))
V4_PROGRAM_FRAME=pd.DataFrame()
if int(_v4_counts[1])>0:
    with duckdb.connect(':memory:') as _db:
        _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
        _v4_q='''SELECT * FROM read_parquet(?) WHERE records_in_cell=1 AND minimum=maximum
             AND minimum>=0 AND ABS(minimum-ROUND(minimum))<1e-8
             AND observation_year IS NOT NULL AND specialty_code!='' '''
        _v4_n=int(_db.execute('SELECT COUNT(*) FROM ('+_v4_q+')',[str(V4_ADMISSION_PATH)]).fetchone()[0])
        if CFG.MODE=='FULL' and CFG.ML54_FULL_NO_STRATA_CAP:
            # ALL verified strata (not 'longest 15000'); 237k observed in V4.
            # Pandas materialization is restricted to verified regional/specialty cells,
            # not the 7.5 million source rows. Never truncate study population.
            V4_PROGRAM_FRAME=_db.execute(_v4_q,[str(V4_ADMISSION_PATH)]).df()
            V4_PROGRAM_FULL_VALID_ROWS=int(len(V4_PROGRAM_FRAME))
        elif _v4_n<=CFG.MAX_PANDAS_ROWS:
            V4_PROGRAM_FRAME=_db.execute(_v4_q,[str(V4_ADMISSION_PATH)]).df()
            V4_PROGRAM_FULL_VALID_ROWS=int(len(V4_PROGRAM_FRAME))
        else:
            raise RuntimeError('ML54_FULL_STRATA_REQUIRED: no discretionary sample allowed')
    if int(_v4_n)!=int(len(V4_PROGRAM_FRAME)):
        raise RuntimeError(f'ML54_COVERAGE_LOSS: eligible={_v4_n}, materialized={len(V4_PROGRAM_FRAME)}')
    qa_table('v5_ml54_full_strata_coverage',pd.DataFrame([{
        'verified_rows':int(_v4_n),'materialized_rows':int(len(V4_PROGRAM_FRAME)),
        'fraction':1.0,'bounded_sample_used':False}]))
if len(V4_PROGRAM_FRAME):
    _f=V4_PROGRAM_FRAME.copy()
    for _c in ['object_name','specialty_code','education_level','education_form','admission_type','university_type']:
        _f[_c]=_f[_c].fillna('__MISSING__').astype(str)
    _fields=['object_name','specialty_code','education_level','education_form','admission_type','university_type']
    _f['entity_id']=_f[_fields].agg('|'.join,axis=1)
    _f['value']=_f['minimum'].astype(float)
    # A unique reporting stratum must not be interpreted as the sum of its components.
    _f=_f.drop_duplicates(['entity_id','observation_year'],keep=False)
    _f=_f[['entity_id','observation_year','value','object_name','specialty_code','education_level']]
    _rows=historical_pair_panel(_f[['entity_id','observation_year','value']],['entity_id'],'value')
    if len(_rows):
        _rows['sector']='VO_REGIONAL_SPECIALTY'
        _rows['target_id']='VPO_2_1_1_REPORTED_ADMITTED_EXACT_STRATUM'
        _rows['geography_grain']='REPORTED_REGION_SPECIALTY'
        qa_table('v4_specialty_model_frame',_rows,allow_empty=True)
        test_predictive('ML-5.4',_rows,nonnegative=True,count_confirmed=True)
        if MODULES['ML-5.4']['status'].startswith('TESTED'):
            MODULES['ML-5.4']['reason']='Exact VPO admitted value regional specialty/level/form/admission_type stratum; not OD and not institution-level'
            V4_SUBEXPERIMENTS.append({'subexperiment':'ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS',
               'status':'MODEL_TESTED','n':len(_rows),'source':VPO_211['asset_id']})
        else:v4_block('ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS','No historically available train/test for exact cells',rows=len(_rows))
    else:v4_block('ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS','Too few historical observations under publication lag',rows=len(_f))
else:v4_block('ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS','No unique regional speciality admission strata',rows=int(_v4_counts[0]))
del V4_PROGRAM_FRAME;gc.collect()

# ML-5.2: do not auto-approve a number of admitted students from name-matched E02.
# A separate observed 'Приведенный контингент студентов' is a permissible alternative target,
# but it CANNOT be reported as organization admissions or interregional entrants.
_v4_e02_p=fetch_asset(E02)
_v4_org_target='Приведенный контингент студентов'
with duckdb.connect(':memory:') as _db:
    _db.execute("SET memory_limit='1024MB'");_db.execute('SET threads=1')
    _org=_db.execute('''SELECT CAST(id AS VARCHAR) id,TRY_CAST(year AS INTEGER) observation_year,
        COUNT(*)::BIGINT n_source_records,COUNT(DISTINCT CAST(area AS VARCHAR)) n_area,
        MIN(TRY_CAST(indicator_value AS DOUBLE)) low_value,MAX(TRY_CAST(indicator_value AS DOUBLE)) high_value
        FROM read_parquet(?) WHERE indicator_name=? AND id IS NOT NULL AND year IS NOT NULL
        GROUP BY 1,2''',[str(_v4_e02_p),_v4_org_target]).df()
qa_table('v4_organization_contingent_candidates',_org,allow_empty=True)
_org=_org[_org.n_source_records.eq(1)&_org.low_value.notna()&_org.low_value.eq(_org.high_value)&_org.low_value.ge(0)].copy()
if len(_org):
    _org['entity_id']='VO|'+norm_code(_org.id)
    _observed=_org[['entity_id','observation_year','low_value']].rename(columns={'low_value':'value'})
    _observed=_observed.drop_duplicates(['entity_id','observation_year'],keep=False)
    _panel=historical_pair_panel(_observed,['entity_id'],'value')
    if len(_panel):
        _panel['sector']='VO';_panel['target_id']='E02_REPORTED_STUDENT_CONTINGENT_NOT_ADMISSIONS'
        _panel['parent_group_id']=_panel['entity_id'];_panel['geography_grain']='ORGANIZATION'
        _split,_why=temporal_partition(_panel,min_train=CFG.MIN_TRAIN_ROWS,min_test=CFG.MIN_TEST_ROWS)
        if _split is not None:
            _tr=_split['train'];_te=_split['test']
            for _model in ('persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm'):
                fit_and_record('ML-5.2',_model,_tr,_te,seed=CFG.RANDOM_SEED,nonnegative=True,
                  protocol='ALT_CONTINGENT_RETROSPECTIVE')
            qa_table('v4_organization_contingent_model_frame',_panel,allow_empty=True)
            V4_SUBEXPERIMENTS.append({'subexperiment':'ML-5.2_ALTERNATIVE_CONTINGENT','status':'MODEL_TESTED_NOT_ADMISSIONS',
                                     'n':len(_te),'target':_v4_org_target})
            MODULES['ML-5.2'].update(status='ALTERNATIVE_TARGET_TESTED_ADMISSIONS_NOT_FEASIBLE',
                  reason='E02 student contingent model evaluated; original number admitted still unavailable for organization × year',n=len(_te))
        else:v4_block('ML-5.2_ALTERNATIVE_CONTINGENT',str(_why),rows=len(_panel))
    else:v4_block('ML-5.2_ALTERNATIVE_CONTINGENT','Insufficient stable organization/year contingent panel',rows=len(_org))
else:v4_block('ML-5.2_ALTERNATIVE_CONTINGENT','No unique organization/year source value; cannot sum area splits',rows=len(_org))
v4_block('ML-5.2_ORGANIZATION_ADMISSIONS','No certified admitted-count × organization × year in Stage I; student contingent is not admissions',
         'exact admission count, organization/branch identifiers, year and verified indicator/units') if len(ORG_YEAR)==0 else None
qa_table('v4_additional_subexperiments',pd.DataFrame(V4_SUBEXPERIMENTS),allow_empty=True)
progress_event('EXTENDED_MODULES','V4: проверка фактических величин ВПО/ВО завершена',announce=True)


## 9Г. ML-5.2 · углублённый временной эксперимент по организации и образовательной страте

Целевая величина — реально опубликованный контингент Е02. Это **не** число зачисленных; наблюдения разных страт не суммируются и не выдаются за организационный итог.

In [ ]:
# V5 · дополнительный строгий эксперимент ML-5.2: контингент организация × образовательная страта.
# Не равно приёму, не суммируется по перекрывающимся направлениям, не подменяет ML-5.2 admission target.
progress_event('EXTENDED_MODULES','V5 ML-5.2: проверка 106 тыс. исходных наблюдений по образовательным стратам',announce=True)
V5_ORG_AREA_STATUS={'status':'NOT_FEASIBLE_SOURCE_GRAIN','source':E02['asset_id']}
_v5_52_file=fetch_asset(E02)
_v5_52_sql='''SELECT CAST(id AS VARCHAR) id,TRY_CAST(year AS INTEGER) observation_year,
    COALESCE(CAST(area AS VARCHAR),'__NA__') area,
    COALESCE(CAST(specialty_code AS VARCHAR),'__NA__') specialty_code,
    COALESCE(CAST(specialty_section AS VARCHAR),'__NA__') specialty_section,
    COALESCE(CAST(indicator_unit AS VARCHAR),'__NA__') indicator_unit,
    COUNT(*)::BIGINT n_original_records,
    MIN(TRY_CAST(indicator_value AS DOUBLE)) min_value,
    MAX(TRY_CAST(indicator_value AS DOUBLE)) max_value
    FROM read_parquet(?) WHERE TRIM(CAST(indicator_name AS VARCHAR))='Приведенный контингент студентов'
      AND id IS NOT NULL AND year IS NOT NULL GROUP BY 1,2,3,4,5,6'''
with duckdb.connect(':memory:') as _con:
    _con.execute("SET memory_limit='1536MB'");_con.execute('SET threads=1')
    _v5_52_raw=_con.execute(_v5_52_sql,[str(_v5_52_file)]).df()
_v5_52_eligible=_v5_52_raw.loc[
    _v5_52_raw.n_original_records.eq(1)&_v5_52_raw.min_value.notna()&
    _v5_52_raw.min_value.eq(_v5_52_raw.max_value)&_v5_52_raw.min_value.ge(0)].copy()
V5_ORG_AREA_STATUS.update(raw_strata=len(_v5_52_raw),uniquely_reported_strata=len(_v5_52_eligible),
    duplicate_or_ambiguous_strata=int(len(_v5_52_raw)-len(_v5_52_eligible)))
if len(_v5_52_eligible):
    # The full combination, including area and specialty, is the statistical entity.
    _v5_52_eligible['entity_id']='VOAREA|'+_v5_52_eligible[
        ['id','area','specialty_code','specialty_section','indicator_unit']].astype(str).agg(
        lambda row:hashlib.sha256(json.dumps(row.tolist(),ensure_ascii=False).encode()).hexdigest()[:24],axis=1)
    _v5_52_eligible=_v5_52_eligible.drop_duplicates(['entity_id','observation_year'],keep=False)
    _v5_52_obs=_v5_52_eligible[['entity_id','observation_year','min_value']].rename(columns={'min_value':'value'})
    _v5_52_panel=historical_pair_panel(_v5_52_obs,['entity_id'],'value')
    V5_ORG_AREA_STATUS['complete_source_entity_year_rows']=len(_v5_52_obs)
    V5_ORG_AREA_STATUS['eligible_lagged_rows']=len(_v5_52_panel)
    qa_table('v5_ml52_organization_area_source_strata',_v5_52_eligible,allow_empty=True)
    if len(_v5_52_panel):
        _v5_52_panel['sector']='VO_CONTINGENT_AREA';_v5_52_panel['target_id']='E02_CONTINGENT_ORGANIZATION_AREA_NOT_ADMISSIONS'
        _v5_52_panel['parent_group_id']=_v5_52_panel.entity_id
        _v5_52_panel['geography_grain']='ORGANIZATION_AREA_SPECIALTY'
        _v5_52_split,_v5_52_reason=temporal_partition(_v5_52_panel)
        if _v5_52_split is not None:
            progress_event('EXTENDED_MODULES','ML-5.2: полный разведочный временной эксперимент по контингенту',announce=True)
            test_predictive('ML-5.2',_v5_52_panel,nonnegative=True,count_confirmed=False)
            if MODULES['ML-5.2']['status'].startswith('TESTED'):
                MODULES['ML-5.2'].update(status='ALTERNATIVE_ORG_AREA_CONTINGENT_TESTED_NOT_ADMISSIONS',
                    reason='Observed E02 student contingent by organization/area/specialty; NOT admitted cohorts, not institutional total')
                V5_ORG_AREA_STATUS.update(status='ALTERNATIVE_RETROSPECTIVE_TESTED_NOT_ADMISSIONS',
                    n_test=int(len(_v5_52_split['test'])),target_year=int(_v5_52_split['test_year']))
        else:
            V5_ORG_AREA_STATUS.update(status='NOT_FEASIBLE_HISTORICAL_SUPPORT',reason=str(_v5_52_reason))
else:
    V5_ORG_AREA_STATUS['reason']='No uniquely reported organization-area-year source cell, no admissible target'
qa_table('v5_ml52_area_model_execution',pd.DataFrame([V5_ORG_AREA_STATUS]),allow_empty=True)
print('ML-5.2 alternative:',V5_ORG_AREA_STATUS['status'],'| scientific admission target still gated')
del _v5_52_raw,_v5_52_eligible
import gc;gc.collect()


## 9В. E0/E1 по муниципальной образовательной инфраструктуре и выпускники
Полный дисковый SQL по БДМО; отдельный групповой тест занятости без выводов о закреплении.

In [ ]:
# V4. ML-5.7 disk-backed E0/E1; ML-5.5 independently held-out geography test.
progress_event('EXTENDED_MODULES','V4: E0/E1 полный муниципальный SQL и выпускники E04',announce=True)
V4_MUNICIPAL_STATUS={};V4_GRADUATE_STATUS={}
_v4_bdmo_file=ROOT/'derived'/'full52_context'/'bdmo_education_municipal.parquet'
if CFG.MODE=='FULL' and _v4_bdmo_file.is_file():
    _mi=MUNI[['oktmo','observation_year','migration_rate','population']].copy()
    _mi['oktmo']=norm_code(_mi.oktmo)
    for _c in ('migration_rate','population'):_mi[_c]=number(_mi[_c])
    _mi=_mi.dropna(subset=['oktmo','observation_year','migration_rate'])
    _mi=_mi.drop_duplicates(['oktmo','observation_year'],keep=False)
    _mi['target_year']=_mi.observation_year.astype(int)
    _mi['origin_year']=_mi.target_year-CFG.HORIZON
    _mi['feature_year']=_mi.origin_year-CFG.FEATURE_PUBLICATION_LAG_YEARS
    _hist=_mi[['oktmo','observation_year','migration_rate','population']].rename(columns={
        'observation_year':'feature_year','migration_rate':'lag_migration_rate','population':'lag_population'})
    _mi=_mi.merge(_hist,on=['oktmo','feature_year'],validate='many_to_one',how='inner')
    _mi=_mi.dropna(subset=['lag_migration_rate','lag_population'])
    _latest=int(_mi.target_year.max()) if len(_mi) else 0
    with duckdb.connect(':memory:') as _db:
        _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
        _db.execute('SET temp_directory='+_sql_literal(DB_TMP))
        _db.register('m',_mi)
        _q='''SELECT m.oktmo,m.target_year,m.origin_year,m.feature_year,m.migration_rate actual,
            m.lag_migration_rate,m.lag_population,CAST(e.indicator_code AS VARCHAR) indicator_code,
            CAST(e.indicator_name AS VARCHAR) indicator_name,CAST(e.indicator_unit AS VARCHAR) indicator_unit,
            e.median_reported education_historic_value
            FROM m JOIN read_parquet(?) e ON CAST(m.oktmo AS VARCHAR)=CAST(e.oktmo AS VARCHAR)
               AND m.feature_year=e.observation_year
            WHERE e.n_raw_rows=1 AND e.min_reported=e.max_reported AND e.median_reported>=0
              AND e.indicator_code IS NOT NULL'''
        _matched=ROOT/'derived'/'v4_municipality_education_exact_support.parquet'
        _db.execute('COPY ('+_q.replace('read_parquet(?)','read_parquet('+_sql_literal(_v4_bdmo_file)+')')+
                    ') TO '+_sql_literal(_matched)+' (FORMAT PARQUET,COMPRESSION ZSTD)')
        # Indicators are selected by training coverage, unit constancy and values, not TEST errors.
        _stats=_db.execute('''SELECT indicator_code,indicator_unit,MIN(indicator_name) AS example_indicator_name,
            COUNT(*) FILTER (WHERE target_year<=?) n_train_candidates,
            COUNT(DISTINCT oktmo) FILTER (WHERE target_year<=?) n_municipalities_train,
            COUNT(DISTINCT target_year) FILTER (WHERE target_year<=?) n_years_train,
            COUNT(DISTINCT education_historic_value) FILTER (WHERE target_year<=?) n_distinct_train,
            COUNT(*) FILTER (WHERE target_year=?) n_test_coverage,
            COUNT(DISTINCT oktmo) FILTER (WHERE target_year=?) n_municipalities_test,
            COUNT(DISTINCT indicator_name) n_indicator_names,
            COUNT(*) as rows_all
            FROM read_parquet(?) GROUP BY 1,2''',
          [_latest-2]*4+[_latest]*2+[str(_matched)]).df()
        _stats=_stats.sort_values(['n_municipalities_train','n_test_coverage','indicator_code'],ascending=[False,False,True])
        qa_table('v4_ml57_indicator_support',_stats,allow_empty=True)
        _eligible=_stats.loc[(_stats.n_train_candidates>=50)&(_stats.n_municipalities_train>=CFG.MIN_MUNICIPALITIES_E0E1)&
            (_stats.n_years_train>=2)&(_stats.n_distinct_train>1)&
            (_stats.n_municipalities_test>=CFG.MIN_MUNICIPALITIES_E0E1)&(_stats.n_indicator_names==1)]
        if len(_eligible):
            _eligible=_eligible.copy()
            _eligible['is_professional_education']=_eligible.example_indicator_name.astype(str).str.contains(
                r'профессиональн|высш|колледж|техникум|университет',case=False,regex=True,na=False)
            _eligible=_eligible.sort_values(['is_professional_education','n_municipalities_train','n_test_coverage','indicator_code'],
                ascending=[False,False,False,True])
            _one=_eligible.iloc[0];_selected=str(_one.indicator_code);_unit=str(_one.indicator_unit)
            V4_MUNICIPAL_INDICATOR_SEMANTIC='PROFESSIONAL_EDUCATION_CONTEXT' if bool(_one.is_professional_education) else 'GENERAL_EDUCATION_PROXY_ONLY'
            _rows=_db.execute('''SELECT * FROM read_parquet(?) WHERE indicator_code=?
                  AND COALESCE(indicator_unit,'')=COALESCE(?,'')''',[str(_matched),_selected,_unit]).df()
        else:_rows=pd.DataFrame();_selected=None
    CONTEXT_PARQUET_FILES.append(_matched)
    V4_MUNICIPAL_STATUS={'candidate_indicators':len(_stats),'matched_municipality_indicator_year_rows':int(pq.ParquetFile(_matched).metadata.num_rows),
       'target_year':_latest,'feasible_indicators':len(_eligible),'selected_code':_selected,'selected_semantic':locals().get('V4_MUNICIPAL_INDICATOR_SEMANTIC','UNAVAILABLE')}
    if len(_rows):
        _rows=_rows.sort_values(['oktmo','target_year','indicator_code'])
        _rows=_rows.drop_duplicates(['oktmo','target_year'],keep=False)
        _tr=_rows[_rows.target_year.le(_latest-2)].dropna(subset=['actual','lag_migration_rate','lag_population','education_historic_value']).copy()
        _te=_rows[_rows.target_year.eq(_latest)].dropna(subset=['actual','lag_migration_rate','lag_population','education_historic_value']).copy()
        if len(_tr)>=50 and len(_te)>=CFG.MIN_MUNICIPALITIES_E0E1:
            # Historical features are at target_year - 1 - publication lag; not current-year infrastructure.
            assert (_tr.feature_year<_tr.target_year).all() and (_te.feature_year<=_te.origin_year).all()
            assert (_tr.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS<=_te.origin_year.min()).all()
            qa_table('v4_ml57_exact_common_support',_rows,allow_empty=True)
            _p0=['lag_migration_rate','lag_population'];_p1=_p0+['education_historic_value']
            _results=[]
            for _label,_features in [('E0',_p0),('E1',_p1)]:
                _mod=Pipeline([('fill',SimpleImputer(strategy='median')),
                     ('scale',StandardScaler()),('ridge',Ridge(alpha=20.0))])
                _mod.fit(_tr[_features].to_numpy(float),_tr.actual.to_numpy(float))
                _yp=_mod.predict(_te[_features].to_numpy(float))
                _f=ROOT/'models'/f'v4_ml57_{_label}.joblib'
                joblib.dump({'model':_mod,'features':_features,'indicator_code':_selected},_f,compress=3)
                _loaded=joblib.load(_f)
                assert np.allclose(_loaded['model'].predict(_te[_features].to_numpy(float)),_yp)
                MODEL_FILES[_f.name]=str(_f)
                MODEL_RECORDS.append({'module':'ML-5.7','sector':'BDMO_EDUCATION_PROXY','model':_label,'seed':-1,
                  'path':_f.name,'file_sha256':hash_file(_f),'feature_num':_features,'feature_cat':[],
                  'training_rows':len(_tr),'test_rows':len(_te),'reloaded':True})
                _p=pd.DataFrame({'entity_id':_te.oktmo.astype(str).to_numpy(),
                  'origin_year':_te.origin_year.to_numpy(),'target_year':_te.target_year.to_numpy(),
                  'information_cutoff_year':_te.origin_year.to_numpy(),
                  'feature_available_year':_te.origin_year.to_numpy(),
                  'target_available_year':_te.target_year.to_numpy()+CFG.LABEL_PUBLICATION_LAG_YEARS,
                  'sector':'BDMO_EDUCATION_PROXY','target_id':'D01_MUNICIPAL_MIGRATION_RATE',
                  'actual':_te.actual.to_numpy()})
                _pred=predictions_frame(_p,'ML-5.7',_label,_yp,protocol='RETROSPECTIVE_PROXY_E0E1')
                _pred['geography_grain']='MUNICIPALITY';record_prediction(_pred)
                _mt=effective_mae(_te.actual,_yp)
                METRICS.append({'module':'ML-5.7','sector':'BDMO_EDUCATION_PROXY','model':_label,
                    'protocol':'RETROSPECTIVE_PROXY_E0E1','seed':-1,'target_year':_latest,
                    'source_vintage':INDEX['created_at'],**_mt})
                _results.append({'variant':_label,'n_test':len(_te),'n_train':len(_tr),'municipalities_test':_te.oktmo.nunique(),**_mt})
            qa_table('v4_ml57_results',pd.DataFrame(_results),allow_empty=True)
            V4_MUNICIPAL_STATUS.update(status='E0_E1_TESTED_PROXY_NOT_VERIFIED_CAMPUS',
                 n_train=len(_tr),n_test=len(_te),MAE_E0=_results[0]['MAE'],MAE_E1=_results[1]['MAE'])
            MODULES['ML-5.7'].update(status='RETROSPECTIVE_MUNICIPAL_PROXY_TEST',
                  reason='BDMO lagged '+V4_MUNICIPAL_INDICATOR_SEMANTIC+' E0/E1; same OKTMO-year support, DOES NOT verify campus or causal effect',n=len(_te))
        else:V4_MUNICIPAL_STATUS.update(status='NOT_FEASIBLE_TEMPORAL_SUPPORT',train=len(_tr),test=len(_te))
    else:V4_MUNICIPAL_STATUS['status']='NOT_FEASIBLE_EXACT_SHARED_SUPPORT'
else:V4_MUNICIPAL_STATUS={'status':'NOT_FEASIBLE_NO_BDMO_FULL_FILE'}
qa_table('v4_ml57_execution',pd.DataFrame([V4_MUNICIPAL_STATUS]),allow_empty=True)

# ML-5.5: out-of-region validation of employment rate on E04's original reporting rows.
# This is deliberately cross-sectional, not a forecast of retention or postgraduation residence.
progress_event('EXTENDED_MODULES','V4: трудоустройство E04, проверка вне регионов обучения',announce=True)
_gpath=fetch_asset(GRAD_ORG)
with duckdb.connect(':memory:') as _db:
    _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
    _gg=_db.execute('''SELECT CAST(object_name AS VARCHAR) region_name,
      CAST(object_level AS VARCHAR) object_level,CAST(university AS VARCHAR) university,
      CAST(gender AS VARCHAR) gender,CAST(education_level AS VARCHAR) education_level,
      TRY_CAST(year AS INTEGER) graduation_year,
      TRY_CAST(count_graduate AS DOUBLE) count_graduate,
      TRY_CAST(percent_employed AS DOUBLE) employed_pct
      FROM read_parquet(?) WHERE regexp_matches(lower(CAST(object_level AS VARCHAR)),'регион|субъект')
      AND TRY_CAST(percent_employed AS DOUBLE) BETWEEN 0 AND 100
      AND TRY_CAST(count_graduate AS DOUBLE)>=0''',[str(_gpath)]).df()
_gkeys=['region_name','university','gender','education_level','graduation_year']
_gduplicates=_gg.duplicated(_gkeys,keep=False).sum()
_gg=_gg.drop_duplicates(_gkeys,keep=False).copy()
_gcount=_gg.region_name.nunique()
V4_GRADUATE_STATUS={'valid_individual_reporting_rows':len(_gg),'ambiguous_reporting_rows_dropped':int(_gduplicates),
                    'n_geographies':int(_gcount),'employment_reference':'2024-12-31','is_longitudinal':False}
if len(_gg)>=100 and _gcount>=8 and _gg.employed_pct.nunique()>1:
    from sklearn.model_selection import GroupShuffleSplit
    _sp=GroupShuffleSplit(n_splits=1,test_size=.2,random_state=CFG.RANDOM_SEED)
    _a,_b=next(_sp.split(_gg,groups=_gg.region_name))
    _gtr=_gg.iloc[_a].copy();_gte=_gg.iloc[_b].copy()
    assert set(_gtr.region_name).isdisjoint(_gte.region_name)
    _cn=['count_graduate','graduation_year'];_cc=['education_level','gender']
    _Xtr=_gtr[_cn+_cc].copy();_Xte=_gte[_cn+_cc].copy()
    for _k in _cc:
        _Xtr[_k]=_Xtr[_k].fillna('UNKNOWN').astype(str);_Xte[_k]=_Xte[_k].fillna('UNKNOWN').astype(str)
    _gpred=[];_gy=_gte.employed_pct.to_numpy(float)
    _gbase=np.full(len(_gte),float(_gtr.employed_pct.mean()))
    _gpred.append(('region_blind_train_mean',_gbase))
    for _kind in ('ridge','lightgbm','catboost'):
        _num=Pipeline([('fill',SimpleImputer(strategy='median')),('scale',StandardScaler())])
        if _kind=='catboost':
            _med={k:float(_Xtr[k].median()) for k in _cn}
            _A=_Xtr.copy();_B=_Xte.copy()
            for _k in _cn:
                _A[_k]=_A[_k].fillna(_med[_k]);_B[_k]=_B[_k].fillna(_med[_k])
            _mod=CatBoostRegressor(iterations=180,depth=4,learning_rate=.06,verbose=False,
                random_seed=CFG.RANDOM_SEED,thread_count=1,allow_writing_files=False)
            _mod.fit(_A,_gtr.employed_pct.to_numpy(float),cat_features=_cc)
            _gp=_mod.predict(_B)
            _art={'model':_mod,'medians':_med,'categoricals':_cc,'features':_cn+_cc}
        else:
            _prep=ColumnTransformer([('n',_num,_cn),('c',OneHotEncoder(handle_unknown='ignore'),_cc)])
            if _kind=='ridge':_est=Ridge(alpha=30.,solver='lsqr')
            else:_est=LGBMRegressor(n_estimators=170,max_depth=4,num_leaves=12,
                   n_jobs=1,verbosity=-1,random_state=CFG.RANDOM_SEED)
            _mod=Pipeline([('prep',_prep),('reg',_est)])
            _mod.fit(_Xtr,_gtr.employed_pct.to_numpy(float));_gp=_mod.predict(_Xte)
            _art={'model':_mod,'features':_cn+_cc}
        _gp=np.clip(_gp,0,100)
        _gpred.append((_kind,_gp))
        _mp=ROOT/'models'/f'v4_ml55_region_heldout_{_kind}.joblib'
        joblib.dump(_art,_mp,compress=3)
        _rt=joblib.load(_mp)
        if _kind=='catboost':
            _verify=_Xte.copy()
            for _k in _cn:_verify[_k]=_verify[_k].fillna(_rt['medians'][_k])
        else:_verify=_Xte
        assert np.allclose(np.clip(_rt['model'].predict(_verify),0,100),_gp)
        MODEL_FILES[_mp.name]=str(_mp)
        MODEL_RECORDS.append({'module':'ML-5.5','sector':'GRAD_E04_CROSS_SECTION',
            'model':_kind,'seed':CFG.RANDOM_SEED,'path':_mp.name,'file_sha256':hash_file(_mp),
            'training_rows':len(_gtr),'test_rows':len(_gte),'reloaded':True,'feature_num':_cn,'feature_cat':_cc})
    _gm=[];_gpframe=_gte[['region_name','university','graduation_year','employed_pct']].reset_index(drop=True).copy()
    for _kind,_yp in _gpred:
        _gpframe['pred_'+_kind]=_yp
        _gm.append({'module':'ML-5.5','model':_kind,'protocol':'REGION_HELD_OUT_CROSS_SECTION_2024',
            'n_test':len(_gte),'n_regions_test':_gte.region_name.nunique(),**effective_mae(_gy,_yp)})
    qa_table('v4_ml55_crosssection_predictions',_gpframe,allow_empty=True)
    qa_table('v4_ml55_crosssection_metrics',pd.DataFrame(_gm),allow_empty=True)
    V4_GRADUATE_STATUS.update(status='REGIONAL_HOLDOUT_EMPLOYMENT_TEST_NOT_RETENTION',n_test=len(_gte),n_train=len(_gtr),
       baseline_mae=_gm[0]['MAE'],tested_methods=len(_gpred))
    MODULES['ML-5.5'].update(status='DESCRIPTIVE_AND_CROSS_SECTIONAL_TEST',
       reason='All four E04/E05 descriptive; E04 regional OOS employment test does not imply retention/temporal forecast',n=len(_gte))
else:V4_GRADUATE_STATUS['status']='NOT_FEASIBLE_REGION_HELDOUT_AFTER_EXACT_STRATA_GATE'
qa_table('v4_ml55_execution',pd.DataFrame([V4_GRADUATE_STATUS]),allow_empty=True)

# ML-5.6: historical node centrality is tested on common regions only; coverage break is an explicit caveat.
_v4_net=pd.DataFrame(NETWORK)
_v4_nresults=[]
if len(_v4_net) and _v4_net.year.nunique()>1:
    _v4_years=sorted(_v4_net.year.unique())
    for _prev,_now in zip(_v4_years[:-1],_v4_years[1:]):
        _p=_v4_net[_v4_net.year.eq(_prev)][['region_code','pagerank']].rename(columns={'pagerank':'prior_pagerank'})
        _n=_v4_net[_v4_net.year.eq(_now)][['region_code','pagerank']].rename(columns={'pagerank':'actual_pagerank'})
        _joined=_p.merge(_n,on='region_code',validate='one_to_one')
        _rho,_pv=(spearmanr(_joined.prior_pagerank,_joined.actual_pagerank) if len(_joined)>=3 else (np.nan,np.nan))
        _v4_nresults.append({'origin_year':int(_prev),'target_year':int(_now),'common_regions':len(_joined),
            'pagerank_persistence_mae':float(np.abs(_joined.prior_pagerank-_joined.actual_pagerank).mean()) if len(_joined) else np.nan,
            'pagerank_rank_spearman':float(_rho) if np.isfinite(_rho) else np.nan,
            'nominal_p_value_NOT_CONFIRMATORY':float(_pv) if np.isfinite(_pv) else np.nan,
            'coverage_break':'UNRESOLVED_2023_OR_OTHER_YEAR_COVERAGE'})
qa_table('v4_ml56_node_forecast',pd.DataFrame(_v4_nresults),allow_empty=True)


## 9Д. ML-5.7 · независимое географическое испытание образовательного муниципального контекста

Дополнительный тест E0/E1 по пространственно отложенным муниципалитетам. Образовательный признак берётся из БДМО в тот же наблюдаемый год и поэтому **не** является историческим прогнозным признаком. Это только диагностическая ассоциация, а не эффект университета или доказательство миграции.

In [ ]:
# V5 · ML-5.7: независимый *пространственный*, не временной E0/E1 diagnostic.
# General education proxy from BDMO; no academic-campus location, no residence effect inferred.
progress_event('EXTENDED_MODULES','V5 ML-5.7: пространственный E0/E1 на наблюдаемом образовательном контексте',announce=True)
V5_MUNICIPAL_CROSS_STATUS={'status':'NOT_IDENTIFIABLE_ON_AVAILABLE_CROSS_SECTION'}
if CFG.MODE=='FULL' and 'BDMO_EDU_PROXY_CANDIDATES' in globals() and len(BDMO_EDU_PROXY_CANDIDATES):
    _v5_e=BDMO_EDU_PROXY_CANDIDATES.copy()
    _v5_e['oktmo']=norm_code(_v5_e.oktmo);_v5_e['observation_year']=safe_year(_v5_e.observation_year)
    _v5_e=_v5_e.loc[_v5_e.n_raw_rows.eq(1)&_v5_e.min_reported.eq(_v5_e.max_reported)&
        _v5_e.median_reported.notna()&_v5_e.median_reported.ge(0)].copy()
    _v5_m=MUNI[['oktmo','observation_year','migration_rate','population']].copy()
    _v5_m['oktmo']=norm_code(_v5_m.oktmo);_v5_m['observation_year']=safe_year(_v5_m.observation_year)
    for _k in ('migration_rate','population'):_v5_m[_k]=number(_v5_m[_k])
    _v5_m=_v5_m.dropna(subset=['oktmo','observation_year','migration_rate','population'])
    _v5_m=_v5_m.drop_duplicates(['oktmo','observation_year'],keep=False)
    _v5_join=_v5_m.merge(_v5_e,on=['oktmo','observation_year'],how='inner',validate='one_to_many')
    _v5_join['is_heldout']=_v5_join.oktmo.astype(str).map(lambda x:int(
         hashlib.sha256(('V5_MUNICIPAL_HOLDOUT|'+x).encode()).hexdigest()[:8],16)%5==0)
    _v5_join['indicator_code']=_v5_join.indicator_code.astype(str)
    _v5_join['indicator_unit']=_v5_join.indicator_unit.astype(str)
    _v5_stats=_v5_join.loc[~_v5_join.is_heldout].groupby(['indicator_code','indicator_unit'],dropna=False).agg(
        n_train_muni=('oktmo','nunique'),n_train_rows=('migration_rate','size'),
        n_train_years=('observation_year','nunique'),value_variation=('median_reported','nunique')).reset_index()
    _v5_stats=_v5_stats.sort_values(['n_train_muni','n_train_years','indicator_code'],ascending=[False,False,True])
    qa_table('v5_ml57_spatial_proxy_eligibility',_v5_stats,allow_empty=True)
    _v5_best=_v5_stats.loc[(_v5_stats.n_train_muni>=CFG.MIN_MUNICIPALITIES_E0E1)&
        (_v5_stats.value_variation>1)]
    if len(_v5_best):
        _v5_code=str(_v5_best.iloc[0].indicator_code);_v5_unit=str(_v5_best.iloc[0].indicator_unit)
        _v5_k=_v5_join.loc[_v5_join.indicator_code.eq(_v5_code)&_v5_join.indicator_unit.eq(_v5_unit)].copy()
        _v5_k=_v5_k.drop_duplicates(['oktmo','observation_year'],keep=False)
        _v5_train=_v5_k.loc[~_v5_k.is_heldout].copy();_v5_test=_v5_k.loc[_v5_k.is_heldout].copy()
        if len(_v5_train)>=100 and _v5_test.oktmo.nunique()>=CFG.MIN_MUNICIPALITIES_E0E1:
            _v5_features={'E0':['population','observation_year'],
                          'E1':['population','observation_year','median_reported']}
            _v5_out=[]
            for _variant,_cols in _v5_features.items():
                _v5_mod=Pipeline([('fill',SimpleImputer(strategy='median')),
                     ('scale',StandardScaler()),('ridge',Ridge(alpha=30.0))])
                _v5_mod.fit(_v5_train[_cols].to_numpy(float),_v5_train.migration_rate.to_numpy(float))
                _v5_pred=_v5_mod.predict(_v5_test[_cols].to_numpy(float))
                _v5_f=ROOT/'models'/('v5_ml57_crosssection_'+_variant+'_'+hashlib.sha256(
                    (_v5_code+'|'+_v5_unit).encode()).hexdigest()[:10]+'.joblib')
                if _v5_f.exists() or _v5_f.name in MODEL_FILES:raise RuntimeError('V5 ML57 model artifact collision')
                joblib.dump({'model':_v5_mod,'features':_cols,'indicator_code':_v5_code,
                   'protocol':'CROSS_SECTION_NOT_TEMPORAL'},_v5_f,compress=3)
                _v5_loaded=joblib.load(_v5_f)
                if not np.allclose(_v5_pred,_v5_loaded['model'].predict(_v5_test[_cols].to_numpy(float))):
                    raise RuntimeError('V5 ML57 reloaded cross-sectional model mismatch')
                MODEL_FILES[_v5_f.name]=str(_v5_f)
                MODEL_RECORDS.append({'module':'ML-5.7','sector':'BDMO_GENERAL_EDUCATION_CONTEXT',
                  'protocol':'SPATIAL_CROSS_SECTION_NOT_FORECAST','model':_variant,'seed':CFG.RANDOM_SEED,
                  'path':_v5_f.name,'file_sha256':hash_file(_v5_f),'feature_num':_cols,'feature_cat':[],
                  'training_rows':len(_v5_train),'test_rows':len(_v5_test),'reloaded':True})
                _v5_pf=pd.DataFrame({'entity_id':_v5_test.oktmo.astype(str).to_numpy(),
                  'origin_year':_v5_test.observation_year.to_numpy(),'target_year':_v5_test.observation_year.to_numpy(),
                  'information_cutoff_year':_v5_test.observation_year.to_numpy(),
                  'feature_available_year':_v5_test.observation_year.to_numpy(),
                  'target_available_year':_v5_test.observation_year.to_numpy(),
                  'sector':'BDMO_GENERAL_EDUCATION_CONTEXT',
                  'target_id':'D01_MUNICIPAL_MIGRATION_RATE_CONTEMPORANEOUS_ASSOCIATION',
                  'actual':_v5_test.migration_rate.to_numpy(float)})
                _v5_pred_rows=predictions_frame(_v5_pf,'ML-5.7',_variant,_v5_pred,
                                   seed=CFG.RANDOM_SEED,protocol='SPATIAL_CROSS_SECTION_NOT_FORECAST')
                _v5_pred_rows['geography_grain']='MUNICIPALITY';record_prediction(_v5_pred_rows)
                _v5_met=effective_mae(_v5_test.migration_rate,_v5_pred)
                METRICS.append({'module':'ML-5.7','sector':'BDMO_GENERAL_EDUCATION_CONTEXT',
                  'model':_variant,'protocol':'SPATIAL_CROSS_SECTION_NOT_FORECAST',
                  'seed':CFG.RANDOM_SEED,'target_year':int(_v5_test.observation_year.max()),**_v5_met})
                _v5_out.append({'variant':_variant,'n_train':len(_v5_train),'n_test':len(_v5_test),
                       'heldout_municipalities':_v5_test.oktmo.nunique(),**_v5_met})
            qa_table('v5_ml57_spatial_E0_E1_results',pd.DataFrame(_v5_out),allow_empty=True)
            V5_MUNICIPAL_CROSS_STATUS={'status':'CROSS_SECTION_ASSOCIATION_TESTED_NOT_TEMPORAL',
                'selected_source_indicator':_v5_code,'unit':_v5_unit,
                'years':int(_v5_k.observation_year.nunique()),'n_train':len(_v5_train),
                'n_test':len(_v5_test),'MAE_E0':_v5_out[0]['MAE'],'MAE_E1':_v5_out[1]['MAE'],
                'target':'municipal_migration_rate','not_campus_metric':True}
            if MODULES['ML-5.7'].get('status','').startswith('NOT_FEASIBLE'):
                MODULES['ML-5.7'].update(status='SPATIAL_ASSOCIATION_ONLY_NOT_E0E1_TEMPORAL',
                    reason='Municipal held-out E0/E1 general education context, no temporal as-of claim and no verified VO/SPO campuses')
        else:V5_MUNICIPAL_CROSS_STATUS['reason']='Insufficient held-out municipality count for spatial test'
    else:V5_MUNICIPAL_CROSS_STATUS['reason']='No valid unique municipal indicator across enough training municipalities'
qa_table('v5_ml57_spatial_E0_E1_execution',pd.DataFrame([V5_MUNICIPAL_CROSS_STATUS]),allow_empty=True)
print('ML-5.7 supplemental:',V5_MUNICIPAL_CROSS_STATUS['status'],'| temporal E0/E1 remains a separate gate')


## V7 · дополнительные предметные проверки: OD, кандидаты приёма, муниципальные E0/E1

In [ ]:
# V7.1.1 · дополнительные проверки научной идентифицируемости и устойчивости.
progress_event('EXTENDED_MODULES','V7: сопоставимость OD, точные целевые показатели, муниципальная поддержка',announce=True)

# 5.1/5.3/5.6: диагностируем устойчивость только на ОБЩИХ фактически наблюдавшихся OD-парах.
# Отсутствующие рёбра никогда не превращаем в нули и не делаем вывод об общей динамике страны.
_v7_od=OD_INTER[['observation_year','origin_region','destination_region','observed_value']].dropna().copy()
_v7_od['observation_year']=pd.to_numeric(_v7_od.observation_year,errors='coerce')
_v7_od=_v7_od[_v7_od.observation_year.notna()].copy()
_v7_od['observation_year']=_v7_od.observation_year.astype(int)
if _v7_od.duplicated(['observation_year','origin_region','destination_region']).any():
    raise RuntimeError('V7 OD: неоднозначный ключ год×откуда×куда')
_v7_years=sorted(_v7_od.observation_year.unique())
V7_OD_COMPARABLE=[]
for _cy in _v7_years[1:]:
    _pr=_v7_od[_v7_od.observation_year.eq(_cy-1)]
    _cr=_v7_od[_v7_od.observation_year.eq(_cy)]
    _pair=_cr.merge(_pr[['origin_region','destination_region','observed_value']].rename(
        columns={'observed_value':'previous_value'}),on=['origin_region','destination_region'],how='inner',validate='one_to_one')
    _shared=int(len(_pair));_now=int(len(_cr));_old=int(len(_pr))
    V7_OD_COMPARABLE.append({'year':int(_cy),'observed_edges_previous':_old,'observed_edges_current':_now,
       'shared_edges':_shared,'previous_coverage':_shared/_old if _old else np.nan,
       'current_coverage':_shared/_now if _now else np.nan,
       'MAE_persistence_on_common_edges':float((_pair.observed_value-_pair.previous_value).abs().mean()) if _shared else np.nan,
       'sum_current_on_common_edges':float(_pair.observed_value.sum()),
       'sum_previous_on_common_edges':float(_pair.previous_value.sum()),
       'interpretation':'INTERSECTION_OF_OBSERVED_EDGES_ONLY_NOT_TOTAL_FLOW',
       'coverage_break_unresolved':bool(_old and _now and (_now/_old>1.5 or _now/_old<.67))})
qa_table('v7_od_common_support_sensitivity',pd.DataFrame(V7_OD_COMPARABLE),allow_empty=True)

# 5.2: полная поисковая проверка утверждаемых целевых индикаторов E02 (в отличие от контингента).
# Публикация названия, значения и числа наблюдений — доказательство кандидата, но НЕ его утверждение.
V7_ORG_ADMISSION_SEARCH=pd.DataFrame()
try:
    _v7_e02=fetch_asset(E02)
    with duck_connect() as _db:
        _v7_g=_db.execute('''SELECT CAST(indicator_name AS VARCHAR) indicator_name,
             CAST(indicator_unit AS VARCHAR) indicator_unit,
             COUNT(*)::BIGINT n_source_rows,COUNT(DISTINCT CAST(id AS VARCHAR)) n_organization_ids,
             COUNT(DISTINCT TRY_CAST(year AS INT)) n_years,
             COUNT(*) FILTER (WHERE TRY_CAST(indicator_value AS DOUBLE) IS NOT NULL) numeric_rows,
             COUNT(*) FILTER (WHERE TRY_CAST(indicator_value AS DOUBLE)<0) negative_rows,
             COUNT(DISTINCT COALESCE(CAST(area AS VARCHAR),'__NULL__')) n_area_values
          FROM read_parquet(?) WHERE regexp_matches(lower(CAST(indicator_name AS VARCHAR)),
            'принят|приём|прием|зачислен|поступив') GROUP BY 1,2''',[str(_v7_e02)]).df()
    _v7_g['indicator_approved']=_v7_g.indicator_name.isin(tuple(CFG.APPROVED_VO_ADMISSION_NAMES))
    _v7_g['has_organization_year_provenance']=_v7_g.n_organization_ids.gt(0)&_v7_g.n_years.ge(2)
    _v7_g['is_certified_admission_total']=False  # name matching and counts never certify statistical semantics
    _v7_g['gate_reason']='REQUIRES_PUBLISHED_METHOD_AND_UNIQUE_ORGANIZATION_YEAR_COHORT'
    V7_ORG_ADMISSION_SEARCH=_v7_g
except Exception as _ex:
    ISSUES.append({'module':'ML-5.2','status':'V7_ADMISSION_DISCOVERY_FAILURE',
                   'reason':type(_ex).__name__+': '+str(_ex)[:160],
                   'required':'repair E02 indicator inventory scan'})
    if CFG.MODE=='FULL': raise
qa_table('v7_organization_admission_discovery',V7_ORG_ADMISSION_SEARCH,allow_empty=True)

# 5.7: строго показываем, почему E0/E1 с учебными площадками выполним или нет.
_v7_muni_status=globals().get('V4_MUNICIPAL_STATUS',{})
_v7_spatial=globals().get('V5_MUNICIPAL_CROSS_STATUS',{})
_v7_campus_n=int(REGISTRY.campus_geo_status.eq('CONFIRMED_STUDY_CAMPUS').sum()) if len(REGISTRY) else 0
V7_MUNICIPAL_SUPPORT=pd.DataFrame([{
    'module':'ML-5.7','municipal_demography_rows':int(len(MUNI)),
    'bdmo_proxy_rows':int(len(BDMO_EDU_PROXY_CANDIDATES)),
    'verified_education_campus_records':_v7_campus_n,
    'temporal_e0e1_status':str(_v7_muni_status.get('status','NOT_VERIFIED')),
    'spatial_e0e1_status':str(_v7_spatial.get('status','NOT_VERIFIED')),
    'original_campus_e0e1_identifiable':bool(_v7_campus_n>0 and MODULES['ML-5.7']['status'].startswith('TESTED')),
    'interpretation':'MUNICIPAL_PROXIES_NOT_IDENTICAL_TO_HIGHER_EDUCATION_CAMPUS_CAPACITY',
    'requirement':'VERIFIED_CAMPUS_OKTMO_BY_YEAR_AND_COMPARABLE_MUNICIPAL_OUTCOME'
}])
qa_table('v7_municipal_identification_gate',V7_MUNICIPAL_SUPPORT,['module'])
print('V7: общая поддержка OD',len(V7_OD_COMPARABLE),'пар лет; кандидатов E02',len(V7_ORG_ADMISSION_SEARCH),
      '; подтверждённых учебных площадок',_v7_campus_n)


## 9Е. Дополнительные проверяемые эксперименты V7.1
При наличии подтверждённого целевого показателя проверяется новый временной прогноз; при отсутствии цель исключается без искусственных значений. Сохраняются все ранее выполненные модели V7.

In [ ]:
# ML-05 V7.1.1 · All defensible new tests, retaining the completed V7 seven-module models.
progress_event('EXTENDED_MODULES','V7.1: новые проверяемые подэксперименты ML-5.1–ML-5.7',announce=True)
ADR_EXPERIMENTS=[]

def adr_record_exp(module,code,status,n_train=0,n_test=0,reason='',source_ids='',scientific_claim=''):
    ADR_EXPERIMENTS.append({'module':module,'experiment':code,'status':status,
        'train_rows':int(n_train),'test_rows':int(n_test),'source_ids':source_ids,
        'scientific_claim':scientific_claim,'reason':str(reason)[:500]})

# ML-5.1/3/6: independent stability diagnostics use the same recorded source E01 only.
if len(ADR_OD_PAIRWISE):
    _adr_break=ADR_OD_PAIRWISE.loc[ADR_OD_PAIRWISE.coverage_break_at_2023.eq(True)]
    _adr_ok_years=ADR_OD_PAIRWISE.loc[(ADR_OD_PAIRWISE.coverage_of_prior>=.8)&
                                      (ADR_OD_PAIRWISE.coverage_of_current>=.8)].copy()
    qa_table('adr_od_high_overlap_year_comparisons',_adr_ok_years,allow_empty=True)
    adr_record_exp('ML-5.1','OD_HIGH_OVERLAP_STABILITY','DESCRIPTIVE_COMPLETED',
        len(ADR_OD_BALANCED),len(_adr_ok_years),
        '2023 coverage break recorded separately; no missing OD zero replacement',E01['asset_id'],
        'Only the intersection of actually reported regional schooling-study edges')
    adr_record_exp('ML-5.3','OUTFLOW_BALANCED_OD_SUPPORT','DESCRIPTIVE_COMPLETED',
        len(ADR_OD_BALANCED),len(_adr_ok_years),'OD annual marginal totals not comparable without common support',E01['asset_id'],
        'Observed outflow from prior schooling region, not permanent population outmigration')
    adr_record_exp('ML-5.6','NETWORK_BALANCED_GRAPH_SUPPORT','DESCRIPTIVE_COMPLETED',
        len(ADR_OD_BALANCED),len(_adr_ok_years),'Temporal graph robustness restricted to comparable reported edges',E01['asset_id'],
        'Observed OD network topology, not residential migration infrastructure')
else:
    adr_record_exp('ML-5.1','OD_HIGH_OVERLAP_STABILITY','SKIPPED_NO_MATCHED_YEARS',reason='No contiguous years of observed OD pairs',source_ids=E01['asset_id'])

# Additional testable V7.1 sensitivity: fixed-source common-edge (and origin marginal) time forecasts.
# Each protocol has a separate artifact ID and only uses observed rows; no TEST-based model choice.
def adr_balanced_time_experiment(module,observation,entity_fields,target,sector,target_id,protocol,min_train=100,min_test=20):
    if not len(observation):
        adr_record_exp(module,protocol,'SKIPPED_NO_COMMON_SOURCE_EDGES',source_ids=E01['asset_id'])
        return
    _f=historical_pair_panel(observation,entity_fields,target)
    if _f.empty:
        adr_record_exp(module,protocol,'SKIPPED_NO_HISTORICAL_LAG',source_ids=E01['asset_id'])
        return
    _f['entity_id']=_f[entity_fields].astype(str).agg('|'.join,axis=1)
    _f['sector']=sector;_f['target_id']=target_id
    _split,_reason=temporal_partition(_f,min_train=min_train,min_test=min_test)
    if _split is None:
        adr_record_exp(module,protocol,'SKIPPED_TIME_SUPPORT',len(_f),0,_reason,E01['asset_id'])
        return
    _tr=_split['train'];_te=_split['test']
    _models=['persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm']
    # PPML/NB are already evaluated exhaustively in the primary V7 OD protocol;
    # the balanced sensitivity adds independent, predeclared non-count-specific checks.
    _before=len(METRICS)
    for _model in _models:
        progress_event('EXTENDED_MODULES',f'{protocol}: {_model}',announce=True)
        fit_and_record(module,_model,_tr,_te,seed=-1,protocol=protocol,
                       nonnegative=True,allow_ppml=False)
    adr_record_exp(module,protocol,'MODEL_COMPLETED',len(_tr),len(_te),
        f'{len(METRICS)-_before} predeclared methods; no selection from TEST',E01['asset_id'],
        'Observed stable OD subpopulation, not a national movement count or source-year admission')

if len(ADR_OD_BALANCED):
    _adr_observed=ADR_OD_BALANCED[['observation_year','origin_region','destination_region','observed_value']].copy()
    adr_balanced_time_experiment('ML-5.1',_adr_observed,['origin_region','destination_region'],
        'observed_value','SPO_BALANCED_OD','E01_BALANCED_OBSERVED_EDGES','ADR_BALANCED_OD_TIME',100,20)
    _adr_balanced_outflow=_adr_observed.groupby(['observation_year','origin_region'],as_index=False).agg(
        balanced_observed_outflow=('observed_value','sum'))
    adr_balanced_time_experiment('ML-5.3',_adr_balanced_outflow,['origin_region'],
        'balanced_observed_outflow','SPO_BALANCED_ORIGIN','E01_BALANCED_ORIGIN_OUTFLOW','ADR_BALANCED_OUTFLOW_TIME',
        CFG.REGION_MIN_TRAIN_ROWS,CFG.REGION_MIN_TEST_ROWS)
else:
    for _m,_protocol in [('ML-5.1','ADR_BALANCED_OD_TIME'),('ML-5.3','ADR_BALANCED_OUTFLOW_TIME')]:
        adr_record_exp(_m,_protocol,'SKIPPED_NO_ALL_YEAR_OD_SUPPORT',0,0,
          'No OD edge appears in every source year; no imputation',E01['asset_id'])

# ML-5.2: actual admissions are modeled only if official indicator labels have been explicitly approved
# and the organizational-year target has no specialty/area duplication.
if len(ADR_ADMISSION_YEAR):
    _adr_approved_frame=ADR_ADMISSION_YEAR[['entity_id','observation_year','actual']].rename(columns={'actual':'observed_admitted'}).copy()
    _adr_org_frame=historical_pair_panel(_adr_approved_frame,['entity_id'],'observed_admitted')
    if len(_adr_org_frame):
        _adr_org_frame['sector']='VO_VERIFIED_ADMISSION'
        _adr_org_frame['target_id']='CERTIFIED_VO_UNIQUE_ORG_YEAR_ADMISSIONS'
        _adr_org_frame['parent_group_id']=_adr_org_frame.entity_id
        _adr_org_frame['historical_target_has_semantic_approval']=True
        _adr_split,_adr_split_reason=temporal_partition(_adr_org_frame,min_train=CFG.MIN_TRAIN_ROWS,min_test=CFG.MIN_TEST_ROWS)
        if _adr_split is not None:
            _adr_ex_before=len(METRICS)
            test_predictive('ML-5.2',_adr_org_frame,nonnegative=True,count_confirmed=True)
            _adr_new_models=len(METRICS)-_adr_ex_before
            adr_record_exp('ML-5.2','APPROVED_VO_ADMISSIONS_RETROSPECTIVE','MODEL_COMPLETED',
                len(_adr_split['train']),len(_adr_split['test']),f'{_adr_new_models} new metric rows',E02['asset_id'],
                'Source-approved admissions by organization and year; not student-stock or applications')
        else:
            adr_record_exp('ML-5.2','APPROVED_VO_ADMISSIONS_RETROSPECTIVE','SKIPPED_TIME_SUPPORT',
                len(_adr_org_frame),0,_adr_split_reason,E02['asset_id'])
    else:
        adr_record_exp('ML-5.2','APPROVED_VO_ADMISSIONS_RETROSPECTIVE','SKIPPED_NO_HISTORICAL_PAIRS',
            len(ADR_ADMISSION_YEAR),0,'No historically available count lag',E02['asset_id'])
else:
    adr_record_exp('ML-5.2','APPROVED_VO_ADMISSIONS_RETROSPECTIVE','SKIPPED_SEMANTICALLY_UNVERIFIED',
        0,0,'Published source data did not certify unique admissions by organization-year; V7 contingent experiment is retained',E02['asset_id'],
        'Candidate labels cannot be converted into real admissions without verified definition')

# ML-5.4/5.5 existing V7 tests already use all matching raw source rows; no invented additional target.
for _adr_mod,_adr_exp,_adr_asset in [('ML-5.4','FULL_SPECIALTY_STRATA_MODEL',VPO_211['asset_id']),
     ('ML-5.5','ALL_FOUR_GRADUATE_FAMILIES',GRAD_ORG['asset_id'])]:
    _adr_success=bool(any(r.get('module')==_adr_mod for r in MODEL_RECORDS) or
                      (_adr_mod=='ML-5.5' and len(PANELS.get('graduate_descriptive_statistics',pd.DataFrame()))))
    adr_record_exp(_adr_mod,_adr_exp,'V7_COMPLETED' if _adr_success else 'SKIPPED_UNAVAILABLE_TARGET',
        reason='Existing V7 observational tests retained; no graduate job-location inference',source_ids=_adr_asset)

# ML-5.7: optional independent E0/E1 *associational* forecasting on geocoded SPO proxies.
# Only available observed SPO institution-years are used. Neither absent organizations nor sites are filled with zero.
ADR_ML57_GEO_STATUS={'status':'SKIPPED_NO_SPATIAL_PROXY','n_spo_location_proxies':len(ADR_GEO_VERIFIED)}
if len(ADR_GEO_VERIFIED):
    _adr_sites=ADR_GEO_VERIFIED[['entity_id','observation_year','candidate_municipal_oktmo']].copy()
    _adr_sites['oktmo']=norm_code(_adr_sites.candidate_municipal_oktmo)
    _adr_sites['observation_year']=safe_year(_adr_sites.observation_year)
    _adr_sites=_adr_sites.dropna(subset=['oktmo','observation_year']).drop_duplicates()
    _adr_site_counts=_adr_sites.groupby(['oktmo','observation_year'],as_index=False).agg(
        observed_geocoded_spo_entities=('entity_id','nunique'))
    qa_table('adr_spo_site_year_observed_counts',_adr_site_counts,['oktmo','observation_year'],allow_empty=True)
    _adr_m=MUNI[['oktmo','observation_year','population','migration_rate']].copy()
    _adr_m['oktmo']=norm_code(_adr_m.oktmo)
    _adr_m['observation_year']=safe_year(_adr_m.observation_year)
    _adr_m['actual']=number(_adr_m.migration_rate)
    _adr_m['population']=number(_adr_m.population)
    _adr_m=_adr_m.dropna(subset=['oktmo','observation_year','actual']).drop_duplicates(['oktmo','observation_year'],keep=False)
    _adr_m['target_year']=_adr_m.observation_year.astype('Int64')
    _adr_m['origin_year']=_adr_m.target_year-CFG.HORIZON
    _adr_m['feature_year']=_adr_m.origin_year-CFG.FEATURE_PUBLICATION_LAG_YEARS
    _adr_lags=_adr_m[['oktmo','observation_year','actual','population']].rename(columns={
        'observation_year':'feature_year','actual':'lag_migration_rate','population':'lag_population'})
    _adr_m=_adr_m.merge(_adr_lags,on=['oktmo','feature_year'],how='inner',validate='many_to_one')
    _adr_sites_lag=_adr_site_counts.rename(columns={'observation_year':'feature_year'})
    _adr_m=_adr_m.merge(_adr_sites_lag,on=['oktmo','feature_year'],how='inner',validate='many_to_one')
    _adr_m=_adr_m.dropna(subset=['lag_migration_rate','lag_population','observed_geocoded_spo_entities'])
    _adr_m['entity_id']='MUNICIPAL|'+_adr_m.oktmo.astype(str)
    _adr_m['sector']='SPO_SPATIAL_PROXY'
    _adr_m['target_id']='D01_OBSERVED_MUNICIPAL_MIGRATION_RATE'
    _adr_m['information_cutoff_year']=_adr_m.origin_year
    _adr_m['target_available_year']=_adr_m.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS
    _adr_m['feature_available_year']=_adr_m.feature_year+CFG.FEATURE_PUBLICATION_LAG_YEARS
    _adr_m['prior_value']=_adr_m.lag_migration_rate
    _adr_m['source_vintage']=INDEX['created_at']
    qa_table('adr_ml57_spo_proxy_common_support',_adr_m,allow_empty=True)
    _adr_years57=sorted(_adr_m.target_year.dropna().astype(int).unique()) if len(_adr_m) else []
    if len(_adr_years57)>=2:
        _adr_testyear=max(_adr_years57)
        _adr_origin=_adr_testyear-CFG.HORIZON
        _adr_tr=_adr_m.loc[(_adr_m.target_year<_adr_testyear)&
             (_adr_m.target_available_year<=_adr_origin)].copy()
        _adr_te=_adr_m.loc[_adr_m.target_year.eq(_adr_testyear)].copy()
        if len(_adr_tr)>=80 and len(_adr_te)>=CFG.MIN_MUNICIPALITIES_E0E1 and _adr_tr.observed_geocoded_spo_entities.nunique()>1:
            _adr_results=[]
            for _variant,_xcols in [('E0',['lag_migration_rate','lag_population']),
                                     ('E1',['lag_migration_rate','lag_population','observed_geocoded_spo_entities'])]:
                _fit=Pipeline([('imputer',SimpleImputer(strategy='median')),
                   ('scale',StandardScaler()),('ridge',Ridge(alpha=20.0))])
                _fit.fit(_adr_tr[_xcols].to_numpy(float),_adr_tr.actual.to_numpy(float))
                _pred=_fit.predict(_adr_te[_xcols].to_numpy(float))
                _saved=ROOT/'models'/f'adr_ml57_spo_geocode_{_variant}.joblib'
                if _saved.name in MODEL_FILES:raise RuntimeError('ADR ML57 artifact name collision')
                joblib.dump({'model':_fit,'features':_xcols,'scientific_scope':'SPO_GEO_PROXY_ASSOCIATION'},_saved,compress=3)
                if not np.allclose(joblib.load(_saved)['model'].predict(_adr_te[_xcols].to_numpy(float)),_pred):
                    raise RuntimeError('ADR ML57 model reload mismatch')
                MODEL_FILES[_saved.name]=str(_saved)
                MODEL_RECORDS.append({'module':'ML-5.7','sector':'SPO_SPATIAL_PROXY',
                    'model':'adr_'+_variant,'protocol':'SPATIAL_PROXY_TEMPORAL_ASOF_NOT_CAUSAL',
                    'seed':-1,'path':_saved.name,'file_sha256':hash_file(_saved),'feature_num':_xcols,
                    'feature_cat':[],'training_rows':len(_adr_tr),'test_rows':len(_adr_te),'reloaded':True})
                _pf=pd.DataFrame({'entity_id':_adr_te.entity_id.to_numpy(),
                     'origin_year':_adr_te.origin_year.to_numpy(),'target_year':_adr_te.target_year.to_numpy(),
                     'information_cutoff_year':_adr_te.information_cutoff_year.to_numpy(),
                     'feature_available_year':_adr_te.feature_available_year.to_numpy(),
                     'target_available_year':_adr_te.target_available_year.to_numpy(),
                     'sector':'SPO_SPATIAL_PROXY','target_id':'D01_OBSERVED_MUNICIPAL_MIGRATION_RATE',
                     'actual':_adr_te.actual.to_numpy()})
                _predrows=predictions_frame(_pf,'ML-5.7','adr_'+_variant,_pred,protocol='SPATIAL_PROXY_TEMPORAL_ASOF_NOT_CAUSAL')
                _predrows['geography_grain']='MUNICIPALITY';record_prediction(_predrows)
                _met=effective_mae(_adr_te.actual,_pred)
                METRICS.append({'module':'ML-5.7','sector':'SPO_SPATIAL_PROXY','model':'adr_'+_variant,
                    'protocol':'SPATIAL_PROXY_TEMPORAL_ASOF_NOT_CAUSAL','seed':-1,'target_year':_adr_testyear,**_met})
                _adr_results.append({'variant':_variant,'n_train':len(_adr_tr),'n_test':len(_adr_te),**_met})
            qa_table('adr_ml57_spatial_e0e1_metrics',pd.DataFrame(_adr_results),allow_empty=True)
            ADR_ML57_GEO_STATUS={'status':'TESTED_ASSOCIATION_NOT_CONFIRMED_CAMPUS',
                'n_train':len(_adr_tr),'n_test':len(_adr_te),'n_spo_location_proxies':len(ADR_GEO_VERIFIED),
                'MAE_E0':_adr_results[0]['MAE'],'MAE_E1':_adr_results[1]['MAE']}
            adr_record_exp('ML-5.7','GEO_RECONSTRUCTED_SPO_E0_E1','MODEL_COMPLETED',len(_adr_tr),len(_adr_te),
                 'Same-support as-of tests; SPO site count only a proximity proxy',
                 SPO_ORG['asset_id']+';'+SPATIAL['asset_id']+';'+MUN['asset_id'],
                 'Predictive association of reported proxy, not causal effect or confirmed campus exposure')
        else:
            ADR_ML57_GEO_STATUS={'status':'SKIPPED_INSUFFICIENT_TIME_OR_SPATIAL_SUPPORT',
              'n_train':len(_adr_tr),'n_test':len(_adr_te)}
    if ADR_ML57_GEO_STATUS.get('status','').startswith('SKIPPED'):
        adr_record_exp('ML-5.7','GEO_RECONSTRUCTED_SPO_E0_E1',ADR_ML57_GEO_STATUS['status'],
           reason='No as-of historical support for unique high-confidence geocoded SPO entities',
           source_ids=SPO_ORG['asset_id']+';'+SPATIAL['asset_id']+';'+MUN['asset_id'])
else:
    adr_record_exp('ML-5.7','GEO_RECONSTRUCTED_SPO_E0_E1','SKIPPED_NO_MATCHED_SPATIAL_SITES',
       reason='No high-confidence spatial links in the actual source; no fabricated campus locations',
       source_ids=SPO_ORG['asset_id']+';'+SPATIAL['asset_id'])

qa_table('adr_ml57_geographic_execution',pd.DataFrame([ADR_ML57_GEO_STATUS]),allow_empty=True)
qa_table('adr_all_feasible_experiments',pd.DataFrame(ADR_EXPERIMENTS),allow_empty=True)
print('V7.1 reconstruction experiments:',len(ADR_EXPERIMENTS),'| new model procedures:',
      sum(x['status']=='MODEL_COMPLETED' for x in ADR_EXPERIMENTS), '| retained all V7 modules')

mark_stage('EXTENDED_MODULES')


## 10. Независимые метрики, canary, статистика
Метрики пересчитываются по прогнозным строкам. При отсутствии законного DEV сравнения не становятся подтверждающими. Отрицательные результаты сохраняются.

In [ ]:
progress_event('STATISTICS','Независимые метрики, канарейки, проверки гипотез')
# Независимые сквозные проверки: источники/будущее/статистика/содержательные выводы.
CANARIES=[]

def test_future_mutation(raw,keys,value_col,module):
    if raw.empty or 'observation_year' not in raw or value_col not in raw:
        return {'module':module,'status':'NOT_APPLICABLE','reason':'Нет валидной временной панели'}
    r=raw[keys+['observation_year',value_col]].dropna(subset=['observation_year']).copy()
    if r.empty or r.observation_year.nunique()<2:return {'module':module,'status':'NOT_APPLICABLE','reason':'Мало лет'}
    r=r.drop_duplicates(keys+['observation_year'])
    if r.empty:return {'module':module,'status':'NOT_APPLICABLE','reason':'Нет уникальных ключей'}
    p1=historical_pair_panel(r,keys,value_col)
    mx=int(r.observation_year.max());r2=r.copy()
    r2.loc[r2.observation_year.eq(mx),value_col]=number(r2.loc[r2.observation_year.eq(mx),value_col]).fillna(0)+10**8
    p2=historical_pair_panel(r2,keys,value_col)
    # Проверяется строгое отсутствие будущих лагов; отдельный тест экономических и сетевых join — ниже.
    if len(p1) and (p1.feature_available_year>p1.information_cutoff_year).any():
        fail('CANARY','Исходная панель содержит будущие признаки: '+module)
    # До мутации будущего итоговые X прежних origins должны быть идентичны.
    common=[c for c in keys+['target_year','prior_value','prior2_value','historical_year'] if c in p1 and c in p2]
    early1=p1[p1.target_year<mx][common].sort_values(keys+['target_year']).reset_index(drop=True)
    early2=p2[p2.target_year<mx][common].sort_values(keys+['target_year']).reset_index(drop=True)
    if not early1.equals(early2):fail('CANARY','Мутация будущего изменила прошлые признаки: '+module)
    return {'module':module,'status':'PASS','n_untouched':len(early1),'mutated_year':mx}

CANARIES.append(test_future_mutation(OD_INTER,['origin_region','destination_region'],'observed_value','ML-5.1'))
CANARIES.append(test_future_mutation(OUTFLOW,['origin_region'],'observed_interregional_routes','ML-5.3'))
if len(ORG_YEAR):CANARIES.append(test_future_mutation(ORG_YEAR,['sector','entity_id'],'actual','ML-5.2'))
if len(PROGRAM_PROFILE):
    prog_test=PROGRAM_PROFILE.drop_duplicates(['specialty_code','indicator_name','observation_year'])
    CANARIES.append(test_future_mutation(prog_test,['specialty_code','indicator_name'],'observed_value','ML-5.4'))
if len(GRADUATES):
    CANARIES.append({'module':'ML-5.5','status':'DESCRIPTIVE_ONLY',
      'reason':'Единственная дата измерения занятости; модель предсказаний не запускалась'})
if len(MUNI):CANARIES.append(test_future_mutation(MUNI,['oktmo'],'migration_rate','ML-5.7'))
CANARIES.append({'module':'ECONOMIC_CONTEXT','status':'STATIC_OR_ASOF_AUDIT',
 'reason':'Использована только экономическая панель observation_year=historical_year; future rows not joined'})
if len(OD_MODEL):
    # Отдельный тест исходных маргиналей: признаки прошлого года должны совпадать с исходными агрегатами.
    src_mass=OD_INTER.groupby(['observation_year','origin_region'],as_index=False).observed_value.sum()
    tmp=OD_MODEL[['origin_region','historical_year','origin_mass']].dropna().drop_duplicates()
    exp=src_mass.rename(columns={'observation_year':'historical_year','observed_value':'expected_mass'})
    comp=tmp.merge(exp,on=['historical_year','origin_region'],how='left',validate='many_to_one')
    if len(comp) and not np.allclose(comp.origin_mass,comp.expected_mass,equal_nan=True):
        fail('CANARY','Региональные маргинали не совпали с историческим исходным годом')
    CANARIES.append({'module':'OD_MARGINS','status':'PASS' if len(comp) else 'NOT_APPLICABLE',
       'verified_origin_year_margins':int(len(comp))})
qa_table('future_mutation_canaries',pd.DataFrame(CANARIES),allow_empty=True)

# Весь стек прогнозов, включая 50 сидов, объединяется НА ДИСКЕ.
PRED=pd.DataFrame();RECALCULATED=[];PRED_STORAGE_FILE=None
if PRED_PARTS:
    progress_event('STATISTICS',f'Независимая проверка {PRED_TOTAL:,} строк из {len(PRED_PARTS)} файлов',announce=True)
    PRED_STORAGE_FILE=ROOT/'derived'/'education_predictions.parquet'
    # read_parquet(list) обрабатывается потоком в DuckDB, не через pandas.concat.
    with duck_connect() as con:
        _files='['+','.join("'"+str(p).replace("'","''")+"'" for p in PRED_PARTS)+']'
        _q=f"SELECT * FROM read_parquet({_files},union_by_name=true)"
        _fp=str(PRED_STORAGE_FILE).replace("'","''")
        con.execute(f"COPY ({_q}) TO '{_fp}' (FORMAT PARQUET, COMPRESSION ZSTD)")
        _num=con.execute('SELECT COUNT(*) FROM read_parquet(?)',[str(PRED_STORAGE_FILE)]).fetchone()[0]
        if _num!=PRED_TOTAL:fail('REPRO',f'Прогнозов в Parquet {_num} вместо {PRED_TOTAL}')
        _invalid=con.execute('''SELECT COUNT(*) FROM read_parquet(?)
            WHERE predicted IS NULL OR actual IS NULL OR NOT isfinite(predicted) OR NOT isfinite(actual)
            OR feature_available_year>information_cutoff_year''',[str(PRED_STORAGE_FILE)]).fetchone()[0]
        if _invalid:fail('REPRO',f'Невалидных прогнозов: {_invalid}')
        _duplicates=con.execute('''SELECT COUNT(*) FROM (
            SELECT module,sector,entity_id,target_year,model,seed,split_id,COUNT(*) AS n
            FROM read_parquet(?) GROUP BY ALL HAVING COUNT(*)>1)''',[str(PRED_STORAGE_FILE)]).fetchone()[0]
        if _duplicates:fail('SUPPORT',f'Повторные натуральные ключи прогнозов: {_duplicates}')
        # Точные MAE/RMSE/WAPE по ВСЕМ строкам всех файлов.
        RECALCULATED=con.execute('''SELECT module,sector,model,seed,target_year,split_id,
            AVG(ABS(actual-predicted)) AS MAE,
            SQRT(AVG(POWER(actual-predicted,2))) AS RMSE,
            SUM(ABS(actual-predicted))/NULLIF(SUM(ABS(actual)),0) AS WAPE,
            COUNT(*)::BIGINT AS n
            FROM read_parquet(?) GROUP BY 1,2,3,4,5,6''',[str(PRED_STORAGE_FILE)]).df().to_dict('records')
        PRED=con.execute('SELECT * FROM read_parquet(?) LIMIT 2000',[str(PRED_STORAGE_FILE)]).df()
    PANEL_FILES['education_predictions']=PRED_STORAGE_FILE
    final=pd.DataFrame(RECALCULATED)
    for r in final.itertuples(index=False):
        original=[m for m in METRICS if m['module']==r.module and m.get('sector')==r.sector
                  and m['model']==r.model and m['seed']==r.seed and m['target_year']==r.target_year and m.get('protocol')==r.split_id]
        if not original:fail('REPRO','Нет строки метрики для '+r.module+' '+r.model)
        for m in original:
            if not np.isclose(m['MAE'],r.MAE,atol=1e-7,rtol=1e-7):fail('REPRO','MAE не совпал: '+r.module+' '+r.model)
qa_table('metrics_recomputed_from_predictions',pd.DataFrame(RECALCULATED),allow_empty=True)

# Предрегистрация Holm: только подтверждающие сравнения, для которых есть валидный p.
CONFIRM=pd.DataFrame(CONFIRMATORY)
if not CONFIRM.empty:
    valid=CONFIRM.p_value.notna()&np.isfinite(CONFIRM.p_value)
    CONFIRM['p_holm']=np.nan
    if valid.any():
        CONFIRM.loc[valid,'p_holm']=multipletests(CONFIRM.loc[valid,'p_value'].to_numpy(float),
                                                method='holm')[1]
    CONFIRM['interpretation']=np.where(CONFIRM.delta_mae.lt(0),'LOWER_MAE_ON_SELECTED_COMMON_SUPPORT',
                                      np.where(CONFIRM.delta_mae.gt(0),'HIGHER_MAE_ON_SELECTED_COMMON_SUPPORT','EQUAL_MAE'))
    CONFIRM['historical_vintage_validated']=False
    CONFIRM['claim_level']='EXPLORATORY_RETROSPECTIVE_SOURCE_2026'

qa_table('confirmatory_comparisons',CONFIRM,allow_empty=True)

# Доказательный реестр 7 гипотез: отдельно результат и предел идентификации.
for i in range(1,8):
    module=f'ML-5.{i}'
    status=MODULES[module]
    subset=CONFIRM[CONFIRM.module.eq(module)] if len(CONFIRM) else pd.DataFrame()
    if len(subset):
        r=subset.iloc[0]
        if r.choice_basis!='DEV_ASOF':
            conclusion='Подтверждающий отбор невозможен: DEV-метки недоступны на момент TEST; сравнения моделей разведочные'
            metric=f'baseline_MAE={r.baseline_mae:.5g}; confirmatory_delta=NOT_APPLICABLE'
        else:
            conclusion=('выбранная на DEV модель не превзошла инерцию на ретроспективном тесте' if r.delta_mae>=0
                        else 'в ретроспективном тесте MAE ниже инерции; исторические версии данных не подтверждены')
            metric=f'baseline_MAE={r.baseline_mae:.5g}; selected_MAE={r.selected_mae:.5g}; delta={r.delta_mae:+.5g}'
    else:conclusion=status['reason'];metric='Нет корректного независимого прогнозного теста'
    HYPOTHESES.append({'hypothesis':f'H5.{i}','module':module,'status':status['status'],
        'metric_or_basis':metric,'scientific_conclusion':conclusion,
        'source_observability':('E01_schooling_to_study_NOT_residence' if i in (1,3,6)
             else 'source_specific_unverified_retention' if i==5 else 'source_specific'),
        'causal_identification':'NOT_ESTABLISHED'})
qa_table('seven_hypotheses',pd.DataFrame(HYPOTHESES),['hypothesis'])
qa_table('split_audit',pd.DataFrame(SPLIT_AUDIT),allow_empty=True)
qa_table('model_metrics',pd.DataFrame(METRICS),allow_empty=True)
qa_table('model_registry',pd.DataFrame(MODEL_RECORDS),allow_empty=True)

# Статусы: техническая полнота != научное подтверждение семи гипотез.
executed_all=all(m['status']!='NOT_STARTED' for m in MODULES.values())
failed_modules=[k for k,m in MODULES.items() if m['status']=='FAILED']
research_full=all(m['status']=='TESTED_FULL' for m in MODULES.values()) and OD_SEMANTIC['yearly_count_unit_confirmed']
STATE['technical']='PASS_ASSEMBLED' if executed_all and not failed_modules else 'FAIL'
STATE['scientific']='FULL_RESEARCH_PASS' if research_full and CFG.MODE=='FULL' else (
    'FULL_EXECUTION_WITH_LIMITATIONS' if CFG.MODE=='FULL' and executed_all and not failed_modules else 'PILOT_OR_PARTIAL')
STATE['overall']=STATE['scientific']

print('Technical / scientific statuses:',STATE['technical'],STATE['scientific'])
print('Confirmed comparisons:',len(CONFIRM),'| exported predictions:',PRED_TOTAL)

# Финальная отметка этапа перенесена после всех его подмодулей


## 10А. Парные сравнения и проверка завершённости программы
Разделение вычислительной полноты и научной идентифицируемости; отсутствующие измерения не фабрикуются.

In [ ]:
# V4 · audit model completion independently of the notebook's technical PASS.
progress_event('STATISTICS','V4: численные сравнения на общих ключах, матрица научной полноты',announce=True)
V4_PAIRED_COMPARISONS=[]
if PRED_STORAGE_FILE and Path(PRED_STORAGE_FILE).is_file():
    with duckdb.connect(':memory:') as _db:
        _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
        _unique=_db.execute('''SELECT DISTINCT module,sector,split_id,model FROM read_parquet(?) ORDER BY 1,2,3,4''',
                           [str(PRED_STORAGE_FILE)]).df()
        _combos=_unique[~_unique.model.isin(['persistence','rolling_mean_2','damped_trend','catboost'])].copy()
        for _rec in _combos.itertuples(index=False):
            _pred=_db.execute('''WITH b AS (SELECT entity_id,target_year,actual,predicted baseline,
                     origin_region,destination_region,parent_group_id FROM read_parquet(?)
                  WHERE module=? AND sector=? AND split_id=? AND model='persistence' AND seed=-1),
                 v AS (SELECT entity_id,target_year,actual,predicted as candidate FROM read_parquet(?)
                  WHERE module=? AND sector=? AND split_id=? AND model=?)
                  SELECT b.*,v.candidate FROM b JOIN v USING(entity_id,target_year)
                  WHERE ABS(b.actual-v.actual)<1e-7''',
               [str(PRED_STORAGE_FILE),_rec.module,_rec.sector,_rec.split_id,
                str(PRED_STORAGE_FILE),_rec.module,_rec.sector,_rec.split_id,_rec.model]).df()
            if not len(_pred):continue
            _a=np.abs(_pred.actual.to_numpy(float)-_pred.baseline.to_numpy(float))
            _v=np.abs(_pred.actual.to_numpy(float)-_pred.candidate.to_numpy(float))
            if _rec.module=='ML-5.1' and {'origin_region','destination_region'}.issubset(_pred):
                _g1=_pred.origin_region.fillna('__UNK__').astype(str)
                _g2=_pred.destination_region.fillna('__UNK__').astype(str)
                _ci=two_way_mae_bootstrap(_a,_v,_g1,_g2,reps=CFG.BOOTSTRAP_REPS,seed=CFG.RANDOM_SEED)
            else:
                _g1=(_pred.parent_group_id.fillna(_pred.entity_id).astype(str) if
                     'parent_group_id' in _pred else _pred.entity_id.astype(str))
                # Only one test target_year; two-way geographic-year bootstrap not identified.
                _unique_groups=_g1.nunique()
                if _unique_groups>=8:
                    _rng=np.random.default_rng(CFG.RANDOM_SEED)
                    _groups=_g1.to_numpy();_base=np.asarray(_v-_a,float)
                    _levels=np.unique(_groups);_boot=[]
                    for _ in range(CFG.BOOTSTRAP_REPS):
                        _mult=Counter(_rng.choice(_levels,len(_levels),replace=True))
                        _weights=np.fromiter((_mult[v] for v in _groups),dtype=float,count=len(_groups))
                        if _weights.sum()>0:_boot.append(float(np.average(_base,weights=_weights)))
                    _vals=np.asarray(_boot)
                    _ci={'delta_MAE':float(np.mean(_base)),'ci_low':float(np.quantile(_vals,.025)) if len(_vals)>20 else np.nan,
                         'ci_high':float(np.quantile(_vals,.975)) if len(_vals)>20 else np.nan,
                         'p_value':min(1.0,2*min((np.count_nonzero(_vals<=0)+1)/(len(_vals)+1),
                            (np.count_nonzero(_vals>=0)+1)/(len(_vals)+1))) if len(_vals)>20 else np.nan,
                         'clusters_a':_unique_groups,'clusters_b':None}
                else:_ci={'delta_MAE':float(np.mean(_v-_a)),'ci_low':np.nan,'ci_high':np.nan,
                           'p_value':np.nan,'clusters_a':_unique_groups,'clusters_b':None}
            V4_PAIRED_COMPARISONS.append({'module':_rec.module,'sector':_rec.sector,
                'split_id':_rec.split_id,'model':_rec.model,'n_common_keys':len(_pred),
                'persistence_MAE':float(_a.mean()),'candidate_MAE':float(_v.mean()),**_ci,
                'inference_type':'EXPLORATORY_PAIRED_RETROSPECTIVE_NOT_CONFIRMATORY',
                'historical_source_vintage_at_prediction':'NOT_ARCHIVED'})
_V4_BOOT=pd.DataFrame(V4_PAIRED_COMPARISONS)
if len(_V4_BOOT):
    _mask=_V4_BOOT['p_value'].notna()&np.isfinite(_V4_BOOT['p_value'])
    _V4_BOOT['p_holm_exploratory']=np.nan
    if _mask.any():_V4_BOOT.loc[_mask,'p_holm_exploratory']=multipletests(
        _V4_BOOT.loc[_mask,'p_value'].to_numpy(float),method='holm')[1]
qa_table('v4_exploratory_paired_comparisons',_V4_BOOT,allow_empty=True)

V4_METHOD_EXECUTIONS=[]
# Check every required model separately, not merely that a module produced one metric.
_v4_contract_base={'persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm','catboost_ensemble'}
_v4_expected={}
for _m in ('ML-5.1','ML-5.3','ML-5.4'):
    _actual={z.get('model') for z in METRICS if z.get('module')==_m and z.get('protocol')=='ROLLING_TIME'}
    if _actual:
        _req=_v4_contract_base.copy()
        # PPML/NB only if their observed-source integer semantics passed the count gate.
        _frame=OD_MODEL if _m=='ML-5.1' else OUTFLOW_MODEL if _m=='ML-5.3' else None
        _count_eligible=bool(_frame is not None and len(_frame) and
             number(_frame.actual).ge(0).all() and
             np.isclose(number(_frame.actual),np.rint(number(_frame.actual)),atol=1e-8,rtol=0).all())
        if _m=='ML-5.4':
            _count_eligible=MODULES[_m]['status'].startswith('TESTED')
        if _count_eligible:_req.update({'ppml','negbin'})
        _v4_expected[_m]=_req
if any(z.get('module')=='ML-5.2' and z.get('protocol')=='ALT_CONTINGENT_RETROSPECTIVE' for z in METRICS):
    _v4_expected['ML-5.2']={'persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm'}
if any(z.get('module')=='ML-5.7' and z.get('protocol')=='RETROSPECTIVE_PROXY_E0E1' for z in METRICS):
    _v4_expected['ML-5.7']={'E0','E1'}
for _module in ['ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7']:
    _r=[m for m in METRICS if m.get('module')==_module]
    _failed=[x for x in ISSUES if x.get('module')==_module and x.get('status') in
            ('MODEL_FAILED','FAMILY_PROFILE_FAILED','SOURCE_PROFILE_FAILURE','SPATIAL_MODEL_FAILURE')]
    _expected=_v4_expected.get(_module,set())
    _actually={z.get('model') for z in _r}
    _missing=sorted(_expected-_actually)
    if _missing:_failed.append({'reason':'Feasible models without metrics: '+','.join(_missing),
                               'status':'METHOD_NOT_EXECUTED'})
    if _module=='ML-5.5' and V4_GRADUATE_STATUS.get('status')=='REGIONAL_HOLDOUT_EMPLOYMENT_TEST_NOT_RETENTION':
        if len(PANELS.get('v4_ml55_crosssection_metrics',pd.DataFrame()))<4:
            _failed.append({'reason':'E04 cross-sectional baseline/Ridge/LightGBM/CatBoost incomplete'})
    if _module=='ML-5.6' and len(NETWORK) and not len(PANELS.get('v4_ml56_node_forecast',pd.DataFrame())):
        _failed.append({'reason':'Node centrality persistence test missing'})
    _feasible=bool(_r) or MODULES[_module].get('status','').startswith('DESCRIPTIVE')
    V4_METHOD_EXECUTIONS.append({'module':_module,'module_status':MODULES[_module].get('status'),
        'models_with_metrics':len(set(x.get('model') for x in _r)),
        'metric_rows':len(_r),'saved_model_artifacts':sum(m.get('module')==_module for m in MODEL_RECORDS),
        'prediction_rows_validated':bool(any(x.get('module')==_module for x in RECALCULATED)),
        'required_models':','.join(sorted(_expected)), 'missing_required_models':','.join(_missing),
        'computational_failures':len(_failed),
        'failure_reason':' | '.join(str(x.get('reason',''))[:120] for x in _failed[:3]),
        'target_is_scientifically_identified':bool(_module in ('ML-5.4','ML-5.6')),
        'historical_vintage_validated':False,
        'scientific_completion':'RETROSPECTIVE_OR_DESCRIPTIVE_WITH_GATES'})
qa_table('v4_methods_execution_ledger',pd.DataFrame(V4_METHOD_EXECUTIONS),['module'])
V4_COMPUTATIONAL_COMPLETE=bool(not V4_SOURCE_ERRORS and not any(x['computational_failures'] for x in V4_METHOD_EXECUTIONS))
V4_SCIENTIFIC_COMPLETE=bool(V4_COMPUTATIONAL_COMPLETE and
                           all(MODULES[m]['status']=='TESTED_FULL' for m in MODULES) and
                           OD_SEMANTIC.get('yearly_count_unit_confirmed',False) and
                           len(ORG_YEAR)>0)
STATE['scientific']='FULL_RESEARCH_PASS' if V4_SCIENTIFIC_COMPLETE else (
    'FULL_COMPUTATIONS_WITH_DOCUMENTED_DATA_LIMITS' if V4_COMPUTATIONAL_COMPLETE and CFG.MODE=='FULL'
     else 'FULL_PARTIAL_COMPUTATIONS_OR_MODEL_FAILURES')
STATE['overall']=STATE['scientific']
qa_table('v4_scientific_acceptance',pd.DataFrame([{'version':CFG.VERSION,'source_assets':len(ASSETS),
    'value_scans_passed':sum(x['content_status']=='FULL_VALUE_COVERAGE_PASS' for x in V4_SOURCE_PROFILES),
    'source_assets_not_in_model':sum(not x['main_prediction_input'] for x in _v4_roles),
    'seven_modules_reported':len(MODULES),'computational_failures':sum(x['computational_failures'] for x in V4_METHOD_EXECUTIONS),
    'full_computations_without_failures':V4_COMPUTATIONAL_COMPLETE,'all_seven_scientifically_identified':V4_SCIENTIFIC_COMPLETE,
    'scientific_status':STATE['scientific'],'human_approval_needed_for_admissions':len(ORG_YEAR)==0,
    'OD_2023_coverage_break_unresolved':bool(coverage_df.potential_break.fillna(False).any()),
    'historical_vintages_not_archived':True,'retention_unobservable_without_work_region':True}]))
print('V4 acceptability:',STATE['scientific'],'| calculations with actual errors:',not V4_COMPUTATIONAL_COMPLETE)

# Rolling-origin feasibility is measured year by year. Lack of archived source vintages forbids a confirmatory claim.
_v4_roll=[]
for _m,_f in [('ML-5.1',OD_MODEL),('ML-5.3',OUTFLOW_MODEL)]:
    if _f is None or not len(_f):continue
    _years=sorted(int(y) for y in _f.target_year.dropna().unique())
    for _y in _years[1:]:
        _cut=_y-CFG.HORIZON
        _train=_f[_f.target_year.lt(_y)&_f.target_available_year.le(_cut)]
        _test=_f[_f.target_year.eq(_y)]
        _gate=len(_train)>=(CFG.REGION_MIN_TRAIN_ROWS if _m=='ML-5.3' else CFG.MIN_TRAIN_ROWS) and \
              len(_test)>=(CFG.REGION_MIN_TEST_ROWS if _m=='ML-5.3' else CFG.MIN_TEST_ROWS)
        _v4_roll.append({'module':_m,'target_year':_y,'n_asof_train':len(_train),
             'n_test_observed':len(_test),'legally_feasible_given_publication_lag':bool(_gate),
             'source_vintage_asof_origin':'NOT_ARCHIVED',
             'scientific_status':'RETROSPECTIVE_ONLY' if _gate else 'NOT_FEASIBLE_SMALL_HISTORICAL_TRAIN'})
qa_table('v4_rolling_origin_feasibility',pd.DataFrame(_v4_roll),allow_empty=True)


# 10Б. Контракт V5: полнота моделей, численная состоятельность и доказательная база

Этот блок проверяет результаты независимо от общей отметки выполнения ноутбука. Модели со сбоями, отсутствующими прогнозами или нестабильными значениями не засчитываются. Нет исходных показателей — `NOT_IDENTIFIABLE`, а не вымышленный результат.

In [ ]:
# ML-05 V5 · fail-closed model registry, numerical diagnostics and evidence matrix.
progress_event('STATISTICS','V5: полный контракт моделей, протоколов и проверка числа сидов',announce=True)
V5_MODEL_QA=[];V5_MODULE_ACCEPTANCE=[]
_artifact_paths=[m['path'] for m in MODEL_RECORDS]
if len(_artifact_paths)!=len(set(_artifact_paths)):
    raise RuntimeError('V5: different model experiments have duplicate output paths')
if len(MODEL_FILES)!=len(set(_artifact_paths)):
    raise RuntimeError('V5: model registry and actual model files differ')
for _m in MODEL_RECORDS:
    _fp=ROOT/'models'/_m['path']
    _ok=_fp.is_file() and _m.get('file_sha256')==hash_file(_fp) and _m.get('reloaded')
    V5_MODEL_QA.append({'module':_m['module'],'model':_m['model'],'protocol':_m.get('protocol','CUSTOM'),
        'seed':_m['seed'],'path':_m['path'],'sha_verified':bool(_ok),
        'guarded_predictions':_m.get('prediction_guard_events',0),
        'guard_fraction':_m.get('prediction_guard_fraction',0.)})
if V5_MODEL_QA and not all(x['sha_verified'] for x in V5_MODEL_QA):
    raise RuntimeError('V5: saved models changed or reload failed')
qa_table('v5_model_files_independently_verified',pd.DataFrame(V5_MODEL_QA),allow_empty=True)

# Reconcile required model names and their independent saved metrics, not a mere code branch.
for _m in ['ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7']:
    _mm=[r for r in METRICS if r.get('module')==_m]
    _issues=[r for r in ISSUES if r.get('module')==_m and r.get('status') in
      ('MODEL_FAILED','SPATIAL_MODEL_FAILURE','SOURCE_PROFILE_FAILURE','FAMILY_PROFILE_FAILED')]
    _status=MODULES[_m].get('status','UNKNOWN')
    _preds=sum(int(x.get('n',0)) for x in _mm)
    _time={x['model'] for x in _mm if x.get('protocol')=='ROLLING_TIME'}
    _required={'persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm','catboost_ensemble','ppml','negbin'}
    _targets_confirmed=bool(_status.startswith('TESTED'))
    if _m in ('ML-5.1','ML-5.3','ML-5.4') and _targets_confirmed:
        _missing=sorted(_required-_time)
        _seeds={int(x['seed']) for x in _mm if x.get('model')=='catboost' and x.get('protocol')=='ROLLING_TIME'}
        _missing_seeds=sorted(set(int(z) for z in CFG.FINAL_SEEDS)-_seeds) if CFG.MODE=='FULL' else []
    else:_missing=[];_missing_seeds=[]
    _numeric_invalid=[x['model'] for x in _mm if not np.isfinite(x.get('MAE',np.nan)) or
        not np.isfinite(x.get('RMSE',np.nan)) or float(x.get('MAE',0))<0]
    _failures=bool(_issues or _missing or _missing_seeds or _numeric_invalid)
    _scientific=('OBSERVED_TARGET_TESTED_RETROSPECTIVELY' if _targets_confirmed else
                 'DESCRIPTIVE_OR_PROXY_TEST_ONLY' if _mm or 'DESCRIPTIVE' in _status or 'PROXY' in _status else
                 'NOT_IDENTIFIABLE_WITH_AVAILABLE_STAGE_I')
    V5_MODULE_ACCEPTANCE.append({'module':_m,'status':_status,'metrics':len(_mm),
        'model_file_count':sum(x['module']==_m for x in MODEL_RECORDS),'failed_methods':len(_issues),
        'missing_methods':','.join(_missing),'missing_final_seeds':','.join(map(str,_missing_seeds)),
        'nonfinite_metrics':','.join(_numeric_invalid),
        'computational_pass':not _failures,'scientific_scope':_scientific,
        'original_target_fully_identified':bool(_m=='ML-5.4' and _targets_confirmed),
        'historical_source_vintage_validated':False})

V5_LEDGER=pd.DataFrame(V5_MODULE_ACCEPTANCE)
qa_table('v5_seven_module_acceptance',V5_LEDGER,['module'])

# Source evidence: 52 original datasets are all profiled but not uniformly identifiable as model features.
V5_SOURCE_EVIDENCE=pd.DataFrame(V4_SOURCE_PROFILES)
if len(V5_SOURCE_EVIDENCE)!=len(ASSETS) or not all(r.get('content_status')=='FULL_VALUE_COVERAGE_PASS' for r in V4_SOURCE_PROFILES):
    if CFG.MODE=='FULL':raise RuntimeError('V5 source coverage is not 52/52')
qa_table('v5_all52_source_evidence',V5_SOURCE_EVIDENCE,allow_empty=True)
V5_COMPUTATIONAL_PASS=bool(not V4_SOURCE_ERRORS and V5_LEDGER.computational_pass.all())
V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS=bool(V5_COMPUTATIONAL_PASS and all(V5_LEDGER.original_target_fully_identified))
V5_ACCEPTANCE_STATUS='FULL_COMPUTATIONS_WITH_SCIENTIFIC_LIMITS' if V5_COMPUTATIONAL_PASS else 'MODEL_OR_EVIDENCE_FAILURE'
STATE['scientific']=V5_ACCEPTANCE_STATUS
print('V5: 52-source integrity',len(V5_SOURCE_EVIDENCE),'/',len(ASSETS),
      '| all computational methods',V5_COMPUTATIONAL_PASS,'| scientifically identified 7/7',V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS)


## V7 · независимая трассировка источников и моделей: 52/52 + научные границы семи модулей

In [ ]:
# V7.1.1 · проверяемая цепочка Stage I → исследовательская панель → сохранённая модель.
# Не называем технически обследованные 52 источника 52 модельными признаками.
progress_event('STATISTICS','V7: проверяемая трассировка 52 источников и 7 модулей',announce=True)

_v7_source_panels={}
def _v7_map(asset_or_list,panels,methods,relationship='PRIMARY_SOURCE'):
    for _a in (asset_or_list if isinstance(asset_or_list,(tuple,list)) else [asset_or_list]):
        if _a is None:continue
        _id=_a['asset_id'] if isinstance(_a,dict) else str(_a)
        _v7_source_panels.setdefault(_id,[]).append({'panels':list(panels),'modules':list(methods),'relationship':relationship})
_v7_map(E01,['edu_od_year','forecast_od_candidates','outflow_observed','education_network_year'],['ML-5.1','ML-5.3','ML-5.6'])
_v7_map(E02,['edu_org_registry','v7_organization_admission_discovery','v4_organization_contingent_candidates'],['ML-5.2'])
_v7_map(SPO_ORG,['edu_org_registry'],['ML-5.2'])
_v7_map(SPO_MASTER,['spo_admission_raw_candidates'],['ML-5.2'])
_v7_map(VPO_211,['vpo_program_description','v4_specialty_model_frame','v7_organization_admission_discovery'],['ML-5.4','ML-5.2'])
_v7_map(GRAD_ORG,['graduate_descriptive_statistics','v4_ml55_crosssection_metrics'],['ML-5.5'])
_v7_map(GRAD_FIELD,['graduate_specialty_description'],['ML-5.5'])
_v7_map(source_asset(segment='05'),['e04_e05_full_family_audit'],['ML-5.5'],'FULL_FAMILY_DESCRIPTION')
_v7_map(MUN,['municipal_demography','municipality_education_context'],['ML-5.7'])
_v7_map(TRANSPORT,['region_distance_proxy'],['ML-5.1'],'CONTEXT_PROXIED_NOT_LITERAL_REGION_DISTANCES')
_v7_map(source_asset(segment='02'),['d02_indicator_coverage'],['ML-5.1','ML-5.7'],'SEPARATE_THEMATIC_CONTEXT')
_v7_map(source_asset(segment='04'),['vpo_all_sections_audit'],['ML-5.2','ML-5.4'],'SEPARATE_VPO_SECTIONS')
_v7_map(source_asset(segment='03',role='education_od_matrix_raw_cells'),['e01_raw_cell_sources'],['ML-5.1'],'SOURCE_CROSSCHECK_NOT_ADDITIVE')
_v7_map(source_asset(segment='06'),['spatial_exact_crosswalk','spatial_D04_preview','spatial_BDMO_HIST_preview'],['ML-5.1','ML-5.7'],'SPATIAL_CONTEXT_WITH_MATCHING_GATE')

# Confirm only *actually used* predictive sources, based on target panel and model feature registry.
_v7_executed={r['module'] for r in MODEL_RECORDS}
_v7_real={}
_v7_real[E01['asset_id']]={'ML-5.1','ML-5.3'} & _v7_executed
_v7_real[VPO_211['asset_id']]={'ML-5.4'} & _v7_executed
_v7_real[E02['asset_id']]={'ML-5.2'} & _v7_executed
_v7_real[GRAD_ORG['asset_id']]={'ML-5.5'} & _v7_executed
_v7_real[MUN['asset_id']]={'ML-5.7'} & _v7_executed
_v7_od_features=set()
for _r in MODEL_RECORDS:
    if _r.get('module')=='ML-5.1':
        _v7_od_features.update(str(f) for f in (_r.get('feature_num') or []))
if 'distance_proxy_km' in _v7_od_features and len(DISTANCE):
    _v7_real[TRANSPORT['asset_id']]={'ML-5.1'}
if any(x.startswith(('origin_econ_','destination_econ_')) for x in _v7_od_features) and len(ECON_PANEL):
    for _id in ECON_PANEL.source_asset_id.dropna().astype(str).unique():
        _v7_real[str(_id)]={'ML-5.1'}

# Model ancestry table: source inputs for every saved trained model, no guessing from source names.
V7_MODEL_LINEAGE=[]
for _r in MODEL_RECORDS:
    _mod=_r.get('module','')
    _src=sorted(_id for _id,_mods in _v7_real.items() if _mod in _mods)
    if not _src and _mod in ('ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.7'):
        ISSUES.append({'module':_mod,'status':'V7_LINEAGE_MISSING','reason':'No verified source linkage for recorded model '+str(_r.get('path')),'required':'exact lineage'})
    V7_MODEL_LINEAGE.append({'module':_mod,'model':_r.get('model'),'protocol':_r.get('protocol'),
      'seed':_r.get('seed'),'model_file':_r.get('path'),'model_sha256':_r.get('file_sha256'),
      'asset_ids':';'.join(_src),'n_sources':len(_src),
      'feature_columns':';'.join([str(x) for x in (_r.get('feature_num') or [])]),
      'feature_categories':';'.join([str(x) for x in (_r.get('feature_cat') or [])]),
      'claim_scope':'AS_STORED_MODEL_FIT_AND_PANEL_DERIVATION_NOT_CAUSAL_SOURCE_ATTRIBUTION'})
V7_MODEL_LINEAGE=pd.DataFrame(V7_MODEL_LINEAGE)
qa_table('v7_model_lineage',V7_MODEL_LINEAGE,allow_empty=True)

V7_SOURCE_LINEAGE=[]
for _a in ASSETS:
    _id=_a['asset_id'];_profiles=_v7_source_panels.get(_id,[])
    _num=V7_SOURCE_AUDIT.loc[V7_SOURCE_AUDIT.asset_id.eq(_id)].iloc[0]
    _panels=sorted({p for x in _profiles for p in x['panels']})
    _modules=sorted({m for x in _profiles for m in x['modules']})
    _direct=sorted(_v7_real.get(_id,set()))
    _derived=ROOT/'derived'/'v4_source_values'/_num.numeric_profile_file
    V7_SOURCE_LINEAGE.append({'asset_id':_id,'segment':_a['segment'],
      'sha256':_a['sha256'],'rows_actual':int(_a['rows_actual']),
      'content_profile_sha256':hash_file(_derived) if _derived.is_file() else '',
      'source_coverage_pass':bool(_num.all_values_profiled and _num.verified_sha256 and _num.footer_rows_verified),
      'source_specific_analysis':'FULL_VALUE_DISTRIBUTION_YEAR_IF_AVAILABLE',
      'analytical_panels':';'.join(_panels),
      'research_modules_possible':';'.join(_modules),
      'actual_saved_model_modules':';'.join(_direct),
      'direct_model_input':bool(_direct),'model_usage_semantics':('PROVEN_PANEL_CHAIN' if _direct else 'THEMATIC_OR_CONTEXT_ONLY'),
      'nonmodel_reason':('MISMATCHED_STATISTICAL_GRAIN_OR_OVERLAPPING_SOURCE_OR_NOT_IDENTIFIED_AS_PREDICTOR'
                         if not _direct else ''),
      'not_invented_source_join':False})
V7_SOURCE_LINEAGE=pd.DataFrame(V7_SOURCE_LINEAGE)
qa_table('v7_source_lineage_52',V7_SOURCE_LINEAGE,['asset_id'])
if CFG.MODE=='FULL' and (len(V7_SOURCE_LINEAGE)!=len(ASSETS) or
    not V7_SOURCE_LINEAGE.source_coverage_pass.all() or
    not V7_MODEL_LINEAGE.n_sources.ge(1).all()):
    raise RuntimeError('V7 LINEAGE: исходники, модели или реестр происхождения не согласованы')

V7_SCIENCE_GATES=[]
for _mod in ('ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7'):
    _row=V5_LEDGER.loc[V5_LEDGER.module.eq(_mod)].iloc[0]
    _raw={
      'ML-5.1':'E01_REGION_OF_PRIOR_SCHOOLING_VS_CURRENT_SPO_STUDY_NOT_RESIDENCE',
      'ML-5.2':'VERIFIED_ADMISSIONS_BY_ORGANIZATION_YEAR_NOT_STUDENT_CONTINGENT',
      'ML-5.3':'OBSERVED_E01_OUTFLOW_SUM_OF_REPORTED_EDGES_NOT_RESIDENTIAL_MIGRATION',
      'ML-5.4':'REGIONAL_SPECIALTY_ADMITTED_COUNT_NOT_ORG_BY_SPECIALTY_OD',
      'ML-5.5':'EMPLOYMENT_SNAPSHOT_NOT_RETENTION_OR_LOCATION_OF_WORK',
      'ML-5.6':'DYNAMIC_OBSERVED_OD_NETWORK_WITH_COVERAGE_BREAK',
      'ML-5.7':'VERIFIED_EDUCATIONAL_CAMPUSES_AND_PREPERIOD_CAPACITY_BY_OKTMO'}[_mod]
    _original_ident=bool(_row.get('original_target_fully_identified',False))
    _status=MODULES[_mod].get('status','UNREPORTED')
    V7_SCIENCE_GATES.append({'module':_mod,'target_definition':_raw,
      'computational_pass':bool(_row.computational_pass),
      'original_scientific_target_identified':_original_ident,
      'scientific_status':str(_row.scientific_scope),
      'module_execution_status':_status,
      'model_records':int(V7_MODEL_LINEAGE.module.eq(_mod).sum()),
      'historical_vintage_proven':False,
      'remaining_identification_dependency':('ORIGINAL_SOURCE_DEFINITION_OR_SPATIAL_CONFIRMATION_REQUIRED' if not _original_ident else ''),
      'no_synthetic_data':True})
V7_SCIENCE_GATES=pd.DataFrame(V7_SCIENCE_GATES)
qa_table('v7_seven_scientific_gates',V7_SCIENCE_GATES,['module'])
print('V7: 52/52 lineages; true direct model sources',int(V7_SOURCE_LINEAGE.direct_model_input.sum()),
      '/52; models with source lineage',len(V7_MODEL_LINEAGE),'/',len(MODEL_RECORDS))


## 10В. Независимая приёмка реконструкций
Состав и происхождение признаков, годовые разрезы, 52/52 источника, фактические и пропущенные эксперименты; доказательные статусы без подмены научного результата технической готовностью.

In [ ]:
# ML-05 V7.1.1 · Per-experiment acceptance without inventing any missing research target.
progress_event('STATISTICS','V7.1: проверка предобработки, 52 источников и всех выполнимых экспериментов',announce=True)
ADR_RECONSTRUCTED_LINEAGE=V7_MODEL_LINEAGE.copy()
if len(ADR_RECONSTRUCTED_LINEAGE):
    for _idx,_r in ADR_RECONSTRUCTED_LINEAGE.iterrows():
        _sources=set(str(_r.asset_ids).split(';')) if pd.notna(_r.asset_ids) else set()
        if _r.module=='ML-5.7' and str(_r.model) in ('adr_E0','adr_E1'):
            _sources.update([MUN['asset_id'],SPO_ORG['asset_id'],SPATIAL['asset_id']])
        if _r.module=='ML-5.2':
            _sources.add(E02['asset_id'])
        _sources={s for s in _sources if s in ASSET_BY_ID}
        ADR_RECONSTRUCTED_LINEAGE.at[_idx,'asset_ids']=';'.join(sorted(_sources))
        ADR_RECONSTRUCTED_LINEAGE.at[_idx,'n_sources']=len(_sources)
    qa_table('adr_reconstructed_model_lineage',ADR_RECONSTRUCTED_LINEAGE,allow_empty=True)

# A fixed research contract: existing V7 calculations and all added observable subexperiments.
_adrex=pd.DataFrame(ADR_EXPERIMENTS)
_adr_qa=[]
for _mod in ('ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7'):
    _ev=_adrex.loc[_adrex.module.eq(_mod)] if len(_adrex) else pd.DataFrame()
    _bad=_ev.loc[_ev.status.str.startswith('FAILED',na=False)] if len(_ev) else pd.DataFrame()
    _v5=V5_LEDGER.loc[V5_LEDGER.module.eq(_mod)]
    _existing=bool(len(_v5) and _v5.iloc[0].computational_pass)
    _ran=bool(len(_ev) and _ev.status.isin(['MODEL_COMPLETED','V7_COMPLETED','DESCRIPTIVE_COMPLETED']).any())
    _scientific=bool(len(_v5) and _v5.iloc[0].original_target_fully_identified)
    _adr_qa.append({'module':_mod,'v7_computational_pass':_existing,'v71_added_experiments':len(_ev),
        'v71_observed_experiment_completed':_ran,'v71_new_failures':len(_bad),
        'original_scientific_target_fully_identified':_scientific,
        'scientific_scope':'ONLY_PUBLISHED_OBSERVED_TARGETS_AND_LEGALLY_MATCHED_PROXIES',
        'missing_data_not_fabricated':True,'missing_data_experiments_skipped':int(_ev.status.str.startswith('SKIPPED',na=False).sum()) if len(_ev) else 0})
ADR_SEVEN_MODULE_FINAL=qa_table('adr_v71_seven_module_acceptance',pd.DataFrame(_adr_qa),['module'])
ADR_COMPUTATIONAL_PASS=bool(V5_COMPUTATIONAL_PASS and ADR_SEVEN_MODULE_FINAL.v71_new_failures.eq(0).all())
ADR_SCIENTIFIC_7_ORIGINAL_IDENTIFIED=bool(ADR_COMPUTATIONAL_PASS and ADR_SEVEN_MODULE_FINAL.original_scientific_target_fully_identified.all())
ADR_SOURCE_AUDIT=pd.DataFrame([{'asset_id':a['asset_id'],'sha256_expected':a['sha256'],
        'sha256_checked':INPUT_SHA.get(a['asset_id']),
        'is_source_checked':INPUT_SHA.get(a['asset_id'])==a['sha256'],
        'source_analytical_role':V7_SOURCE_LINEAGE.loc[V7_SOURCE_LINEAGE.asset_id.eq(a['asset_id']),'model_usage_semantics'].iloc[0]
            if V7_SOURCE_LINEAGE.asset_id.eq(a['asset_id']).any() else 'UNMAPPED',
        'source_methodological_grain':a.get('grain_status','UNVERIFIED'),
        'direct_model_input':bool(V7_SOURCE_LINEAGE.loc[V7_SOURCE_LINEAGE.asset_id.eq(a['asset_id']),'direct_model_input'].iloc[0])
            if V7_SOURCE_LINEAGE.asset_id.eq(a['asset_id']).any() else False} for a in ASSETS])
qa_table('adr_source_52_completeness',ADR_SOURCE_AUDIT,['asset_id'])

# Exact original inputs are immutable: compare expected Stage I fingerprint and do not reinterpret source rows as new observations.
if CFG.MODE=='FULL' and (len(ADR_SOURCE_AUDIT)!=52 or not ADR_SOURCE_AUDIT.is_source_checked.all()):
    raise RuntimeError('ADR SOURCE_GATE: source coverage failure')
if ADR_COMPUTATIONAL_PASS and len(ADR_RECONSTRUCTED_LINEAGE):
    if not ADR_RECONSTRUCTED_LINEAGE.n_sources.ge(1).all():
        raise RuntimeError('ADR MODEL_GATE: saved model lacks verified source lineage')
if len(ADR_ADMISSION_YEAR) and not (ADR_ADMISSION_YEAR.area_total.fillna(False).all() and
        ADR_ADMISSION_YEAR.specialty_total.fillna(False).all() and ADR_ADMISSION_YEAR.n_reporting_cells.eq(1).all()):
    raise RuntimeError('ADR SEMANTIC_GATE: source-reported admission cell is not an unambiguous total')
if len(ADR_GEO_VERIFIED) and not ADR_GEO_VERIFIED.identity_status.eq('PROXIMITY_MATCH_NOT_OFFICIAL_CAMPUS_LOCATION').all():
    raise RuntimeError('ADR SPATIAL_GATE: proxy falsely asserted as confirmed campus')

ADR_FINAL_REPORT={
 'version':CFG.VERSION,'source_catalog_entries':len(ASSETS),'source_sha_passed':int(ADR_SOURCE_AUDIT.is_source_checked.sum()),
 'all_seven_modules_reported':len(ADR_SEVEN_MODULE_FINAL)==7,'v7_technical_methods_retained':V5_COMPUTATIONAL_PASS,
 'v71_additional_experiment_count':len(ADR_EXPERIMENTS),'v71_additional_model_experiment_count':sum(
    r['status']=='MODEL_COMPLETED' for r in ADR_EXPERIMENTS),
 'v71_computational_pass':ADR_COMPUTATIONAL_PASS,
 'seven_original_scientific_targets_identified':ADR_SCIENTIFIC_7_ORIGINAL_IDENTIFIED,
 'OD_all_year_balanced_edges':int(len(ADR_OD_BALANCED)),
 'VO_source_approved_org_year_admissions':int(len(ADR_ADMISSION_YEAR)),
 'SPO_spatially_reconstructed_site_year_proxies':int(len(ADR_GEO_VERIFIED)),
 'municipal_geocode_E0E1':ADR_ML57_GEO_STATUS,
 'methodological_rule':'MISSING_NOT_OBSERVED_IS_SKIPPED_NOT_ZERO_FILLED_AND_NOT_FABRICATED'
}
qa_table('adr_final_scientific_diagnostics',pd.DataFrame([{
    k:(json.dumps(v,ensure_ascii=False,default=str) if isinstance(v,(dict,list)) else v)
    for k,v in ADR_FINAL_REPORT.items()}]),allow_empty=True)
print('V7.1: source coverage',ADR_FINAL_REPORT['source_sha_passed'],'/52 | additional experiments',
    ADR_FINAL_REPORT['v71_additional_experiment_count'],'| computational gate',ADR_COMPUTATIONAL_PASS,
    '| scientific 7/7',ADR_SCIENTIFIC_7_ORIGINAL_IDENTIFIED)

mark_stage('STATISTICS')


## 11. Научные изображения
Диаграммы привязаны к исходному показателю и его периоду, без фиктивной геометрии и трактовки отсутствующей связи как нулевой.

In [ ]:
progress_event('VISUALS','Визуализация фактических E01, сетей и тестовых ошибок')
# Фигуры только на реально наблюдаемых исходных величинах; без псевдогеографических карт.
plt.rcParams.update({'font.family':'DejaVu Sans','figure.figsize':(10,5),'axes.grid':True})

def figure_save(stem):
    path=ROOT/'figures'/(stem+'.png')
    plt.tight_layout();plt.savefig(path,dpi=160,bbox_inches='tight');plt.close()
    FIGURES.append(str(path))
    return path

# Композиция берётся из неизменяемого PANELS, а не из локального counts (он изменяется в FEATURES).
od_comp=PANELS.get('od_composition',pd.DataFrame()).copy()
required={'observation_year','code_status','total_value'}
if not required.issubset(od_comp.columns):
    if {'observation_year','code_status','observed_value'}.issubset(OD.columns):
        od_comp=OD.groupby(['observation_year','code_status'],dropna=False,as_index=False).agg(total_value=('observed_value','sum'))
    else:fail('VISUALS','Потеряна первичная схема E01 / code_status')
if not od_comp.empty:
    od_comp['total_value']=pd.to_numeric(od_comp.total_value,errors='coerce')
    od_comp=od_comp.dropna(subset=['observation_year','code_status','total_value'])
    tab=od_comp.groupby(['observation_year','code_status']).total_value.sum().unstack(fill_value=0)
    tab=tab.reindex(columns=['INTER_REGION','INTRA_REGION','UNKNOWN_CODE'],fill_value=0).sort_index()
    if len(tab):
        fig,ax=plt.subplots(figsize=(10,5));tab.plot(kind='bar',stacked=True,ax=ax)
        ax.set(title='E01: состав зарегистрированного показателя (не постоянная миграция)',xlabel='Отчётный год',ylabel='Наблюдаемое значение E01')
        ax.legend(['Межрегиональные','Внутрирегиональные','Неопределённые коды'],fontsize=8)
        ax.tick_params(axis='x',rotation=0)
        figure_save('01_od_value_composition')
if len(coverage_df):
    cv=coverage_df.copy();cv['year']=pd.to_numeric(cv.year,errors='coerce');cv['observed_edges']=pd.to_numeric(cv.observed_edges,errors='coerce')
    cv=cv.dropna(subset=['year','observed_edges']).sort_values('year')
    if len(cv):
        fig,ax=plt.subplots(figsize=(10,4));ax.plot(cv.year,cv.observed_edges,'o-')
        flagged=cv.get('potential_break',pd.Series(False,index=cv.index)).fillna(False).astype(bool)
        for r in cv.loc[flagged].itertuples():
            ax.annotate('Изменение покрытия',(r.year,r.observed_edges),xytext=(0,12),textcoords='offset points',ha='center',fontsize=8)
        ax.set(title='E01: число зарегистрированных межрегиональных направлений',xlabel='Год',ylabel='Наблюдаемые пары регионов')
        ax.set_xticks(sorted(cv.year.astype(int).unique()))
        figure_save('02_edge_coverage')
if len(NETWORK):
    net=pd.DataFrame(NETWORK);year=int(net.year.max())
    x=net[net.year==year].nlargest(15,'in_strength').sort_values('in_strength')
    if len(x):
        fig,ax=plt.subplots(figsize=(11,6));ax.barh(x.region_code.astype(str),x.in_strength)
        ax.set(title=f'E01: крупнейшие регионы обучения, {year}',xlabel='Сумма исходного E01',ylabel='Код региона')
        figure_save('03_destination_centres')
if len(CONFIRM):
    q=CONFIRM[['module','baseline_mae','selected_mae']].copy()
    q['baseline_mae']=pd.to_numeric(q.baseline_mae,errors='coerce');q['selected_mae']=pd.to_numeric(q.selected_mae,errors='coerce')
    q=q.dropna();q=q[q.baseline_mae.gt(0)&q.selected_mae.ge(0)].copy()
    q['ratio']=q.selected_mae/q.baseline_mae;q=q[np.isfinite(q.ratio)]
    if len(q):
        fig,ax=plt.subplots(figsize=(10,5));at=np.arange(len(q));w=.34
        ax.bar(at-w/2,np.ones(len(q)),width=w,label='Инерция = 1,0')
        ax.bar(at+w/2,q.ratio,width=w,label='Выбранная до теста модель')
        ax.axhline(1,color='gray',linewidth=1,alpha=.7)
        ax.set_xticks(at);ax.set_xticklabels(q.module.astype(str).tolist())
        ax.set(ylabel='MAE / MAE инерции',title='Относительная ошибка на общих тестовых ключах')
        ax.legend();figure_save('04_mae_preselected_models')
print('Generated figures:',len(FIGURES))

mark_stage('VISUALS')


## 12. Экспорт: полный evidence в GitHub, аналитический Excel и воспроизводимый встроенный ноутбук

In [ ]:
memory_point('before_export')
progress_event('EXPORT','Excel, evidence-Parquet, модели, ноутбук и манифест')
import zipfile
from itertools import chain
EXCEL=ROOT/'exports'/'ML5_EDUCATION_MIGRATION_RESULTS.xlsx'
EVIDENCE=ROOT/'exports'/'ML5_EVIDENCE.zip'
NOTEBOOK=ROOT/'exports'/'ML5_EDUCATION_MIGRATION.ipynb'
MANIFEST=ROOT/'exports'/'run_manifest.json'

# Экспорт строго типизирован: смешанный int/string test_year не допускается.
STRING_HINTS={'entity_id','parent_group_id','origin_region','destination_region','region_code',
              'source_asset_id','oktmo','organization_id','indicator_name','test_period_label',
              'object_oktmo','specialty_code','area','campus_id','region_oktmo'}
INT_HINTS={'target_year','origin_year','observation_year','historical_year','historical_year_2',
           'information_cutoff_year','feature_available_year','target_available_year',
           'test_year','year','horizon','seed','graduation_year','salary_reference_year'}

def prepare_for_parquet(name,frame):
    f=frame.copy()
    f.columns=[str(c) for c in f.columns]
    if len(set(f.columns))!=len(f.columns):fail('EXPORT','Дубли названий столбцов '+name)
    if 'test_year' in f and f['test_year'].dtype in ('object','string'):
        original=f['test_year'].copy();numeric_year=number(original)
        descriptive=original.where(original.notna() & numeric_year.isna()).astype('string')
        if 'test_period_label' not in f:f['test_period_label']=descriptive.fillna('')
        else:f['test_period_label']=f['test_period_label'].astype('string').fillna(descriptive).fillna('')
        f['test_year']=numeric_year.astype('Int64')
    for c in f.columns:
        if c in INT_HINTS:
            z=number(f[c]);invalid=f[c].notna()&z.isna()
            if invalid.any():fail('PARQUET_SCHEMA',f'{name}.{c}: нечисловые годы/сиды')
            f[c]=z.astype('Int64')
        elif c in STRING_HINTS:f[c]=f[c].astype('string')
        elif pd.api.types.is_object_dtype(f[c]):
            sample=f[c].dropna()
            kinds={type(t) for t in sample.head(2000)}
            if not kinds or kinds<={str}:
                f[c]=f[c].astype('string')
            elif kinds<={int,float,np.int64,np.float64}:
                f[c]=number(f[c])
            elif any(isinstance(t,(dict,list,tuple,set)) for t in sample.head(2000)):
                f[c]=f[c].map(lambda v:json.dumps(v,ensure_ascii=False,default=str,sort_keys=True)
                                    if isinstance(v,(dict,list,tuple,set)) else None if v is None else str(v)).astype('string')
            elif all(isinstance(t,(str,bytes,int,float,complex,np.number)) for t in sample.head(2000)):
                # Смешанный административный код переводится в строку только как metadata,
                # строгие числовые признаки/метрики не трогаем.
                if re.search(r'(value|actual|predicted|MAE|RMSE|salary|rate|population)',c,re.I):
                    fail('PARQUET_SCHEMA',f'{name}.{c}: смешанный тип числового поля')
                f[c]=f[c].astype('string')
            else:
                f[c]=f[c].map(lambda v:str(v) if pd.notna(v) else None).astype('string')
    return f

# Регрессионный тест известной ArrowInvalid: числа и текстовый исторический период.
_schema_probe=prepare_for_parquet('group_protocols',pd.DataFrame({
    'test_year':[2025,'historical',pd.NA],
    'test_period_label':['',pd.NA,''],
    'entity_id':['001','002','003']}))
if str(_schema_probe.test_year.dtype)!='Int64' or _schema_probe.test_period_label.iloc[1]!='historical':
    fail('PARQUET_SCHEMA','Смешанные test_year снова повреждают метку исторического периода')
del _schema_probe

EVIDENCE_PARQUET={}
def parquet_pack(name,frame):
    progress_event('EXPORT',f'Parquet {name}: {len(frame):,} строк, порциями',announce=True)
    # Физически пишем по 10 тыс. строк; никакой pa.Table из полного DataFrame.
    clean=prepare_for_parquet(name,frame)
    path=ROOT/'derived'/(name+'.parquet')
    write_df_chunks(clean,path)
    pf=pq.ParquetFile(path)
    if pf.metadata.num_rows!=len(clean):fail('EXPORT','row count changed in '+name)
    if pf.schema_arrow.names!=list(clean.columns):fail('EXPORT','schema changed in '+name)
    sample=pf.read_row_group(0).slice(0,min(100,len(clean))).to_pandas()
    if len(sample)!=min(len(clean),100):fail('EXPORT','roundtrip failed '+name)
    EVIDENCE_PARQUET[name]={'rows':len(clean),'columns':len(clean.columns),
        'schema':str(pf.schema_arrow),'sha256':hash_file(path),'path':str(path)}
    del sample,clean,pf;gc.collect()
    return path

def excel_table(writer,source,sheet,cap=None):
    if not isinstance(source,pd.DataFrame) or source.empty:return False
    # XlsxWriter constant_memory + последовательная запись строк (pandas.to_excel с ним несовместим).
    cap=int(cap or CFG.EXCEL_PREVIEW_ROWS)
    x=source.head(cap)
    ws=writer.book.add_worksheet(sheet[:31]);ws.freeze_panes(1,0)
    header=writer.book.add_format({'bold':True,'bg_color':'#294862','font_color':'#FFFFFF','text_wrap':True,'valign':'vcenter'})
    ws.write_row(0,0,[str(c) for c in x.columns],header)
    ws.set_row(0,30)
    for j,c in enumerate(x.columns):ws.set_column(j,j,min(43,max(11,len(str(c))+2)))
    for idx,row in enumerate(x.itertuples(index=False,name=None),1):
        values=[]
        for v in row:
            if v is None or v is pd.NA:values.append(None)
            elif isinstance(v,(dict,list,tuple,set)):values.append(json.dumps(v,ensure_ascii=False,default=str))
            elif isinstance(v,(datetime,pd.Timestamp)):values.append(str(v))
            elif isinstance(v,(np.integer,)):values.append(int(v))
            elif isinstance(v,(np.floating,)):values.append(float(v) if np.isfinite(v) else None)
            elif isinstance(v,(int,float,bool,str)):values.append(v if not isinstance(v,float) or np.isfinite(v) else None)
            else:values.append(str(v))
        ws.write_row(idx,0,values)
    ws.autofilter(0,0,len(x),len(x.columns)-1)
    return True

V7_EXCEL_EXTRA=[
 ('v7_all52_scientific_source_audit','V7 научный аудит 52'),
 ('v7_od_common_support_sensitivity','V7 OD общие пары'),
 ('v7_organization_admission_discovery','V7 кандидаты приема'),
 ('v7_municipal_identification_gate','V7 муниципалитеты'),
 ('v7_source_lineage_52','V7 происхождение 52'),
 ('v7_model_lineage','V7 модели источники'),
 ('v7_seven_scientific_gates','V7 семь целей')]
EXCEL_TABLES=[
    ('adr_preprocessing_provenance','V71 происхождение реконструкций'),
    ('adr_od_pairwise_comparable','V71 OD сопоставимые пары'),
    ('adr_od_balanced_edges_by_year','V71 OD общие рёбра'),
    ('adr_organizational_admission_dictionary','V71 поиск приёма E02'),
    ('adr_admission_organization_year','V71 приём подтвержден'),
    ('adr_spo_geolocation_candidates','V71 география СПО'),
    ('adr_spo_geolocation_high_confidence','V71 география высокая'),
    ('adr_all_feasible_experiments','V71 все эксперименты'),
    ('adr_ml57_geographic_execution','V71 E0 E1 география'),
    ('adr_ml57_spatial_e0e1_metrics','V71 E0 E1 результаты'),
    ('adr_v71_seven_module_acceptance','V71 7 модулей'),
    ('adr_source_52_completeness','V71 источники 52'),
    ('adr_reconstructed_model_lineage','V71 происхождение моделей'),
    ('adr_final_scientific_diagnostics','V71 научные границы'),

 ('v4_all52_source_value_audit','V4 данные 52 из 52'),
 ('v4_all52_research_use','V4 использование 52'),
 ('v4_source_scan_failures','V4 ошибки источников'),
 ('v4_specialty_admission_gate','V4 прием спец. данные'),
 ('v4_specialty_model_frame','V4 прогноз спец. база'),
 ('v4_organization_contingent_candidates','V4 контингент ВО'),
 ('v4_organization_contingent_model_frame','V4 прогноз контингента'),
 ('v4_additional_subexperiments','V4 дополнительные задачи'),
 ('v4_ml57_indicator_support','V4 охват БДМО'),
 ('v4_ml57_exact_common_support','V4 E0 E1 состав'),
 ('v4_ml57_results','V4 E0 E1 метрики'),
 ('v4_ml57_execution','V4 E0 E1 статус'),
 ('v4_ml55_crosssection_metrics','V4 занятость тест'),
 ('v4_ml55_execution','V4 выпуск тест'),
 ('v4_ml56_node_forecast','V4 сеть тест'),
 ('v4_exploratory_paired_comparisons','V4 парные сравнения'),
 ('v4_methods_execution_ledger','V4 выполнение методов'),
 ('v4_rolling_origin_feasibility','V4 скользящие годы'),
 ('v4_scientific_acceptance','V4 научная приемка'),
 ('full52_source_inventory', 'Все 52 источника'),
 ('seven_module_status_final', 'Семь модулей итог'),
 ('ml51_spatial_protocols', 'Пространственный тест OD'),
 ('ml54_history_persistence_diagnostic', 'Тренды спец. диагностика'),
 ('ml52_admission_candidates', 'Кандидаты приема'),
 ('full52_year_coverage', 'Все годы источников'),
 ('full52_theme_coverage', 'Охват тематик'),
 ('full52_raw_originals', 'Оригинальные XLSX'),
 ('bdmo_education_candidates', 'БДМО образование'),
 ('municipal_infrastructure_candidates', 'Инфраструктура ОКТМО'),
 ('spatial_exact_crosswalk', 'География ОКТМО'),
 ('e04_e05_full_family_audit', 'Все массивы выпускников'),
 ('vpo_all_sections_audit', 'Все разделы ВПО'),
 ('spo_annual_vs_master', 'СПО годовые сверки'),
 ('d02_indicator_coverage', 'Экономика 11 источников'),
 ('module_52_extension', 'Доп модули'),
 ('ml54_specialty_history', 'История специальностей'),
 ('ml55_outcome_disparities', 'Выпускники различия'),
 ('ml56_network_stress', 'Устойчивость сети'),
 ('ml57_common_support','БДМО E0 E1 исходные'),

 ('stage1_sources','Источники Stage I'),('semantic_indicator_catalog','Семантика индикаторов'),
 ('admission_approval_register','Утверждение индикаторов'),('registry_coverage_audit','Охват реестров'),
 ('organization_sector_subtasks','ВО СПО итоги модулей'),
 ('e01_original_headers','E01 первичные заголовки'),('e01_raw_cell_sources','E01 raw-проверка'),
 ('od_composition','Состав OD'),('od_edge_coverage','Покрытие OD'),
 ('od_edge_transitions','Новые и исчезнувшие OD'),('edu_org_registry','Реестр организаций'),
 ('edu_org_year','Приём ВО СПО'),('vpo_admission_audit','Приём ВПО-1'),
 ('program_profile','Программы'),('program_concentration','Концентрация программ'),
 ('vpo_program_description','ВПО программы отдельно'),
 ('graduate_full_scan_audit','E04 E05 полное сканирование'),('graduate_descriptive_statistics','Занятость выпускников'),('graduate_specialty_description','Выпуск по специальностям'),('municipality_education_context','Муниципалитеты E0 E1'),
 ('economic_indicator_candidates','Экономика источники'),('regional_economy','Региональные факторы'),
 ('region_distance_proxy','Транспорт и расстояния'),('education_network_year','Сетевые показатели'),
 ('education_network_communities','Сообщества'),('network_stability','Динамика сети'),
 ('seven_module_status','Семь модулей'),('module_diagnostics','Статусы модулей'),
 ('seven_hypotheses','Семь гипотез'),('model_metrics','Метрики'),
 ('confirmatory_comparisons','Подтверждающие тесты'),('split_audit','Временные разбиения'),
 ('future_mutation_canaries','Защита от утечек'),('metrics_recomputed_from_predictions','Независимый пересчёт'),
 ('entity_match_quarantine','Спорные совпадения'),('graduate_outcomes','Выпускники исходные'),
 ('edu_od_interregional','Рёбра межрегиональные')]

EXCEL_TABLES += V7_EXCEL_EXTRA

passport=pd.DataFrame([
  {'Параметр':'Проект','Значение':'Демографический советник · ML-05 Образовательная миграция'},
  {'Параметр':'Версия','Значение':CFG.VERSION},
  {'Параметр':'Режим','Значение':CFG.MODE},
  {'Параметр':'Снапшот Stage I','Значение':SOURCE_COMMIT},
  {'Параметр':'Каталог Stage I','Значение':index_info['sha']},
  {'Параметр':'Наборы в каталоге','Значение':len(ASSETS)},
  {'Параметр':'Наблюдаемая OD-связка','Значение':'Регион общего образования → регион СПО; не является фактом переезда'},
  {'Параметр':'E01 зачисления','Значение':'Не подтверждены первичным определением'},
  {'Параметр':'Статус научного исполнения','Значение':STATE['scientific']},
  {'Параметр':'Ограничение временного теста','Значение':'Исторические source vintages не доступны; ретроспективная проверка'},
  {'Параметр':'Публикация','Значение':'Только FULL при локальной приёмке и явно включённом разрешении'},
])

# При повторном запуске после ошибки NOTEBOOK_CAPTURE не пересоздаём огромный evidence.
def _completed_evidence_available():
    if not all(p.is_file() and p.stat().st_size>100 for p in (EXCEL,EVIDENCE)):return False
    try:
        with zipfile.ZipFile(EVIDENCE) as z:
            cfg=json.loads(z.read('metadata/config.json'))
            names=z.namelist()
            return (cfg.get('VERSION')==CFG.VERSION and 'metadata/v5_module_acceptance.json' in names
                    and 'predictions/education_predictions.parquet' in names
                    and any(p.startswith('models/') for p in names)
                    and any(p.startswith('panels/') for p in names))
    except Exception:return False

_REUSE_EXPORT=_completed_evidence_available()
if _REUSE_EXPORT:
    progress_event('EXPORT','Готовые Excel/evidence V7.1.1 обнаружены: повторная упаковка не требуется',announce=True)
    # Восстанавливаем реестр Parquet, сброшенный в начале этой ячейки.
    with zipfile.ZipFile(EVIDENCE) as z:
        for member in z.namelist():
            if not member.endswith('.parquet') or not member.startswith(('panels/','predictions/')):continue
            key=Path(member).stem
            p=Path(PRED_STORAGE_FILE) if key=='education_predictions' and PRED_STORAGE_FILE else Path(PANEL_FILES.get(key,ROOT/'derived'/(key+'.parquet')))
            if not p.is_file():
                p=ROOT/'derived'/(key+'.parquet');p.parent.mkdir(parents=True,exist_ok=True)
                with z.open(member) as source,p.open('wb') as target:shutil.copyfileobj(source,target,4*1024**2)
            pf=pq.ParquetFile(p)
            EVIDENCE_PARQUET[key]={'rows':pf.metadata.num_rows,'columns':len(pf.schema_arrow.names),
              'schema':str(pf.schema_arrow),'sha256':hash_file(p),'path':str(p)}
    progress_event('EXPORT',f'Восстановлен реестр {len(EVIDENCE_PARQUET)} Parquet без пересчёта',announce=True)

if not _REUSE_EXPORT:
    progress_event('EXPORT','Сборка многолистового Excel',announce=True)
    with pd.ExcelWriter(EXCEL,engine='xlsxwriter',engine_kwargs={'options':{'constant_memory':True,'strings_to_formulas':False,'strings_to_urls':False}}) as writer:
        excel_table(writer,passport,'Паспорт')
        for _i,(key,title) in enumerate(EXCEL_TABLES,1):
            progress_event('EXPORT',f'Excel {_i}/{len(EXCEL_TABLES)}: {key}',_i,len(EXCEL_TABLES),announce=False)
            excel_table(writer,PANELS.get(key),title)
        excel_table(writer,pd.DataFrame(TABLE_QA),'Контроль таблиц')
        excel_table(writer,pd.DataFrame(ISSUES),'Ограничения')
        excel_table(writer,pd.DataFrame(LOG),'Журнал диагностики')

METHODOLOGY='''# ML-05 V7.1.1. Научный протокол

Восемь различных событий мобильности не отождествляются.
E01 region_of_previous_general_education × region_of_current_study НЕ является доказанным годовым приёмом.
Ретроспективные модели оценивают только измеренную E01 величину; нулевые отсутствующие рёбра не создаются.
Разбиения train/DEV/TEST подчинены доступности меток по консервативному лагу. Если DEV не был известен к test origin,
победителя из теста не выбирают; persistence остаётся предзаданной основной моделью.
Историческая версия источника 2026 не доказывает фактическую доступность значений в 2021–2025.
Сети описывают только наблюдаемые связи регионов предыдущего образования и обучения.
Выпускники E04/E05 — отдельный employment_reference 2024-12-31: место работы не известно, retention нельзя оценить.
Пересекающиеся географические/гендерные агрегаты E04/E05 нельзя суммировать, поэтому описаны медианы исходных строк по разрезам.
В пространственном E0/E1 отсутствие подтверждённой образовательной площадки не равно отсутствию площадки.
Экономические и транспортные признаки включаются только при доказанном grain и лаге.
Отрицательные результаты сохраняются, причинные эффекты по прогнозной важности не идентифицируются.
'''

if not _REUSE_EXPORT:
    progress_event('EXPORT','Сборка evidence-ZIP и контроль отдельных Parquet',announce=True)
    with zipfile.ZipFile(EVIDENCE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
        z.writestr('research_methods.md',METHODOLOGY)
        z.writestr('metadata/config.json',json.dumps(asdict(CFG),ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/plan.json',json.dumps(EXPERIMENT_PLAN,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/semantics_e01.json',json.dumps(OD_SEMANTIC,ensure_ascii=False,indent=2))
        z.writestr('metadata/qa.json',json.dumps(TABLE_QA,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/source_hashes.json',json.dumps(INPUT_SHA,ensure_ascii=False,indent=2))
        z.writestr('metadata/issues.json',json.dumps(ISSUES,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/module_status.json',json.dumps(MODULES,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/splits.json',json.dumps(SPLIT_AUDIT,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/catalog_52_sha.json',json.dumps(INPUT_SHA,ensure_ascii=False,indent=2))
        z.writestr('metadata/v5_module_acceptance.json',V5_LEDGER.to_json(orient='records',force_ascii=False,indent=2))
        z.writestr('metadata/v5_model_registry.json',json.dumps(MODEL_RECORDS,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/catalog_52_protocol.md', 'Каждый из 52 Parquet: SHA-256, число строк, schema, годовой охват. '+'Годовые shards СПО не суммируются с общей панелью. Другие уровни данных не приписываются образовательному OD.')
        _export_panels=list(PANELS.items())
        for _i,(key,df) in enumerate(_export_panels,1):
            progress_event('EXPORT',f'Evidence {_i}/{len(_export_panels)}: {key}',_i,len(_export_panels),announce=False)
            if not isinstance(df,pd.DataFrame) or df.empty:continue
            if key in PANEL_FILES:
                p=Path(PANEL_FILES[key]);pf=pq.ParquetFile(p)
                EVIDENCE_PARQUET[key]={'rows':pf.metadata.num_rows,'columns':len(pf.schema_arrow.names),
                    'schema':str(pf.schema_arrow),'sha256':hash_file(p),'path':str(p)}
            else:p=parquet_pack(key,df)
            z.write(p,'panels/'+key+'.parquet')
        if PRED_STORAGE_FILE is not None:
            p=Path(PRED_STORAGE_FILE);pf=pq.ParquetFile(p)
            EVIDENCE_PARQUET['education_predictions']={'rows':pf.metadata.num_rows,'columns':len(pf.schema_arrow.names),
                'schema':str(pf.schema_arrow),'sha256':hash_file(p),'path':str(p)}
            z.write(p,'predictions/education_predictions.parquet')
        # Large source-grouped context: include every successfully built Parquet, not only Excel preview.
        for p in CONTEXT_PARQUET_FILES:
            if not Path(p).exists():fail('EXPORT','Отсутствует исходный контекстный Parquet '+str(p))
            z.write(p,'context/'+Path(p).name)
        for path in FIGURES:z.write(path,'figures/'+Path(path).name)
        for name,path in MODEL_FILES.items():z.write(path,'models/'+name)
        z.writestr('logs/run_events.json',json.dumps(LOG,ensure_ascii=False,indent=2))

# V6 SOURCE CAPTURE: восстанавливаем полный код из самого ноутбука; Colab get_ipynb НЕ используется.
progress_event('EXPORT','Сохранение встроенного неизменяемого V7.1.1 notebook',announce=True)
snap=notebook_from_embedded_source()
for _cell in snap.cells:
    if _cell.cell_type=='code':
        _cell.outputs=[]
        _cell.execution_count=None
nbformat.validate(snap)
clean_text=nbformat.writes(snap)
_token=globals().get('_TOKEN')
if (isinstance(_token,str) and len(_token)>15 and _token in clean_text) or re.search(r'(?:ghp_|github_pat_)[A-Za-z0-9_]{20,}',clean_text):
    raise RuntimeError('V7.1.1 SECURITY: notebook contains GitHub token; export blocked')
_tmp_notebook=NOTEBOOK.with_suffix('.tmp.ipynb')
nbformat.write(snap,_tmp_notebook)
_tmp_notebook.replace(NOTEBOOK)
_saved_nb=nbformat.read(NOTEBOOK,as_version=4)
if [c.source for c in _saved_nb.cells]!=[c.source for c in snap.cells]:
    raise RuntimeError('V7.1.1: notebook roundtrip changed sources')
progress_event('EXPORT',f'V7.1.1 notebook saved, {len(snap.cells)} cells',announce=True)

# Manifest создаётся после локальных артефактов и содержит их реальные SHA; его собственный SHA нельзя включить в него же.
output_hashes={p.name:hash_file(p) for p in [EXCEL,EVIDENCE,NOTEBOOK] if p.exists()}
manifest={
 'version':CFG.VERSION,'mode':CFG.MODE,'created_utc':stamp(),
 'source_repo':CFG.REPO,'source_commit':SOURCE_COMMIT,'source_index_git_blob_sha':index_info['sha'],
 'source_index_version':INDEX['version'],'source_hashes':INPUT_SHA,
 'context_derived_files':{Path(p).name:{'rows':pq.ParquetFile(p).metadata.num_rows,'sha256':hash_file(p)} for p in CONTEXT_PARQUET_FILES},
 'catalog_52_coverage':{'registered':len(ASSETS),'sha_verified':len(INPUT_SHA),
 'verified_every_asset':len(INPUT_SHA)==len(ASSETS),
 'year_profiled':len(FULL52_YEAR_STATS)},
 'experiment_plan_sha256':PLAN_HASH,'v5_science_gates':{'all52_source_content_verified':not V4_SOURCE_ERRORS,'all_feasible_computations_succeeded':V5_COMPUTATIONAL_PASS,'all_seven_scientifically_identified':V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS,'model_failure_count':int((~V5_LEDGER.computational_pass).sum())},'status':STATE.copy(),'module_status':MODULES,
 'observed_od_counts':{'all':len(OD),'inter':len(OD_INTER),'intra':len(OD_INTRA),'unknown':len(OD_UNKNOWN)},
 'real_prediction_rows':PRED_TOTAL,'v5_model_file_count':len(MODEL_FILES),
 'v5_module_acceptance':V5_MODULE_ACCEPTANCE,
 'v5_original_seven_target_identifiable':V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS,'metric_rows':len(METRICS),'panels':{
      k:{'rows':v['rows'],'columns':v['columns'],'sha256':v['sha256']} for k,v in EVIDENCE_PARQUET.items()},
 'adr_advanced_data_reconstruction':ADR_FINAL_REPORT,
 'adr_experiment_ledger':ADR_EXPERIMENTS,
 'output_sha256':output_hashes,'runtime_seconds':round(time.monotonic()-START_TIME,1),
 'limitations':ISSUES,'publication_status':'NOT_PUBLISHED',
 'commit_self_reference':'Commit SHA can be reported after Git commit; cannot be embedded in the same hashed blob'
}
progress_event('EXPORT','Манифест с SHA-256 локальных результатов',announce=True)
MANIFEST.write_text(json.dumps(manifest,ensure_ascii=False,default=str,indent=2),encoding='utf-8')
print('Local package:',EXCEL.name,round(EXCEL.stat().st_size/1e6,2),'MB;',
      EVIDENCE.name,round(EVIDENCE.stat().st_size/1e6,2),'MB; notebook:',NOTEBOOK.exists())

memory_point('after_export')
mark_stage('EXPORT')


## 13. Строгая локальная приёмка семи модулей, численных результатов и 52 источников

In [ ]:
progress_event('QA_AND_PUBLICATION','Локальная приёмка, воспроизводимость и безопасная публикация')
# Финальный QA запускается до первого GitHub-запроса на запись.
LOCAL_CHECKS=[]
def check(name,condition,why=''):
    ok=bool(condition)
    LOCAL_CHECKS.append({'check':name,'status':'PASS' if ok else 'FAIL','reason':why})
    if not ok:note('QA',name+': '+why,'FAIL')
    return ok

check('DUCKDB_SQL_PREFLIGHT_PRODUCTION_QUERIES',bool(globals().get('SQL_PREFLIGHT_PASSED')),
      'Production COPY, candidate SUMMARY and org-year TOTAL must parse and execute before Stage I download')
check('INPUT_SOURCE_PIN',bool(SOURCE_COMMIT and index_info['sha']==CFG.EXPECTED_INDEX_BLOB_SHA))
check('SEVEN_MODULES_REPORTED',len(MODULES)==7 and all(v['status']!='NOT_STARTED' for v in MODULES.values()))
check('EXPECTED_CATALOG',len(ASSETS)==52)
check('FULL52_SHA_COVERAGE',CFG.MODE!='FULL' or (len(INPUT_SHA)==len(ASSETS) and all(a['asset_id'] in INPUT_SHA for a in ASSETS)))
check('FULL52_ROW_COVERAGE',CFG.MODE!='FULL' or (len(FULL52_INVENTORY)==len(ASSETS) and all(r.get('footer_rows_ok') for r in FULL52_INVENTORY)))
check('FULL52_YEAR_DISTRIBUTION',CFG.MODE!='FULL' or all(r['year_scan_status'] in ('SCANNED_FULL','NO_TIME_KEY') for r in FULL52_INVENTORY))
check('FULL52_D02_PROFILE',CFG.MODE!='FULL' or (len(D02_PROFILE)==11 and all(r['status']=='GROUPED_ALL_SOURCE_ROWS_NO_DOUBLE_COUNT' for r in D02_PROFILE)))
check('FULL52_VPO_PROFILE',CFG.MODE!='FULL' or (len(VPO_ALL_PROFILE)==11 and all(r['status']=='PASS' for r in VPO_ALL_PROFILE)))
check('FULL52_E04_E05_PROFILE',CFG.MODE!='FULL' or (len(FAMILY_E04E05)==4 and all(r['status']=='PASS_NOT_ADDITIVE' for r in FAMILY_E04E05)))
check('FULL52_CONTEXT_EXPORT',CFG.MODE!='FULL' or len(CONTEXT_PARQUET_FILES)>=26)
check('V4_SOURCE_VALUES_52',CFG.MODE!='FULL' or (len(V4_SOURCE_PROFILES)==52 and not V4_SOURCE_ERRORS))
check('V4_MODEL_FAILURES_ZERO',V4_COMPUTATIONAL_COMPLETE,'Failed model or source value analysis is a real computational failure')
check('V5_COMPLETE_COMPUTATIONAL_CONTRACT',V5_COMPUTATIONAL_PASS,'All feasible models and checks must complete; no hidden failures')
check('V5_UNIQUE_MODEL_ARTIFACTS',len(MODEL_FILES)==len(MODEL_RECORDS),'No overwritten spatial model files')
check('V5_ALL_MODELS_VERIFIED',bool(V5_MODEL_QA) and all(x['sha_verified'] for x in V5_MODEL_QA))
check('V5_SEVEN_MODULE_LEDGER',len(V5_LEDGER)==7 and V5_LEDGER.module.nunique()==7)
check('V5_ALL52_SOURCE_EVIDENCE',CFG.MODE!='FULL' or len(V5_SOURCE_EVIDENCE)==52)
check('V5_ML54_NO_SAMPLE',CFG.MODE!='FULL' or (len(PANELS.get('v5_ml54_full_strata_coverage',pd.DataFrame()))==1))
check('V4_FINAL_LEDGER',len(V4_METHOD_EXECUTIONS)==7)
check('V4_SCIENTIFIC_TRUTHFUL',not V4_SCIENTIFIC_COMPLETE or (len(ORG_YEAR)>0 and OD_SEMANTIC.get('yearly_count_unit_confirmed',False)))
check('E01_PRIMARY_OPENED',CFG.MODE!='FULL' or bool(OD_SEMANTIC.get('original_opened')))
check('SEMANTIC_APPROVAL_GATED',not len(ORG_YEAR) or all(
    name in set(CFG.APPROVED_VO_ADMISSION_NAMES)|set(CFG.APPROVED_SPO_ADMISSION_NAMES)
    for name in ORG_YEAR.indicator_name.dropna().astype(str).unique()))
check('EDGES_PARTITION',len(OD)==len(OD_INTER)+len(OD_INTRA)+len(OD_UNKNOWN))
check('INTRA_EXCLUDED',OD_INTER.origin_region.ne(OD_INTER.destination_region).all())
check('NO_FUTURE_FEATURES',all(v.get('status')!='FAIL' for v in CANARIES))
check('CANARIES_EXECUTED',any(v.get('module')=='ML-5.1' and v.get('status')=='PASS' for v in CANARIES) and
      any(v.get('module')=='ML-5.3' and v.get('status')=='PASS' for v in CANARIES))
check('NO_TEST_SELECTED_MODEL',not len(CONFIRM) or
      (CONFIRM.choice_basis.isin(['DEV_ASOF','NO_CONFIRMATORY_COMPARISON_NO_DEV']).all() and
       CONFIRM.loc[CONFIRM.choice_basis.eq('NO_CONFIRMATORY_COMPARISON_NO_DEV'),
                   'selected_model'].eq('persistence').all()))
check('NEGATIVE_FINDINGS_PRESERVED',len(HYPOTHESES)==7)
check('ZIP_CRC',zipfile.ZipFile(EVIDENCE).testzip() is None)
with zipfile.ZipFile(EXCEL) as z:
    check('EXCEL_OPENS',z.testzip() is None and 'xl/workbook.xml' in z.namelist())
check('MODEL_RELOAD',all(m.get('reloaded') for m in MODEL_RECORDS))
check('NOTEBOOK_CAPTURED',NOTEBOOK.exists(),'Необходимо получить очищенный самостоятельный Colab notebook')
if NOTEBOOK.exists():
    try:
        nb=nbformat.read(NOTEBOOK,as_version=4)
        nbformat.validate(nb)
        content=NOTEBOOK.read_text(encoding='utf-8')
        suspicious=re.findall(r'\b(?:ghp_|github_pat_)[A-Za-z0-9_]{20,}\b',content)
        check('NOTEBOOK_NO_TOKEN',not suspicious and _TOKEN not in content)
        check('NOTEBOOK_NO_OUTPUTS',sum(len(c.get('outputs',[])) for c in nb.cells if c.cell_type=='code')==0)
        check('NOTEBOOK_SELF_CONTAINED',not re.search(r'(?m)^\s*(?:%run|%%writefile)\b',content))
        check('NOTEBOOK_REQUIRED_SECTION_COUNT',sum(c.cell_type=='code' for c in nb.cells)>=12)
        check('NOTEBOOK_REASONABLE_SIZE',NOTEBOOK.stat().st_size<2_000_000)
    except Exception as ex:check('NOTEBOOK_VALIDATE',False,type(ex).__name__)

# Проверяем независимые Parquet исходя из физического ZIP, не доверяя только собственным логам.
with zipfile.ZipFile(EVIDENCE) as z:
    for name,info in EVIDENCE_PARQUET.items():
        target=('predictions/' if name=='education_predictions' else 'panels/')+name+'.parquet'
        if target not in z.namelist():
            check('PARQUET_'+name,False,'Нет файла в evidence');continue
        try:
            # Не z.read(): весь Parquet в bytes может мгновенно исчерпать RAM.
            p=Path(info['path']);pf=pq.ParquetFile(p)
            member=z.getinfo(target)
            check('PARQUET_'+name,int(pf.metadata.num_rows)==info['rows'] and
                  len(pf.schema_arrow.names)==info['columns'] and
                  int(member.file_size)==p.stat().st_size and hash_file(p)==info['sha256'])
        except Exception as ex:check('PARQUET_'+name,False,type(ex).__name__)

# V7: repeat independent source/model lineage checks on physical exported tables.
check('V7_SOURCE_AUDIT_ALL52',CFG.MODE!='FULL' or
      (len(V7_SOURCE_AUDIT)==52 and V7_SOURCE_AUDIT.verified_sha256.all() and
       V7_SOURCE_AUDIT.all_values_profiled.all() and V7_SOURCE_AUDIT.numeric_profile_exists.all()))
check('V7_LINEAGE_52_UNIQUE',len(V7_SOURCE_LINEAGE)==len(ASSETS) and
      V7_SOURCE_LINEAGE.asset_id.nunique()==len(ASSETS) and
      V7_SOURCE_LINEAGE.source_coverage_pass.all())
check('V7_ALL_MODELS_LINKED',len(V7_MODEL_LINEAGE)==len(MODEL_RECORDS) and
      (V7_MODEL_LINEAGE.n_sources.ge(1).all() if len(V7_MODEL_LINEAGE) else True),
      'Every recorded model needs a traceable source input')
check('V7_ORIGINAL_7_GATES_REPORTED',len(V7_SCIENCE_GATES)==7 and
      V7_SCIENCE_GATES.module.nunique()==7 and
      V7_SCIENCE_GATES.computational_pass.all())
check('V7_OD_COMPARABLE_COMMON_SUPPORT',bool(len(V7_OD_COMPARABLE)) and
      all(x['shared_edges']<=min(x['observed_edges_previous'],x['observed_edges_current'])
          for x in V7_OD_COMPARABLE))
check('V71_PUBLISH_CHECK_SEPARATED_FROM_LOCAL_QA',
      bool(MANIFEST.is_file() and EXCEL.is_file() and EVIDENCE.is_file() and
           manifest.get('version')==CFG.VERSION),
      'Remote publication is checked only after uploading and reading the GitHub Release')
check('V71_ADVANCED_SOURCE_COVERAGE',len(ADR_SOURCE_AUDIT)==52 and ADR_SOURCE_AUDIT.is_source_checked.all())
check('V71_SEVEN_MODULE_RECONSTRUCTION_LEDGER',len(ADR_SEVEN_MODULE_FINAL)==7 and ADR_SEVEN_MODULE_FINAL.module.nunique()==7)
check('V71_RECONSTRUCTION_MODEL_AND_SEMANTICS',ADR_COMPUTATIONAL_PASS,
      'All actually executed reconstruction models passed; missing source targets are explicitly skipped')
check('V71_NO_INVENTED_ADMISSIONS',not len(ADR_ADMISSION_YEAR) or bool(CFG.APPROVED_VO_ADMISSION_NAMES))
check('V71_NO_INVENTED_CAMPUSES',not len(ADR_GEO_VERIFIED) or
      ADR_GEO_VERIFIED.identity_status.eq('PROXIMITY_MATCH_NOT_OFFICIAL_CAMPUS_LOCATION').all())

LOCAL_QA=pd.DataFrame(LOCAL_CHECKS)
progress_event('QA_AND_PUBLICATION',f'Проверено {len(LOCAL_QA):,} контролей; ошибок: {int(LOCAL_QA.status.eq("FAIL").sum())}',announce=True)
# Перезаписываем локальный manifest после QA, но Excel/ZIP уже фиксированы по SHA.
manifest['local_qa']=LOCAL_CHECKS
manifest['status']['technical']='PASS' if LOCAL_QA.status.eq('PASS').all() else 'FAIL'
manifest['status']['publication']='NOT_ATTEMPTED'
manifest['status']['overall']=(STATE['scientific'] if manifest['status']['technical']=='PASS'
                              else 'PARTIAL_LOCAL_QA')
MANIFEST.write_text(json.dumps(manifest,ensure_ascii=False,default=str,indent=2),encoding='utf-8')
print('Independent QA:',int(LOCAL_QA.status.eq('PASS').sum()),'PASS /',len(LOCAL_QA),'checks')

# Публикация отсюда не выполняется, даже при FAIL локальной QA.
# Новый финальный этап публикует артефакты с честным статусом QA; сами данные не теряются.
PUBLICATION_RECEIPT={'status':'PENDING_FINAL_STAGE'}
STATE['publication']='PENDING_FINAL_STAGE'

print('ML-05 V7.1.1:',STATE['scientific'],'| technical:',manifest['status']['technical'],
      '| GitHub:',STATE['publication'])
print('Output paths:',EXCEL,EVIDENCE,NOTEBOOK,MANIFEST)

mark_stage('QA_AND_PUBLICATION')
# Heartbeat работает и во время сборки/загрузки итогового ZIP; закрываем после последней ячейки.


## Приёмочный контракт V7.1.1

- **TECHNICAL_PASS**: все 52 исходных набора проверены по SHA, 7 модулей имеют журналы исполнения, обязательные по данным расчёты завершены, модельные файлы сверены по SHA и локальная QA без ошибок.
- **SCIENTIFIC_LIMITED**: измеряемая цель не идентифицируется в первичных данных. Этот статус честно сохраняется в отчёте и не подменяется иной величиной.
- **REPLACE**: если локальные вычисления и QA успешны, публикуется новый Git-коммит, заменяющий только целевой ML-05 сегмент. При техническом FAIL полные результаты могут быть сохранены как черновой Release, но старая рабочая версия не удаляется.


- **SQL_PREFLIGHT**: рабочие DuckDB-запросы E02 сначала исполняются на маленьком наборе, затем на полном источнике; ошибка синтаксиса приводит к остановке до многогигабайтных операций.

## 14. Итог: компактная автозагрузка отчёта → проверенный GitHub Release → атомарная замена старого ML-05

In [ ]:
# ML-05 V7.1.1 · COMPLETE PIPELINE / VERIFIED GITHUB REPLACEMENT / COMPACT DOWNLOAD
# FULL -> 52 SHA verified -> seven modules -> QA -> small report -> draft Release -> atomic segment replace.
# Does not delete Stage I, unrelated projects or historical git commits. Every destructive change is prefix-scoped.
import json, os, math, hashlib, base64, zipfile, shutil, re
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

progress_event('DOWNLOAD_RESULTS','V7.1: итоговый QA, компактная аналитика и безопасная замена ML-05',announce=True)
if CFG.MODE != 'FULL': raise RuntimeError('V7.1: для публикации необходим FULL')
if not CFG.PUBLISH_RESULTS_TO_PRIVATE_GITHUB: raise RuntimeError('V7.1: публикация в приватный GitHub выключена')
if not CFG.STORE_DERIVED_RESTRICTED_DATA: raise RuntimeError('V7.1: не разрешено публиковать производные результаты')
if not api_json('GET',BASE).get('private'): raise RuntimeError('V7.1: репозиторий не приватный')
if len(ASSETS)!=52 or len(INPUT_SHA)!=52 or len(V5_SOURCE_EVIDENCE)!=52:
    raise RuntimeError('V7.1: 52 источника не подтверждены SHA и профилями')
if len(V5_LEDGER)!=7 or V5_LEDGER.module.nunique()!=7:
    raise RuntimeError('V7.1: отсутствует одна или несколько приёмок семи модулей')

out=ROOT/'exports'; out.mkdir(parents=True,exist_ok=True)
needed=[Path(p) for p in (EXCEL,EVIDENCE,NOTEBOOK,MANIFEST)]
for p in needed:
    if not p.is_file() or p.stat().st_size==0: raise RuntimeError('V7.1: отсутствует файл '+str(p))
_saved=json.loads(Path(MANIFEST).read_text(encoding='utf-8'))
if _saved.get('version')!=CFG.VERSION or _saved.get('source_commit')!=SOURCE_COMMIT:
    raise RuntimeError('V7.1: манифест не соответствует активному запуску')
for p in (EXCEL,EVIDENCE,NOTEBOOK):
    if _saved.get('output_sha256',{}).get(Path(p).name)!=hash_file(p):
        raise RuntimeError('V7.1: нарушена контрольная сумма '+str(p))

# Source capture is a deterministic embedded copy of this notebook, not a stale Colab get_ipynb.
recovered=notebook_from_embedded_source()
if hash_file(NOTEBOOK)!=hashlib.sha256(nbformat.writes(recovered).encode('utf-8')).hexdigest():
    # nbformat serialization may differ in insignificant line breaks. Revalidate actual source instead.
    _check=nbformat.read(NOTEBOOK,as_version=4)
    if sum(c.cell_type=='code' for c in _check.cells)!=sum(c.cell_type=='code' for c in recovered.cells):
        raise RuntimeError('V7.1: сохранённый ноутбук не совпал с встроенным источником')
    _check_src=[c.source for c in _check.cells]
    if _check_src != [c.source for c in recovered.cells]:
        raise RuntimeError('V7.1: исходный код ноутбука отличается от эталона')

sha_evidence=hash_file(EVIDENCE)
run_key=hashlib.sha256((CFG.VERSION+'|'+SOURCE_COMMIT+'|'+sha_evidence).encode()).hexdigest()[:20]
release_tag='ml05-v7.1.1-full-'+run_key
prefix=CFG.OUTPUT_PREFIX.rstrip('/')+'/'
if prefix!='stage2/ml05_education/': raise RuntimeError('V7.1: удаление разрешено ТОЛЬКО в stage2/ml05_education/')
qa_ok=bool(len(LOCAL_QA) and LOCAL_QA.status.eq('PASS').all())
compute_ok=bool(V5_COMPUTATIONAL_PASS and ADR_COMPUTATIONAL_PASS)
science_ok=bool(V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS)
can_replace=bool(qa_ok and compute_ok)

# V7 source/model lineage is derived from exact input panels and stored feature lists.
# No false claim that all 52 sources enter predictive X; all 52 have full numerical research profiles.
source_ledger=V7_SOURCE_LINEAGE.to_dict('records')
adr_ledger_csv=out/'ML5_V711_RECONSTRUCTED_EXPERIMENTS.csv'
pd.DataFrame(ADR_EXPERIMENTS).to_csv(adr_ledger_csv,index=False,encoding='utf-8-sig')
adr_summary_file=out/'ML5_V711_RECONSTRUCTION_ACCEPTANCE.json'
adr_summary_file.write_text(json.dumps(ADR_FINAL_REPORT,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
adr_lineage_csv=out/'ML5_V711_RECONSTRUCTED_MODEL_LINEAGE.csv'
ADR_RECONSTRUCTED_LINEAGE.to_csv(adr_lineage_csv,index=False,encoding='utf-8-sig')
source_csv=out/'ML5_V711_SOURCE_COVERAGE_52.csv'
V7_SOURCE_LINEAGE.to_csv(source_csv,index=False,encoding='utf-8-sig')
lineage_csv=out/'ML5_V711_MODEL_LINEAGE.csv'
V7_MODEL_LINEAGE.to_csv(lineage_csv,index=False,encoding='utf-8-sig')
gates_csv=out/'ML5_V711_SEVEN_SCIENTIFIC_GATES.csv'
V7_SCIENCE_GATES.to_csv(gates_csv,index=False,encoding='utf-8-sig')
source_stats_csv=out/'ML5_V711_SOURCE_RESEARCH_52.csv'
V7_SOURCE_AUDIT.to_csv(source_stats_csv,index=False,encoding='utf-8-sig')
module_csv=out/'ML5_V711_SEVEN_MODULES.csv';V5_LEDGER.to_csv(module_csv,index=False,encoding='utf-8-sig')
metrics_csv=out/'ML5_V711_METRICS.csv';pd.DataFrame(METRICS).to_csv(metrics_csv,index=False,encoding='utf-8-sig')
models_csv=out/'ML5_V711_MODELS.csv';pd.DataFrame(V5_MODEL_QA).to_csv(models_csv,index=False,encoding='utf-8-sig')
qa_csv=out/'ML5_V711_TECHNICAL_QA.csv';LOCAL_QA.to_csv(qa_csv,index=False,encoding='utf-8-sig')
issues_csv=out/'ML5_V711_SCIENTIFIC_LIMITATIONS.csv';pd.DataFrame(ISSUES).to_csv(issues_csv,index=False,encoding='utf-8-sig')

summary={
    'version':CFG.VERSION,'run_key':run_key,'git_source_commit':SOURCE_COMMIT,
    'catalog_assets':len(ASSETS),'sha_verified':len(INPUT_SHA),
    'content_profiled':sum(x.get('content_status')=='FULL_VALUE_COVERAGE_PASS' for x in V4_SOURCE_PROFILES),
    'direct_model_sources':int(V7_SOURCE_LINEAGE.direct_model_input.sum()),
    'model_lineage_records':len(V7_MODEL_LINEAGE),'source_profiles_researched':int(V7_SOURCE_AUDIT.all_values_profiled.sum()),
    'seven_modules':V5_MODULE_ACCEPTANCE,'v7_scientific_gates':V7_SCIENCE_GATES.to_dict('records'),'qa_pass':int(LOCAL_QA.status.eq('PASS').sum()),
    'qa_fail':int(LOCAL_QA.status.eq('FAIL').sum()),'computational_pass':can_replace,
    'original_seven_targets_scientifically_identified':science_ok,
    'prediction_rows':PRED_TOTAL,'metric_rows':len(METRICS),'model_count':len(MODEL_FILES),
    'evidence_sha256':sha_evidence,'evidence_bytes':EVIDENCE.stat().st_size,
    'scientific_caveat':'Невозможный по имеющимся данным прогноз не подменяется иным целевым показателем.',
    'adr_reconstruction':ADR_FINAL_REPORT
}
summary_file=out/'ML5_V711_FINAL_ACCEPTANCE.json'
summary_file.write_text(json.dumps(summary,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

# Reports are downloaded before remote upload; complete evidence and heavy models NEVER enter this ZIP.
plots=sorted({Path(p) for p in FIGURES if Path(p).is_file()},key=lambda p:p.name)
reports=[Path(EXCEL),Path(NOTEBOOK),Path(MANIFEST),source_csv,lineage_csv,gates_csv,source_stats_csv,module_csv,metrics_csv,models_csv,qa_csv,issues_csv,summary_file,adr_ledger_csv,adr_summary_file,adr_lineage_csv]+plots
small_zip=out/('ML5_V711_ANALYTICS_'+run_key+'.zip')
with zipfile.ZipFile(small_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
    for p in reports:
        z.write(p,('figures/' if p in plots else 'reports/')+p.name)
    z.writestr('reports/checksums.json',json.dumps({p.name:hash_file(p) for p in reports},indent=2))
with zipfile.ZipFile(small_zip) as z:
    if z.testzip() is not None: raise RuntimeError('V7.1: повреждён отчётный ZIP')
if small_zip.stat().st_size>100*2**20:raise RuntimeError('V7.1: отчётный архив >100 MiB, проверьте содержимое')
progress_event('DOWNLOAD_RESULTS',f'V7.1: готов отчётный архив {small_zip.stat().st_size/2**20:.1f} МиБ',announce=True)
print('АНАЛИТИКА:',small_zip,'|','EVIDENCE только в GitHub:',EVIDENCE.stat().st_size/2**20,'МиБ',flush=True)
if CFG.DOWNLOAD_COMPACT_REPORTS:
    try:
        from google.colab import files as colab_files
        colab_files.download(str(small_zip))
    except ImportError:
        from IPython.display import FileLink,display
        display(FileLink(str(small_zip)))

# Precompute resumable chunks on disk; verify recombined SHA before GitHub upload.
size_part=int(CFG.GH_RELEASE_PART_MIB)*2**20
if not 16<=int(CFG.GH_RELEASE_PART_MIB)<=256:raise RuntimeError('V7.1: part size вне 16..256 MiB')
chunk_dir=out/('release_chunks_'+run_key);chunk_dir.mkdir(parents=True,exist_ok=True)
chunk_index={'version':CFG.VERSION,'source_evidence':Path(EVIDENCE).name,
    'evidence_sha256':sha_evidence,'evidence_bytes':EVIDENCE.stat().st_size,'parts':[]}
with Path(EVIDENCE).open('rb') as src:
    total=math.ceil(EVIDENCE.stat().st_size/size_part)
    for i in range(total):
        name=f'ML5_EVIDENCE.zip.part{i+1:03d}'
        p=chunk_dir/name
        n=min(size_part,EVIDENCE.stat().st_size-i*size_part)
        if p.exists() and p.stat().st_size==n:
            src.seek(n,1)
        else:
            tmp=p.with_suffix('.part.tmp')
            with tmp.open('wb') as dest:
                remaining=n
                while remaining:
                    buf=src.read(min(4*2**20,remaining))
                    if not buf:raise IOError('V7.1: преждевременный конец evidence')
                    dest.write(buf);remaining-=len(buf)
            tmp.replace(p)
        chunk_index['parts'].append({'name':name,'bytes':n,'sha256':hash_file(p)})
        progress_event('DOWNLOAD_RESULTS',f'Локальная часть evidence {i+1}/{total}',announce=True)
verify_hash=hashlib.sha256()
for part in chunk_index['parts']:
    with (chunk_dir/part['name']).open('rb') as f:
        for buf in iter(lambda:f.read(4*2**20),b''):verify_hash.update(buf)
if verify_hash.hexdigest()!=sha_evidence:raise RuntimeError('V7.1: разбитый evidence SHA не совпадает с исходным')
index_file=out/'ML5_V711_RELEASE_PARTS.json'
index_file.write_text(json.dumps(chunk_index,ensure_ascii=False,indent=2),encoding='utf-8')

# Helpers for GitHub. An existing asset is reused ONLY if GitHub's SHA-256 and size match.
def remote_release_assets(release_id):
    return {a['name']:a for a in api_json('GET',BASE+f'/releases/{release_id}/assets',params={'per_page':100})}

def upload_release_asset(release_id,p,mime):
    p=Path(p);sha=hash_file(p)
    already=remote_release_assets(release_id).get(p.name)
    if already and already.get('state')=='uploaded' and already.get('digest')=='sha256:'+sha and int(already.get('size',-1))==p.stat().st_size:
        progress_event('DOWNLOAD_RESULTS','GitHub SHA уже совпадает: '+p.name,announce=True)
        return already
    if already:
        resp=SESS.delete(BASE+'/releases/assets/'+str(already['id']),timeout=(30,120))
        status=resp.status_code;resp.close()
        if status!=204:raise RuntimeError('V7.1: не удалён повреждённый Release-asset '+p.name)
    upload_url=f'https://uploads.github.com/repos/{CFG.REPO}/releases/{release_id}/assets'
    progress_event('DOWNLOAD_RESULTS',f'GitHub: {p.name} · {p.stat().st_size/2**20:.1f} МиБ',announce=True)
    try:
        with p.open('rb') as f:
            resp=SESS.post(upload_url,params={'name':p.name},headers={'Content-Type':mime},data=f,timeout=(30,3600))
        status=resp.status_code
        if status==201:obj=resp.json()
        else:obj=None
        resp.close()
    except Exception:
        obj=None
    if obj is None:
        obj=remote_release_assets(release_id).get(p.name)
    if not obj or obj.get('state')!='uploaded' or obj.get('digest')!='sha256:'+sha or int(obj.get('size',-1))!=p.stat().st_size:
        raise RuntimeError('V7.1: ошибка GitHub asset '+p.name+'; можно повторить только финальную ячейку')
    return obj

branch=CFG.TARGET_BRANCH
remote_head=repo_head(branch)
if remote_head!=TARGET_HEAD_AT_PREFLIGHT:
    # A rerun after the successful commit is allowed, but unrelated changes must not be overwritten.
    try:
        existing=json.loads(read_git_blob(prefix+'publication_receipt.json',ref=remote_head))
        already_done=(existing.get('run_key')==run_key and existing.get('evidence_sha256')==sha_evidence)
    except Exception:
        already_done=False
    if not already_done:raise RuntimeError('V7.1: ветка изменилась после preflight. Не перезаписываю чужие изменения.')
else:already_done=False

# Atomic replacement is permitted ONLY if all computational checks and local QA passed.
# Otherwise publish a diagnostic draft Release; leave old active segment unchanged.
if not can_replace:
    print('V7.1: QA/вычисления не полностью пройдены. Отчёты скачаны. Старый рабочий сегмент НЕ удаляю.',flush=True)

publication_plan=out/'ML5_V711_PUBLICATION_PLAN.json'
planned=None
if publication_plan.is_file():
    try:
        j=json.loads(publication_plan.read_text('utf-8'))
        if j.get('run_key')==run_key and j.get('base_sha')==TARGET_HEAD_AT_PREFLIGHT:planned=j
    except Exception:pass

publication_receipt={
    'version':CFG.VERSION,'run_key':run_key,'release_tag':release_tag,
    'release_url':f'https://github.com/{CFG.REPO}/releases/tag/{release_tag}',
    'evidence_sha256':sha_evidence,'parts_manifest':index_file.name,
    'source_commit':SOURCE_COMMIT,'source_count':52,'computational_pass':can_replace,
    'scientific_targets_all_identified':science_ok,
    'reconstruction_computational_pass':ADR_COMPUTATIONAL_PASS,'target_prefix':prefix,
    'replacement_policy':'atomic current ML-05 tree only; preserve git history and Stage I',
    'remote_verification_state':'PREPARED_NOT_YET_VERIFIED'
}
receipt_path=out/'publication_receipt.json'
receipt_path.write_text(json.dumps(publication_receipt,ensure_ascii=False,indent=2),encoding='utf-8')
pubmanifest=out/'run_manifest_publication.json'
pubm=json.loads(Path(MANIFEST).read_text('utf-8'))
pubm['publication_status']='PENDING_GITHUB_VERIFICATION' if can_replace else 'DRAFT_PARTIAL_FAIL'
pubm['status']['publication']=pubm['publication_status']
pubm['publication_reference']={'release_tag':release_tag,'evidence_sha256':sha_evidence}
pubmanifest.write_text(json.dumps(pubm,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

if already_done:
    _remote_manifest=read_git_blob(prefix+'run_manifest.json',ref=remote_head)
    _remote_receipt=read_git_blob(prefix+'publication_receipt.json',ref=remote_head)
    if json.loads(_remote_manifest).get('status',{}).get('publication')!='PUBLISHED_GITHUB_VERIFIED':
        raise RuntimeError('V7.1: final verified status missing on a re-run')
    pubmanifest.write_bytes(_remote_manifest)
    receipt_path.write_bytes(_remote_receipt)

# What remains in active stage2/ml05_education AFTER successful update.
target_files={
    prefix+'ML5_EDUCATION_MIGRATION.ipynb':Path(NOTEBOOK),
    prefix+'ML5_EDUCATION_MIGRATION_RESULTS.xlsx':Path(EXCEL),
    prefix+'run_manifest.json':pubmanifest,
    prefix+'publication_receipt.json':receipt_path,
    prefix+index_file.name:index_file,
    prefix+summary_file.name:summary_file,
    prefix+source_csv.name:source_csv,
    prefix+lineage_csv.name:lineage_csv,
    prefix+gates_csv.name:gates_csv,
    prefix+source_stats_csv.name:source_stats_csv,
    prefix+module_csv.name:module_csv,
    prefix+metrics_csv.name:metrics_csv,
    prefix+qa_csv.name:qa_csv,
    prefix+adr_ledger_csv.name:adr_ledger_csv,
    prefix+adr_summary_file.name:adr_summary_file,
    prefix+adr_lineage_csv.name:adr_lineage_csv,
}
if not all(k.startswith(prefix) for k in target_files): raise RuntimeError('V7.1: нарушена изоляция сегмента')

# Prepare a candidate Git commit *before* creating a release, so new tag points to V6 code, not old V2.
commit_sha=remote_head if already_done else None
if can_replace and not already_done:
    if planned:
        candidate=api_json('GET',BASE+'/git/commits/'+planned['candidate_sha'])
        commit_sha=planned['candidate_sha']
        if candidate['parents'][0]['sha']!=remote_head:raise RuntimeError('V7.1: кандидат не является потомком HEAD')
    else:
        old_tree=api_json('GET',BASE+'/git/trees/'+remote_head,params={'recursive':1})
        if old_tree.get('truncated'):raise RuntimeError('V7.1: неполное дерево Git, удаление запрещено')
        old_paths={x['path'] for x in old_tree.get('tree',[]) if x.get('type')=='blob' and x.get('path','').startswith(prefix)}
        obsolete=sorted(old_paths-set(target_files))
        changes=[]
        for filename,path in sorted(target_files.items()):
            binary=Path(path).read_bytes() # Small files only; evidence is Release-only.
            if len(binary)>45*2**20:raise RuntimeError('V7.1: слишком большой Git blob '+filename)
            blob=api_json('POST',BASE+'/git/blobs',json={'content':base64.b64encode(binary).decode(),'encoding':'base64'})
            changes.append({'path':filename,'mode':'100644','type':'blob','sha':blob['sha']})
        for filename in obsolete:
            changes.append({'path':filename,'mode':'100644','type':'blob','sha':None})
        old_commit=api_json('GET',BASE+'/git/commits/'+remote_head)
        tree=api_json('POST',BASE+'/git/trees',json={'base_tree':old_commit['tree']['sha'],'tree':changes})
        candidate=api_json('POST',BASE+'/git/commits',json={
            'message':f'ML-05 V7.1.1 FULL verified replacement: {run_key} (old segment files retired)',
            'tree':tree['sha'],'parents':[remote_head]})
        commit_sha=candidate['sha']
        plan={'run_key':run_key,'base_sha':remote_head,'candidate_sha':commit_sha,
              'obsolete_paths':obsolete,'new_files':sorted(target_files)}
        publication_plan.write_text(json.dumps(plan,ensure_ascii=False,indent=2),encoding='utf-8')
        progress_event('DOWNLOAD_RESULTS',f'Кандидат Git commit готов, старых файлов к замене {len(obsolete)}',announce=True)

# Create a draft Release. The tag MUST point to the prepared V6 commit.
all_releases=api_json('GET',BASE+'/releases',params={'per_page':100})
release=next((x for x in all_releases if x.get('tag_name')==release_tag),None)
if release is None:
    release=api_json('POST',BASE+'/releases',json={
        'tag_name':release_tag,'target_commitish':commit_sha or remote_head,
        'name':'ML-05 V7.1.1 FULL '+run_key,'draft':True,'prerelease':True,
        'body':f'Stage I 52/52. Computational PASS: {can_replace}. Scientific 7/7 identified: {science_ok}. '
               f'Evidence SHA-256 {sha_evidence}. Full results belong to this private repository.'})
release_id=release['id']
for p,mime in [(index_file,'application/json')]+[(chunk_dir/x['name'],'application/octet-stream') for x in chunk_index['parts']]+[(small_zip,'application/zip')]:
    upload_release_asset(release_id,p,mime)
assets=remote_release_assets(release_id)
needed_assets=[index_file,small_zip]+[chunk_dir/x['name'] for x in chunk_index['parts']]
for p in needed_assets:
    a=assets.get(p.name)
    if not a or a.get('state')!='uploaded' or a.get('digest')!='sha256:'+hash_file(p) or int(a.get('size',-1))!=p.stat().st_size:
        raise RuntimeError('V7.1: удалённый Release-asset не прошёл приёмку '+p.name)

if can_replace:
    if not already_done:
        if repo_head(branch)!=remote_head:raise RuntimeError('V7.1: конфликт обновления Git ветки, старый сегмент не изменён')
        api_json('PATCH',BASE+'/git/refs/heads/'+quote(branch,safe='/'),json={'sha':commit_sha,'force':False})
        if repo_head(branch)!=commit_sha:raise RuntimeError('V7.1: не подтверждён Git HEAD после коммита')
    # Check EVERY committed file by round-trip SHA256 from GitHub, then ensure old files are absent.
    for filename,path in target_files.items():
        remote=read_git_blob(filename,ref=commit_sha)
        if hashlib.sha256(remote).hexdigest()!=hash_file(path):
            raise RuntimeError('V7.1: GitHub roundtrip SHA mismatch '+filename)
    finaltree=api_json('GET',BASE+'/git/trees/'+commit_sha,params={'recursive':1})
    if finaltree.get('truncated'):raise RuntimeError('V7.1: Git tree truncated in final check')
    now={x['path'] for x in finaltree['tree'] if x['type']=='blob' and x['path'].startswith(prefix)}
    if now!=set(target_files):raise RuntimeError('V7.1: остались устаревшие файлы ML-05 или пропали новые')
    if release.get('draft'):
        release=api_json('PATCH',BASE+'/releases/'+str(release_id),json={'draft':False,'prerelease':True})
    published=api_json('GET',BASE+'/releases/'+str(release_id))
    if published.get('draft') or published.get('tag_name')!=release_tag:
        raise RuntimeError('V7.1: новый Release не подтвердился')
    # Confirm the tag points to the new commit, not the retired V2 commit.
    tagref=api_json('GET',BASE+'/git/ref/tags/'+quote(release_tag,safe=''))
    tagsha=tagref.get('object',{}).get('sha')
    if tagsha!=(commit_sha if not already_done else existing.get('prepared_candidate_sha')):
        raise RuntimeError('V7.1: release tag does not point to the checked code commit')
    # V7.1: second metadata-only commit records independently verified publication.
    # The first verified commit atomically replaces only the ML-05 tree; this second one updates
    # the publication state after (not before) the GitHub/Release round-trip succeeds.
    if not already_done:
        pubm['status']['publication']='PUBLISHED_GITHUB_VERIFIED'
        pubm['publication_status']='PUBLISHED_GITHUB_VERIFIED'
        pubm['publication_reference']['candidate_commit_sha']=commit_sha
        pubm['publication_reference']['release_id']=release_id
        pubm['publication_reference']['verification']='GIT_TREE_AND_RELEASE_ASSET_DIGESTS_PASS'
        pubmanifest.write_text(json.dumps(pubm,ensure_ascii=False,default=str,indent=2),encoding='utf-8')
        publication_receipt.update({'remote_verification_state':'PASS_GITHUB',
            'prepared_candidate_sha':commit_sha,'release_id':release_id,
            'release_digests_confirmed':True,'stage1_unchanged':True})
        receipt_path.write_text(json.dumps(publication_receipt,ensure_ascii=False,indent=2),encoding='utf-8')
        _base_commit=api_json('GET',BASE+'/git/commits/'+commit_sha)
        _changes=[]
        for _path,_file in ((prefix+'run_manifest.json',pubmanifest),
                            (prefix+'publication_receipt.json',receipt_path)):
            _data=Path(_file).read_bytes()
            _blob=api_json('POST',BASE+'/git/blobs',json={
                'content':base64.b64encode(_data).decode(),'encoding':'base64'})
            _changes.append({'path':_path,'mode':'100644','type':'blob','sha':_blob['sha']})
        _tree=api_json('POST',BASE+'/git/trees',json={
            'base_tree':_base_commit['tree']['sha'],'tree':_changes})
        _receipt_commit=api_json('POST',BASE+'/git/commits',json={
            'message':'ML-05 V7.1.1: verified release and GitHub publication receipt '+run_key,
            'tree':_tree['sha'],'parents':[commit_sha]})['sha']
        if repo_head(branch)!=commit_sha:
            raise RuntimeError('V7.1: concurrent branch modification before final receipt')
        api_json('PATCH',BASE+'/git/refs/heads/'+quote(branch,safe='/'),json={
            'sha':_receipt_commit,'force':False})
        if repo_head(branch)!=_receipt_commit:
            raise RuntimeError('V7.1: final publication receipt commit was not installed')
        for _path,_file in ((prefix+'run_manifest.json',pubmanifest),
                            (prefix+'publication_receipt.json',receipt_path)):
            if hashlib.sha256(read_git_blob(_path,ref=_receipt_commit)).hexdigest()!=hash_file(_file):
                raise RuntimeError('V7.1: final publication receipt failed round-trip '+_path)
        commit_sha=_receipt_commit
        print('V7.1: final GitHub publication receipt independently verified',commit_sha,flush=True)
    else:
        existing=json.loads(read_git_blob(prefix+'publication_receipt.json',ref=remote_head))
        if existing.get('remote_verification_state')!='PASS_GITHUB':
            raise RuntimeError('V7.1: rerun found incomplete metadata receipt; manual recovery required')
        commit_sha=remote_head
    STATE['publication']='PASS_GITHUB_V71_REPLACED'
    print('ЗАМЕНА ML-05 ПОДТВЕРЖДЕНА:',commit_sha,flush=True)
    print('Только новый набор файлов в',prefix,'; старые файлы удалены из текущего дерева',flush=True)
else:
    STATE['publication']='DRAFT_REVIEW_ONLY_OLD_SEGMENT_PRESERVED'
    print('Новая доказательная база в черновом Release. Старый сегмент сохранён из-за FAIL.',flush=True)

receipt={**publication_receipt,'commit_sha':commit_sha,'release_id':release_id,
    'final_publication_status':STATE['publication'],
    'time_utc':datetime.now(timezone.utc).isoformat()}
(out/'ML5_V711_PUBLICATION_CONFIRMATION.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2),encoding='utf-8')
print('GitHub Release:',publication_receipt['release_url'],flush=True)
print('Небольшой отчётный архив:',small_zip,flush=True)
print('Научная приёмка:', 'PASS' if science_ok else 'LIMITED_BY_OBSERVABILITY',flush=True)
if can_replace:mark_stage('DOWNLOAD_RESULTS')
else:progress_event('DOWNLOAD_RESULTS','Публикация частичного исследования; 100% исполнения не подтверждены',announce=True)
bar=globals().get('MAIN_PROGRESS')
if bar is not None: bar.close()
